In [1]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/aqi_framework'
LINK = '/content/aqi_framework'
os.makedirs(DRIVE_DIR, exist_ok=True)

if os.path.islink(LINK):
    os.unlink(LINK)
elif os.path.isdir(LINK):                      # anything already created this session
    shutil.copytree(LINK, DRIVE_DIR, dirs_exist_ok=True)
    shutil.rmtree(LINK)
os.symlink(DRIVE_DIR, LINK)
print("Saving to:", os.path.realpath(LINK))

Mounted at /content/drive
Saving to: /content/drive/MyDrive/aqi_framework


In [2]:
# =============================================================================
# City- and Season-Aware Multi-Pollutant Air Quality Assessment and Forecasting
# Framework for Indian Cities Using Deep Learning
#
# PART 1 of 7: data loading, validation, hourly O3/CO retrieval,
#              8-hour maxima, and saving a clean master dataset.
#
# Runs end-to-end in Google Colab (also works locally). Paste into one cell or
# save as a .py file and run.
#
# DATA NOTE: All O3/CO hourly values come from the Open-Meteo Air Quality API
# (CAMS-based model estimates). They are NOT direct CPCB ground-station readings.
# =============================================================================

# ----------------------------------------------------------------------------
# 0. Imports, configuration, reproducibility
# ----------------------------------------------------------------------------
import os
import sys
import glob
import json
import time
import random
import shutil
import hashlib
import itertools
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import requests
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---- project layout --------------------------------------------------------
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
BASE_DIR = "/content/aqi_framework" if IN_COLAB else os.path.abspath("./aqi_framework")

RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
PROC_DIR = os.path.join(BASE_DIR, "data", "processed")
FIG_DIR = os.path.join(BASE_DIR, "figures")
CACHE_DIR = os.path.join(RAW_DIR, "openmeteo_cache")          # per-chunk API cache (resumable)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
VERSION_DIR = os.path.join(PROC_DIR, "versions", f"part1_{RUN_ID}")  # versioned archive copy

for d in (RAW_DIR, PROC_DIR, FIG_DIR, CACHE_DIR, VERSION_DIR):
    os.makedirs(d, exist_ok=True)

# ---- required output paths (as specified) ----------------------------------
MASTER_CSV = os.path.join(PROC_DIR, "master_real_units.csv")
HOURLY_CSV = os.path.join(RAW_DIR, "hourly_o3_co_openmeteo.csv")
QUALITY_CSV = os.path.join(PROC_DIR, "data_quality_report.csv")
FIG_PNG = os.path.join(FIG_DIR, "data_validation_plots.png")
MANIFEST_JSON = os.path.join(PROC_DIR, "part1_handoff_manifest.json")

# ---- input file ------------------------------------------------------------
INPUT_NAME = "AQI_Dataset_2022_2026_Clean-1.xlsx"
INPUT_GLOB = "AQI_Dataset_2022_2026_Clean*.xlsx"   # also matches e.g. "..._Clean__1_.xlsx"

# ---- data schema -----------------------------------------------------------
REQUIRED_COLS = ["Date", "City", "Region", "Latitude", "Longitude",
                 "PM2_5", "PM10", "CO", "NO2", "SO2", "O3",
                 "Temperature_C", "Humidity_pct", "WindSpeed_kmh", "Precipitation_mm"]
POLLUTANTS = ["PM2_5", "PM10", "CO", "NO2", "SO2", "O3"]
SEASON_MAP = {12: "Winter", 1: "Winter", 2: "Winter",
              3: "Summer", 4: "Summer", 5: "Summer",
              6: "Monsoon", 7: "Monsoon", 8: "Monsoon", 9: "Monsoon",
              10: "Post-monsoon", 11: "Post-monsoon"}

# ---- Open-Meteo settings ---------------------------------------------------
FETCH_HOURLY = os.environ.get("AQI_FETCH_HOURLY", "1") != "0"   # set env AQI_FETCH_HOURLY=0 to skip API
OPENMETEO_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
HOURLY_VARS = ["ozone", "carbon_monoxide"]
TIMEZONE = "Asia/Kolkata"            # hourly timestamps returned in IST so daily grouping is local-date based
CHUNK_DAYS = 366                      # one request per ~year per location
REQUEST_PAUSE_S = 0.4                 # polite pacing between calls
MAX_RETRIES = 4
WAIT_ON_HOURLY_LIMIT = False          # if True, sleeps ~1h on an hourly-limit 429 instead of stopping

# ---- 8-hour metric settings ------------------------------------------------
ROLL_WINDOW_H = 8
ROLL_MIN_HOURS = 6                    # a trailing 8-h mean needs >= 6 valid hours (75 %)
MIN_ROLL_PER_DAY = 18                 # a daily max needs >= 18 valid rolling values (75 % of 24)

# ---- extreme-value flagging ------------------------------------------------
EXTREME_IQR_K = 3.0                   # per-city Tukey "far-out" fence: Q3 + 3*IQR

DATA_DISCLAIMER = ("Hourly O3 and CO are Open-Meteo/CAMS-based model estimates, "
                   "not direct CPCB ground-station readings.")


# ----------------------------------------------------------------------------
# 1. Small helpers
# ----------------------------------------------------------------------------
class Report:
    """Collects rows for data_quality_report.csv (long format)."""
    def __init__(self):
        self.rows = []

    def add(self, section, check, item="", value="", note=""):
        self.rows.append({"section": section, "check": check, "item": item,
                          "value": value, "note": note})

    def to_df(self):
        return pd.DataFrame(self.rows, columns=["section", "check", "item", "value", "note"])


def sha256_of(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


def coord_key(lat, lon):
    return (round(float(lat), 4), round(float(lon), 4))


def banner(text):
    print("\n" + "=" * 78 + f"\n{text}\n" + "=" * 78)


# ----------------------------------------------------------------------------
# 2. Locate and load the original Excel file (read-only; never overwritten)
# ----------------------------------------------------------------------------
def find_input_file():
    search_dirs = [os.getcwd(), "/content", "/content/drive/MyDrive",
                   "/mnt/user-data/uploads", BASE_DIR, RAW_DIR]
    for d in search_dirs:                                   # exact name first
        p = os.path.join(d, INPUT_NAME)
        if os.path.isfile(p):
            return p
    for d in search_dirs:                                   # then tolerant glob
        hits = sorted(glob.glob(os.path.join(d, INPUT_GLOB)))
        if hits:
            return hits[0]
    if IN_COLAB:                                            # finally, ask for an upload
        from google.colab import files
        print("Excel file not found - please upload it now.")
        up = files.upload()
        for name in up:
            if name.lower().endswith(".xlsx"):
                return os.path.abspath(name)
    raise FileNotFoundError(f"Could not find '{INPUT_NAME}'. Upload it to /content or the working directory.")


def load_daily(path):
    df = pd.read_excel(path, engine="openpyxl")
    missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    df = df[REQUIRED_COLS].copy()
    df["City"] = df["City"].astype(str).str.strip()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
    return df


# ----------------------------------------------------------------------------
# 3. Validation
# ----------------------------------------------------------------------------
def _series_hash(sub, cols):
    t = sub[cols].copy()
    for c in t.columns:
        if pd.api.types.is_float_dtype(t[c]):
            t[c] = t[c].round(6)
    return hashlib.sha1(pd.util.hash_pandas_object(t, index=False).values.tobytes()).hexdigest()


def find_identical_city_series(df, cols):
    """Exact-duplicate city series: (a) all pollutants together, (b) each pollutant individually."""
    by_city = {c: g.sort_values("Date") for c, g in df.groupby("City")}
    full = {}
    for city, g in by_city.items():
        full.setdefault(_series_hash(g, ["Date"] + cols), []).append(city)
    full_pairs = [p for grp in full.values() if len(grp) > 1 for p in itertools.combinations(sorted(grp), 2)]

    col_pairs = {}
    for col in cols:
        groups = {}
        for city, g in by_city.items():
            groups.setdefault(_series_hash(g, ["Date", col]), []).append(city)
        col_pairs[col] = [p for grp in groups.values() if len(grp) > 1 for p in itertools.combinations(sorted(grp), 2)]
    return full_pairs, col_pairs


def validate_daily(df, rep):
    """Runs all Task-4 checks, adds flags (Task 5) and Season (Task 6). Returns flagged df."""
    info = {}

    # --- shape / range / cities
    info["raw_shape"] = df.shape
    rep.add("overview", "shape_raw", "rows x cols", f"{df.shape[0]} x {df.shape[1]}")
    bad_dates = int(df["Date"].isna().sum())
    rep.add("overview", "unparseable_dates", "Date", bad_dates,
            "Rows with unparseable Date are dropped." if bad_dates else "")
    df = df.dropna(subset=["Date"]).copy()
    dmin, dmax = df["Date"].min(), df["Date"].max()
    n_cities = df["City"].nunique()
    info.update(date_min=dmin, date_max=dmax, n_cities=n_cities)
    rep.add("overview", "date_range", "Date", f"{dmin.date()} to {dmax.date()}")
    rep.add("overview", "n_cities", "City", n_cities,
            "" if n_cities == 101 else "Expected 101 cities; please inspect.")
    expected_days = (dmax - dmin).days + 1
    rep.add("overview", "calendar_days_in_range", "Date", expected_days)

    # --- missing values
    miss = df.isna().sum()
    for col, n in miss.items():
        rep.add("missing_values", "n_missing", col, int(n), f"{100 * n / len(df):.3f}% of rows")

    # --- duplicate City-Date rows
    dup_mask = df.duplicated(subset=["City", "Date"], keep=False)
    exact_dup = int(df.duplicated(keep="first").sum())
    key_dup = int(df.duplicated(subset=["City", "Date"], keep="first").sum())
    rep.add("duplicates", "duplicate_city_date_rows (excess)", "City+Date", key_dup)
    rep.add("duplicates", "exact_duplicate_rows (excess)", "all columns", exact_dup)
    if key_dup:
        conflicting = int(dup_mask.sum()) - exact_dup
        rep.add("duplicates", "conflicting_duplicates", "City+Date", conflicting,
                "First occurrence kept after sorting; inspect if non-zero.")
        df = df.drop_duplicates(subset=["City", "Date"], keep="first")

    # --- sort
    df = df.sort_values(["City", "Date"]).reset_index(drop=True)
    info["clean_shape"] = df.shape
    rep.add("overview", "shape_after_date_parse_and_dedup", "rows x cols", f"{df.shape[0]} x {df.shape[1]}")

    # --- negative values
    df["Flag_Negative"] = (df[POLLUTANTS] < 0).any(axis=1)
    for col in POLLUTANTS:
        rep.add("negative_values", "n_negative", col, int((df[col] < 0).sum()),
                "Negatives are flagged, not altered.")

    # --- PM2.5 > PM10
    df["Flag_PM25_gt_PM10"] = df["PM2_5"] > df["PM10"]
    n_pm = int(df["Flag_PM25_gt_PM10"].sum())
    rep.add("physical_consistency", "PM2_5 > PM10 rows", "PM2_5,PM10", n_pm,
            f"{100 * n_pm / len(df):.3f}% of rows. This indicates possible measurement/estimation "
            "inconsistency; rows are flagged, not deleted.")

    # --- extreme-value summary + flags (kept, not removed)
    for col in POLLUTANTS:
        s = df[col]
        grp = df.groupby("City")[col]
        q1, q3 = grp.transform(lambda x: x.quantile(0.25)), grp.transform(lambda x: x.quantile(0.75))
        iqr = q3 - q1
        fence = q3 + EXTREME_IQR_K * iqr
        df[f"Flag_Extreme_{col}"] = (s > fence) & (iqr > 0)
        qs = s.quantile([0.01, 0.5, 0.99, 0.999])
        rep.add("extreme_summary", "min/p1/median/p99/p99.9/max", col,
                f"{s.min():.2f}/{qs[0.01]:.2f}/{qs[0.5]:.2f}/{qs[0.99]:.2f}/{qs[0.999]:.2f}/{s.max():.2f}",
                "ug/m3 daily means")
        n_ext = int(df[f"Flag_Extreme_{col}"].sum())
        rep.add("extreme_summary", f"n_flagged (per-city Q3+{EXTREME_IQR_K:g}*IQR)", col, n_ext,
                "Flagged and retained; high values may reflect genuine pollution episodes.")
        if n_ext:
            top = df.loc[df[f"Flag_Extreme_{col}"], "City"].value_counts().head(5)
            rep.add("extreme_summary", "top_cities_by_flag_count", col,
                    "; ".join(f"{c}:{n}" for c, n in top.items()))
    df["Flag_Extreme_any"] = df[[f"Flag_Extreme_{c}" for c in POLLUTANTS]].any(axis=1)
    rep.add("extreme_summary", "rows_with_any_extreme_flag", "all pollutants", int(df["Flag_Extreme_any"].sum()))

    # --- city date coverage
    cov = df.groupby("City")["Date"].agg(n_dates="nunique", first="min", last="max")
    cov["coverage_pct"] = 100 * cov["n_dates"] / expected_days
    incomplete = cov[cov["n_dates"] < expected_days]
    rep.add("coverage", "cities_with_incomplete_date_coverage", "City", len(incomplete),
            f"Expected {expected_days} dates per city.")
    for city, r in incomplete.sort_values("coverage_pct").iterrows():
        rep.add("coverage", "city_incomplete", city, f"{int(r['n_dates'])}/{expected_days}",
                f"{r['coverage_pct']:.2f}% | {r['first'].date()} to {r['last'].date()}")
    info["coverage"] = cov

    # --- identical city-series pairs
    full_pairs, col_pairs = find_identical_city_series(df, POLLUTANTS)
    rep.add("identical_series", "identical_pairs_all_pollutants", "City pairs", len(full_pairs))
    for a, b in full_pairs[:100]:
        rep.add("identical_series", "identical_pair_all_pollutants", f"{a} | {b}", "identical")
    for col, pairs in col_pairs.items():
        rep.add("identical_series", "identical_pairs_single_pollutant", col, len(pairs),
                "Exactly identical full-period series in two cities is unusual and suggests "
                "possible duplication or shared source data; not a definitive finding.")
        for a, b in pairs[:50]:
            rep.add("identical_series", "identical_pair_single_pollutant", f"{col}: {a} | {b}", "identical")

    # --- shared coordinates (relevant to API dedup)
    ll = df.drop_duplicates("City")[["City", "Latitude", "Longitude"]].copy()
    ll["key"] = [coord_key(a, b) for a, b in zip(ll["Latitude"], ll["Longitude"])]
    shared = ll.groupby("key")["City"].apply(list)
    shared = shared[shared.apply(len) > 1]
    rep.add("coordinates", "n_unique_coordinates", "City", ll["key"].nunique(), f"of {len(ll)} cities")
    rep.add("coordinates", "coordinate_groups_shared_by_multiple_cities", "City", len(shared),
            "; ".join("/".join(v) for v in shared.head(10)))
    rep.add("coordinates", "lat_lon_missing_cities", "City", int(ll[["Latitude", "Longitude"]].isna().any(axis=1).sum()))

    # --- season (Task 6)
    df["Season"] = df["Date"].dt.month.map(SEASON_MAP)
    for s, n in df["Season"].value_counts().items():
        rep.add("season", "rows_per_season", s, int(n))

    return df, info


# ----------------------------------------------------------------------------
# 4. Open-Meteo hourly O3 / CO retrieval (resumable, cached per chunk)
# ----------------------------------------------------------------------------
class RateLimitStop(Exception):
    pass


def make_chunks(start, end, days=CHUNK_DAYS):
    chunks, s = [], start
    while s <= end:
        e = min(s + pd.Timedelta(days=days - 1), end)
        chunks.append((s, e))
        s = e + pd.Timedelta(days=1)
    return chunks


def fetch_chunk(session, lat, lon, s, e):
    """Returns (DataFrame|None, status_message). Raises RateLimitStop for hourly/daily limits."""
    params = {"latitude": lat, "longitude": lon, "hourly": ",".join(HOURLY_VARS),
              "start_date": s.strftime("%Y-%m-%d"), "end_date": e.strftime("%Y-%m-%d"),
              "timezone": TIMEZONE}
    last_msg = ""
    for attempt in range(MAX_RETRIES):
        try:
            r = session.get(OPENMETEO_URL, params=params, timeout=90)
            if r.status_code == 200:
                js = r.json()
                h = js.get("hourly")
                if not h or "time" not in h:
                    return None, "empty_response"
                units = js.get("hourly_units", {})
                out = pd.DataFrame({"time": pd.to_datetime(h["time"]),
                                    "ozone": h.get("ozone"),
                                    "carbon_monoxide": h.get("carbon_monoxide")})
                u_o3, u_co = units.get("ozone", ""), units.get("carbon_monoxide", "")
                if "g/m" not in u_o3 or "g/m" not in u_co:
                    return None, f"unexpected_units O3={u_o3} CO={u_co}"
                return out, f"ok ({u_o3},{u_co})"
            reason = ""
            try:
                reason = r.json().get("reason", "")
            except Exception:
                reason = r.text[:200]
            last_msg = f"HTTP {r.status_code}: {reason}"
            if r.status_code == 429:
                low = reason.lower()
                if "hourly" in low or "daily" in low:
                    if "hourly" in low and WAIT_ON_HOURLY_LIMIT:
                        print("  Hourly limit hit; sleeping 61 min ...")
                        time.sleep(3660)
                        continue
                    raise RateLimitStop(last_msg)
                time.sleep(65)                       # minutely limit
                continue
            if 400 <= r.status_code < 500:
                return None, last_msg                # bad request, e.g. date outside available range
            time.sleep(2 ** attempt * 2)
        except RateLimitStop:
            raise
        except requests.RequestException as ex:
            last_msg = f"request_error: {type(ex).__name__}"
            time.sleep(2 ** attempt * 2)
    return None, last_msg or "failed"


def fetch_hourly_all(cities_df, fetch_start, fetch_end, rep):
    """Fetch for each unique coordinate; return dict coord_key -> hourly DataFrame (or None)."""
    coords = sorted({coord_key(a, b) for a, b in zip(cities_df["Latitude"], cities_df["Longitude"])
                     if pd.notna(a) and pd.notna(b)})
    chunks = make_chunks(fetch_start, fetch_end)
    print(f"Unique coordinates: {len(coords)} | chunks per coordinate: {len(chunks)} "
          f"| max API calls: {len(coords) * len(chunks)}")
    results, log = {}, []
    session = requests.Session()
    stopped = False
    for i, (lat, lon) in enumerate(coords, 1):
        parts = []
        for (s, e) in chunks:
            tag = f"{lat:.4f}_{lon:.4f}_{s:%Y%m%d}_{e:%Y%m%d}"
            cpath = os.path.join(CACHE_DIR, tag + ".csv")
            if os.path.exists(cpath):
                parts.append(pd.read_csv(cpath, parse_dates=["time"]))
                log.append((lat, lon, f"{s:%Y-%m-%d}..{e:%Y-%m-%d}", "cached", len(parts[-1])))
                continue
            if stopped or not FETCH_HOURLY:
                log.append((lat, lon, f"{s:%Y-%m-%d}..{e:%Y-%m-%d}", "skipped", 0))
                continue
            try:
                part, msg = fetch_chunk(session, lat, lon, s, e)
            except RateLimitStop as ex:
                stopped = True
                msg, part = f"rate_limit_stop: {ex}", None
                print(f"  ! API limit reached ({ex}). Remaining requests skipped; re-run later to resume from cache.")
            if part is not None:
                part.to_csv(cpath, index=False)
                parts.append(part)
            log.append((lat, lon, f"{s:%Y-%m-%d}..{e:%Y-%m-%d}", msg, 0 if part is None else len(part)))
            time.sleep(REQUEST_PAUSE_S)
        results[(lat, lon)] = (pd.concat(parts, ignore_index=True).drop_duplicates("time").sort_values("time")
                               if parts else None)
        if i % 10 == 0 or i == len(coords):
            ok = sum(v is not None for v in results.values())
            print(f"  progress {i}/{len(coords)} coordinates | with data: {ok}")

    logdf = pd.DataFrame(log, columns=["lat", "lon", "chunk", "status", "rows"])
    n_ok = int(logdf["status"].str.startswith(("ok", "cached")).sum())
    rep.add("openmeteo_fetch", "chunks_ok_or_cached", "all", f"{n_ok}/{len(logdf)}")
    for st, n in logdf["status"].str.split(":").str[0].str.split(" ").str[0].value_counts().items():
        rep.add("openmeteo_fetch", "chunk_status_count", st, int(n))
    failed = logdf[~logdf["status"].str.startswith(("ok", "cached"))]
    for _, r in failed.head(60).iterrows():
        rep.add("openmeteo_fetch", "chunk_failed", f"({r['lat']},{r['lon']}) {r['chunk']}", r["status"])
    return results


def build_hourly_table(cities_df, results):
    frames = []
    for _, r in cities_df.drop_duplicates("City").iterrows():
        h = results.get(coord_key(r["Latitude"], r["Longitude"]))
        if h is None or h.empty:
            continue
        t = h.copy()
        t.insert(0, "City", r["City"])
        t.insert(1, "Latitude", r["Latitude"])
        t.insert(2, "Longitude", r["Longitude"])
        frames.append(t)
    if not frames:
        return pd.DataFrame(columns=["City", "Latitude", "Longitude", "time", "ozone", "carbon_monoxide"])
    return pd.concat(frames, ignore_index=True)


# ----------------------------------------------------------------------------
# 5. Rolling 8-hour averages -> daily maxima
# ----------------------------------------------------------------------------
def compute_8hr_daily(hourly):
    """Trailing 8-h mean (hour h uses h-7..h), assigned to the local date of hour h; daily max.
    Hours are reindexed to a complete hourly grid so gaps stay NaN rather than being bridged."""
    cols = ["City", "Date", "O3_8hr_max", "CO_8hr_max", "O3_hourly_daily_mean", "CO_hourly_daily_mean"]
    if hourly.empty:
        e = pd.DataFrame({"City": pd.Series(dtype="object"), "Date": pd.Series(dtype="datetime64[ns]")})
        for c in cols[2:]:
            e[c] = pd.Series(dtype="float64")
        return e[cols]
    out = []
    for city, g in hourly.groupby("City", sort=False):
        s = g.drop_duplicates("time").set_index("time").sort_index()[HOURLY_VARS]
        idx = pd.date_range(s.index.min().normalize(), s.index.max().normalize() + pd.Timedelta(hours=23), freq="h")
        s = s.reindex(idx)
        roll = s.rolling(ROLL_WINDOW_H, min_periods=ROLL_MIN_HOURS).mean()
        day = idx.normalize()
        d_max, d_cnt, d_mean = roll.groupby(day).max(), roll.groupby(day).count(), s.groupby(day).mean()
        res = pd.DataFrame({
            "O3_8hr_max": d_max["ozone"].where(d_cnt["ozone"] >= MIN_ROLL_PER_DAY),
            "CO_8hr_max": d_max["carbon_monoxide"].where(d_cnt["carbon_monoxide"] >= MIN_ROLL_PER_DAY),
            "O3_hourly_daily_mean": d_mean["ozone"],
            "CO_hourly_daily_mean": d_mean["carbon_monoxide"],
        })
        res.index.name = "Date"
        res["City"] = city
        out.append(res.reset_index())
    return pd.concat(out, ignore_index=True)[cols]


# ----------------------------------------------------------------------------
# 6. Merge, fallback, coverage documentation
# ----------------------------------------------------------------------------
def merge_master(daily, eight, rep):
    daily_keys = daily[["City", "Date"]]
    m = daily.merge(eight, on=["City", "Date"], how="left")
    for c in ("O3_8hr_max", "CO_8hr_max", "O3_hourly_daily_mean", "CO_hourly_daily_mean"):
        m[c] = pd.to_numeric(m[c], errors="coerce")
    assert len(m) == len(daily), "Merge changed row count"

    # agreement between hourly-derived daily mean and the dataset's own daily mean (diagnostic only)
    for pol, hcol in (("O3", "O3_hourly_daily_mean"), ("CO", "CO_hourly_daily_mean")):
        both = m[[pol, hcol]].dropna()
        if len(both) > 100:
            rho = both[pol].corr(both[hcol], method="spearman")
            ratio = float((both[pol] / both[hcol].replace(0, np.nan)).median())
            rep.add("diagnostics", f"spearman_{pol}_dataset_vs_openmeteo_daily_mean", pol, f"{rho:.3f}",
                    "Descriptive only; a moderate or low value would suggest the two sources differ in "
                    "origin or processing.")
            rep.add("diagnostics", f"median_ratio_{pol}_dataset_over_openmeteo", pol, f"{ratio:.3f}")

    # coverage before fallback
    n = len(m)
    for var in ("O3_8hr_max", "CO_8hr_max"):
        rep.add("hourly_coverage", f"rows_with_{var}_from_hourly", var, int(m[var].notna().sum()),
                f"{100 * m[var].notna().mean():.2f}% of {n} city-days")
    per_city = m.groupby("City")[["O3_8hr_max", "CO_8hr_max"]].apply(lambda g: g.notna().mean() * 100)
    short = per_city[(per_city < 99.99).any(axis=1)]
    rep.add("hourly_coverage", "cities_with_incomplete_8hr_coverage", "City", len(short))
    for city, r in short.sort_values("O3_8hr_max").head(120).iterrows():
        rep.add("hourly_coverage", "city_8hr_coverage_pct", city, f"O3 {r['O3_8hr_max']:.1f} | CO {r['CO_8hr_max']:.1f}")

    # sources + provisional fallback (daily mean) where the hourly-derived value is absent
    m["O3_8hr_max_source"] = np.select(
        [m["O3_8hr_max"].notna(), m["O3"].notna()],
        ["openmeteo_hourly", "PROVISIONAL_daily_mean_fallback"], default="missing")
    m["CO_8hr_max_source"] = np.select(
        [m["CO_8hr_max"].notna(), m["CO"].notna()],
        ["openmeteo_hourly", "PROVISIONAL_daily_mean_fallback"], default="missing")
    m["O3_8hr_max"] = m["O3_8hr_max"].fillna(m["O3"])        # ug/m3
    m["CO_8hr_max"] = m["CO_8hr_max"].fillna(m["CO"])        # ug/m3 (fallback: daily mean)
    m["CO_8hr_max_mg_m3"] = m["CO_8hr_max"] / 1000.0         # Task 10
    m["Provisional_8hr_any"] = (m["O3_8hr_max_source"].str.startswith("PROVISIONAL") |
                                m["CO_8hr_max_source"].str.startswith("PROVISIONAL"))

    for col in ("O3_8hr_max_source", "CO_8hr_max_source"):
        for src, c in m[col].value_counts().items():
            rep.add("fallback", f"{col}_counts", src, int(c))
    n_prov = int(m["Provisional_8hr_any"].sum())
    rep.add("fallback", "rows_with_provisional_8hr_values", "all", n_prov,
            "Fallback daily means are not peak-based 8-h maxima and likely understate peaks; "
            "treat as provisional." if n_prov else "No fallback needed.")

    keep = [c for c in daily.columns] + ["O3_8hr_max", "CO_8hr_max_mg_m3",
                                         "O3_8hr_max_source", "CO_8hr_max_source", "Provisional_8hr_any"]
    # keep CO_8hr_max (ug/m3) too, for traceability
    keep.insert(keep.index("CO_8hr_max_mg_m3"), "CO_8hr_max")
    return m[keep]


# ----------------------------------------------------------------------------
# 7. Validation figure
# ----------------------------------------------------------------------------
def make_figure(raw_df, master, info, path):
    rng = np.random.RandomState(SEED)
    fig, ax = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Part 1 - Data validation overview (Open-Meteo/CAMS-based O3/CO estimates; "
                 "not direct CPCB readings)", fontsize=12)

    miss = (raw_df[REQUIRED_COLS].isna().mean() * 100).sort_values()
    ax[0, 0].barh(miss.index, miss.values, color="#4C72B0")
    ax[0, 0].set_title("Missing values by column (%)")
    ax[0, 0].set_xlabel("% missing")

    cov = info["coverage"]["coverage_pct"]
    ax[0, 1].hist(cov, bins=20, color="#55A868")
    ax[0, 1].set_title("Date coverage per city (%)")
    ax[0, 1].set_xlabel("% of calendar days present")
    ax[0, 1].set_ylabel("Number of cities")

    samp = master.sample(min(20000, len(master)), random_state=SEED)
    f = samp["Flag_PM25_gt_PM10"]
    ax[0, 2].scatter(samp.loc[~f, "PM10"], samp.loc[~f, "PM2_5"], s=3, alpha=0.3, color="#4C72B0", label="PM2.5 <= PM10")
    ax[0, 2].scatter(samp.loc[f, "PM10"], samp.loc[f, "PM2_5"], s=8, alpha=0.8, color="#C44E52", label="PM2.5 > PM10")
    lim = float(np.nanmax([samp["PM10"].max(), samp["PM2_5"].max()]))
    ax[0, 2].plot([0, lim], [0, lim], "k--", lw=0.8)
    ax[0, 2].set_title("PM2.5 vs PM10 (random sample, seed 42)")
    ax[0, 2].set_xlabel("PM10 (ug/m3)")
    ax[0, 2].set_ylabel("PM2.5 (ug/m3)")
    ax[0, 2].legend(markerscale=3)

    ext = pd.Series({c: int(master[f"Flag_Extreme_{c}"].sum()) for c in POLLUTANTS})
    ax[1, 0].bar(ext.index, ext.values, color="#DD8452")
    ax[1, 0].set_title(f"Flagged extreme days (per-city Q3+{EXTREME_IQR_K:g}*IQR)")
    ax[1, 0].set_ylabel("Count (retained, not deleted)")

    order = ["Winter", "Summer", "Monsoon", "Post-monsoon"]
    data = [master.loc[master["Season"] == s, "PM2_5"].dropna().values for s in order]
    try:
        ax[1, 1].boxplot(data, tick_labels=order, showfliers=False)   # matplotlib >= 3.9
    except TypeError:
        ax[1, 1].boxplot(data, labels=order, showfliers=False)
    ax[1, 1].set_title("PM2.5 daily mean by season (outliers hidden)")
    ax[1, 1].set_ylabel("ug/m3")

    hv = master[master["O3_8hr_max_source"] == "openmeteo_hourly"]
    if len(hv) > 0:
        hs = hv.sample(min(15000, len(hv)), random_state=SEED)
        ax[1, 2].scatter(hs["O3"], hs["O3_8hr_max"], s=3, alpha=0.3, color="#8172B2")
        ax[1, 2].set_xlabel("Dataset daily-mean O3 (ug/m3)")
        ax[1, 2].set_ylabel("Open-Meteo O3_8hr_max (ug/m3)")
        pct = 100 * (master["O3_8hr_max_source"] == "openmeteo_hourly").mean()
        ax[1, 2].set_title(f"O3: daily mean vs 8-h max ({pct:.1f}% hourly-derived)")
    else:
        ax[1, 2].text(0.5, 0.5, "No hourly data retrieved.\nDaily-mean fallback (PROVISIONAL) used.",
                      ha="center", va="center", fontsize=11)
        ax[1, 2].set_title("O3 8-h maximum coverage")
        ax[1, 2].axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(path, dpi=150)
    plt.close(fig)


# ----------------------------------------------------------------------------
# 8. Main pipeline
# ----------------------------------------------------------------------------
def main():
    rep = Report()
    banner("PART 1 - Data loading, validation and hourly O3/CO 8-hour maxima")
    print(DATA_DISCLAIMER)
    rep.add("meta", "data_source_note", "O3/CO hourly", DATA_DISCLAIMER)
    rep.add("meta", "seed", "", SEED)
    rep.add("meta", "run_id", "", RUN_ID)

    # --- load (original file is only read; hash recorded before and after)
    src = find_input_file()
    hash_before = sha256_of(src)
    print(f"Input file: {src}")
    raw = load_daily(src)
    rep.add("meta", "input_file", "path", src)
    rep.add("meta", "input_sha256", "before_read", hash_before)

    # --- validate and flag
    banner("Validation")
    df, info = validate_daily(raw, rep)
    print(f"Shape: {info['clean_shape']} | Dates: {info['date_min'].date()} to {info['date_max'].date()} "
          f"| Cities: {info['n_cities']}")
    qr = rep.to_df()
    for sect in ("missing_values", "duplicates", "negative_values", "physical_consistency", "coverage", "identical_series"):
        sub = qr[(qr["section"] == sect) & (~qr["check"].isin(["city_incomplete", "identical_pair_all_pollutants",
                                                               "identical_pair_single_pollutant"]))]
        for _, r in sub.iterrows():
            if str(r["value"]) not in ("0", "0.0"):
                print(f"  [{sect}] {r['check']} | {r['item']}: {r['value']}")

    # --- hourly retrieval
    banner("Open-Meteo hourly O3 / CO retrieval")
    # one extra day before the start so trailing 8-h windows at 00:00-06:00 are complete
    fetch_start = info["date_min"] - pd.Timedelta(days=1)
    fetch_end = info["date_max"]
    rep.add("openmeteo_fetch", "requested_period", "UTC+05:30 local dates",
            f"{fetch_start.date()} to {fetch_end.date()}", "Includes one buffer day for rolling windows.")
    cities_df = df.drop_duplicates("City")[["City", "Latitude", "Longitude"]]
    results = fetch_hourly_all(cities_df, fetch_start, fetch_end, rep)
    hourly = build_hourly_table(cities_df, results)
    hourly_out = hourly.copy()
    for c in HOURLY_VARS:
        hourly_out[c] = hourly_out[c].round(2)
    hourly_out.to_csv(HOURLY_CSV, index=False, date_format="%Y-%m-%d %H:%M")
    n_city_hourly = hourly["City"].nunique() if len(hourly) else 0
    rep.add("hourly_coverage", "cities_with_any_hourly_data", "City", f"{n_city_hourly}/{info['n_cities']}")
    rep.add("hourly_coverage", "hourly_rows_saved", HOURLY_CSV, len(hourly))
    if len(hourly):
        exp_hours = ((fetch_end - fetch_start).days + 1) * 24
        hc = hourly.groupby("City")[HOURLY_VARS].count() / exp_hours * 100
        rep.add("hourly_coverage", "median_hourly_completeness_pct", "ozone", f"{hc['ozone'].median():.2f}")
        rep.add("hourly_coverage", "median_hourly_completeness_pct", "carbon_monoxide",
                f"{hc['carbon_monoxide'].median():.2f}")
        for city, r in hc[(hc < 99.9).any(axis=1)].head(120).iterrows():
            rep.add("hourly_coverage", "city_hourly_completeness_pct", city,
                    f"O3 {r['ozone']:.1f} | CO {r['carbon_monoxide']:.1f}")
    print(f"Hourly rows: {len(hourly):,} | cities with data: {n_city_hourly}/{info['n_cities']}")

    # --- 8-hour metrics and merge
    banner("8-hour maxima and merge")
    eight = compute_8hr_daily(hourly)
    master = merge_master(df, eight, rep)
    # column order: original 15 -> Season + validation flags -> new 8-hour columns
    flag_cols = ["Season", "Flag_Negative", "Flag_PM25_gt_PM10"] + \
                [f"Flag_Extreme_{c}" for c in POLLUTANTS] + ["Flag_Extreme_any"]
    new_cols = ["O3_8hr_max", "CO_8hr_max", "CO_8hr_max_mg_m3",
                "O3_8hr_max_source", "CO_8hr_max_source", "Provisional_8hr_any"]
    master = master[REQUIRED_COLS + flag_cols + new_cols]

    # sanity checks
    assert master.duplicated(["City", "Date"]).sum() == 0
    assert (master["CO_8hr_max_mg_m3"].dropna() >= 0).all() or master["CO_8hr_max_mg_m3"].isna().all()
    assert np.allclose((master["CO_8hr_max"] / 1000).fillna(-1), master["CO_8hr_max_mg_m3"].fillna(-1))
    orig_o3_unchanged = master["O3"].equals(df["O3"]) and master["CO"].equals(df["CO"])
    rep.add("integrity", "original_daily_mean_O3_CO_unchanged", "O3,CO", orig_o3_unchanged)
    rep.add("integrity", "master_shape", "rows x cols", f"{master.shape[0]} x {master.shape[1]}")
    print(f"Master shape: {master.shape} | provisional rows: {int(master['Provisional_8hr_any'].sum())}")

    # --- confirm the original Excel is untouched
    hash_after = sha256_of(src)
    rep.add("meta", "input_sha256", "after_read", hash_after, "unchanged" if hash_after == hash_before else "CHANGED!")
    assert hash_after == hash_before, "Original Excel file hash changed!"

    # --- save outputs
    master.to_csv(MASTER_CSV, index=False, date_format="%Y-%m-%d")
    rep.add("meta", "disclaimer", "", "Findings suggest patterns only; no definitive source attribution is made.")
    qdf = rep.to_df()
    qdf.to_csv(QUALITY_CSV, index=False)
    make_figure(raw, master, info, FIG_PNG)

    # versioned archive copies
    ver_master = os.path.join(VERSION_DIR, "master_real_units.csv")
    ver_quality = os.path.join(VERSION_DIR, "data_quality_report.csv")
    shutil.copy2(MASTER_CSV, ver_master)
    shutil.copy2(QUALITY_CSV, ver_quality)

    # --- manifest
    files = [MASTER_CSV, HOURLY_CSV, QUALITY_CSV, FIG_PNG, ver_master, ver_quality]
    manifest = {
        "part": 1, "total_parts": 7, "run_id": RUN_ID, "seed": SEED,
        "project_title": "City- and Season-Aware Multi-Pollutant Air Quality Assessment and Forecasting "
                         "Framework for Indian Cities Using Deep Learning",
        "data_disclaimer": DATA_DISCLAIMER,
        "base_dir": BASE_DIR,
        "input_file": {"path": src, "sha256": hash_before, "modified": False},
        "master_columns": list(master.columns),
        "units": {"pollutants_daily": "ug/m3 (daily mean)", "O3_8hr_max": "ug/m3",
                  "CO_8hr_max": "ug/m3", "CO_8hr_max_mg_m3": "mg/m3 (= CO_8hr_max / 1000)"},
        "seasons": {"Winter": "Dec-Feb", "Summer": "Mar-May", "Monsoon": "Jun-Sep", "Post-monsoon": "Oct-Nov"},
        "eight_hour_method": f"trailing {ROLL_WINDOW_H}-h mean (min {ROLL_MIN_HOURS} valid hours), assigned to local "
                             f"date of the window-end hour; daily max requires >= {MIN_ROLL_PER_DAY} valid windows",
        "provisional_fallback_rows": int(master["Provisional_8hr_any"].sum()),
        "n_rows_master": int(len(master)), "n_cities": int(master["City"].nunique()),
        "date_range": [str(info["date_min"].date()), str(info["date_max"].date())],
        "files_created": [{"path": p, "rel_path": os.path.relpath(p, BASE_DIR),
                           "size_bytes": os.path.getsize(p)} for p in files] +
                         [{"path": MANIFEST_JSON, "rel_path": os.path.relpath(MANIFEST_JSON, BASE_DIR),
                           "size_bytes": None}],
        "api_cache_dir": CACHE_DIR,
        "next_part_should_load": MASTER_CSV,
    }
    with open(MANIFEST_JSON, "w") as f:
        json.dump(manifest, f, indent=2, default=str)
    shutil.copy2(MANIFEST_JSON, os.path.join(VERSION_DIR, "part1_handoff_manifest.json"))

    # --- final printout
    banner("PART 1 COMPLETE - files created (exact paths for Part 2)")
    for p in files + [MANIFEST_JSON, os.path.join(VERSION_DIR, "part1_handoff_manifest.json")]:
        print(p)
    print(f"\nBASE_DIR for later parts: {BASE_DIR}")
    print(f"Hourly API cache (resumable): {CACHE_DIR}")
    if manifest["provisional_fallback_rows"]:
        print(f"\nNOTE: {manifest['provisional_fallback_rows']:,} city-days use PROVISIONAL daily-mean fallback "
              "for O3/CO. Re-running this script resumes from the cache and may fill the gaps.")
    print("\n" + DATA_DISCLAIMER)
    return master


if __name__ == "__main__" or IN_COLAB:
    master_df = main()


PART 1 - Data loading, validation and hourly O3/CO 8-hour maxima
Hourly O3 and CO are Open-Meteo/CAMS-based model estimates, not direct CPCB ground-station readings.
Excel file not found - please upload it now.


Saving AQI_Dataset_2022_2026_Clean (1).xlsx to AQI_Dataset_2022_2026_Clean (1).xlsx
Input file: /content/AQI_Dataset_2022_2026_Clean (1).xlsx

Validation
Shape: (146147, 15) | Dates: 2022-08-04 to 2026-07-20 | Cities: 101
  [identical_series] identical_pairs_all_pollutants | City pairs: 1
  [identical_series] identical_pairs_single_pollutant | PM2_5: 7
  [identical_series] identical_pairs_single_pollutant | PM10: 7
  [identical_series] identical_pairs_single_pollutant | CO: 1
  [identical_series] identical_pairs_single_pollutant | NO2: 7
  [identical_series] identical_pairs_single_pollutant | SO2: 7
  [identical_series] identical_pairs_single_pollutant | O3: 7

Open-Meteo hourly O3 / CO retrieval
Unique coordinates: 101 | chunks per coordinate: 4 | max API calls: 404
  progress 10/101 coordinates | with data: 10
  progress 20/101 coordinates | with data: 20
  progress 30/101 coordinates | with data: 30
  progress 40/101 coordinates | with data: 40
  progress 50/101 coordinates | with d

In [3]:
# =============================================================================
# City- and Season-Aware Multi-Pollutant Air Quality Assessment and Forecasting
# Framework for Indian Cities Using Deep Learning
#
# PART 2 of 7: six-pollutant estimated AQI (CPCB sub-index methodology),
#              dominant pollutant, EDA, and the primary City x Season table.
#
# Continues directly from Part 1 (same BASE_DIR, reads Part 1's manifest,
# master_real_units.csv and data_quality_report.csv). The Part 1 files are only
# read, never modified. No train/test split and no scaling are done here.
#
# LABEL : "Six-pollutant estimated AQI using CPCB sub-index methodology."
# CAVEAT: NH3 and Pb are unavailable, so this is NOT the official CPCB AQI.
# DATA  : O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct
#         CPCB ground readings (rows using daily-mean fallback are flagged).
# =============================================================================

# ----------------------------------------------------------------------------
# 0. Imports, configuration, reproducibility, link to Part 1
# ----------------------------------------------------------------------------
import os
import sys
import json
import random
import hashlib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.RandomState(SEED)

# ---- same project root as Part 1 --------------------------------------------
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
BASE_DIR = "/content/aqi_framework" if IN_COLAB else os.path.abspath("./aqi_framework")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
PROC_DIR = os.path.join(BASE_DIR, "data", "processed")
RES_DIR = os.path.join(BASE_DIR, "results")
FIG_DIR = os.path.join(BASE_DIR, "figures")
EDA_DIR = os.path.join(FIG_DIR, "eda_aqi")
for d in (PROC_DIR, RES_DIR, FIG_DIR, EDA_DIR):
    os.makedirs(d, exist_ok=True)

# ---- Part 1 inputs ----------------------------------------------------------
P1_MASTER = os.path.join(PROC_DIR, "master_real_units.csv")
P1_QUALITY = os.path.join(PROC_DIR, "data_quality_report.csv")
P1_MANIFEST = os.path.join(PROC_DIR, "part1_handoff_manifest.json")

# ---- Part 2 outputs (as specified) -------------------------------------------
AQI_V1_CSV = os.path.join(PROC_DIR, "aqi_estimated_v1.csv")
DAILY_CSV = os.path.join(RES_DIR, "daily_aqi_dominant_pollutant.csv")
SUMMARY_CSV = os.path.join(RES_DIR, "city_season_pollutant_summary.csv")
RANKING_CSV = os.path.join(RES_DIR, "city_aqi_ranking.csv")
BREAKPOINT_CSV = os.path.join(RES_DIR, "cpcb_breakpoints_used.csv")
METHOD_NOTE_TXT = os.path.join(RES_DIR, "AQI_METHOD_NOTE.txt")
MANIFEST_JSON = os.path.join(RES_DIR, "part2_handoff_manifest.json")

# ---- labels ------------------------------------------------------------------
AQI_LABEL = "Six-pollutant estimated AQI using CPCB sub-index methodology."
AQI_CAVEAT = ("NH3 and Pb are unavailable, so this is not the official CPCB AQI. "
              "O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct CPCB ground readings.")
FOOTER = "Six-pollutant estimated AQI (CPCB sub-index methodology); not official CPCB AQI (NH3, Pb unavailable)."

# ---- schema ------------------------------------------------------------------
POLLUTANTS = ["PM2_5", "PM10", "NO2", "SO2", "O3", "CO"]          # tie-break order for dominance
DAILY_POLL_COLS = ["PM2_5", "PM10", "CO", "NO2", "SO2", "O3"]
WEATHER_COLS = ["Temperature_C", "Humidity_pct", "WindSpeed_kmh", "Precipitation_mm"]
SUBIDX = {p: f"{p}_subindex" for p in POLLUTANTS}
# concentration column actually used for each sub-index (CPCB: 24-h mean, or 8-h max for O3/CO)
AQI_INPUT = {"PM2_5": "PM2_5", "PM10": "PM10", "NO2": "NO2", "SO2": "SO2",
             "O3": "O3_8hr_max", "CO": "CO_8hr_max_mg_m3"}
SEASONS = ["Winter", "Summer", "Monsoon", "Post-monsoon"]
SEASON_MAP = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Summer", 4: "Summer", 5: "Summer",
              6: "Monsoon", 7: "Monsoon", 8: "Monsoon", 9: "Monsoon", 10: "Post-monsoon", 11: "Post-monsoon"}
CATEGORIES = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
CAT_COLORS = {"Good": "#009644", "Satisfactory": "#8CC63F", "Moderate": "#F2C200",
              "Poor": "#F58220", "Very Poor": "#E31E24", "Severe": "#7E0023"}
POLL_COLORS = {"PM2_5": "#C44E52", "PM10": "#DD8452", "NO2": "#4C72B0",
               "SO2": "#937860", "O3": "#55A868", "CO": "#8172B2"}
MIN_SUBINDICES = 3          # CPCB: at least 3 pollutants, one of which must be PM2.5 or PM10

# ---- CPCB breakpoints: (BP_lo, BP_hi, I_lo, I_hi) per band ------------------------------
# Units: ug/m3 (PM2.5, PM10, NO2, SO2: 24-h mean; O3: 8-h max) and mg/m3 (CO: 8-h max).
# The CPCB table is open-ended in the "Severe" band (e.g. PM2.5 > 250). To interpolate 401-500 and to
# define "exceeds the highest breakpoint", the upper end of the Severe band is an ASSUMPTION of this
# framework: lower bound + width of the previous band (documented in outputs; editable below).
BREAKPOINTS = {
    "PM2_5": [(0, 30, 0, 50), (31, 60, 51, 100), (61, 90, 101, 200), (91, 120, 201, 300),
              (121, 250, 301, 400), (251, 380, 401, 500)],
    "PM10":  [(0, 50, 0, 50), (51, 100, 51, 100), (101, 250, 101, 200), (251, 350, 201, 300),
              (351, 430, 301, 400), (431, 510, 401, 500)],
    "NO2":   [(0, 40, 0, 50), (41, 80, 51, 100), (81, 180, 101, 200), (181, 280, 201, 300),
              (281, 400, 301, 400), (401, 520, 401, 500)],
    "SO2":   [(0, 40, 0, 50), (41, 80, 51, 100), (81, 380, 101, 200), (381, 800, 201, 300),
              (801, 1600, 301, 400), (1601, 2400, 401, 500)],
    "O3":    [(0, 50, 0, 50), (51, 100, 51, 100), (101, 168, 101, 200), (169, 208, 201, 300),
              (209, 748, 301, 400), (749, 1288, 401, 500)],
    "CO":    [(0, 1.0, 0, 50), (1.1, 2.0, 51, 100), (2.1, 10, 101, 200), (10, 17, 201, 300),
              (17, 34, 301, 400), (34, 51, 401, 500)],
}
SEVERE_TOP_ASSUMED = {p: b[-1][1] for p, b in BREAKPOINTS.items()}


def banner(text):
    print("\n" + "=" * 78 + f"\n{text}\n" + "=" * 78)


def sha256_of(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


# ----------------------------------------------------------------------------
# 1. Load Part 1 outputs (read-only) and verify continuity
# ----------------------------------------------------------------------------
def load_part1():
    for p in (P1_MASTER, P1_QUALITY):
        if not os.path.isfile(p):
            raise FileNotFoundError(
                f"Missing Part 1 output: {p}\nRun Part 1 first (same BASE_DIR={BASE_DIR}). "
                "In Colab, a restarted runtime loses /content - re-run Part 1 or restore the folder.")
    manifest = {}
    if os.path.isfile(P1_MANIFEST):
        with open(P1_MANIFEST) as f:
            manifest = json.load(f)
    master = pd.read_csv(P1_MASTER, parse_dates=["Date"])
    quality = pd.read_csv(P1_QUALITY)

    need = (["Date", "City", "Region", "Latitude", "Longitude"] + DAILY_POLL_COLS + WEATHER_COLS +
            ["Season", "O3_8hr_max", "CO_8hr_max_mg_m3"])
    missing = [c for c in need if c not in master.columns]
    if missing:
        raise ValueError(f"master_real_units.csv is missing columns: {missing}")

    # continuity checks against Part 1's manifest
    if manifest:
        if manifest.get("n_rows_master") not in (None, len(master)):
            raise ValueError(f"Row count {len(master)} differs from Part 1 manifest {manifest['n_rows_master']}.")
        if manifest.get("n_cities") not in (None, master["City"].nunique()):
            raise ValueError("City count differs from Part 1 manifest.")
    if master.duplicated(["City", "Date"]).any():
        raise ValueError("Duplicate City-Date rows found in master file.")
    season_check = master["Date"].dt.month.map(SEASON_MAP)
    n_bad_season = int((season_check != master["Season"]).sum())
    if n_bad_season:
        print(f"  ! {n_bad_season} rows have a Season that differs from the Part 1 definition; "
              "using the Part 1 Season column as-is.")

    # provisional-fallback information carried over from Part 1
    if "Provisional_8hr_any" in master.columns:
        master["Provisional_8hr_any"] = master["Provisional_8hr_any"].astype(str).str.lower().isin(["true", "1"])
    else:
        master["Provisional_8hr_any"] = False
    master = master.sort_values(["City", "Date"]).reset_index(drop=True)
    return master, quality, manifest


def part1_highlights(quality):
    """Pull a few Part 1 quality facts to carry into the Part 2 manifest (descriptive only)."""
    out = {}
    q = quality
    def pick(check, item=None):
        s = q[q["check"] == check]
        if item is not None:
            s = s[s["item"] == item]
        return None if s.empty else str(s.iloc[0]["value"])
    out["PM25_gt_PM10_rows"] = pick("PM2_5 > PM10 rows")
    out["identical_pairs_all_pollutants"] = pick("identical_pairs_all_pollutants")
    out["rows_with_provisional_8hr_values"] = pick("rows_with_provisional_8hr_values")
    out["cities_with_incomplete_date_coverage"] = pick("cities_with_incomplete_date_coverage")
    return out


# ----------------------------------------------------------------------------
# 2. CPCB-style sub-indices, estimated AQI, dominant pollutant
# ----------------------------------------------------------------------------
def subindex(conc, bands):
    """Official per-band linear interpolation:
       I = (I_hi - I_lo) / (BP_hi - BP_lo) * (C - BP_lo) + I_lo.
       Band = first band whose BP_hi >= C. C above the top (assumed) Severe breakpoint -> 500 and capped.
       Negative or missing concentrations -> NaN. Returns (subindex, capped_flag)."""
    c = np.asarray(conc, dtype=float)
    his = np.array([b[1] for b in bands])
    los = np.array([b[0] for b in bands], dtype=float)
    ilo = np.array([b[2] for b in bands], dtype=float)
    ihi = np.array([b[3] for b in bands], dtype=float)
    out = np.full(c.shape, np.nan)
    capped = np.zeros(c.shape, dtype=bool)
    valid = ~np.isnan(c) & (c >= 0)
    idx = np.searchsorted(his, c, side="left")            # first band with BP_hi >= C
    over = valid & (idx >= len(bands))
    ok = valid & ~over
    k = idx[ok]
    val = (ihi[k] - ilo[k]) / (his[k] - los[k]) * (c[ok] - los[k]) + ilo[k]
    out[ok] = np.clip(val, ilo[k], ihi[k])                # gap values between bands stay inside their band
    out[over] = 500.0
    capped[over] = True
    return out, capped


def categorize(aqi):
    a = np.asarray(aqi, dtype=float)
    cat = np.select([a <= 50, a <= 100, a <= 200, a <= 300, a <= 400, a > 400], CATEGORIES, default="")
    cat = pd.Series(cat, index=getattr(aqi, "index", None)).replace("", np.nan)
    return pd.Categorical(cat, categories=CATEGORIES, ordered=True)


def compute_aqi(master):
    df = master.copy()
    capped_cols = []
    for p in POLLUTANTS:
        si, cap = subindex(df[AQI_INPUT[p]].values, BREAKPOINTS[p])
        df[SUBIDX[p]] = si
        df[f"{p}_SevereCapped"] = cap
        capped_cols.append(f"{p}_SevereCapped")
    sub = df[[SUBIDX[p] for p in POLLUTANTS]]
    n_valid = sub.notna().sum(axis=1)
    has_pm = df[[SUBIDX["PM2_5"], SUBIDX["PM10"]]].notna().any(axis=1)
    ok = (n_valid >= MIN_SUBINDICES) & has_pm
    raw_max = sub.max(axis=1)
    df["N_SubIndices"] = n_valid
    df["Estimated_AQI"] = np.floor(raw_max + 0.5).where(ok)               # round half up, integer AQI
    dom = sub.fillna(-1).idxmax(axis=1).str.replace("_subindex", "", regex=False)
    df["Dominant_Pollutant"] = dom.where(ok)
    df["AQI_Category"] = categorize(df["Estimated_AQI"])
    df["Severe_Capped_Flag"] = df[capped_cols].any(axis=1)
    # transparency flags
    prov_o3 = df["O3_8hr_max_source"].astype(str).str.startswith("PROVISIONAL") if "O3_8hr_max_source" in df else False
    prov_co = df["CO_8hr_max_source"].astype(str).str.startswith("PROVISIONAL") if "CO_8hr_max_source" in df else False
    df["AQI_Uses_Provisional_8hr"] = (df["Provisional_8hr_any"] | prov_o3 | prov_co)
    n_ties = int((sub.eq(raw_max, axis=0).sum(axis=1).where(ok, 0) > 1).sum())
    df["AQI_Type"] = "Estimated_6pollutant_CPCB_style_not_official"
    return df, {"n_aqi_valid": int(ok.sum()), "n_aqi_missing": int((~ok).sum()), "n_dominance_ties": n_ties}


def breakpoints_table():
    rows = []
    for p, bands in BREAKPOINTS.items():
        for i, (lo, hi, ilo, ihi) in enumerate(bands):
            rows.append({"Pollutant": p, "Input_Column": AQI_INPUT[p],
                         "Unit": "mg/m3" if p == "CO" else "ug/m3",
                         "Averaging": "8-h max" if p in ("O3", "CO") else "24-h mean",
                         "Band": CATEGORIES[i], "BP_lo": lo, "BP_hi": hi, "I_lo": ilo, "I_hi": ihi,
                         "Note": "Upper end ASSUMED by this framework (CPCB band is open-ended)" if i == 5 else ""})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# 3. Primary City x Season table (101 x 4 = 404 rows)
# ----------------------------------------------------------------------------
REQUIRED_SUMMARY_COLS = [
    "City", "Season", "Dominant_Pollutant", "Dominant_Pollutant_Mean_Concentration", "Percent_Days_Dominant",
    "Mean_PM2_5", "Mean_PM10", "Mean_CO", "Mean_NO2", "Mean_SO2", "Mean_O3",
    "Mean_Temperature", "Mean_Humidity", "Mean_WindSpeed", "Mean_Rainfall",
    "Mean_AQI", "Percent_Days_AQI_Above200"]


def build_city_season_summary(df):
    rows = []
    meta = df.drop_duplicates("City").set_index("City")[["Region", "Latitude", "Longitude"]]
    cities = sorted(df["City"].unique())
    for city in cities:
        for season in SEASONS:
            g = df[(df["City"] == city) & (df["Season"] == season)]
            r = {"City": city, "Season": season, "Region": meta.loc[city, "Region"],
                 "Latitude": meta.loc[city, "Latitude"], "Longitude": meta.loc[city, "Longitude"],
                 "N_Days": len(g)}
            v = g[g["Estimated_AQI"].notna()]
            r["N_Valid_AQI_Days"] = len(v)
            for p in DAILY_POLL_COLS:
                r[f"Mean_{p}"] = g[p].mean()
                r[f"Median_{p}"] = g[p].median()
            r["Mean_O3_8hr_max"] = g["O3_8hr_max"].mean()
            r["Mean_CO_8hr_max_mg_m3"] = g["CO_8hr_max_mg_m3"].mean()
            r["Mean_Temperature"] = g["Temperature_C"].mean()
            r["Mean_Humidity"] = g["Humidity_pct"].mean()
            r["Mean_WindSpeed"] = g["WindSpeed_kmh"].mean()
            r["Mean_Rainfall"] = g["Precipitation_mm"].mean()
            r["Mean_AQI"] = v["Estimated_AQI"].mean()
            r["Median_AQI"] = v["Estimated_AQI"].median()
            r["Percent_Days_AQI_Above200"] = 100 * (v["Estimated_AQI"] > 200).mean() if len(v) else np.nan
            for c in CATEGORIES:
                r[f"Pct_Days_{c.replace(' ', '_')}"] = 100 * (v["AQI_Category"] == c).mean() if len(v) else np.nan
            shares = (v["Dominant_Pollutant"].value_counts(normalize=True).reindex(POLLUTANTS).fillna(0) * 100
                      if len(v) else pd.Series(np.nan, index=POLLUTANTS))
            for p in POLLUTANTS:
                r[f"Pct_Days_Dominant_{p}"] = shares[p]
            if len(v):
                dom = shares.idxmax()                              # ties resolved by POLLUTANTS order
                r["Dominant_Pollutant"] = dom
                r["Percent_Days_Dominant"] = shares[dom]
                r["Dominant_Pollutant_Mean_Concentration"] = g[dom if dom != "CO" else "CO"].mean() \
                    if dom in DAILY_POLL_COLS else np.nan          # daily-mean basis, same unit as Mean_* columns
                r["Dominant_Pollutant_Mean_AQI_Input"] = g[AQI_INPUT[dom]].mean()   # value used in AQI (O3 8h; CO mg/m3)
                r["Dominant_Pollutant_Unit"] = "mg/m3 (8-h max)" if dom == "CO" else (
                    "ug/m3 (8-h max)" if dom == "O3" else "ug/m3 (24-h mean)")
                r["Dominant_Pollutant_Mean_Subindex"] = g[SUBIDX[dom]].mean()
            else:
                r["Dominant_Pollutant"] = np.nan
            r["Pct_Days_Provisional_8hr"] = 100 * g["AQI_Uses_Provisional_8hr"].mean() if len(g) else np.nan
            r["Pct_Days_Severe_Capped"] = 100 * g["Severe_Capped_Flag"].mean() if len(g) else np.nan
            rows.append(r)
    s = pd.DataFrame(rows)
    others = [c for c in s.columns if c not in REQUIRED_SUMMARY_COLS]
    s = s[REQUIRED_SUMMARY_COLS + others]
    s["Season"] = pd.Categorical(s["Season"], categories=SEASONS, ordered=True)
    s = s.sort_values(["City", "Season"]).reset_index(drop=True)
    s["Season"] = s["Season"].astype(str)
    return s


def build_city_ranking(df, summary):
    v = df[df["Estimated_AQI"].notna()]
    g = v.groupby("City")
    rk = pd.DataFrame({
        "Mean_AQI": g["Estimated_AQI"].mean(),
        "Median_AQI": g["Estimated_AQI"].median(),
        "P95_AQI": g["Estimated_AQI"].quantile(0.95),
        "Percent_Days_AQI_Above200": g["Estimated_AQI"].apply(lambda s: 100 * (s > 200).mean()),
        "N_Valid_Days": g.size()})
    mode = v.groupby("City")["Dominant_Pollutant"].agg(lambda s: s.value_counts().reindex(POLLUTANTS).fillna(0).idxmax())
    rk["Most_Frequent_Dominant_Pollutant"] = mode
    rk = rk.join(df.drop_duplicates("City").set_index("City")[["Region", "Latitude", "Longitude"]])
    rk = rk.sort_values("Mean_AQI", ascending=False)
    rk.insert(0, "Rank_Highest_AQI", np.arange(1, len(rk) + 1))
    return rk.reset_index()


# ----------------------------------------------------------------------------
# 4. EDA figures (each wrapped so one failure does not stop the pipeline)
# ----------------------------------------------------------------------------
FIG_LOG = []


def save_fig(fig, name, title=None, dpi=130):
    if title:
        fig.suptitle(title, fontsize=13, y=0.995)
    fig.text(0.5, 0.003, FOOTER, ha="center", va="bottom", fontsize=7.5, color="dimgray")
    path = os.path.join(EDA_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return path


def run_plot(fn, df, ctx):
    name = fn.__name__
    try:
        paths = fn(df, ctx)
        paths = [paths] if isinstance(paths, str) else (paths or [])
        FIG_LOG.extend(paths)
        print(f"  [ok] {name}")
    except Exception as ex:                                    # keep the pipeline running
        print(f"  [FAILED] {name}: {type(ex).__name__}: {ex}")
        FIG_LOG.append(f"FAILED:{name}:{type(ex).__name__}")


def binned_stat(x, y, nbins=20):
    d = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(d) < 200:
        return None
    d["b"] = pd.qcut(d["x"], nbins, duplicates="drop")
    return d.groupby("b", observed=True).agg(xm=("x", "median"), ym=("y", "median"),
                                             q1=("y", lambda s: s.quantile(.25)),
                                             q3=("y", lambda s: s.quantile(.75)))


def plot_distributions(df, ctx):
    fig, ax = plt.subplots(2, 3, figsize=(16, 8))
    for a, p in zip(ax.ravel(), DAILY_POLL_COLS):
        x = df[p].dropna()
        x = x[x > 0]
        a.hist(np.log10(x), bins=60, color=POLL_COLORS[p], alpha=0.85)
        a.set_title(f"{p} daily mean (log10 scale)")
        a.set_xlabel("log10(ug/m3)")
        a.set_ylabel("City-days")
    p1 = save_fig(fig, "01_pollutant_distributions.png", "Pollutant distributions (all cities, daily means)")

    fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
    data = [df[p].dropna().clip(lower=0.01) for p in DAILY_POLL_COLS]
    ax[0].boxplot(data, showfliers=True, flierprops=dict(markersize=1.5, alpha=0.3))
    ax[0].set_xticks(range(1, 7), DAILY_POLL_COLS)
    ax[0].set_yscale("log")
    ax[0].set_title("Daily-mean concentrations (log scale; extreme days retained)")
    ax[0].set_ylabel("ug/m3")
    sub = [df[SUBIDX[p]].dropna() for p in POLLUTANTS]
    ax[1].boxplot(sub, showfliers=True, flierprops=dict(markersize=1.5, alpha=0.3))
    ax[1].set_xticks(range(1, 7), POLLUTANTS)
    ax[1].set_title("CPCB-style sub-indices")
    ax[1].set_ylabel("Sub-index")
    p2 = save_fig(fig, "02_pollutant_boxplots.png", "Boxplots of concentrations and sub-indices")
    return [p1, p2]


def plot_correlations(df, ctx):
    cols = DAILY_POLL_COLS + ["O3_8hr_max", "CO_8hr_max_mg_m3"] + WEATHER_COLS
    labels = DAILY_POLL_COLS + ["O3_8h", "CO_8h"] + ["Temp", "Humid", "Wind", "Rain"]
    out = []
    fig, ax = plt.subplots(1, 2, figsize=(17, 7.5))
    for a, m in zip(ax, ["pearson", "spearman"]):
        c = df[cols].corr(method=m)
        im = a.imshow(c.values, cmap="RdBu_r", vmin=-1, vmax=1)
        a.set_xticks(range(len(cols)), labels, rotation=60, ha="right")
        a.set_yticks(range(len(cols)), labels)
        for i in range(len(cols)):
            for j in range(len(cols)):
                a.text(j, i, f"{c.values[i, j]:.2f}", ha="center", va="center", fontsize=6.5)
        a.set_title(m.capitalize())
    fig.colorbar(im, ax=ax, shrink=0.7)
    out.append(save_fig(fig, "03_correlation_heatmaps_overall.png",
                        "Correlation heatmaps (associations only; do not by themselves indicate causation)"))
    fig, ax = plt.subplots(2, 2, figsize=(15, 13))
    for a, s in zip(ax.ravel(), SEASONS):
        c = df.loc[df["Season"] == s, cols].corr(method="spearman")
        im = a.imshow(c.values, cmap="RdBu_r", vmin=-1, vmax=1)
        a.set_xticks(range(len(cols)), labels, rotation=60, ha="right")
        a.set_yticks(range(len(cols)), labels)
        a.set_title(f"{s} (Spearman)")
    fig.colorbar(im, ax=ax, shrink=0.6)
    out.append(save_fig(fig, "04_correlation_heatmaps_by_season.png", "Seasonal Spearman correlations"))
    return out


def plot_monthly_seasonal(df, ctx):
    out = []
    cols = DAILY_POLL_COLS + ["Estimated_AQI"]
    d = df.copy()
    d["Month"] = d["Date"].dt.month
    cm = d.groupby(["City", "Month"])[cols].mean().reset_index()
    fig, ax = plt.subplots(2, 4, figsize=(19, 8))
    for a, c in zip(ax.ravel(), cols):
        g = cm.groupby("Month")[c]
        a.plot(g.mean().index, g.mean().values, marker="o", color="#333")
        a.fill_between(g.mean().index, g.quantile(.25).values, g.quantile(.75).values, alpha=0.25)
        a.set_title(f"{c}: mean of city monthly means (band = city IQR)")
        a.set_xticks(range(1, 13))
        a.set_xlabel("Month")
    ax.ravel()[-1].axis("off")
    out.append(save_fig(fig, "05_monthly_climatology.png", "Monthly pattern across cities"))

    d["YearMonth"] = d["Date"].dt.to_period("M").dt.to_timestamp()
    ts = d.groupby("YearMonth")[["PM2_5", "PM10", "Estimated_AQI"]].mean()
    fig, ax = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    for a, c in zip(ax, ts.columns):
        a.plot(ts.index, ts[c], marker=".", color="#333")
        a.set_ylabel(c)
    ax[-1].set_xlabel("Month")
    out.append(save_fig(fig, "06_monthly_timeseries_national_mean.png", "Month-by-month mean across all cities"))

    sm = d.groupby("Season")[cols].mean().reindex(SEASONS)
    fig, ax = plt.subplots(2, 4, figsize=(18, 7))
    for a, c in zip(ax.ravel(), cols):
        a.bar(SEASONS, sm[c].values, color="#4C72B0")
        a.set_title(c)
        a.tick_params(axis="x", rotation=25)
    ax.ravel()[-1].axis("off")
    out.append(save_fig(fig, "07_seasonal_means.png", "Seasonal means across all city-days"))
    return out


def plot_city_comparison(df, ctx):
    cm = df.groupby("City")[DAILY_POLL_COLS].mean()
    fig, ax = plt.subplots(2, 3, figsize=(18, 10))
    for a, p in zip(ax.ravel(), DAILY_POLL_COLS):
        top = cm[p].sort_values(ascending=False).head(15)[::-1]
        a.barh(top.index, top.values, color=POLL_COLORS[p])
        a.set_title(f"Highest mean {p} (15 cities)")
        a.set_xlabel("ug/m3")
    return save_fig(fig, "08_city_comparison_top15_by_pollutant.png", "City-wise pollutant comparison (study-period means)")


def plot_city_ranking(df, ctx):
    rk = ctx["ranking"]
    fig, ax = plt.subplots(1, 2, figsize=(17, 10))
    top = rk.head(25)[::-1]
    bot = rk.tail(25)
    ax[0].barh(top["City"], top["Mean_AQI"], color=[POLL_COLORS.get(p, "#999") for p in top["Most_Frequent_Dominant_Pollutant"]])
    ax[0].set_title("25 cities with highest mean estimated AQI")
    ax[1].barh(bot["City"], bot["Mean_AQI"], color=[POLL_COLORS.get(p, "#999") for p in bot["Most_Frequent_Dominant_Pollutant"]])
    ax[1].set_title("25 cities with lowest mean estimated AQI")
    handles = [Patch(color=POLL_COLORS[p], label=f"Most frequent dominant: {p}") for p in POLLUTANTS]
    ax[1].legend(handles=handles, loc="lower right")
    for a in ax:
        a.set_xlabel("Mean estimated AQI")
    return save_fig(fig, "09_city_aqi_ranking.png", "City-level estimated AQI ranking")


def plot_category_distribution(df, ctx):
    v = df[df["Estimated_AQI"].notna()]
    ov = v["AQI_Category"].value_counts(normalize=True).reindex(CATEGORIES) * 100
    se = (pd.crosstab(v["Season"], v["AQI_Category"], normalize="index") * 100).reindex(SEASONS)[CATEGORIES]
    fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
    ax[0].bar(CATEGORIES, ov.values, color=[CAT_COLORS[c] for c in CATEGORIES])
    ax[0].set_ylabel("% of valid city-days")
    ax[0].set_title("Overall")
    ax[0].tick_params(axis="x", rotation=25)
    se.plot(kind="bar", stacked=True, ax=ax[1], color=[CAT_COLORS[c] for c in CATEGORIES], width=0.7)
    ax[1].set_ylabel("% of city-days")
    ax[1].set_title("By season")
    ax[1].tick_params(axis="x", rotation=0)
    return save_fig(fig, "10_aqi_category_distribution.png", "Estimated AQI category distribution")


def plot_maps(df, ctx):
    out = []
    summ = ctx["summary"]
    rk = ctx["ranking"]
    # static map (no basemap needed)
    fig, ax = plt.subplots(2, 2, figsize=(15, 14))
    vmax = max(summ["Mean_AQI"].max(), 1)
    for a, s in zip(ax.ravel(), SEASONS):
        g = summ[summ["Season"] == s]
        sc = a.scatter(g["Longitude"], g["Latitude"], c=g["Mean_AQI"], cmap="YlOrRd", s=45,
                       edgecolor="k", linewidth=0.3, vmin=0, vmax=vmax)
        a.set_title(f"{s}: mean estimated AQI")
        a.set_xlabel("Longitude")
        a.set_ylabel("Latitude")
        a.set_aspect("equal")
    fig.colorbar(sc, ax=ax, shrink=0.6, label="Mean estimated AQI")
    out.append(save_fig(fig, "11_india_map_static_by_season.png", "City mean estimated AQI by season (lon/lat scatter)"))

    try:
        import plotly.express as px
        d = summ.copy()
        d["Season"] = pd.Categorical(d["Season"], SEASONS, ordered=True)
        d = d.sort_values("Season")
        d["Mean_AQI_r"] = d["Mean_AQI"].round(1)
        fig = px.scatter_geo(d, lat="Latitude", lon="Longitude", color="Mean_AQI", animation_frame="Season",
                             hover_name="City", hover_data={"Dominant_Pollutant": True, "Mean_AQI_r": True,
                                                            "Percent_Days_AQI_Above200": ":.1f",
                                                            "Latitude": False, "Longitude": False, "Mean_AQI": False},
                             color_continuous_scale="YlOrRd", range_color=(0, float(d["Mean_AQI"].max())),
                             title=f"{AQI_LABEL} (city x season means; not official CPCB AQI)")
        fig.update_geos(scope="asia", lataxis_range=[5, 38], lonaxis_range=[66, 100],
                        showcountries=True, showland=True, landcolor="#f3f3f3")
        p = os.path.join(EDA_DIR, "12_india_map_plotly_interactive.html")
        fig.write_html(p, include_plotlyjs="cdn")
        out.append(p)
    except Exception as ex:
        print(f"    (plotly map skipped: {type(ex).__name__})")
    try:
        import folium
        m = folium.Map(location=[22.5, 79], zoom_start=5, tiles="CartoDB positron")
        for _, r in rk.iterrows():
            cat = categorize(pd.Series([r["Mean_AQI"]]))[0]
            col = CAT_COLORS.get(str(cat), "#999999")
            folium.CircleMarker([r["Latitude"], r["Longitude"]], radius=6, color="black", weight=0.5,
                                fill=True, fill_color=col, fill_opacity=0.85,
                                popup=folium.Popup(f"<b>{r['City']}</b><br>Mean AQI {r['Mean_AQI']:.0f} ({cat})<br>"
                                                   f"Most frequent dominant: {r['Most_Frequent_Dominant_Pollutant']}<br>"
                                                   f"Days AQI>200: {r['Percent_Days_AQI_Above200']:.1f}%", max_width=260)
                                ).add_to(m)
        p = os.path.join(EDA_DIR, "13_india_map_folium_city_mean_aqi.html")
        m.save(p)
        out.append(p)
    except Exception as ex:
        print(f"    (folium map skipped: {type(ex).__name__})")
    return out


def plot_city_timeseries(df, ctx):
    rk = ctx["ranking"]
    preferred = ["Delhi", "Mumbai", "Kolkata", "Chennai", "Bengaluru", "Hyderabad", "Lucknow", "Patna", "Ahmedabad"]
    avail = {c.lower().replace("_", " "): c for c in df["City"].unique()}
    chosen = [avail[p.lower()] for p in preferred if p.lower() in avail][:6]
    for c in (rk.iloc[0]["City"], rk.iloc[-1]["City"]):
        if c not in chosen:
            chosen.append(c)
    n = len(chosen)
    fig, ax = plt.subplots((n + 1) // 2, 2, figsize=(16, 3.4 * ((n + 1) // 2)), sharex=True)
    for a, c in zip(ax.ravel(), chosen):
        g = df[df["City"] == c].set_index("Date")["Estimated_AQI"]
        a.plot(g.index, g.values, lw=0.5, color="#999", label="Daily")
        a.plot(g.index, g.rolling(30, min_periods=15).mean(), lw=1.6, color="#C44E52", label="30-day mean")
        a.axhline(200, ls="--", lw=0.8, color="k")
        a.set_title(c)
        a.set_ylabel("Estimated AQI")
    for a in ax.ravel()[n:]:
        a.axis("off")
    ax.ravel()[0].legend()
    return save_fig(fig, "14_timeseries_representative_cities.png",
                    "Estimated AQI time series: representative cities plus highest and lowest mean AQI "
                    "(dashed line = AQI 200)")


def plot_dominant_heatmap(df, ctx):
    s = ctx["summary"]
    reg = s.drop_duplicates("City").set_index("City")["Region"]
    piv = s.pivot(index="City", columns="Season", values="Dominant_Pollutant")[SEASONS]
    order = sorted(piv.index, key=lambda c: (str(reg[c]), c))
    piv = piv.loc[order]
    code = piv.apply(lambda col: col.map({p: i for i, p in enumerate(POLLUTANTS)})).astype(float)
    cmap = ListedColormap([POLL_COLORS[p] for p in POLLUTANTS])
    cmap.set_bad("#DDDDDD")
    fig, ax = plt.subplots(figsize=(7.5, 0.22 * len(piv) + 3))
    ax.imshow(np.ma.masked_invalid(code.values), cmap=cmap, vmin=-0.5, vmax=len(POLLUTANTS) - 0.5, aspect="auto")
    ax.set_xticks(range(4), SEASONS)
    ax.set_yticks(range(len(piv)), [f"{c} ({reg[c]})" for c in piv.index], fontsize=6.5)
    ax.xaxis.tick_top()
    ax.legend(handles=[Patch(color=POLL_COLORS[p], label=p) for p in POLLUTANTS], loc="upper center",
              bbox_to_anchor=(0.5, -0.01), ncol=6, fontsize=8)
    p1 = save_fig(fig, "15_city_season_dominant_pollutant_heatmap.png",
                  "Most frequent dominant pollutant by city and season (404 cells)")

    v = df[df["Dominant_Pollutant"].notna()]
    sh = (pd.crosstab(v["Season"], v["Dominant_Pollutant"], normalize="index") * 100).reindex(SEASONS)[POLLUTANTS]
    fig, ax = plt.subplots(figsize=(9, 5))
    sh.plot(kind="bar", stacked=True, ax=ax, color=[POLL_COLORS[p] for p in POLLUTANTS], width=0.7)
    ax.set_ylabel("% of city-days")
    ax.tick_params(axis="x", rotation=0)
    p2 = save_fig(fig, "16_dominant_pollutant_share_by_season.png", "Share of city-days by dominant pollutant")
    return [p1, p2]


def plot_weather(df, ctx):
    fig, ax = plt.subplots(2, 2, figsize=(15, 11))

    def rho(x, y):
        d = df[[x, y]].dropna()
        return d[x].corr(d[y], method="spearman")

    a = ax[0, 0]
    for y, col in (("O3", "#55A868"), ("O3_8hr_max", "#2E7D32")):
        g = binned_stat(df["Temperature_C"], df[y])
        if g is not None:
            a.plot(g.xm, g.ym, marker="o", color=col, label=f"{y} (rho={rho('Temperature_C', y):.2f})")
            a.fill_between(g.xm, g.q1, g.q3, color=col, alpha=0.12)
    a.set_xlabel("Temperature (C)")
    a.set_ylabel("O3 (ug/m3)")
    a.set_title("Temperature vs O3 (binned median, IQR band)")
    a.legend()

    a = ax[0, 1]
    g = binned_stat(df["Humidity_pct"], df["PM2_5"])
    if g is not None:
        a.plot(g.xm, g.ym, marker="o", color=POLL_COLORS["PM2_5"])
        a.fill_between(g.xm, g.q1, g.q3, color=POLL_COLORS["PM2_5"], alpha=0.15)
    a.set_xlabel("Relative humidity (%)")
    a.set_ylabel("PM2.5 (ug/m3)")
    a.set_title(f"Humidity vs PM2.5 (Spearman rho={rho('Humidity_pct', 'PM2_5'):.2f})")

    a = ax[1, 0]
    for y in ("PM2_5", "PM10"):
        g = binned_stat(df["WindSpeed_kmh"], df[y])
        if g is not None:
            a.plot(g.xm, g.ym, marker="o", color=POLL_COLORS[y], label=f"{y} (rho={rho('WindSpeed_kmh', y):.2f})")
            a.fill_between(g.xm, g.q1, g.q3, color=POLL_COLORS[y], alpha=0.12)
    a.set_xlabel("Wind speed (km/h)")
    a.set_ylabel("ug/m3")
    a.set_title("Wind speed vs PM2.5 / PM10")
    a.legend()

    a = ax[1, 1]
    cls = pd.cut(df["Precipitation_mm"], [-0.001, 0, 1, 5, 20, np.inf], labels=["0", "0-1", "1-5", "5-20", ">20"])
    data = [df.loc[cls == k, "PM10"].dropna() for k in cls.cat.categories]
    a.boxplot(data, showfliers=False)
    a.set_xticks(range(1, 6), [f"{k}\n(n={len(d):,})" for k, d in zip(cls.cat.categories, data)])
    a.set_xlabel("Daily rainfall class (mm)")
    a.set_ylabel("PM10 (ug/m3)")
    a.set_title(f"Rainfall vs PM10 (Spearman rho={rho('Precipitation_mm', 'PM10'):.2f})")
    return save_fig(fig, "17_weather_pollutant_relationships.png",
                    "Weather-pollutant relationships (patterns are associations; "
                    "they suggest, but do not establish, causal links)")


def plot_mean_vs_8hr(df, ctx):
    d = df[~df["AQI_Uses_Provisional_8hr"]].copy()
    fig, ax = plt.subplots(2, 2, figsize=(15, 11))
    if len(d) < 200:
        for a in ax.ravel():
            a.axis("off")
        ax[0, 0].text(0.5, 0.5, "Too few hourly-derived (non-provisional) rows.\n"
                      "Daily-mean fallback rows are excluded because they would be identical by construction.",
                      ha="center", va="center", transform=ax[0, 0].transAxes)
        return save_fig(fig, "18_daily_mean_vs_8hr_max.png", "Daily mean vs 8-hour maximum (insufficient hourly data)")
    s = d.sample(min(20000, len(d)), random_state=SEED)
    ax[0, 0].scatter(s["O3"], s["O3_8hr_max"], s=3, alpha=0.25, color=POLL_COLORS["O3"])
    lim = float(np.nanpercentile(s[["O3", "O3_8hr_max"]].values, 99.5))
    ax[0, 0].plot([0, lim], [0, lim], "k--", lw=0.8)
    ax[0, 0].set_xlim(0, lim)
    ax[0, 0].set_ylim(0, lim)
    ax[0, 0].set_xlabel("Dataset daily-mean O3 (ug/m3)")
    ax[0, 0].set_ylabel("O3 8-h max (Open-Meteo, ug/m3)")
    ax[0, 0].set_title("O3: daily mean vs 8-h max")
    co_mean_mg = s["CO"] / 1000.0
    ax[0, 1].scatter(co_mean_mg, s["CO_8hr_max_mg_m3"], s=3, alpha=0.25, color=POLL_COLORS["CO"])
    lim = float(np.nanpercentile(np.r_[co_mean_mg.dropna(), s["CO_8hr_max_mg_m3"].dropna()], 99.5))
    ax[0, 1].plot([0, lim], [0, lim], "k--", lw=0.8)
    ax[0, 1].set_xlim(0, lim)
    ax[0, 1].set_ylim(0, lim)
    ax[0, 1].set_xlabel("Dataset daily-mean CO (mg/m3)")
    ax[0, 1].set_ylabel("CO 8-h max (Open-Meteo, mg/m3)")
    ax[0, 1].set_title("CO: daily mean vs 8-h max")
    for a, (num, den, ttl) in zip(ax[1], [("O3_8hr_max", "O3", "O3 ratio: 8-h max / daily mean"),
                                           ("CO_8hr_max_mg_m3", "CO_mgm3", "CO ratio: 8-h max / daily mean")]):
        dd = d.copy()
        dd["CO_mgm3"] = dd["CO"] / 1000.0
        ratio = (dd[num] / dd[den].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
        data = [ratio[dd["Season"] == s_].dropna().clip(upper=ratio.quantile(0.995)) for s_ in SEASONS]
        a.boxplot(data, showfliers=False)
        a.set_xticks(range(1, 5), SEASONS)
        a.axhline(1, ls="--", lw=0.8, color="k")
        a.set_title(ttl)
    return save_fig(fig, "18_daily_mean_vs_8hr_max.png",
                    "Daily mean vs 8-hour maximum (differences reflect the averaging window and the data source; "
                    "dataset means and Open-Meteo/CAMS estimates are different products)")


def plot_aqi_inputs_overview(df, ctx):
    """Which pollutant sets the AQI, and how often is the severe cap reached."""
    v = df[df["Dominant_Pollutant"].notna()]
    cnt = v["Dominant_Pollutant"].value_counts().reindex(POLLUTANTS)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ax[0].bar(cnt.index, 100 * cnt.values / cnt.sum(), color=[POLL_COLORS[p] for p in cnt.index])
    ax[0].set_ylabel("% of valid city-days")
    ax[0].set_title("Dominant pollutant (all city-days)")
    ax[1].hist(v["Estimated_AQI"], bins=50, color="#4C72B0")
    ax[1].set_title("Distribution of estimated AQI")
    ax[1].set_xlabel("Estimated AQI")
    return save_fig(fig, "19_dominant_pollutant_and_aqi_histogram.png", "Estimated AQI overview")


# ----------------------------------------------------------------------------
# 5. Main pipeline
# ----------------------------------------------------------------------------
def main():
    banner("PART 2 - Estimated AQI (CPCB sub-index methodology), EDA, City x Season table")
    print(AQI_LABEL)
    print(AQI_CAVEAT)

    # -- load Part 1 ----------------------------------------------------------
    master, quality, p1_manifest = load_part1()
    p1_hash = sha256_of(P1_MASTER)
    print(f"Loaded Part 1 master: {master.shape} | cities: {master['City'].nunique()} | "
          f"{master['Date'].min().date()} to {master['Date'].max().date()}")
    n_prov = int(master["Provisional_8hr_any"].sum())
    print(f"Rows using provisional daily-mean fallback for O3/CO (from Part 1): {n_prov:,}")
    if n_prov == len(master):
        print("  ! No hourly-derived O3/CO values are present. AQI for O3 and CO is therefore provisional "
              "(daily means), which likely understates 8-hour peaks. Re-run Part 1 after the API data "
              "is retrieved, then re-run Part 2.")

    # -- AQI ------------------------------------------------------------------
    banner("Sub-indices, estimated AQI, dominant pollutant")
    aqi_df, aqi_info = compute_aqi(master)
    print(f"Valid AQI days: {aqi_info['n_aqi_valid']:,} | missing: {aqi_info['n_aqi_missing']:,} | "
          f"dominance ties (first pollutant in order kept): {aqi_info['n_dominance_ties']:,}")
    n_cap = int(aqi_df["Severe_Capped_Flag"].sum())
    print(f"Severe_Capped_Flag rows (AQI set to 500): {n_cap:,}")
    print("AQI category counts:\n" + aqi_df["AQI_Category"].value_counts().reindex(CATEGORIES).to_string())
    print("Dominant pollutant counts:\n" + aqi_df["Dominant_Pollutant"].value_counts().to_string())

    # -- tables ---------------------------------------------------------------
    banner("City x Season summary (primary table)")
    summary = build_city_season_summary(aqi_df)
    ranking = build_city_ranking(aqi_df, summary)
    n_expected = master["City"].nunique() * 4
    print(f"Summary rows: {len(summary)} (expected {n_expected})")
    empty_cells = int(summary["Dominant_Pollutant"].isna().sum())
    if len(summary) != n_expected:
        print("  ! Unexpected number of City x Season rows.")
    if empty_cells:
        print(f"  ! {empty_cells} City x Season cells have no valid AQI days.")
    assert list(summary.columns[:len(REQUIRED_SUMMARY_COLS)]) == REQUIRED_SUMMARY_COLS

    # -- save data files ------------------------------------------------------
    banner("Saving data outputs")
    v1_cols = list(master.columns) + [SUBIDX[p] for p in POLLUTANTS] + \
              ["N_SubIndices", "Estimated_AQI", "AQI_Category", "Dominant_Pollutant", "Severe_Capped_Flag"] + \
              [f"{p}_SevereCapped" for p in POLLUTANTS] + ["AQI_Uses_Provisional_8hr", "AQI_Type"]
    v1_cols = list(dict.fromkeys(v1_cols))
    aqi_df[v1_cols].to_csv(AQI_V1_CSV, index=False, date_format="%Y-%m-%d")

    daily_cols = (["Date", "City", "Season"] + DAILY_POLL_COLS + WEATHER_COLS +
                  [SUBIDX[p] for p in ["PM2_5", "PM10", "NO2", "SO2", "O3", "CO"]] +
                  ["Estimated_AQI", "AQI_Category", "Dominant_Pollutant"])
    extra_daily = ["Severe_Capped_Flag", "AQI_Uses_Provisional_8hr", "O3_8hr_max", "CO_8hr_max_mg_m3", "N_SubIndices"]
    aqi_df[daily_cols + extra_daily].to_csv(DAILY_CSV, index=False, date_format="%Y-%m-%d")
    summary.to_csv(SUMMARY_CSV, index=False)
    ranking.to_csv(RANKING_CSV, index=False)
    breakpoints_table().to_csv(BREAKPOINT_CSV, index=False)
    with open(METHOD_NOTE_TXT, "w") as f:
        f.write(f"{AQI_LABEL}\n{AQI_CAVEAT}\n\n"
                "Method: CPCB-style linear sub-index interpolation per pollutant; Estimated_AQI = maximum sub-index "
                "(integer, round half up); Dominant_Pollutant = pollutant with the highest sub-index (ties go to the "
                "first of PM2_5, PM10, NO2, SO2, O3, CO). PM2.5, PM10, NO2, SO2 use 24-h (daily) means; O3 uses "
                "O3_8hr_max (ug/m3); CO uses CO_8hr_max_mg_m3. An AQI is reported only when at least "
                f"{MIN_SUBINDICES} sub-indices are available and one of them is PM2.5 or PM10.\n"
                "Severe band: the CPCB table is open-ended there; the upper end used for interpolation and for the "
                f"500 cap is an assumption: {SEVERE_TOP_ASSUMED}.\n"
                "Extreme observations flagged in Part 1 are retained. Rows using daily-mean fallback for O3/CO are "
                "flagged in AQI_Uses_Provisional_8hr.\n")
    for p in (AQI_V1_CSV, DAILY_CSV, SUMMARY_CSV, RANKING_CSV, BREAKPOINT_CSV):
        print(f"  saved {p}")

    # -- EDA ------------------------------------------------------------------
    banner("EDA figures")
    ctx = {"summary": summary, "ranking": ranking}
    for fn in (plot_distributions, plot_correlations, plot_monthly_seasonal, plot_city_comparison,
               plot_city_ranking, plot_category_distribution, plot_maps, plot_city_timeseries,
               plot_dominant_heatmap, plot_weather, plot_mean_vs_8hr, plot_aqi_inputs_overview):
        run_plot(fn, aqi_df, ctx)

    # -- sanity checks ----------------------------------------------------------
    assert len(summary) == n_expected, "City x Season row count mismatch"
    assert aqi_df["Estimated_AQI"].dropna().between(0, 500).all(), "AQI outside 0-500"
    assert len(aqi_df) == len(master), "Row count changed"
    assert sha256_of(P1_MASTER) == p1_hash, "Part 1 master file was modified!"

    # -- manifest ---------------------------------------------------------------
    data_files = [AQI_V1_CSV, DAILY_CSV, SUMMARY_CSV, RANKING_CSV, BREAKPOINT_CSV, METHOD_NOTE_TXT]
    fig_files = [p for p in FIG_LOG if not p.startswith("FAILED:")]
    failed = [p for p in FIG_LOG if p.startswith("FAILED:")]
    manifest = {
        "part": 2, "total_parts": 7, "run_id": RUN_ID, "seed": SEED,
        "label": AQI_LABEL, "caveats": AQI_CAVEAT,
        "base_dir": BASE_DIR,
        "inputs_from_part1": {"master": P1_MASTER, "master_sha256": p1_hash, "quality_report": P1_QUALITY,
                              "part1_manifest": P1_MANIFEST if p1_manifest else None,
                              "part1_run_id": p1_manifest.get("run_id"),
                              "part1_quality_highlights": part1_highlights(quality),
                              "rows_with_provisional_8hr": n_prov},
        "aqi_method": {
            "inputs": AQI_INPUT, "min_subindices": MIN_SUBINDICES,
            "rounding": "Estimated_AQI rounded half-up to integer",
            "dominance_tie_order": POLLUTANTS,
            "severe_band_top_assumed": SEVERE_TOP_ASSUMED,
            "categories": {"Good": "0-50", "Satisfactory": "51-100", "Moderate": "101-200",
                           "Poor": "201-300", "Very Poor": "301-400", "Severe": "401-500"}},
        "results": {**aqi_info, "severe_capped_rows": n_cap,
                    "city_season_rows": int(len(summary)),
                    "city_season_cells_without_valid_aqi": empty_cells},
        "key_columns": {
            "aqi_estimated_v1": list(aqi_df[v1_cols].columns),
            "daily_aqi_dominant_pollutant": daily_cols + extra_daily,
            "city_season_pollutant_summary": list(summary.columns)},
        "no_split_no_scaling_in_part2": True,
        "figures_failed": failed,
        "files_created": [{"path": p, "rel_path": os.path.relpath(p, BASE_DIR),
                           "size_bytes": os.path.getsize(p)} for p in data_files + fig_files] +
                         [{"path": MANIFEST_JSON, "rel_path": os.path.relpath(MANIFEST_JSON, BASE_DIR), "size_bytes": None}],
        "next_part_should_load": {"daily": DAILY_CSV, "full_daily_with_flags": AQI_V1_CSV,
                                  "city_season_summary": SUMMARY_CSV},
    }
    with open(MANIFEST_JSON, "w") as f:
        json.dump(manifest, f, indent=2, default=str)

    # -- final printout ---------------------------------------------------------
    banner("PART 2 COMPLETE - files created (exact paths for Part 3)")
    for p in data_files + fig_files + [MANIFEST_JSON]:
        print(p)
    if failed:
        print("\nFigures that failed (pipeline continued):", failed)
    print(f"\nBASE_DIR: {BASE_DIR}")
    print("\n" + AQI_LABEL)
    print(AQI_CAVEAT)
    return aqi_df, summary


if __name__ == "__main__" or IN_COLAB:
    aqi_df, city_season_summary = main()


PART 2 - Estimated AQI (CPCB sub-index methodology), EDA, City x Season table
Six-pollutant estimated AQI using CPCB sub-index methodology.
NH3 and Pb are unavailable, so this is not the official CPCB AQI. O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct CPCB ground readings.
Loaded Part 1 master: (146147, 31) | cities: 101 | 2022-08-04 to 2026-07-20
Rows using provisional daily-mean fallback for O3/CO (from Part 1): 101

Sub-indices, estimated AQI, dominant pollutant
Valid AQI days: 146,147 | missing: 0 | dominance ties (first pollutant in order kept): 17
Severe_Capped_Flag rows (AQI set to 500): 2,385
AQI category counts:
AQI_Category
Good             2305
Satisfactory    30140
Moderate        85125
Poor            19229
Very Poor        6431
Severe           2917
Dominant pollutant counts:
Dominant_Pollutant
O3       124112
PM2_5     13497
PM10       6926
SO2        1273
CO          237
NO2         102

City x Season summary (primary table)
Summary rows: 404 (exp

/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning:

CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.



  [ok] plot_maps
  [ok] plot_city_timeseries
  [ok] plot_dominant_heatmap
  [ok] plot_weather
  [ok] plot_mean_vs_8hr
  [ok] plot_aqi_inputs_overview

PART 2 COMPLETE - files created (exact paths for Part 3)
/content/aqi_framework/data/processed/aqi_estimated_v1.csv
/content/aqi_framework/results/daily_aqi_dominant_pollutant.csv
/content/aqi_framework/results/city_season_pollutant_summary.csv
/content/aqi_framework/results/city_aqi_ranking.csv
/content/aqi_framework/results/cpcb_breakpoints_used.csv
/content/aqi_framework/results/AQI_METHOD_NOTE.txt
/content/aqi_framework/figures/eda_aqi/01_pollutant_distributions.png
/content/aqi_framework/figures/eda_aqi/02_pollutant_boxplots.png
/content/aqi_framework/figures/eda_aqi/03_correlation_heatmaps_overall.png
/content/aqi_framework/figures/eda_aqi/04_correlation_heatmaps_by_season.png
/content/aqi_framework/figures/eda_aqi/05_monthly_climatology.png
/content/aqi_framework/figures/eda_aqi/06_monthly_timeseries_national_mean.png
/content/aqi

In [4]:
# =============================================================================
# City- and Season-Aware Multi-Pollutant Air Quality Assessment and Forecasting
# Framework for Indian Cities Using Deep Learning
#
# PART 3 of 7: weather-based cause explanations and city-specific policy
#              recommendations (rule-based), applied to every daily row and
#              every City x Season summary row.
#
# Continues directly from Part 2 (same BASE_DIR; reads Part 2's manifest,
# daily_aqi_dominant_pollutant.csv and city_season_pollutant_summary.csv).
# Part 2 files are only read, never modified. No modelling, splitting or
# scaling is done here.
#
# LABEL : "Six-pollutant estimated AQI using CPCB sub-index methodology."
# CAVEAT: NH3 and Pb are unavailable, so this is NOT the official CPCB AQI.
#         O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct
#         CPCB ground readings.
# CAUTION: Cause statements are rule-based associations between simultaneous
#         weather and concentration patterns. They suggest possible
#         contributors; they do NOT establish source attribution.
# =============================================================================

# ----------------------------------------------------------------------------
# 0. Imports, configuration, reproducibility, link to Part 2
# ----------------------------------------------------------------------------
import os
import sys
import json
import random
import hashlib
import textwrap
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, FancyBboxPatch

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.RandomState(SEED)

# ---- same project root as Parts 1-2 ------------------------------------------
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
BASE_DIR = "/content/aqi_framework" if IN_COLAB else os.path.abspath("./aqi_framework")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

RES_DIR = os.path.join(BASE_DIR, "results")
FIG_DIR = os.path.join(BASE_DIR, "figures")
CP_DIR = os.path.join(FIG_DIR, "cause_policy")
for d in (RES_DIR, FIG_DIR, CP_DIR):
    os.makedirs(d, exist_ok=True)

# ---- Part 2 inputs -------------------------------------------------------------
P2_DAILY = os.path.join(RES_DIR, "daily_aqi_dominant_pollutant.csv")
P2_SUMMARY = os.path.join(RES_DIR, "city_season_pollutant_summary.csv")
P2_MANIFEST = os.path.join(RES_DIR, "part2_handoff_manifest.json")

# ---- Part 3 outputs (as specified + supporting files) --------------------------------
DAILY_OUT = os.path.join(RES_DIR, "daily_aqi_dominant_pollutant_with_causes.csv")
SUMMARY_OUT = os.path.join(RES_DIR, "city_season_dominant_pollutant_summary.csv")
DOM_TABLE_CSV = os.path.join(RES_DIR, "dominant_pollutant_city_season_table.csv")
THRESH_CSV = os.path.join(RES_DIR, "cause_policy_thresholds_used.csv")
RULES_CSV = os.path.join(RES_DIR, "cause_policy_rules.csv")
CARDS_CSV = os.path.join(RES_DIR, "explanation_cards_10_cities.csv")
RULE_FREQ_CSV = os.path.join(RES_DIR, "cause_policy_rule_frequencies.csv")
METHOD_NOTE_TXT = os.path.join(RES_DIR, "CAUSE_POLICY_METHOD_NOTE.txt")
MANIFEST_JSON = os.path.join(RES_DIR, "part3_handoff_manifest.json")

# ---- labels --------------------------------------------------------------------
AQI_LABEL = "Six-pollutant estimated AQI using CPCB sub-index methodology."
AQI_CAVEAT = ("NH3 and Pb are unavailable, so this is not the official CPCB AQI. "
              "O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct CPCB ground readings.")
CAUSE_CAVEAT = ("Likely_Cause text is a rule-based, suggestive interpretation of simultaneous weather and "
                "concentration patterns; it does not establish source attribution.")
FOOTER = ("Six-pollutant estimated AQI (CPCB sub-index methodology); not official CPCB AQI (NH3, Pb unavailable). "
          "Causes are suggestive, rule-based associations, not source attribution.")

# ---- schema ----------------------------------------------------------------------
POLLUTANTS = ["PM2_5", "PM10", "NO2", "SO2", "O3", "CO"]
SEASONS = ["Winter", "Summer", "Monsoon", "Post-monsoon"]
CATEGORIES = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
POOR_CATEGORIES = {"Poor", "Very Poor", "Severe"}
POLL_COLORS = {"PM2_5": "#C44E52", "PM10": "#DD8452", "NO2": "#4C72B0",
               "SO2": "#937860", "O3": "#55A868", "CO": "#8172B2"}
SEASON_COLORS = {"Winter": "#4C72B0", "Summer": "#DD8452", "Monsoon": "#55A868", "Post-monsoon": "#8172B2"}

# ---- thresholds ---------------------------------------------------------------------
# "High" pollutant values are dataset-relative: >= this percentile of the study-period national city-day
# distribution (computed from the loaded Part 2 file; saved to cause_policy_thresholds_used.csv).
HIGH_POLLUTANT_PERCENTILE = 75
# Weather thresholds are fixed, documented values. Two profiles: individual days, and City x Season means
# (means smooth out extremes, so the 'high rainfall' cut-off is lower for the means).
WEATHER_THRESHOLDS = {
    "daily":       {"low_wind_kmh": 6.0, "low_rain_mm": 1.0, "high_rain_mm": 10.0,
                    "low_humidity_pct": 40.0, "high_humidity_pct": 80.0, "high_temp_c": 30.0},
    "season_mean": {"low_wind_kmh": 6.0, "low_rain_mm": 1.0, "high_rain_mm": 5.0,
                    "low_humidity_pct": 40.0, "high_humidity_pct": 80.0, "high_temp_c": 30.0},
}
PM_COARSE_RATIO = 2.0                 # PM10 / PM2.5 >= 2 -> "PM10 much greater than PM2.5"
SUMMARY_FREQUENT_POOR_PCT = 20.0      # extension for City x Season rows: >= 20 % of days with AQI > 200
THRESH = {"pollutant": {}, "weather": WEATHER_THRESHOLDS, "pm_coarse_ratio": PM_COARSE_RATIO}

# ----------------------------------------------------------------------------
# 1. Rule text (cause explanations use the wording specified for the project)
# ----------------------------------------------------------------------------
CAUSE_RULES = {
    "C1": ("High PM2.5/PM10 + low wind + low rainfall",
           "Low wind speed and low rainfall favour accumulation of particulate pollution."),
    "C2": ("High PM2.5/PM10 + Winter",
           "Winter conditions may worsen particulate accumulation due to reduced atmospheric ventilation."),
    "C3": ("High PM10 + low humidity",
           "Dry conditions and dust resuspension may contribute to elevated PM10."),
    "C4": ("High PM10 with PM10 >> PM2.5",
           "The pattern suggests a dust-dominated contribution."),
    "C5": ("High NO2 or CO",
           "Elevated NO2 and CO patterns are commonly associated with traffic and combustion-related emissions."),
    "C6": ("High SO2",
           "Elevated SO2 may indicate industrial or fossil-fuel combustion influence."),
    "C7": ("High O3 + high temperature",
           "Warm and sunny conditions favour photochemical ozone formation."),
    "C8": ("High rainfall",
           "Rainfall may reduce pollutant concentrations through wet deposition."),
    "C9": ("High humidity + high PM2.5",
           "High humidity may support secondary particle formation and pollutant retention."),
}
CAUSE_NONE = ("No strong weather- or emission-linked pattern is flagged by the available values under the "
              "thresholds used; the pattern is inconclusive.")
CAUSE_NODATA = "Insufficient data to assess a likely cause."
# which dominant pollutants each cause rule is most relevant to (used only to order the sentences)
CAUSE_RELEVANT = {"C1": {"PM2_5", "PM10"}, "C2": {"PM2_5", "PM10"}, "C3": {"PM2_5", "PM10"},
                  "C4": {"PM2_5", "PM10"}, "C9": {"PM2_5", "PM10"}, "C5": {"NO2", "CO"},
                  "C6": {"SO2"}, "C7": {"O3"}, "C8": set()}

# policy rules: id -> (label, [action templates]) ; placeholders: {City} {Season} {AQI_Category} {H_detail} {regime}
ACTION_RULES = {
    "H": ("Health advisory (AQI Poor/Very Poor/Severe)", [
        "Issue a public-health advisory for {City} ({Season}): limit prolonged outdoor exertion; {H_detail}.",
        "Sensitive-group caution: children, older adults and people with respiratory or cardiac conditions should reduce outdoor exposure.",
        "Send school and hospital alerts in {City} so outdoor activities and patient-care plans can be adjusted."]),
    "PM": ("Particulate controls (PM2.5/PM10 dominant)", [
        "Intensify dust control (e.g., water sprinkling on exposed soil and unpaved surfaces).",
        "Increase road cleaning on busy roads, preferring methods that limit dust resuspension.",
        "Enforce construction-dust control measures (covering material, barriers, water suppression).",
        "Issue public advisories on particulate pollution in {City}."]),
    "TR": ("Traffic measures (NO2/CO dominant)", [
        "Apply traffic-management measures at congestion hotspots (e.g., signal coordination).",
        "Issue congestion alerts during peak hours.",
        "Promote public transport and non-motorised options."]),
    "SO": ("Industrial/combustion inspection flag (SO2 dominant)", [
        "Flag possible industrial or combustion influence in {City} for inspection and verification (a prompt to check, not a source attribution)."]),
    "OZ": ("Ozone measures (O3 dominant)", [
        "Advise reducing afternoon outdoor activity.",
        "Monitor ozone (O3) levels, especially in the afternoon."]),
    "ACC": ("Accumulation watch (low wind + low rainfall)", [
        "Low wind and low rainfall: monitor pollutant accumulation and review local emission controls."]),
    "RAIN": ("Routine monitoring (high rainfall)", [
        "High rainfall: continue routine monitoring; wet deposition may temporarily reduce pollutant concentrations."]),
    "WPM": ("Winter particulate priority", [
        "Winter: prioritize particulate-matter advisories and accumulation monitoring."]),
    "SOZ": ("Summer ozone priority", [
        "Summer: prioritize afternoon ozone advisories."]),
    "REG": ("Regime note (optional regime supplied)", [
        "Interpret these recommendations alongside the '{regime}' pollution regime."]),
    "ROUT": ("Routine monitoring (nothing else triggered)", [
        "No additional action is indicated by these rules; continue routine monitoring."]),
}
ACTION_ORDER = ["H", "PM", "TR", "SO", "OZ", "ACC", "RAIN", "WPM", "SOZ", "REG", "ROUT"]
ACTION_NODATA = "Insufficient data to recommend actions; continue routine monitoring."


# ----------------------------------------------------------------------------
# 2. Small helpers
# ----------------------------------------------------------------------------
def banner(text):
    print("\n" + "=" * 78 + f"\n{text}\n" + "=" * 78)


def sha256_of(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


def _num(x):
    try:
        v = float(x)
    except (TypeError, ValueError):
        return np.nan
    return v


def _ge(x, t):
    x, t = _num(x), _num(t)
    return bool(x >= t) if not (np.isnan(x) or np.isnan(t)) else False


def _lt(x, t):
    x, t = _num(x), _num(t)
    return bool(x < t) if not (np.isnan(x) or np.isnan(t)) else False


def _le(x, t):
    x, t = _num(x), _num(t)
    return bool(x <= t) if not (np.isnan(x) or np.isnan(t)) else False


def category_from_aqi(a):
    a = _num(a)
    if np.isnan(a):
        return np.nan
    for edge, name in zip([50, 100, 200, 300, 400], CATEGORIES[:5]):
        if a <= edge:
            return name
    return "Severe"


def derive_thresholds(daily):
    """Dataset-relative 'high' thresholds for pollutants (documented, reproducible)."""
    cols = {"PM2_5": "PM2_5", "PM10": "PM10", "NO2": "NO2", "SO2": "SO2", "O3": "O3", "CO": "CO",
            "O3_8hr_max": "O3_8hr_max", "CO_8hr_max_mg_m3": "CO_8hr_max_mg_m3"}
    out = {}
    for k, c in cols.items():
        s = daily[c].dropna() if c in daily.columns else pd.Series(dtype=float)
        out[k] = float(np.percentile(s, HIGH_POLLUTANT_PERCENTILE)) if len(s) else np.nan
    THRESH["pollutant"] = out
    return out


def thresholds_table():
    rows = []
    for k, v in THRESH["pollutant"].items():
        unit = "mg/m3" if "mg_m3" in k else ("ug/m3 (daily mean, CO in ug/m3)" if k == "CO" else "ug/m3")
        rows.append({"group": "pollutant_high", "name": k, "value": v, "unit": unit,
                     "definition": f">= P{HIGH_POLLUTANT_PERCENTILE} of study-period city-days (dataset-relative)"})
    for prof, d in WEATHER_THRESHOLDS.items():
        for k, v in d.items():
            rows.append({"group": f"weather_{prof}", "name": k, "value": v, "unit": k.split("_")[-1],
                         "definition": "fixed value; 'low_*' is strictly below, 'high_*' is at or above"})
    rows.append({"group": "pm", "name": "pm_coarse_ratio", "value": PM_COARSE_RATIO, "unit": "ratio",
                 "definition": "PM10/PM2.5 at or above this value -> 'PM10 much greater than PM2.5'"})
    rows.append({"group": "summary_extension", "name": "frequent_poor_pct", "value": SUMMARY_FREQUENT_POOR_PCT,
                 "unit": "% days", "definition": "City x Season rows: health advisory also triggered if >= this % of "
                                                  "days have AQI > 200 (extension to the AQI-category rule)"})
    return pd.DataFrame(rows)


def rules_table():
    rows = [{"type": "cause", "id": k, "condition_or_label": v[0], "text": v[1]} for k, v in CAUSE_RULES.items()]
    for k in ACTION_ORDER:
        lab, acts = ACTION_RULES[k]
        rows.append({"type": "action", "id": k, "condition_or_label": lab, "text": " | ".join(acts)})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# 3. Function 1: weather-based cause explanation
# ----------------------------------------------------------------------------
def explain_cause(City, Season, Dominant_Pollutant,
                  PM2_5, PM10, CO, NO2, SO2, O3,
                  Temperature_C, Humidity_pct, WindSpeed_kmh, Precipitation_mm,
                  O3_8hr_max=None, CO_8hr_max_mg_m3=None,
                  profile="daily", thresholds=None, return_ids=False):
    """Rule-based, suggestive explanation of a day (profile='daily') or a City x Season mean
    (profile='season_mean'). City is accepted for interface consistency; cause wording does not depend on it.
    O3/CO: the 8-hour metrics are used when supplied (matching the AQI inputs), otherwise the daily means.
    Returns the explanation string (and the list of triggered rule IDs if return_ids=True)."""
    T = thresholds or THRESH
    P, W = T["pollutant"], T["weather"][profile]
    dom = Dominant_Pollutant if isinstance(Dominant_Pollutant, str) else None

    o3_val, o3_thr = (O3_8hr_max, P.get("O3_8hr_max")) if not np.isnan(_num(O3_8hr_max)) else (O3, P.get("O3"))
    co_val, co_thr = ((CO_8hr_max_mg_m3, P.get("CO_8hr_max_mg_m3")) if not np.isnan(_num(CO_8hr_max_mg_m3))
                      else (CO, P.get("CO")))

    pm25_high = _ge(PM2_5, P.get("PM2_5"))
    pm10_high = _ge(PM10, P.get("PM10"))
    pm_high = pm25_high or pm10_high
    low_wind = _lt(WindSpeed_kmh, W["low_wind_kmh"])
    low_rain = _lt(Precipitation_mm, W["low_rain_mm"])
    high_rain = _ge(Precipitation_mm, W["high_rain_mm"])
    low_hum = _lt(Humidity_pct, W["low_humidity_pct"])
    high_hum = _ge(Humidity_pct, W["high_humidity_pct"])
    high_temp = _ge(Temperature_C, W["high_temp_c"])
    pm2, pm10v = _num(PM2_5), _num(PM10)
    coarse = (not np.isnan(pm2) and not np.isnan(pm10v) and pm2 > 0 and pm10v / pm2 >= T["pm_coarse_ratio"])

    ids = []
    if pm_high and low_wind and low_rain:
        ids.append("C1")
    if pm_high and Season == "Winter":
        ids.append("C2")
    if pm10_high and low_hum:
        ids.append("C3")
    if pm10_high and coarse:
        ids.append("C4")
    if _ge(NO2, P.get("NO2")) or _ge(co_val, co_thr):
        ids.append("C5")
    if _ge(SO2, P.get("SO2")):
        ids.append("C6")
    if _ge(o3_val, o3_thr) and high_temp:
        ids.append("C7")
    if high_rain:
        ids.append("C8")
    if high_hum and pm25_high:
        ids.append("C9")

    # order: rules relevant to the dominant pollutant first, then the rest (stable)
    ids = sorted(ids, key=lambda i: (0 if (dom in CAUSE_RELEVANT[i]) else 1, i))
    if dom is None and not ids:
        text = CAUSE_NODATA
    elif not ids:
        text = CAUSE_NONE
    else:
        text = " ".join(CAUSE_RULES[i][1] for i in ids)
    return (text, ids) if return_ids else text


# ----------------------------------------------------------------------------
# 4. Function 2: city-specific policy recommendation
# ----------------------------------------------------------------------------
def recommend_actions(City, Season, AQI_Category, Dominant_Pollutant,
                      Temperature_C, Humidity_pct, WindSpeed_kmh, Precipitation_mm,
                      regime=None, pct_days_above200=None,
                      profile="daily", thresholds=None, return_ids=False):
    """Rule-based, city-specific recommendations; duplicates removed (case-insensitive).
    Temperature and humidity are accepted (and reserved for later refinements); the current rules use
    AQI category, dominant pollutant, season, wind and rainfall. 'regime' (optional) only adds an
    interpretation note. pct_days_above200 (optional, City x Season rows) can also trigger the health rule."""
    T = thresholds or THRESH
    W = T["weather"][profile]
    dom = Dominant_Pollutant if isinstance(Dominant_Pollutant, str) else None
    cat = AQI_Category if isinstance(AQI_Category, str) else None
    if dom is None and cat is None:
        return (ACTION_NODATA, []) if return_ids else ACTION_NODATA

    triggered = []
    frequent_poor = (pct_days_above200 is not None and _ge(pct_days_above200, SUMMARY_FREQUENT_POOR_PCT))
    if cat in POOR_CATEGORIES or frequent_poor:
        triggered.append("H")
    if dom in ("PM2_5", "PM10"):
        triggered.append("PM")
    if dom in ("NO2", "CO"):
        triggered.append("TR")
    if dom == "SO2":
        triggered.append("SO")
    if dom == "O3":
        triggered.append("OZ")
    if _lt(WindSpeed_kmh, W["low_wind_kmh"]) and _lt(Precipitation_mm, W["low_rain_mm"]):
        triggered.append("ACC")
    if _ge(Precipitation_mm, W["high_rain_mm"]):
        triggered.append("RAIN")
    if Season == "Winter" and dom in ("PM2_5", "PM10"):
        triggered.append("WPM")
    if Season == "Summer" and dom == "O3":
        triggered.append("SOZ")
    if isinstance(regime, str) and regime.strip():
        triggered.append("REG")
    if not triggered:
        triggered.append("ROUT")

    seen, actions, used_ids = set(), [], []
    if cat in POOR_CATEGORIES:
        h_detail = f"the estimated AQI is in the {cat} range"
    elif frequent_poor:
        h_detail = f"the estimated AQI exceeds 200 on about {_num(pct_days_above200):.0f}% of days in this season"
    else:
        h_detail = "the estimated AQI is elevated"
    fmt = {"City": City, "Season": Season, "AQI_Category": cat or "unclassified", "H_detail": h_detail,
           "regime": regime if isinstance(regime, str) else ""}
    for rid in [r for r in ACTION_ORDER if r in triggered]:
        added = False
        for tpl in ACTION_RULES[rid][1]:
            s = tpl.format(**fmt)
            key = " ".join(s.lower().split())
            if key not in seen:
                seen.add(key)
                actions.append(s)
                added = True
        if added:
            used_ids.append(rid)
    text = " | ".join(actions)
    return (text, used_ids) if return_ids else text


# ----------------------------------------------------------------------------
# 5. Load Part 2 (read-only) and verify continuity
# ----------------------------------------------------------------------------
def load_part2():
    for p in (P2_DAILY, P2_SUMMARY):
        if not os.path.isfile(p):
            raise FileNotFoundError(
                f"Missing Part 2 output: {p}\nRun Part 2 first (same BASE_DIR={BASE_DIR}). "
                "In Colab, a restarted runtime loses /content - re-run Parts 1-2 or restore the folder.")
    manifest = {}
    if os.path.isfile(P2_MANIFEST):
        with open(P2_MANIFEST) as f:
            manifest = json.load(f)
    daily = pd.read_csv(P2_DAILY, parse_dates=["Date"])
    summary = pd.read_csv(P2_SUMMARY)

    need_d = (["Date", "City", "Season", "PM2_5", "PM10", "CO", "NO2", "SO2", "O3", "Temperature_C", "Humidity_pct",
               "WindSpeed_kmh", "Precipitation_mm", "Estimated_AQI", "AQI_Category", "Dominant_Pollutant"])
    need_s = ["City", "Season", "Dominant_Pollutant", "Percent_Days_Dominant", "Mean_PM2_5", "Mean_PM10", "Mean_CO",
              "Mean_NO2", "Mean_SO2", "Mean_O3", "Mean_Temperature", "Mean_Humidity", "Mean_WindSpeed",
              "Mean_Rainfall", "Mean_AQI", "Percent_Days_AQI_Above200"]
    for name, df, need in (("daily", daily, need_d), ("summary", summary, need_s)):
        miss = [c for c in need if c not in df.columns]
        if miss:
            raise ValueError(f"Part 2 {name} file is missing columns: {miss}")
    if manifest:
        res = manifest.get("results", {})
        exp_rows = res.get("n_aqi_valid", 0) + res.get("n_aqi_missing", 0)
        if exp_rows and exp_rows != len(daily):
            raise ValueError(f"Daily rows {len(daily)} differ from Part 2 manifest ({exp_rows}).")
        if res.get("city_season_rows") not in (None, len(summary)):
            raise ValueError("City x Season rows differ from Part 2 manifest.")
    if len(summary) != daily["City"].nunique() * 4:
        print(f"  ! Summary has {len(summary)} rows; expected {daily['City'].nunique() * 4}.")
    daily["AQI_Category"] = daily["AQI_Category"].where(daily["AQI_Category"].isin(CATEGORIES))
    return daily, summary, manifest


# ----------------------------------------------------------------------------
# 6. Apply both functions
# ----------------------------------------------------------------------------
def _col(df, name):
    return df[name].tolist() if name in df.columns else [None] * len(df)


def apply_to_daily(daily):
    d = daily
    n = len(d)
    causes, cause_ids, acts, act_ids = [None] * n, [None] * n, [None] * n, [None] * n
    rows = zip(d["City"], d["Season"], d["Dominant_Pollutant"], d["AQI_Category"],
               d["PM2_5"], d["PM10"], d["CO"], d["NO2"], d["SO2"], d["O3"],
               d["Temperature_C"], d["Humidity_pct"], d["WindSpeed_kmh"], d["Precipitation_mm"],
               _col(d, "O3_8hr_max"), _col(d, "CO_8hr_max_mg_m3"))
    for i, (city, season, dom, cat, pm25, pm10, co, no2, so2, o3, t, h, w, p, o3m, com) in enumerate(rows):
        c, cid = explain_cause(city, season, dom, pm25, pm10, co, no2, so2, o3, t, h, w, p,
                               O3_8hr_max=o3m, CO_8hr_max_mg_m3=com, profile="daily", return_ids=True)
        a, aid = recommend_actions(city, season, cat, dom, t, h, w, p, profile="daily", return_ids=True)
        causes[i], cause_ids[i], acts[i], act_ids[i] = c, "|".join(cid), a, "|".join(aid)
    out = d.copy()
    out["Likely_Cause"] = causes
    out["Recommended_Actions"] = acts
    out["Cause_Rule_IDs"] = cause_ids
    out["Action_Rule_IDs"] = act_ids
    return out


def apply_to_summary(summary):
    s = summary.copy()
    s["Mean_AQI_Category"] = [category_from_aqi(a) for a in s["Mean_AQI"]]
    causes, cids, acts, aids = [], [], [], []
    for r in s.itertuples(index=False):
        rd = r._asdict()
        c, cid = explain_cause(rd["City"], rd["Season"], rd["Dominant_Pollutant"],
                               rd["Mean_PM2_5"], rd["Mean_PM10"], rd["Mean_CO"], rd["Mean_NO2"], rd["Mean_SO2"],
                               rd["Mean_O3"], rd["Mean_Temperature"], rd["Mean_Humidity"], rd["Mean_WindSpeed"],
                               rd["Mean_Rainfall"], O3_8hr_max=rd.get("Mean_O3_8hr_max"),
                               CO_8hr_max_mg_m3=rd.get("Mean_CO_8hr_max_mg_m3"),
                               profile="season_mean", return_ids=True)
        a, aid = recommend_actions(rd["City"], rd["Season"], rd["Mean_AQI_Category"], rd["Dominant_Pollutant"],
                                   rd["Mean_Temperature"], rd["Mean_Humidity"], rd["Mean_WindSpeed"],
                                   rd["Mean_Rainfall"], pct_days_above200=rd["Percent_Days_AQI_Above200"],
                                   profile="season_mean", return_ids=True)
        causes.append(c)
        cids.append("|".join(cid))
        acts.append(a)
        aids.append("|".join(aid))
    s["Likely_Cause"] = causes
    s["Recommended_Actions"] = acts
    s["Cause_Rule_IDs"] = cids
    s["Action_Rule_IDs"] = aids
    return s


def dominant_table(summary):
    piv = summary.pivot(index="City", columns="Season", values="Dominant_Pollutant")[SEASONS]
    pct = summary.pivot(index="City", columns="Season", values="Percent_Days_Dominant")[SEASONS]
    pct.columns = [f"{c}_Pct_Days_Dominant" for c in pct.columns]
    piv.columns = [f"{c}_Dominant" for c in piv.columns]
    meta = summary.drop_duplicates("City").set_index("City")[[c for c in ["Region", "Latitude", "Longitude"] if c in summary]]
    mean_aqi = summary.groupby("City")["Mean_AQI"].mean().rename("Mean_AQI_All_Seasons")
    return meta.join(piv).join(pct).join(mean_aqi).sort_values("Mean_AQI_All_Seasons", ascending=False).reset_index()


def rule_frequency_table(daily_out, summary_out):
    rows = []
    for level, df in (("daily", daily_out), ("city_season", summary_out)):
        n = len(df)
        for kind, col, labels in (("cause", "Cause_Rule_IDs", {k: v[0] for k, v in CAUSE_RULES.items()}),
                                  ("action", "Action_Rule_IDs", {k: v[0] for k, v in ACTION_RULES.items()})):
            ids = df[col].fillna("").str.split("|").explode()
            ids = ids[ids != ""]
            cnt = ids.value_counts()
            for rid, lab in labels.items():
                rows.append({"level": level, "type": kind, "rule_id": rid, "label": lab,
                             "n_rows": int(cnt.get(rid, 0)), "pct_rows": 100 * cnt.get(rid, 0) / n})
            none = int((df[col].fillna("") == "").sum())
            rows.append({"level": level, "type": kind, "rule_id": "none", "label": "No rule triggered / no data",
                         "n_rows": none, "pct_rows": 100 * none / n})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# 7. Figures
# ----------------------------------------------------------------------------
FIG_LOG = []


def save_fig(fig, name, title=None, dpi=130):
    if title:
        fig.suptitle(title, fontsize=13, y=0.998)
    fig.text(0.5, 0.002, FOOTER, ha="center", va="bottom", fontsize=7, color="dimgray")
    path = os.path.join(CP_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return path


def run_plot(fn, ctx):
    try:
        paths = fn(ctx)
        paths = [paths] if isinstance(paths, str) else (paths or [])
        FIG_LOG.extend(paths)
        print(f"  [ok] {fn.__name__}")
    except Exception as ex:
        print(f"  [FAILED] {fn.__name__}: {type(ex).__name__}: {ex}")
        FIG_LOG.append(f"FAILED:{fn.__name__}:{type(ex).__name__}")


def plot_dominant_by_season(ctx):
    s = ctx["summary"]
    out = []
    # (a) counts of cities per dominant pollutant, per season, stacked by region
    regs = sorted(s["Region"].dropna().unique()) if "Region" in s else []
    fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
    cnt = (s.groupby(["Season", "Dominant_Pollutant"]).size().unstack(fill_value=0)
           .reindex(SEASONS).reindex(columns=POLLUTANTS, fill_value=0))
    cnt.plot(kind="bar", stacked=True, ax=ax[0], color=[POLL_COLORS[p] for p in POLLUTANTS], width=0.7)
    ax[0].set_ylabel("Number of cities")
    ax[0].set_title("Most frequent dominant pollutant: cities per season")
    ax[0].tick_params(axis="x", rotation=0)
    mean_share = s.groupby("Season")[[f"Pct_Days_Dominant_{p}" for p in POLLUTANTS]].mean().reindex(SEASONS)
    mean_share.columns = POLLUTANTS
    mean_share.plot(kind="bar", stacked=True, ax=ax[1], color=[POLL_COLORS[p] for p in POLLUTANTS], width=0.7)
    ax[1].set_ylabel("Mean % of days dominant (average across cities)")
    ax[1].set_title("Share of days each pollutant is dominant")
    ax[1].tick_params(axis="x", rotation=0)
    out.append(save_fig(fig, "20_dominant_pollutant_by_season_bars.png", "Dominant pollutants by season"))

    # (b) region x season small multiples
    if regs:
        fig, ax = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
        for a, season in zip(ax, SEASONS):
            g = (s[s["Season"] == season].groupby(["Region", "Dominant_Pollutant"]).size()
                 .unstack(fill_value=0).reindex(regs).reindex(columns=POLLUTANTS, fill_value=0))
            g.plot(kind="barh", stacked=True, ax=a, color=[POLL_COLORS[p] for p in POLLUTANTS], legend=False, width=0.75)
            a.set_title(season)
            a.set_xlabel("Cities")
        ax[0].set_ylabel("Region")
        ax[-1].legend(handles=[Patch(color=POLL_COLORS[p], label=p) for p in POLLUTANTS], loc="lower right")
        out.append(save_fig(fig, "21_dominant_pollutant_by_region_season.png",
                            "Cities by dominant pollutant: region x season"))
    return out


def plot_dominant_table(ctx):
    t = ctx["dom_table"].head(25)
    fig, ax = plt.subplots(figsize=(12, 0.42 * len(t) + 2))
    ax.axis("off")
    cell_text, cell_colors = [], []
    for _, r in t.iterrows():
        row, cols = [r["City"]], ["white"]
        for s in SEASONS:
            d, pc = r[f"{s}_Dominant"], r[f"{s}_Pct_Days_Dominant"]
            row.append(f"{d} ({pc:.0f}%)" if isinstance(d, str) else "n/a")
            cols.append(POLL_COLORS.get(d, "#DDDDDD") if isinstance(d, str) else "#DDDDDD")
        row.append(f"{r['Mean_AQI_All_Seasons']:.0f}")
        cols.append("white")
        cell_text.append(row)
        cell_colors.append(cols)
    tb = ax.table(cellText=cell_text, colLabels=["City"] + SEASONS + ["Mean AQI"], cellColours=cell_colors,
                  loc="center", cellLoc="center")
    tb.auto_set_font_size(False)
    tb.set_fontsize(8.5)
    tb.scale(1, 1.35)
    for (i, j), c in tb.get_celld().items():
        if i > 0 and 1 <= j <= 4:
            c.get_text().set_color("white")
    return save_fig(fig, "22_dominant_pollutant_table_top25_cities.png",
                    "Dominant pollutant (and % of days dominant) by season: 25 cities with highest mean estimated AQI")


def plot_dominant_city_bars(ctx):
    s = ctx["summary"]
    top = ctx["ranking_cities"][:20]
    fig, ax = plt.subplots(2, 2, figsize=(17, 13), sharex=True)
    for a, season in zip(ax.ravel(), SEASONS):
        g = s[(s["Season"] == season) & (s["City"].isin(top))].set_index("City").reindex(top[::-1])
        left = np.zeros(len(g))
        for p in POLLUTANTS:
            v = g[f"Pct_Days_Dominant_{p}"].fillna(0).values
            a.barh(g.index, v, left=left, color=POLL_COLORS[p], label=p)
            left += v
        a.set_title(season)
        a.set_xlabel("% of days dominant")
    ax[0, 0].legend(ncol=3, fontsize=8)
    return save_fig(fig, "23_dominant_pollutant_city_bars_top20.png",
                    "Percent of days each pollutant is dominant: 20 cities with highest mean estimated AQI")


def _wrap(text, width):
    return "\n".join(textwrap.fill(p, width=width) for p in str(text).split("\n"))


def pick_representative_cities(summary, n=10):
    mean_aqi = summary.groupby("City")["Mean_AQI"].mean().sort_values(ascending=False)
    preferred = ["Delhi", "Mumbai", "Kolkata", "Chennai", "Bengaluru", "Hyderabad", "Lucknow", "Patna", "Ahmedabad", "Pune"]
    avail = {c.lower().replace("_", " "): c for c in mean_aqi.index}
    chosen = [avail[p.lower()] for p in preferred if p.lower() in avail][:6]
    for c in (mean_aqi.index[0], mean_aqi.index[-1]):
        if c not in chosen:
            chosen.append(c)
    reg = summary.drop_duplicates("City").set_index("City")["Region"] if "Region" in summary else pd.Series(dtype=str)
    pool = [c for c in mean_aqi.index if c not in chosen]
    RNG.shuffle(pool)                                                  # seed 42 -> reproducible
    used_regions = set(reg.reindex(chosen).dropna())
    for c in list(pool):                                               # prefer regions not yet represented
        if len(chosen) >= n:
            break
        if reg.get(c) not in used_regions:
            chosen.append(c)
            used_regions.add(reg.get(c))
    for c in pool:
        if len(chosen) >= n:
            break
        if c not in chosen:
            chosen.append(c)
    return chosen[:n]


def build_cards(summary_out, cities):
    rows = []
    for c in cities:
        g = summary_out[summary_out["City"] == c]
        r = g.loc[g["Mean_AQI"].idxmax()]                              # season with the highest mean AQI
        rows.append({
            "City": c, "Season": r["Season"], "Region": r.get("Region", ""),
            "Dominant_Pollutant": r["Dominant_Pollutant"],
            "Percent_Days_Dominant": r["Percent_Days_Dominant"],
            "Mean_AQI": r["Mean_AQI"], "Mean_AQI_Category": r["Mean_AQI_Category"],
            "Percent_Days_AQI_Above200": r["Percent_Days_AQI_Above200"],
            "Mean_PM2_5": r["Mean_PM2_5"], "Mean_PM10": r["Mean_PM10"], "Mean_NO2": r["Mean_NO2"],
            "Mean_SO2": r["Mean_SO2"], "Mean_O3": r["Mean_O3"], "Mean_CO": r["Mean_CO"],
            "Mean_Temperature": r["Mean_Temperature"], "Mean_Humidity": r["Mean_Humidity"],
            "Mean_WindSpeed": r["Mean_WindSpeed"], "Mean_Rainfall": r["Mean_Rainfall"],
            "Likely_Cause": r["Likely_Cause"], "Recommended_Actions": r["Recommended_Actions"]})
    return pd.DataFrame(rows)


def plot_explanation_cards(ctx):
    cards = ctx["cards"]
    out = []
    for part, chunk in enumerate([cards.iloc[:5], cards.iloc[5:10]], start=1):
        fig, axes = plt.subplots(len(chunk), 1, figsize=(15, 4.9 * len(chunk)))
        axes = np.atleast_1d(axes)
        for ax, (_, r) in zip(axes, chunk.iterrows()):
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.axis("off")
            ax.add_patch(FancyBboxPatch((0.005, 0.01), 0.99, 0.97, boxstyle="round,pad=0.005,rounding_size=0.02",
                                        fc="#FAFAFA", ec=POLL_COLORS.get(r["Dominant_Pollutant"], "#888"), lw=2.5))
            head = (f"{r['City']}  |  {r['Season']} (season with highest mean AQI)  |  Dominant: {r['Dominant_Pollutant']} "
                    f"on {r['Percent_Days_Dominant']:.0f}% of days  |  Mean AQI {r['Mean_AQI']:.0f} "
                    f"({r['Mean_AQI_Category']})  |  AQI > 200 on {r['Percent_Days_AQI_Above200']:.0f}% of days")
            cond = (f"Mean conditions: temperature {r['Mean_Temperature']:.1f} C, humidity {r['Mean_Humidity']:.0f}%, "
                    f"wind {r['Mean_WindSpeed']:.1f} km/h, rainfall {r['Mean_Rainfall']:.1f} mm/day   |   "
                    f"PM2.5 {r['Mean_PM2_5']:.0f}, PM10 {r['Mean_PM10']:.0f}, NO2 {r['Mean_NO2']:.0f}, "
                    f"SO2 {r['Mean_SO2']:.0f}, O3 {r['Mean_O3']:.0f} ug/m3, CO {r['Mean_CO'] / 1000:.2f} mg/m3 (daily means)")
            acts = "\n".join("  - " + a for a in str(r["Recommended_Actions"]).split(" | "))
            body = (_wrap(head, 150) + "\n" + _wrap(cond, 160) + "\n\n"
                    + "Likely cause (suggestive, rule-based):\n" + _wrap("  " + str(r["Likely_Cause"]), 150) + "\n\n"
                    + "Suggested actions:\n" + "\n".join(_wrap(l, 150) for l in acts.split("\n")))
            ax.text(0.02, 0.95, body, va="top", ha="left", fontsize=8.6, family="DejaVu Sans", linespacing=1.35)
        out.append(save_fig(fig, f"24_explanation_cards_part{part}.png",
                            f"Example explanation cards: representative cities ({part}/2)"))
    return out


def plot_policy_frequencies(ctx):
    d, s = ctx["daily_out"], ctx["summary_out"]
    out = []
    labels = {k: ACTION_RULES[k][0] for k in ACTION_ORDER}

    ex = d[["Season", "Dominant_Pollutant", "Action_Rule_IDs"]].copy()
    ex["id"] = ex["Action_Rule_IDs"].fillna("").str.split("|")
    ex = ex.explode("id")
    ex = ex[ex["id"] != ""]
    n_days = d.groupby("Season").size().reindex(SEASONS)
    by_season = (ex.groupby(["id", "Season"]).size().unstack(fill_value=0).reindex(columns=SEASONS, fill_value=0)
                 .reindex([k for k in ACTION_ORDER if k in ex["id"].unique()]))
    pct_season = by_season.div(n_days, axis=1) * 100

    fig, ax = plt.subplots(1, 2, figsize=(18, 6.5))
    overall = by_season.sum(axis=1) / len(d) * 100
    ax[0].barh([labels[i] for i in overall.index][::-1], overall.values[::-1], color="#4C72B0")
    ax[0].set_xlabel("% of city-days triggering the recommendation")
    ax[0].set_title("Daily rows: recommendation frequency")
    y = np.arange(len(pct_season))
    w = 0.2
    for k, sname in enumerate(SEASONS):
        ax[1].barh(y + (k - 1.5) * w, pct_season[sname].values, height=w, color=SEASON_COLORS[sname], label=sname)
    ax[1].set_yticks(y, [labels[i] for i in pct_season.index])
    ax[1].invert_yaxis()
    ax[1].set_xlabel("% of days within season")
    ax[1].set_title("By season")
    ax[1].legend()
    out.append(save_fig(fig, "25_policy_recommendation_frequency_daily.png",
                        "Policy-recommendation frequency (all city-days)"))

    # by dominant pollutant (heatmap)
    hm = (ex.groupby(["id", "Dominant_Pollutant"]).size().unstack(fill_value=0).reindex(columns=POLLUTANTS, fill_value=0))
    tot = d.groupby("Dominant_Pollutant").size().reindex(POLLUTANTS)
    hm = hm.div(tot, axis=1).mul(100).reindex([k for k in ACTION_ORDER if k in hm.index])
    fig, ax = plt.subplots(figsize=(9.5, 6))
    im = ax.imshow(hm.values, cmap="YlOrRd", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(len(POLLUTANTS)), POLLUTANTS)
    ax.set_yticks(range(len(hm)), [labels[i] for i in hm.index])
    for i in range(hm.shape[0]):
        for j in range(hm.shape[1]):
            ax.text(j, i, f"{hm.values[i, j]:.0f}", ha="center", va="center", fontsize=8)
    ax.set_xlabel("Dominant pollutant on the day")
    fig.colorbar(im, ax=ax, label="% of days with that dominant pollutant")
    out.append(save_fig(fig, "26_policy_recommendation_by_dominant_pollutant.png",
                        "Recommendation frequency by dominant pollutant"))

    # City x Season summary rows
    es = s[["Season", "Action_Rule_IDs"]].copy()
    es["id"] = es["Action_Rule_IDs"].fillna("").str.split("|")
    es = es.explode("id")
    es = es[es["id"] != ""]
    cs = (es.groupby(["id", "Season"]).size().unstack(fill_value=0).reindex(columns=SEASONS, fill_value=0)
          .reindex([k for k in ACTION_ORDER if k in es["id"].unique()]))
    fig, ax = plt.subplots(1, 2, figsize=(18, 6))
    cs.rename(index=labels).iloc[::-1].plot(kind="barh", stacked=True, ax=ax[0],
                                            color=[SEASON_COLORS[x] for x in SEASONS], width=0.7)
    ax[0].set_xlabel("Number of City x Season cells (of 404)")
    ax[0].set_title("City x Season rows: recommendations triggered")
    health = (d.assign(h=d["Action_Rule_IDs"].fillna("").str.contains(r"(?:^|\|)H(?:\||$)"))
              .groupby("City")["h"].mean().mul(100).sort_values(ascending=False).head(20)[::-1])
    ax[1].barh(health.index, health.values, color="#C44E52")
    ax[1].set_xlabel("% of days with health-advisory recommendation")
    ax[1].set_title("20 cities with the most health-advisory days")
    out.append(save_fig(fig, "27_policy_recommendation_frequency_city_season.png",
                        "Policy recommendations: City x Season rows and city-level health advisories"))
    return out


def plot_cause_frequencies(ctx):
    d = ctx["daily_out"]
    ex = d[["Season", "Cause_Rule_IDs"]].copy()
    ex["id"] = ex["Cause_Rule_IDs"].fillna("").str.split("|")
    ex = ex.explode("id")
    ex = ex[ex["id"] != ""]
    n_days = d.groupby("Season").size().reindex(SEASONS)
    cnt = ex.groupby(["id", "Season"]).size().unstack(fill_value=0).reindex(columns=SEASONS, fill_value=0)
    cnt = cnt.reindex(list(CAUSE_RULES)).fillna(0)
    pct = cnt.div(n_days, axis=1) * 100
    fig, ax = plt.subplots(figsize=(13, 6))
    y = np.arange(len(pct))
    for k, sname in enumerate(SEASONS):
        ax.barh(y + (k - 1.5) * 0.2, pct[sname].values, height=0.2, color=SEASON_COLORS[sname], label=sname)
    ax.set_yticks(y, [f"{i}: {CAUSE_RULES[i][0]}" for i in pct.index])
    ax.invert_yaxis()
    ax.set_xlabel("% of days within season where the rule is triggered")
    ax.legend()
    none = 100 * (d["Cause_Rule_IDs"].fillna("") == "").mean()
    ax.set_title(f"Rule triggers by season ({none:.1f}% of days trigger no cause rule)")
    return save_fig(fig, "28_cause_rule_frequency_by_season.png",
                    "Cause-explanation rules triggered (suggestive associations)")


# ----------------------------------------------------------------------------
# 8. Main pipeline
# ----------------------------------------------------------------------------
def main():
    banner("PART 3 - Weather-based cause explanations and city-specific policy recommendations")
    print(AQI_LABEL)
    print(AQI_CAVEAT)
    print(CAUSE_CAVEAT)

    daily, summary, p2_manifest = load_part2()
    h_daily, h_summary = sha256_of(P2_DAILY), sha256_of(P2_SUMMARY)
    print(f"Loaded Part 2 daily: {daily.shape} | summary: {summary.shape} | cities: {daily['City'].nunique()}")
    n_prov = int(daily["AQI_Uses_Provisional_8hr"].astype(str).str.lower().isin(["true", "1"]).sum()) \
        if "AQI_Uses_Provisional_8hr" in daily else None
    if n_prov:
        print(f"Rows whose O3/CO inputs are PROVISIONAL (daily-mean fallback) in Part 2: {n_prov:,}. "
              "Ozone/CO-related explanations on those rows are provisional too.")

    banner("Thresholds")
    thr = derive_thresholds(daily)
    for k, v in thr.items():
        print(f"  high {k}: >= {v:.2f}")
    print(f"  weather (daily): {WEATHER_THRESHOLDS['daily']}")
    print(f"  weather (season_mean): {WEATHER_THRESHOLDS['season_mean']}")

    banner("Applying cause and policy functions")
    daily_out = apply_to_daily(daily)
    summary_out = apply_to_summary(summary)
    print(f"Daily rows processed: {len(daily_out):,} | City x Season rows processed: {len(summary_out)}")
    for col in ("Likely_Cause", "Recommended_Actions"):
        assert daily_out[col].notna().all() and (daily_out[col] != "").all(), f"Empty {col} in daily rows"
        assert summary_out[col].notna().all() and (summary_out[col] != "").all(), f"Empty {col} in summary rows"
    assert len(daily_out) == len(daily) and len(summary_out) == len(summary)
    dup_check = daily_out["Recommended_Actions"].head(5000).str.split(" \\| ").apply(lambda l: len(l) == len(set(x.lower() for x in l)))
    assert dup_check.all(), "Duplicate recommendations found"

    # ---- save data outputs ----------------------------------------------------------
    banner("Saving data outputs")
    daily_out.to_csv(DAILY_OUT, index=False, date_format="%Y-%m-%d")
    summary_out.to_csv(SUMMARY_OUT, index=False)
    dom_table = dominant_table(summary_out)
    dom_table.to_csv(DOM_TABLE_CSV, index=False)
    thresholds_table().to_csv(THRESH_CSV, index=False)
    rules_table().to_csv(RULES_CSV, index=False)
    freq = rule_frequency_table(daily_out, summary_out)
    freq.to_csv(RULE_FREQ_CSV, index=False)
    cities10 = pick_representative_cities(summary_out, 10)
    cards = build_cards(summary_out, cities10)
    cards.to_csv(CARDS_CSV, index=False)
    with open(METHOD_NOTE_TXT, "w") as f:
        f.write(f"{AQI_LABEL}\n{AQI_CAVEAT}\n{CAUSE_CAVEAT}\n\n"
                "Cause function: nine rules (C1-C9, see cause_policy_rules.csv) compare each day's (or each City x "
                "Season mean's) concentrations and weather with the thresholds in cause_policy_thresholds_used.csv. "
                "Pollutant 'high' means >= the study-period national percentile "
                f"P{HIGH_POLLUTANT_PERCENTILE} (dataset-relative, not a regulatory limit). Sentences relevant to the "
                "dominant pollutant are listed first. The text suggests possible contributors and does not establish "
                "source attribution.\n"
                "Policy function: rules in cause_policy_rules.csv; duplicates removed; temperature and humidity are "
                "accepted but not used by the current rules; an optional regime only adds an interpretation note. "
                "For City x Season rows the AQI category is taken from Mean_AQI, and the health advisory is also "
                f"triggered when >= {SUMMARY_FREQUENT_POOR_PCT:g}% of days have AQI > 200 (extension).\n"
                "Recommendations are suggestions for review by local authorities, not mandates.\n")
    for p in (DAILY_OUT, SUMMARY_OUT, DOM_TABLE_CSV, THRESH_CSV, RULES_CSV, RULE_FREQ_CSV, CARDS_CSV):
        print(f"  saved {p}")

    # ---- figures ------------------------------------------------------------------------
    banner("Figures")
    ranking_cities = list(dom_table["City"])
    ctx = {"summary": summary_out, "daily_out": daily_out, "summary_out": summary_out, "dom_table": dom_table,
           "ranking_cities": ranking_cities, "cards": cards}
    for fn in (plot_dominant_by_season, plot_dominant_table, plot_dominant_city_bars, plot_explanation_cards,
               plot_policy_frequencies, plot_cause_frequencies):
        run_plot(fn, ctx)

    # ---- integrity: Part 2 files unchanged ------------------------------------------------
    assert sha256_of(P2_DAILY) == h_daily and sha256_of(P2_SUMMARY) == h_summary, "Part 2 files were modified!"

    # ---- manifest ------------------------------------------------------------------------
    data_files = [DAILY_OUT, SUMMARY_OUT, DOM_TABLE_CSV, THRESH_CSV, RULES_CSV, RULE_FREQ_CSV, CARDS_CSV, METHOD_NOTE_TXT]
    fig_files = [p for p in FIG_LOG if not p.startswith("FAILED:")]
    failed = [p for p in FIG_LOG if p.startswith("FAILED:")]
    top_causes = freq[(freq["level"] == "daily") & (freq["type"] == "cause")].set_index("rule_id")["pct_rows"].round(2).to_dict()
    top_actions = freq[(freq["level"] == "daily") & (freq["type"] == "action")].set_index("rule_id")["pct_rows"].round(2).to_dict()
    manifest = {
        "part": 3, "total_parts": 7, "run_id": RUN_ID, "seed": SEED,
        "label": AQI_LABEL, "caveats": AQI_CAVEAT, "cause_caveat": CAUSE_CAVEAT,
        "base_dir": BASE_DIR,
        "inputs_from_part2": {"daily": P2_DAILY, "daily_sha256": h_daily,
                              "summary": P2_SUMMARY, "summary_sha256": h_summary,
                              "part2_manifest": P2_MANIFEST if p2_manifest else None,
                              "part2_run_id": p2_manifest.get("run_id"),
                              "rows_with_provisional_8hr": n_prov},
        "thresholds": {"pollutant_high": THRESH["pollutant"], "pollutant_high_percentile": HIGH_POLLUTANT_PERCENTILE,
                       "weather": WEATHER_THRESHOLDS, "pm_coarse_ratio": PM_COARSE_RATIO,
                       "summary_frequent_poor_pct": SUMMARY_FREQUENT_POOR_PCT},
        "results": {"daily_rows": int(len(daily_out)), "city_season_rows": int(len(summary_out)),
                    "representative_cities": cities10,
                    "daily_cause_rule_pct_rows": top_causes, "daily_action_rule_pct_rows": top_actions},
        "added_columns": {"daily": ["Likely_Cause", "Recommended_Actions", "Cause_Rule_IDs", "Action_Rule_IDs"],
                          "city_season": ["Mean_AQI_Category", "Likely_Cause", "Recommended_Actions",
                                          "Cause_Rule_IDs", "Action_Rule_IDs"]},
        "functions": ["explain_cause(...)", "recommend_actions(...)"],
        "no_modelling_split_or_scaling_in_part3": True,
        "figures_failed": failed,
        "files_created": [{"path": p, "rel_path": os.path.relpath(p, BASE_DIR),
                           "size_bytes": os.path.getsize(p)} for p in data_files + fig_files] +
                         [{"path": MANIFEST_JSON, "rel_path": os.path.relpath(MANIFEST_JSON, BASE_DIR), "size_bytes": None}],
        "next_part_should_load": {"daily_with_causes": DAILY_OUT, "city_season_summary": SUMMARY_OUT,
                                  "dominant_pollutant_table": DOM_TABLE_CSV},
    }
    with open(MANIFEST_JSON, "w") as f:
        json.dump(manifest, f, indent=2, default=str)

    banner("PART 3 COMPLETE - files created (exact paths for Part 4)")
    for p in data_files + fig_files + [MANIFEST_JSON]:
        print(p)
    if failed:
        print("\nFigures that failed (pipeline continued):", failed)
    print(f"\nBASE_DIR: {BASE_DIR}")
    print("\n" + AQI_LABEL)
    print(AQI_CAVEAT)
    print(CAUSE_CAVEAT)
    return daily_out, summary_out


if __name__ == "__main__" or IN_COLAB:
    daily_with_causes, city_season_with_causes = main()


PART 3 - Weather-based cause explanations and city-specific policy recommendations
Six-pollutant estimated AQI using CPCB sub-index methodology.
NH3 and Pb are unavailable, so this is not the official CPCB AQI. O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct CPCB ground readings.
Likely_Cause text is a rule-based, suggestive interpretation of simultaneous weather and concentration patterns; it does not establish source attribution.
Loaded Part 2 daily: (146147, 27) | summary: (404, 48) | cities: 101
Rows whose O3/CO inputs are PROVISIONAL (daily-mean fallback) in Part 2: 101. Ozone/CO-related explanations on those rows are provisional too.

Thresholds
  high PM2_5: >= 53.92
  high PM10: >= 84.92
  high NO2: >= 21.10
  high SO2: >= 18.20
  high O3: >= 98.83
  high CO: >= 537.75
  high O3_8hr_max: >= 151.38
  high CO_8hr_max_mg_m3: >= 0.74
  weather (daily): {'low_wind_kmh': 6.0, 'low_rain_mm': 1.0, 'high_rain_mm': 10.0, 'low_humidity_pct': 40.0, 'high_humidity_pct':

In [5]:
# =============================================================================
# City- and Season-Aware Multi-Pollutant Air Quality Assessment and Forecasting
# Framework for Indian Cities Using Deep Learning
#
# PART 4 of 7: past-only feature engineering, forecasting targets, ONE global
#              chronological 70/15/15 split, and a train-only RobustScaler.
#
# Continues directly from Parts 1-3 (same BASE_DIR). Inputs are only read.
#
# LABEL : "Six-pollutant estimated AQI using CPCB sub-index methodology."
# CAVEAT: NH3 and Pb are unavailable, so this is NOT the official CPCB AQI.
#         O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct
#         CPCB ground readings (rows using daily-mean fallback stay flagged).
#
# LEAKAGE PROTOCOL
#   * Row date t = last day whose observations are known. Targets refer to t+1,
#     t+3, t+7 (and the max over t+1..t+3).
#   * Current-day values (t) and lags 1,2,3,7 are allowed; rolling statistics are
#     computed on the series shifted by 1 day (windows end at t-1), so no window
#     ever touches t+1 or later. A perturbation test proves this before saving.
#   * Lags/shifts are calendar-day based (each city is reindexed to a complete
#     daily grid first), so a gap can never silently pull in a distant day.
#   * ONE global chronological split by date; the last 7 dates of train and of
#     validation are purged (their t+1..t+7 labels would reach into the next
#     split). The scaler is fit on train rows only.
# =============================================================================

# ----------------------------------------------------------------------------
# 0. Imports, configuration, reproducibility, link to Part 3
# ----------------------------------------------------------------------------
import os
import sys
import json
import random
import hashlib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.RandomState(SEED)

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
BASE_DIR = "/content/aqi_framework" if IN_COLAB else os.path.abspath("./aqi_framework")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

PROC_DIR = os.path.join(BASE_DIR, "data", "processed")
RES_DIR = os.path.join(BASE_DIR, "results")
FIG_DIR = os.path.join(BASE_DIR, "figures")
MODEL_DIR = os.path.join(BASE_DIR, "models")
for d in (PROC_DIR, RES_DIR, FIG_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

# ---- inputs (Parts 1 and 3) -------------------------------------------------------
P1_MASTER = os.path.join(PROC_DIR, "master_real_units.csv")
P3_DAILY = os.path.join(RES_DIR, "daily_aqi_dominant_pollutant_with_causes.csv")
P3_SUMMARY = os.path.join(RES_DIR, "city_season_dominant_pollutant_summary.csv")
P3_MANIFEST = os.path.join(RES_DIR, "part3_handoff_manifest.json")
# optional K-Means regime labels (searched in this order; skipped if none exists)
REGIME_CANDIDATES = [os.path.join(RES_DIR, "kmeans_regime_labels.csv"),
                     os.path.join(RES_DIR, "city_season_regimes.csv"),
                     os.path.join(PROC_DIR, "kmeans_regime_labels.csv")]
USE_REGIME_IF_AVAILABLE = True

# ---- outputs (as specified + supporting) ------------------------------------------------
FEATURES_CSV = os.path.join(PROC_DIR, "features_v1.csv")
TRAIN_CSV = os.path.join(PROC_DIR, "train.csv")
VAL_CSV = os.path.join(PROC_DIR, "validation.csv")
TEST_CSV = os.path.join(PROC_DIR, "test.csv")
TRAIN_S_CSV = os.path.join(PROC_DIR, "train_scaled.csv")
VAL_S_CSV = os.path.join(PROC_DIR, "validation_scaled.csv")
TEST_S_CSV = os.path.join(PROC_DIR, "test_scaled.csv")
SCALER_PKL = os.path.join(MODEL_DIR, "standard_scaler.pkl")       # holds a RobustScaler (name fixed by spec)
FEATURE_SUMMARY_CSV = os.path.join(RES_DIR, "feature_summary.csv")
HEATMAP_PNG = os.path.join(FIG_DIR, "feature_correlation_heatmap.png")
TOPCORR_PNG = os.path.join(FIG_DIR, "feature_target_correlation_top30.png")
SPLIT_PNG = os.path.join(FIG_DIR, "chronological_split_timeline.png")
FEATURE_COLS_JSON = os.path.join(RES_DIR, "feature_columns.json")
CITY_MAP_CSV = os.path.join(RES_DIR, "city_id_map.csv")
SPLIT_SUMMARY_CSV = os.path.join(RES_DIR, "split_summary.csv")
SCALER_META_JSON = os.path.join(MODEL_DIR, "scaler_feature_list.json")
MANIFEST_JSON = os.path.join(RES_DIR, "part4_handoff_manifest.json")

# ---- labels -----------------------------------------------------------------------------
AQI_LABEL = "Six-pollutant estimated AQI using CPCB sub-index methodology."
AQI_CAVEAT = ("NH3 and Pb are unavailable, so this is not the official CPCB AQI. "
              "O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct CPCB ground readings.")
FOOTER = "Six-pollutant estimated AQI (CPCB sub-index methodology); not official CPCB AQI (NH3, Pb unavailable)."

# ---- schema -------------------------------------------------------------------------------
POLL6 = ["PM2_5", "PM10", "CO", "NO2", "SO2", "O3"]
WEATHER = ["Temperature_C", "Humidity_pct", "WindSpeed_kmh", "Precipitation_mm"]
EIGHT_HR = ["O3_8hr_max", "CO_8hr_max_mg_m3"]
CURRENT_VARS = POLL6 + EIGHT_HR + WEATHER                       # observed on day t
AQI_VAR = "Estimated_AQI"
LAG_VARS = CURRENT_VARS + [AQI_VAR]                             # variables that get lags / rolling stats
POLLUTANT_ORDER = ["PM2_5", "PM10", "NO2", "SO2", "O3", "CO"]   # same order as Parts 2-3
SEASONS = ["Winter", "Summer", "Monsoon", "Post-monsoon"]
SEASON_MAP = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Summer", 4: "Summer", 5: "Summer",
              6: "Monsoon", 7: "Monsoon", 8: "Monsoon", 9: "Monsoon", 10: "Post-monsoon", 11: "Post-monsoon"}
CATEGORIES = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
CAT_CODE = {c: i for i, c in enumerate(CATEGORIES)}

LAGS = [1, 2, 3, 7]
ROLL_MEAN_WINDOWS = [3, 7, 14, 30]
ROLL_STD_WINDOWS = [7, 30]
ROLL_SHIFT = 1                                                  # rolling windows end at t-1
HORIZONS = [1, 3, 7]
HIGH_RISK_THRESHOLD = 200.0
HIGH_RISK_DAYS = 3
SPLIT_FRACTIONS = (0.70, 0.15, 0.15)
PURGE_DAYS = max(HORIZONS)                                      # 7; set 0 to disable the purge

TARGET_REG = ["Estimated_AQI_tplus1", "Estimated_AQI_tplus3", "Estimated_AQI_tplus7"]
TARGET_CLS = ["Estimated_AQI_Category_tplus1", "Estimated_AQI_Category_tplus1_code", "HighRisk_AQI_gt200_next3d"]
TARGET_COLS = TARGET_REG + TARGET_CLS
CSV_FLOAT_FORMAT = "%.6g"


def banner(text):
    print("\n" + "=" * 78 + f"\n{text}\n" + "=" * 78)


def sha256_of(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


# ----------------------------------------------------------------------------
# 1. Load Parts 1 and 3 (read-only) and verify continuity
# ----------------------------------------------------------------------------
def load_inputs():
    for p in (P1_MASTER, P3_DAILY, P3_SUMMARY):
        if not os.path.isfile(p):
            raise FileNotFoundError(
                f"Missing input: {p}\nRun the earlier parts first (same BASE_DIR={BASE_DIR}). "
                "In Colab, a restarted runtime loses /content - re-run Parts 1-3 or restore the folder.")
    p3_manifest = {}
    if os.path.isfile(P3_MANIFEST):
        with open(P3_MANIFEST) as f:
            p3_manifest = json.load(f)

    daily = pd.read_csv(P3_DAILY, parse_dates=["Date"])
    summary = pd.read_csv(P3_SUMMARY)
    master_cols = ["Date", "City", "Region", "Latitude", "Longitude"] + POLL6 + WEATHER + EIGHT_HR
    master = pd.read_csv(P1_MASTER, usecols=lambda c: c in master_cols, parse_dates=["Date"])

    need = ["Date", "City", "Season"] + POLL6 + WEATHER + [AQI_VAR, "AQI_Category", "Dominant_Pollutant"]
    missing = [c for c in need if c not in daily.columns]
    if missing:
        raise ValueError(f"Part 3 daily file is missing columns: {missing}")
    if p3_manifest:
        exp = p3_manifest.get("results", {}).get("daily_rows")
        if exp and exp != len(daily):
            raise ValueError(f"Daily rows {len(daily)} differ from Part 3 manifest ({exp}).")
    if len(summary) != daily["City"].nunique() * 4:
        print(f"  ! Part 3 summary has {len(summary)} rows; expected {daily['City'].nunique() * 4}.")
    if daily.duplicated(["City", "Date"]).any():
        raise ValueError("Duplicate City-Date rows in Part 3 daily file.")

    # master vs Part 3: same rows and same values (Part 3 values were derived from the master)
    merged = daily[["City", "Date"] + POLL6 + WEATHER].merge(
        master[["City", "Date", "Region", "Latitude", "Longitude"] + POLL6 + WEATHER],
        on=["City", "Date"], how="outer", suffixes=("", "_m"), indicator=True)
    n_only = int((merged["_merge"] != "both").sum())
    both = merged[merged["_merge"] == "both"]
    max_diff = max(float(np.nanmax(np.abs(both[c] - both[c + "_m"]))) for c in POLL6 + WEATHER)
    print(f"  Part 1 master vs Part 3 daily: rows not in both = {n_only}, max abs difference = {max_diff:.2e}")
    if n_only or max_diff > 1e-3:
        print("  ! Master and Part 3 files disagree; Part 3 values are used for features.")

    # base table: Part 3 daily + city-level context (Region, Lat, Lon) from the master
    ctx = master.drop_duplicates("City")[["City", "Region", "Latitude", "Longitude"]]
    keep = (["Date", "City", "Season"] + POLL6 + WEATHER + [c for c in EIGHT_HR if c in daily.columns] +
            [AQI_VAR, "AQI_Category", "Dominant_Pollutant"] +
            (["AQI_Uses_Provisional_8hr"] if "AQI_Uses_Provisional_8hr" in daily.columns else []))
    base = daily[keep].merge(ctx, on="City", how="left")
    for c in EIGHT_HR:                                              # fall back to master if absent in Part 3
        if c not in base.columns:
            base = base.merge(master[["City", "Date", c]], on=["City", "Date"], how="left")
    if "AQI_Uses_Provisional_8hr" in base.columns:
        base["AQI_Uses_Provisional_8hr"] = base["AQI_Uses_Provisional_8hr"].astype(str).str.lower().isin(["true", "1"]).astype(int)
    else:
        base["AQI_Uses_Provisional_8hr"] = 0
    base["AQI_Category"] = base["AQI_Category"].where(base["AQI_Category"].isin(CATEGORIES))
    return base, summary, p3_manifest


def find_regimes(summary):
    """Optional K-Means regime labels. Returns (DataFrame with City/[Season|Date]/Regime_KMeans_raw, source) or (None, None)."""
    def pick_col(df):
        for c in df.columns:
            if c.lower() in ("regime", "kmeans_regime", "regime_label", "regime_id", "cluster", "kmeans_cluster",
                             "regime_kmeans"):
                return c
        return None
    for p in REGIME_CANDIDATES:
        if os.path.isfile(p):
            r = pd.read_csv(p)
            c = pick_col(r)
            if c is not None and "City" in r.columns:
                keys = [k for k in ("City", "Date", "Season") if k in r.columns]
                if "Date" in keys:
                    r["Date"] = pd.to_datetime(r["Date"])
                return r[keys + [c]].rename(columns={c: "Regime_raw"}), p
    c = pick_col(summary)
    if c is not None:
        return summary[["City", "Season", c]].rename(columns={c: "Regime_raw"}), P3_SUMMARY
    return None, None


# ----------------------------------------------------------------------------
# 2. Feature engineering (pure function of the base table -> testable for leakage)
# ----------------------------------------------------------------------------
def to_full_grid(base):
    """Reindex each city to a complete daily calendar so shifts are calendar-day based."""
    cities = sorted(base["City"].unique())
    dates = pd.date_range(base["Date"].min(), base["Date"].max(), freq="D")
    grid = pd.MultiIndex.from_product([cities, dates], names=["City", "Date"]).to_frame(index=False)
    g = grid.merge(base.assign(_observed=True), on=["City", "Date"], how="left")
    g["_observed"] = g["_observed"].fillna(False).astype(bool)
    static = base.drop_duplicates("City").set_index("City")[["Region", "Latitude", "Longitude"]]
    for c in static.columns:
        g[c] = g["City"].map(static[c])
    g["Season"] = g["Date"].dt.month.map(SEASON_MAP)
    return g.sort_values(["City", "Date"]).reset_index(drop=True)


def build_features(base):
    """Returns (table, feature_cols, meta) where table has Date, City, Season, Region + features + targets.
    meta: dict feature -> (group, description, scaled_bool)."""
    g = to_full_grid(base)
    grp = g.groupby("City", sort=False)
    F, meta = {}, {}

    def add(name, series, group, desc, scaled=True):
        F[name] = series
        meta[name] = (group, desc, scaled)

    city_ids = {c: i for i, c in enumerate(sorted(g["City"].unique()))}
    region_ids = {r: i for i, r in enumerate(sorted(g["Region"].dropna().unique()))}

    # identifiers / context
    add("City_ID", g["City"].map(city_ids), "identifier", "Integer city index (sorted names); for embeddings", False)
    add("Region_ID", g["Region"].map(region_ids), "context", "Integer region index (optional context)", False)
    add("Latitude", g["Latitude"], "context", "City latitude (optional context)")
    add("Longitude", g["Longitude"], "context", "City longitude (optional context)")

    # current-day (t) observations
    for v in CURRENT_VARS + [AQI_VAR]:
        add(v, g[v], "current", f"{v} observed on day t")
    add("Dominant_Pollutant_t_code", g["Dominant_Pollutant"].map({p: i for i, p in enumerate(POLLUTANT_ORDER)}).fillna(-1),
        "current", "Dominant pollutant on day t (0=PM2_5,1=PM10,2=NO2,3=SO2,4=O3,5=CO,-1=unknown)", False)
    add("Provisional_8hr_flag_t", g["AQI_Uses_Provisional_8hr"].fillna(0), "current",
        "1 if day-t O3/CO 8-hour values are provisional daily-mean fallbacks", False)

    # lags
    for k in LAGS:
        sh = grp[LAG_VARS].shift(k)
        for v in LAG_VARS:
            add(f"{v}_lag{k}", sh[v], "lag", f"{v} at t-{k}")

    # rolling statistics on the series shifted by ROLL_SHIFT (windows end at t-1)
    sh1 = grp[LAG_VARS].shift(ROLL_SHIFT)
    city_key = g["City"]
    for w in ROLL_MEAN_WINDOWS:
        r = sh1.groupby(city_key).rolling(w, min_periods=w).mean().reset_index(level=0, drop=True).reindex(g.index)
        for v in LAG_VARS:
            add(f"{v}_roll{w}_mean", r[v], "rolling_mean", f"Mean of {v} over days t-{w} .. t-1")
    for w in ROLL_STD_WINDOWS:
        r = sh1.groupby(city_key).rolling(w, min_periods=w).std().reset_index(level=0, drop=True).reindex(g.index)
        for v in LAG_VARS:
            add(f"{v}_roll{w}_std", r[v], "rolling_std", f"Std of {v} over days t-{w} .. t-1")

    # calendar (for day t)
    d = g["Date"]
    month, wd, doy = d.dt.month, d.dt.weekday, d.dt.dayofyear
    add("month", month, "calendar", "Month 1-12", False)
    add("weekday", wd, "calendar", "Weekday 0=Mon", False)
    add("is_weekend", (wd >= 5).astype(int), "calendar", "1 if Saturday/Sunday", False)
    add("day_of_year", doy, "calendar", "Day of year", False)
    add("week_of_year", d.dt.isocalendar().week.astype(int), "calendar", "ISO week of year", False)
    add("Season_code", g["Season"].map({s: i for i, s in enumerate(SEASONS)}), "calendar",
        "Season code (0=Winter,1=Summer,2=Monsoon,3=Post-monsoon)", False)
    for s in SEASONS:
        add(f"Season_{s.replace('-', '_')}", (g["Season"] == s).astype(int), "calendar", f"1 if season is {s}", False)
    add("month_sin", np.sin(2 * np.pi * month / 12), "calendar_cyclic", "sin(2*pi*month/12)", False)
    add("month_cos", np.cos(2 * np.pi * month / 12), "calendar_cyclic", "cos(2*pi*month/12)", False)
    add("doy_sin", np.sin(2 * np.pi * doy / 365.25), "calendar_cyclic", "sin(2*pi*day_of_year/365.25)", False)
    add("doy_cos", np.cos(2 * np.pi * doy / 365.25), "calendar_cyclic", "cos(2*pi*day_of_year/365.25)", False)
    add("weekday_sin", np.sin(2 * np.pi * wd / 7), "calendar_cyclic", "sin(2*pi*weekday/7)", False)
    add("weekday_cos", np.cos(2 * np.pi * wd / 7), "calendar_cyclic", "cos(2*pi*weekday/7)", False)

    # interactions (day-t values; CO is the daily mean in ug/m3)
    pm10 = g["PM10"].replace(0, np.nan)
    pm25 = g["PM2_5"].replace(0, np.nan)
    add("PM2_5_to_PM10_ratio", g["PM2_5"] / pm10, "interaction", "PM2.5 / PM10")
    add("NO2_plus_CO", g["NO2"] + g["CO"], "interaction", "NO2 + CO (CO daily mean in ug/m3, so CO dominates the sum)")
    add("SO2_to_PM2_5_ratio", g["SO2"] / pm25, "interaction", "SO2 / PM2.5")
    add("Temperature_x_O3", g["Temperature_C"] * g["O3"], "interaction", "Temperature_C x O3")
    add("WindSpeed_x_PM2_5", g["WindSpeed_kmh"] * g["PM2_5"], "interaction", "WindSpeed_kmh x PM2.5")
    add("Rainfall_x_PM10", g["Precipitation_mm"] * g["PM10"], "interaction", "Precipitation_mm x PM10")
    add("Humidity_x_PM2_5", g["Humidity_pct"] * g["PM2_5"], "interaction", "Humidity_pct x PM2.5")

    feat = pd.DataFrame(F, index=g.index)
    feature_cols = list(feat.columns)

    # targets (future values; never used as features)
    T = {}
    for h in HORIZONS:
        T[f"Estimated_AQI_tplus{h}"] = grp[AQI_VAR].shift(-h)
    nxt_cat = grp["AQI_Category"].shift(-1)
    T["Estimated_AQI_Category_tplus1"] = nxt_cat
    T["Estimated_AQI_Category_tplus1_code"] = nxt_cat.map(CAT_CODE)
    fut = pd.concat([grp[AQI_VAR].shift(-k) for k in range(1, HIGH_RISK_DAYS + 1)], axis=1)
    full_window = fut.notna().all(axis=1)
    T["HighRisk_AQI_gt200_next3d"] = (fut.max(axis=1) > HIGH_RISK_THRESHOLD).astype(float).where(full_window)
    tgt = pd.DataFrame(T, index=g.index)

    table = pd.concat([g[["Date", "City", "Region", "Season", "_observed"]], feat, tgt], axis=1)
    return table, feature_cols, meta


# ----------------------------------------------------------------------------
# 3. Leakage tests (run before anything is saved)
# ----------------------------------------------------------------------------
def leakage_tests(base, table, feature_cols):
    res = {}
    obs = table[table["_observed"]]

    # (a) target alignment: tplus1 equals the AQI of the next calendar day of the same city
    chk = obs.dropna(subset=["Estimated_AQI_tplus1"]).sample(2000, random_state=SEED)[["City", "Date", "Estimated_AQI_tplus1"]]
    nxt = chk.assign(Date=chk["Date"] + pd.Timedelta(days=1)).merge(
        base[["City", "Date", AQI_VAR]], on=["City", "Date"], how="left")
    res["target_tplus1_matches_next_day_aqi"] = bool(np.allclose(nxt["Estimated_AQI_tplus1"], nxt[AQI_VAR]))

    # (b) manual recomputation of rolling features and lags for random rows
    ok_roll = True
    cand = obs.dropna(subset=["PM2_5_roll30_mean", "PM2_5_lag7", "Estimated_AQI_roll7_std"]).sample(25, random_state=SEED)
    for _, r in cand.iterrows():
        h = base[(base["City"] == r["City"])].set_index("Date")
        t = r["Date"]
        past30 = h.loc[t - pd.Timedelta(days=30): t - pd.Timedelta(days=1), "PM2_5"]
        past7 = h.loc[t - pd.Timedelta(days=7): t - pd.Timedelta(days=1), AQI_VAR]
        ok_roll &= np.isclose(past30.mean(), r["PM2_5_roll30_mean"], rtol=1e-9) and len(past30) == 30
        ok_roll &= np.isclose(past7.std(), r["Estimated_AQI_roll7_std"], rtol=1e-9)
        ok_roll &= np.isclose(h.loc[t - pd.Timedelta(days=7), "PM2_5"], r["PM2_5_lag7"])
    res["rolling_and_lag_values_match_manual_past_only_computation"] = bool(ok_roll)

    # (c) perturbation test: scramble every observation after a cutoff and rebuild; features up to the cutoff must not move
    dates = np.sort(base["Date"].unique())
    cutoff = pd.Timestamp(dates[int(len(dates) * 0.6)])
    pert = base.copy()
    future = pert["Date"] > cutoff
    for v in LAG_VARS:
        pert.loc[future, v] = RNG.uniform(1e3, 1e4, future.sum())
    pert.loc[future, "AQI_Category"] = "Severe"
    t2, _, _ = build_features(pert)
    a = table[table["_observed"] & (table["Date"] <= cutoff)].reset_index(drop=True)
    b = t2[t2["_observed"] & (t2["Date"] <= cutoff)].reset_index(drop=True)
    same = np.allclose(a[feature_cols].to_numpy(float), b[feature_cols].to_numpy(float), equal_nan=True)
    res["features_unchanged_when_future_observations_are_scrambled"] = bool(same)
    changed = not np.allclose(a["Estimated_AQI_tplus1"].to_numpy(float), b["Estimated_AQI_tplus1"].to_numpy(float), equal_nan=True)
    res["test_is_sensitive_targets_do_change"] = bool(changed)
    res["perturbation_cutoff_date"] = str(cutoff.date())

    # (d) rolling features never include day t itself: day-t value change must not move roll features of day t
    for k, v in res.items():
        if isinstance(v, bool) and not v:
            raise AssertionError(f"Leakage test failed: {k}")
    return res


# ----------------------------------------------------------------------------
# 4. Split, scaling
# ----------------------------------------------------------------------------
def chronological_split(model_df):
    dates = np.sort(model_df["Date"].unique())
    n = len(dates)
    i1 = int(round(SPLIT_FRACTIONS[0] * n))
    i2 = int(round((SPLIT_FRACTIONS[0] + SPLIT_FRACTIONS[1]) * n))
    train_dates, val_dates, test_dates = dates[:i1], dates[i1:i2], dates[i2:]
    split = pd.Series("test", index=model_df.index)
    split[model_df["Date"].isin(train_dates)] = "train"
    split[model_df["Date"].isin(val_dates)] = "validation"
    purged = []
    if PURGE_DAYS > 0:
        for dset in (train_dates, val_dates):
            gap = dset[-PURGE_DAYS:]
            purged.extend(gap)
            split[model_df["Date"].isin(gap)] = "purged"
    info = {"n_modelling_dates": int(n),
            "train_dates": [str(pd.Timestamp(train_dates[0]).date()), str(pd.Timestamp(train_dates[-1]).date())],
            "validation_dates": [str(pd.Timestamp(val_dates[0]).date()), str(pd.Timestamp(val_dates[-1]).date())],
            "test_dates": [str(pd.Timestamp(test_dates[0]).date()), str(pd.Timestamp(test_dates[-1]).date())],
            "purged_dates": [str(pd.Timestamp(x).date()) for x in purged]}
    return split, info


# ----------------------------------------------------------------------------
# 5. Feature summary and figures
# ----------------------------------------------------------------------------
def feature_summary(train, feature_cols, meta, scaled_cols, scaler, all_rows):
    rank = train[feature_cols].rank()
    y = train["Estimated_AQI_tplus1"].rank()
    rho = rank.corrwith(y)
    centers = dict(zip(scaled_cols, scaler.center_))
    scales = dict(zip(scaled_cols, scaler.scale_))
    rows = []
    for c in feature_cols:
        s = train[c]
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        rows.append({"Feature": c, "Group": meta[c][0], "Description": meta[c][1], "Scaled": meta[c][2],
                     "N_Missing_All_Rows": int(all_rows[c].isna().sum()),
                     "Train_Mean": s.mean(), "Train_Std": s.std(), "Train_Min": s.min(), "Train_Q1": q1,
                     "Train_Median": s.median(), "Train_Q3": q3, "Train_Max": s.max(),
                     "Scaler_Center": centers.get(c, np.nan), "Scaler_Scale": scales.get(c, np.nan),
                     "Spearman_with_AQI_tplus1_train": rho.get(c, np.nan)})
    return pd.DataFrame(rows)


def make_heatmap(train, path):
    cols = (["Estimated_AQI"] + POLL6 + WEATHER + EIGHT_HR +
            ["Estimated_AQI_lag1", "Estimated_AQI_lag7", "Estimated_AQI_roll7_mean", "Estimated_AQI_roll30_mean",
             "Estimated_AQI_roll7_std", "PM2_5_lag1", "PM2_5_roll7_mean", "month_sin", "month_cos",
             "PM2_5_to_PM10_ratio", "NO2_plus_CO", "SO2_to_PM2_5_ratio", "Temperature_x_O3", "WindSpeed_x_PM2_5",
             "Rainfall_x_PM10", "Humidity_x_PM2_5"] + TARGET_REG + ["HighRisk_AQI_gt200_next3d"])
    cols = [c for c in dict.fromkeys(cols) if c in train.columns]
    c = train[cols].corr(method="spearman")
    fig, ax = plt.subplots(figsize=(19, 16))
    im = ax.imshow(c.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(cols)), cols, rotation=75, ha="right", fontsize=8)
    ax.set_yticks(range(len(cols)), cols, fontsize=8)
    for i in range(len(cols)):
        for j in range(len(cols)):
            ax.text(j, i, f"{c.values[i, j]:.2f}", ha="center", va="center", fontsize=5.5)
    fig.colorbar(im, ax=ax, shrink=0.7, label="Spearman correlation")
    ax.set_title("Feature and target correlations (Spearman, TRAIN split only; associations, not causation)", fontsize=12)
    fig.text(0.5, 0.005, FOOTER, ha="center", fontsize=8, color="dimgray")
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)


def make_topcorr(summary_df, path):
    s = summary_df[~summary_df["Feature"].isin(["Estimated_AQI_tplus1"])].copy()
    s["abs"] = s["Spearman_with_AQI_tplus1_train"].abs()
    top = s.sort_values("abs", ascending=False).head(30).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, 9))
    ax.barh(top["Feature"], top["Spearman_with_AQI_tplus1_train"],
            color=np.where(top["Spearman_with_AQI_tplus1_train"] >= 0, "#C44E52", "#4C72B0"))
    ax.set_xlabel("Spearman correlation with Estimated_AQI_tplus1 (train split)")
    ax.set_title("Strongest feature associations with next-day estimated AQI")
    fig.text(0.5, 0.0, FOOTER, ha="center", fontsize=7, color="dimgray")
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)


def make_split_plot(model_df, info, path):
    daily = model_df.groupby("Date").agg(aqi=("Estimated_AQI", "mean"), split=("Split", "first"))
    colors = {"train": "#4C72B0", "validation": "#DD8452", "test": "#55A868", "purged": "#999999"}
    fig, ax = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [3, 1.4]})
    for s, c in colors.items():
        sub = daily[daily["split"] == s]
        ax[0].scatter(sub.index, sub["aqi"], s=6, color=c, label=s)
    ax[0].set_ylabel("Mean estimated AQI (all cities)")
    ax[0].set_title("Global chronological split (one cut for all cities; purged dates shown in grey)")
    ax[0].legend(markerscale=3)
    cnt = model_df["Split"].value_counts().reindex(["train", "validation", "test", "purged"]).fillna(0)
    ax[1].bar(cnt.index, cnt.values, color=[colors[k] for k in cnt.index])
    ax[1].set_ylabel("City-day rows")
    for i, v in enumerate(cnt.values):
        ax[1].text(i, v, f"{int(v):,}", ha="center", va="bottom")
    fig.text(0.5, 0.0, FOOTER, ha="center", fontsize=7, color="dimgray")
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)


# ----------------------------------------------------------------------------
# 6. Main pipeline
# ----------------------------------------------------------------------------
def main():
    banner("PART 4 - Features, targets, chronological split, RobustScaler")
    print(AQI_LABEL)
    print(AQI_CAVEAT)

    # ---- load -------------------------------------------------------------------------
    base, summary, p3_manifest = load_inputs()
    hashes_in = {p: sha256_of(p) for p in (P1_MASTER, P3_DAILY, P3_SUMMARY)}
    print(f"Base table: {base.shape} | cities: {base['City'].nunique()} | "
          f"{base['Date'].min().date()} to {base['Date'].max().date()}")
    n_prov = int(base["AQI_Uses_Provisional_8hr"].sum())
    if n_prov:
        print(f"  Note: {n_prov:,} rows carry provisional daily-mean O3/CO fallbacks (feature Provisional_8hr_flag_t).")

    # ---- features and targets --------------------------------------------------------------
    banner("Building past-only features and targets")
    table, feature_cols, meta = build_features(base)
    print(f"Feature columns built: {len(feature_cols)} (before optional regime)")

    banner("Leakage tests")
    leak = leakage_tests(base, table, feature_cols)
    for k, v in leak.items():
        print(f"  {k}: {v}")

    # ---- optional K-Means regime ---------------------------------------------------------------
    regime_info = {"available": False}
    reg, reg_src = (find_regimes(summary) if USE_REGIME_IF_AVAILABLE else (None, None))
    if reg is not None:
        keys = [k for k in ("City", "Date", "Season") if k in reg.columns]
        table = table.merge(reg, on=keys, how="left")
        codes = {v: i for i, v in enumerate(sorted(table["Regime_raw"].dropna().unique(), key=str))}
        table["Regime_KMeans"] = table["Regime_raw"].map(codes)
        table = table.drop(columns=["Regime_raw"])
        feature_cols.append("Regime_KMeans")
        meta["Regime_KMeans"] = ("regime", "K-Means regime label (integer code)", False)
        regime_info = {"available": True, "source": reg_src, "merge_keys": keys, "label_codes": {str(k): v for k, v in codes.items()}}
        print(f"K-Means regime labels found in {reg_src} (keys: {keys}).")
        if "Date" not in keys:
            print("  ! Caution: City x Season regime labels summarise the whole study period (including validation/test "
                  "dates). They are static descriptors; for a strictly leakage-free design, re-derive them from the "
                  "training split only.")
    else:
        print("K-Means regime labels: not available yet; feature skipped (add results/kmeans_regime_labels.csv to include it).")

    # ---- modelling table: observed days, complete features, tplus1 target ------------------------------
    banner("Selecting modelling rows")
    obs = table[table["_observed"]].copy()
    n0 = len(obs)
    no_t1 = obs["Estimated_AQI_tplus1"].isna() | obs["Estimated_AQI_Category_tplus1"].isna()
    incomplete = obs[feature_cols].isna().any(axis=1)
    model_df = obs[~no_t1 & ~incomplete].copy()
    print(f"Observed city-days: {n0:,} | dropped (no t+1 target): {int(no_t1.sum()):,} | "
          f"dropped (incomplete history/features, mostly the first 30 days per city): {int((incomplete & ~no_t1).sum()):,} | "
          f"kept: {len(model_df):,}")
    assert np.isfinite(model_df[feature_cols].to_numpy(float)).all(), "Non-finite feature values remain"
    model_df["Estimated_AQI_Category_tplus1_code"] = model_df["Estimated_AQI_Category_tplus1_code"].astype(int)
    model_df = model_df.drop(columns=["_observed"]).sort_values(["Date", "City"]).reset_index(drop=True)

    # ---- global chronological split -----------------------------------------------------------------
    banner("Global chronological split (70/15/15 by date, all cities together)")
    model_df["Split"], split_info = chronological_split(model_df)
    train = model_df[model_df["Split"] == "train"].copy()
    val = model_df[model_df["Split"] == "validation"].copy()
    test = model_df[model_df["Split"] == "test"].copy()
    parts = {"train": train, "validation": val, "test": test}

    summary_rows = []
    for name, df_ in parts.items():
        print(f"{name:>10}: {df_['Date'].min().date()} to {df_['Date'].max().date()} | rows {len(df_):,} | "
              f"dates {df_['Date'].nunique()} | cities {df_['City'].nunique()}")
        summary_rows.append({"split": name, "date_min": df_["Date"].min().date(), "date_max": df_["Date"].max().date(),
                             "n_rows": len(df_), "n_dates": df_["Date"].nunique(), "n_cities": df_["City"].nunique()})
    n_purged = int((model_df["Split"] == "purged").sum())
    print(f"{'purged':>10}: {len(split_info['purged_dates'])} dates, {n_purged:,} rows removed from train/validation "
          f"(last {PURGE_DAYS} dates of each) because their t+1..t+{max(HORIZONS)} labels reach into the next split")
    print(f"Rows: train {len(train):,} | validation {len(val):,} | test {len(test):,} | purged {n_purged:,} | "
          f"total kept {len(model_df):,}")
    print(f"Number of cities in the dataset: {model_df['City'].nunique()}")

    print("\nSeason distribution (rows):")
    ct = pd.crosstab(model_df["Split"], model_df["Season"]).reindex(["train", "validation", "test", "purged"])[SEASONS]
    print(ct.to_string())
    print("Season distribution (% within split):")
    print((ct.div(ct.sum(axis=1), axis=0) * 100).round(1).to_string())
    for name, df_ in parts.items():
        row = {r["split"]: r for r in summary_rows}[name]
        for s in SEASONS:
            row[f"n_{s}"] = int((df_["Season"] == s).sum())
        row["pct_high_risk_next3d"] = float(df_["HighRisk_AQI_gt200_next3d"].mean() * 100) if len(df_) and df_["HighRisk_AQI_gt200_next3d"].notna().any() else np.nan
        row["n_missing_tplus3"] = int(df_["Estimated_AQI_tplus3"].isna().sum())
        row["n_missing_tplus7"] = int(df_["Estimated_AQI_tplus7"].isna().sum())
    pd.DataFrame(summary_rows).to_csv(SPLIT_SUMMARY_CSV, index=False)

    print("\nHigh-risk target (AQI > 200 within next 3 days), % positive:")
    for name, df_ in parts.items():
        print(f"  {name}: {df_['HighRisk_AQI_gt200_next3d'].mean() * 100:.1f}% (n valid {int(df_['HighRisk_AQI_gt200_next3d'].notna().sum()):,})")
    print("Next-day AQI category distribution (%):")
    for name, df_ in parts.items():
        print(f"  {name}: " + ", ".join(f"{c} {100 * (df_['Estimated_AQI_Category_tplus1'] == c).mean():.1f}" for c in CATEGORIES))

    # ---- split integrity ------------------------------------------------------------------------------
    sets = {k: set(v["Date"].unique()) for k, v in parts.items()}
    overlap = (sets["train"] & sets["validation"]) | (sets["train"] & sets["test"]) | (sets["validation"] & sets["test"])
    assert not overlap, f"Dates appear in more than one split: {sorted(overlap)[:5]}"
    assert train["Date"].max() < val["Date"].min() and val["Date"].max() < test["Date"].min(), "Chronology violated"
    assert (np.diff(model_df["Date"].values.astype("datetime64[D]").astype(int)) >= 0).all()
    print("\nCONFIRMED: no date appears in more than one split; "
          f"train max {train['Date'].max().date()} < validation min {val['Date'].min().date()} and "
          f"validation max {val['Date'].max().date()} < test min {test['Date'].min().date()}.")
    # the split is global: every split must contain the same set of cities (all cities)
    cities_per = {k: v["City"].nunique() for k, v in parts.items()}
    print(f"Cities per split: {cities_per}")

    # ---- scaling (fit on train only) ----------------------------------------------------------------------
    banner("RobustScaler (median / IQR), fit on TRAIN only")
    scaled_cols = [c for c in feature_cols if meta[c][2]]
    unscaled_feature_cols = [c for c in feature_cols if not meta[c][2]]
    scaler = RobustScaler(with_centering=True, with_scaling=True, quantile_range=(25.0, 75.0))
    scaler.fit(train[scaled_cols])
    print(f"Scaled features: {len(scaled_cols)} | left unscaled (IDs, flags, calendar, cyclic, codes): {len(unscaled_feature_cols)}")
    zero_iqr = [c for c, s in zip(scaled_cols, scaler.scale_) if s == 1.0 and train[c].quantile(.75) == train[c].quantile(.25)]
    if zero_iqr:
        print(f"  {len(zero_iqr)} feature(s) have zero IQR in train (scale set to 1 by sklearn), e.g. {zero_iqr[:4]}")
    joblib.dump(scaler, SCALER_PKL)

    def scale(df_):
        out = df_.copy()
        out[scaled_cols] = scaler.transform(df_[scaled_cols])
        return out
    train_s, val_s, test_s = scale(train), scale(val), scale(test)
    assert np.isfinite(train_s[scaled_cols].to_numpy(float)).all() and np.isfinite(test_s[scaled_cols].to_numpy(float)).all()
    med = np.abs(train_s[scaled_cols].median().to_numpy()).max()
    assert med < 1e-6, f"Scaled train median not ~0 ({med})"
    for c in TARGET_COLS:                                   # targets must be untouched
        assert train_s[c].equals(train[c]) and test_s[c].equals(test[c])
    print("Scaler fitted on train only; train medians ~0; targets verified unscaled (original AQI units / labels).")

    # ---- column order and saving ---------------------------------------------------------------------------------
    meta_cols = ["Date", "City", "Region", "Season", "Split"]
    ordered = meta_cols + feature_cols + TARGET_COLS
    banner("Saving outputs")
    out_all = model_df[ordered]
    out_all.to_csv(FEATURES_CSV, index=False, date_format="%Y-%m-%d", float_format=CSV_FLOAT_FORMAT)
    for df_, path in ((train, TRAIN_CSV), (val, VAL_CSV), (test, TEST_CSV),
                      (train_s, TRAIN_S_CSV), (val_s, VAL_S_CSV), (test_s, TEST_S_CSV)):
        df_[ordered].to_csv(path, index=False, date_format="%Y-%m-%d", float_format=CSV_FLOAT_FORMAT)
        print(f"  saved {path}")

    fs = feature_summary(train, feature_cols, meta, scaled_cols, scaler, model_df)
    fs.to_csv(FEATURE_SUMMARY_CSV, index=False)
    city_map = (base.drop_duplicates("City")[["City", "Region", "Latitude", "Longitude"]].sort_values("City").reset_index(drop=True))
    city_map.insert(1, "City_ID", city_map.index)
    city_map["Region_ID"] = city_map["Region"].map({r: i for i, r in enumerate(sorted(city_map["Region"].dropna().unique()))})
    city_map.to_csv(CITY_MAP_CSV, index=False)
    groups = fs.groupby("Group")["Feature"].apply(list).to_dict()
    with open(FEATURE_COLS_JSON, "w") as f:
        json.dump({"meta_columns": meta_cols, "feature_columns": feature_cols, "scaled_feature_columns": scaled_cols,
                   "unscaled_feature_columns": unscaled_feature_cols, "target_regression": TARGET_REG,
                   "target_classification": TARGET_CLS, "feature_groups": groups,
                   "embedding_id_column": "City_ID", "n_cities": int(city_map.shape[0])}, f, indent=2)
    with open(SCALER_META_JSON, "w") as f:
        json.dump({"scaler_class": "sklearn.preprocessing.RobustScaler", "quantile_range": [25.0, 75.0],
                   "fit_on": "train split only", "scaled_columns": scaled_cols,
                   "center_": [float(x) for x in scaler.center_], "scale_": [float(x) for x in scaler.scale_],
                   "sklearn_version": sklearn.__version__}, f, indent=2)
    print(f"  saved {FEATURES_CSV}\n  saved {SCALER_PKL}\n  saved {FEATURE_SUMMARY_CSV}")

    make_heatmap(train, HEATMAP_PNG)
    make_topcorr(fs, TOPCORR_PNG)
    make_split_plot(model_df, split_info, SPLIT_PNG)

    # ---- integrity of inputs ------------------------------------------------------------------------------------------
    assert all(sha256_of(p) == h for p, h in hashes_in.items()), "An input file was modified!"

    # ---- manifest ---------------------------------------------------------------------------------------------------
    data_files = [FEATURES_CSV, TRAIN_CSV, VAL_CSV, TEST_CSV, TRAIN_S_CSV, VAL_S_CSV, TEST_S_CSV, SCALER_PKL,
                  SCALER_META_JSON, FEATURE_SUMMARY_CSV, FEATURE_COLS_JSON, CITY_MAP_CSV, SPLIT_SUMMARY_CSV,
                  HEATMAP_PNG, TOPCORR_PNG, SPLIT_PNG]
    manifest = {
        "part": 4, "total_parts": 7, "run_id": RUN_ID, "seed": SEED, "label": AQI_LABEL, "caveats": AQI_CAVEAT,
        "base_dir": BASE_DIR,
        "inputs": {"master": P1_MASTER, "part3_daily": P3_DAILY, "part3_summary": P3_SUMMARY,
                   "part3_manifest": P3_MANIFEST if p3_manifest else None, "part3_run_id": p3_manifest.get("run_id"),
                   "sha256": hashes_in, "rows_with_provisional_8hr": n_prov},
        "feature_design": {
            "n_features": len(feature_cols), "n_scaled": len(scaled_cols), "n_unscaled": len(unscaled_feature_cols),
            "lags": LAGS, "rolling_mean_windows": ROLL_MEAN_WINDOWS, "rolling_std_windows": ROLL_STD_WINDOWS,
            "rolling_series_shift_days": ROLL_SHIFT, "lag_variables": LAG_VARS,
            "row_date_semantics": "Row date t = last known day; current-day values, lags and t-1-ending rolling "
                                  "statistics are features; targets are t+1, t+3, t+7 and max(t+1..t+3) > 200.",
            "regime": regime_info, "leakage_tests": leak,
            "note": "No feature uses City x Season summary statistics from Part 3, because those were computed on the full period."},
        "targets": {"regression": TARGET_REG, "classification": TARGET_CLS,
                    "high_risk_definition": "1 if max Estimated_AQI over t+1..t+3 > 200 (all 3 future days required, else NaN)",
                    "optional_targets_missing_near_end": "tplus3/tplus7/high-risk are NaN in the last days of each city; filter before use"},
        "split": {**split_info, "fractions": SPLIT_FRACTIONS, "purge_days": PURGE_DAYS,
                  "rows": {k: int(len(v)) for k, v in parts.items()}, "rows_purged": n_purged,
                  "cities_per_split": cities_per, "no_date_overlap": True,
                  "method": "one global chronological cut over unique modelling dates; no per-city split; no random K-Fold"},
        "scaler": {"path": SCALER_PKL, "class": "RobustScaler", "fit_on": "train only",
                   "scaled_columns_file": SCALER_META_JSON, "targets_scaled": False, "sklearn_version": sklearn.__version__},
        "files_created": [{"path": p, "rel_path": os.path.relpath(p, BASE_DIR), "size_bytes": os.path.getsize(p)}
                          for p in data_files] +
                         [{"path": MANIFEST_JSON, "rel_path": os.path.relpath(MANIFEST_JSON, BASE_DIR), "size_bytes": None}],
        "next_part_should_load": {"train_scaled": TRAIN_S_CSV, "validation_scaled": VAL_S_CSV, "test_scaled": TEST_S_CSV,
                                  "train": TRAIN_CSV, "validation": VAL_CSV, "test": TEST_CSV, "scaler": SCALER_PKL,
                                  "feature_columns": FEATURE_COLS_JSON, "city_id_map": CITY_MAP_CSV},
    }
    with open(MANIFEST_JSON, "w") as f:
        json.dump(manifest, f, indent=2, default=str)

    banner("PART 4 COMPLETE - files created (exact paths for Part 5)")
    for p in data_files + [MANIFEST_JSON]:
        print(p)
    print(f"\nBASE_DIR: {BASE_DIR}")
    print("\n" + AQI_LABEL)
    print(AQI_CAVEAT)
    return model_df


if __name__ == "__main__" or IN_COLAB:
    model_table = main()


PART 4 - Features, targets, chronological split, RobustScaler
Six-pollutant estimated AQI using CPCB sub-index methodology.
NH3 and Pb are unavailable, so this is not the official CPCB AQI. O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct CPCB ground readings.
  Part 1 master vs Part 3 daily: rows not in both = 0, max abs difference = 0.00e+00
Base table: (146147, 22) | cities: 101 | 2022-08-04 to 2026-07-20
  Note: 101 rows carry provisional daily-mean O3/CO fallbacks (feature Provisional_8hr_flag_t).

Building past-only features and targets
Feature columns built: 172 (before optional regime)

Leakage tests
  target_tplus1_matches_next_day_aqi: True
  rolling_and_lag_values_match_manual_past_only_computation: True
  features_unchanged_when_future_observations_are_scrambled: True
  test_is_sensitive_targets_do_change: True
  perturbation_cutoff_date: 2024-12-19
K-Means regime labels: not available yet; feature skipped (add results/kmeans_regime_labels.csv to include

/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning:

invalid value encountered in divide

/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning:

invalid value encountered in divide



  saved /content/aqi_framework/data/processed/features_v1.csv
  saved /content/aqi_framework/models/standard_scaler.pkl
  saved /content/aqi_framework/results/feature_summary.csv

PART 4 COMPLETE - files created (exact paths for Part 5)
/content/aqi_framework/data/processed/features_v1.csv
/content/aqi_framework/data/processed/train.csv
/content/aqi_framework/data/processed/validation.csv
/content/aqi_framework/data/processed/test.csv
/content/aqi_framework/data/processed/train_scaled.csv
/content/aqi_framework/data/processed/validation_scaled.csv
/content/aqi_framework/data/processed/test_scaled.csv
/content/aqi_framework/models/standard_scaler.pkl
/content/aqi_framework/models/scaler_feature_list.json
/content/aqi_framework/results/feature_summary.csv
/content/aqi_framework/results/feature_columns.json
/content/aqi_framework/results/city_id_map.csv
/content/aqi_framework/results/split_summary.csv
/content/aqi_framework/figures/feature_correlation_heatmap.png
/content/aqi_framework/fi

In [7]:
# =============================================================================
# City- and Season-Aware Multi-Pollutant Air Quality Assessment and Forecasting
# Framework for Indian Cities Using Deep Learning
#
# PART 5 of 7: persistence baselines and classical ML models
#   * Regression      -> Estimated_AQI_tplus1        (Ridge, Random Forest, XGBoost)
#   * Classification  -> Estimated_AQI_Category_tplus1 (Logistic, Random Forest, XGBoost)
#   * High-risk       -> AQI > 200 within next 3 days (Random Forest, XGBoost)
#
# Continues directly from Part 4 (same BASE_DIR). The Part 4 split files and
# scaler are only read, never modified (verified by hash at the end).
# No deep learning here.
#
# PROTOCOL: every model is fit on TRAIN only; hyper-parameters / thresholds are
# chosen on VALIDATION only; each model's TEST predictions are generated exactly
# once, after all selection is finished. Permutation importance and SHAP use
# validation data, so the test set is never re-scored.
#
# LABEL : "Six-pollutant estimated AQI using CPCB sub-index methodology."
# CAVEAT: NH3 and Pb are unavailable, so this is NOT the official CPCB AQI.
#         O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct
#         CPCB ground readings (provisional fallback rows remain flagged).
#
# Colab setup (once):  !pip -q install xgboost shap
# Profiles: AQI_PROFILE=standard (default, full grids) | fast (smoke test only)
# =============================================================================

# ----------------------------------------------------------------------------
# 0. Imports, configuration, reproducibility, link to Part 4
# ----------------------------------------------------------------------------
import os
import sys
import json
import time
import random
import hashlib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score,
                             balanced_accuracy_score, confusion_matrix, precision_score, recall_score,
                             roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
                             precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    xgb, HAS_XGB = None, False
try:
    import shap
    HAS_SHAP = True
except Exception:
    shap, HAS_SHAP = None, False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.RandomState(SEED)

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
BASE_DIR = "/content/aqi_framework" if IN_COLAB else os.path.abspath("./aqi_framework")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PROFILE = os.environ.get("AQI_PROFILE", "standard")

PROC_DIR = os.path.join(BASE_DIR, "data", "processed")
RES_DIR = os.path.join(BASE_DIR, "results")
FIG_DIR = os.path.join(BASE_DIR, "figures")
P5_FIG_DIR = os.path.join(FIG_DIR, "part5_models")
MODEL_DIR = os.path.join(BASE_DIR, "models")
CLASSICAL_DIR = os.path.join(MODEL_DIR, "classical")
for d in (RES_DIR, FIG_DIR, P5_FIG_DIR, MODEL_DIR, CLASSICAL_DIR):
    os.makedirs(d, exist_ok=True)

# ---- Part 4 inputs ---------------------------------------------------------------------
TRAIN_CSV = os.path.join(PROC_DIR, "train.csv")
VAL_CSV = os.path.join(PROC_DIR, "validation.csv")
TEST_CSV = os.path.join(PROC_DIR, "test.csv")
TRAIN_S_CSV = os.path.join(PROC_DIR, "train_scaled.csv")
VAL_S_CSV = os.path.join(PROC_DIR, "validation_scaled.csv")
TEST_S_CSV = os.path.join(PROC_DIR, "test_scaled.csv")
SCALER_PKL = os.path.join(MODEL_DIR, "standard_scaler.pkl")
FEATURE_COLS_JSON = os.path.join(RES_DIR, "feature_columns.json")
P4_MANIFEST = os.path.join(RES_DIR, "part4_handoff_manifest.json")

# ---- Part 5 outputs (as specified + supporting) ---------------------------------------------
METRICS_CSV = os.path.join(RES_DIR, "model_metrics.csv")
CITY_CSV = os.path.join(RES_DIR, "city_wise_metrics.csv")
SEASON_CSV = os.path.join(RES_DIR, "season_wise_metrics.csv")
CATEGORY_CSV = os.path.join(RES_DIR, "aqi_category_wise_metrics.csv")
PVA_CSV = os.path.join(RES_DIR, "predicted_vs_actual.csv")
IMPORTANCE_CSV = os.path.join(RES_DIR, "feature_importance.csv")
HR_PRED_CSV = os.path.join(RES_DIR, "high_risk_predictions.csv")
COMPARISON_CSV = os.path.join(RES_DIR, "model_comparison_table.csv")
SELECTION_CSV = os.path.join(RES_DIR, "model_selection_log.csv")
CONFUSION_CSV = os.path.join(RES_DIR, "confusion_matrices.csv")
MODEL_INDEX_JSON = os.path.join(CLASSICAL_DIR, "classical_models_index.json")
MANIFEST_JSON = os.path.join(RES_DIR, "part5_handoff_manifest.json")
FIG_COMPARISON = os.path.join(FIG_DIR, "model_comparison_plots.png")
FIG_CONFUSION = os.path.join(FIG_DIR, "confusion_matrix_plots.png")

# ---- labels ----------------------------------------------------------------------------------
AQI_LABEL = "Six-pollutant estimated AQI using CPCB sub-index methodology."
AQI_CAVEAT = ("NH3 and Pb are unavailable, so this is not the official CPCB AQI. "
              "O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct CPCB ground readings.")
FOOTER = ("Six-pollutant estimated AQI (CPCB sub-index methodology); not official CPCB AQI (NH3, Pb unavailable). "
          "Model results describe predictive performance only; they do not establish causes.")

# ---- schema ---------------------------------------------------------------------------------------
CATEGORIES = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
SEASONS = ["Winter", "Summer", "Monsoon", "Post-monsoon"]
POLLUTANT_ORDER = ["PM2_5", "PM10", "NO2", "SO2", "O3", "CO"]
LAG_VARS = ["PM2_5", "PM10", "CO", "NO2", "SO2", "O3", "O3_8hr_max", "CO_8hr_max_mg_m3", "Temperature_C",
            "Humidity_pct", "WindSpeed_kmh", "Precipitation_mm", "Estimated_AQI"]
# nominal integer codes with no numeric meaning -> excluded from the shared feature set
EXCLUDE_NOMINAL = ["City_ID", "Region_ID", "Season_code", "Dominant_Pollutant_t_code", "Regime_KMeans"]
TARGET_REG, TARGET_CLS_CODE, TARGET_CLS_NAME, TARGET_HR = (
    "Estimated_AQI_tplus1", "Estimated_AQI_Category_tplus1_code", "Estimated_AQI_Category_tplus1",
    "HighRisk_AQI_gt200_next3d")
MODEL_COLORS = {"Persistence": "#999999", "Ridge": "#4C72B0", "LogisticRegression": "#4C72B0",
                "RandomForest": "#55A868", "XGBoost": "#C44E52"}

# ---- run profiles ----------------------------------------------------------------------------------
PROFILES = {
    "standard": dict(
        ridge_alphas=[0.1, 1.0, 10.0, 100.0, 1000.0, 3000.0],
        lr_grid=[(C, cw) for C in (0.01, 0.1, 1.0) for cw in (None, "balanced")],
        rf_trees=150, rf_max_samples=0.5,
        rf_grid=[dict(max_depth=12, min_samples_leaf=20, max_features=0.25),
                 dict(max_depth=18, min_samples_leaf=10, max_features=0.25)],
        rf_class_weights=[None, "balanced_subsample"],
        xgb_lr=0.05, xgb_rounds=1500, xgb_early=50,
        xgb_reg_depths=[4, 6], xgb_cls_grid=[(4, False), (6, False), (5, True)],
        xgb_hr_grid=[(4, "none"), (6, "none"), (5, "pos_weight")],
        perm_rows=3000, perm_repeats=3, shap_rows_xgb=1000, shap_rows_rf=400, train_subsample=None),
    "fast": dict(                                   # smoke test only - not for reporting results
        ridge_alphas=[10.0, 1000.0], lr_grid=[(0.1, None)],
        rf_trees=30, rf_max_samples=0.3, rf_grid=[dict(max_depth=10, min_samples_leaf=20, max_features=0.25)],
        rf_class_weights=[None], xgb_lr=0.2, xgb_rounds=120, xgb_early=15,
        xgb_reg_depths=[4], xgb_cls_grid=[(4, False)], xgb_hr_grid=[(4, "pos_weight")],
        perm_rows=1000, perm_repeats=2, shap_rows_xgb=300, shap_rows_rf=150, train_subsample=20000),
}
CFG = PROFILES[PROFILE]


def banner(text):
    print("\n" + "=" * 78 + f"\n{text}\n" + "=" * 78)


def sha256_of(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


# ----------------------------------------------------------------------------
# 1. Load Part 4 outputs (read-only) and verify continuity
# ----------------------------------------------------------------------------
def load_part4():
    needed = [TRAIN_CSV, VAL_CSV, TEST_CSV, TRAIN_S_CSV, VAL_S_CSV, TEST_S_CSV, SCALER_PKL, FEATURE_COLS_JSON]
    for p in needed:
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Missing Part 4 output: {p}\nRun Parts 1-4 first (same BASE_DIR={BASE_DIR}). "
                                    "In Colab, a restarted runtime loses /content - re-run the earlier parts.")
    with open(FEATURE_COLS_JSON) as f:
        fc = json.load(f)
    p4 = json.load(open(P4_MANIFEST)) if os.path.isfile(P4_MANIFEST) else {}
    frames = {}
    for key, path in (("train", TRAIN_CSV), ("validation", VAL_CSV), ("test", TEST_CSV),
                      ("train_s", TRAIN_S_CSV), ("validation_s", VAL_S_CSV), ("test_s", TEST_S_CSV)):
        frames[key] = pd.read_csv(path, parse_dates=["Date"])
    for k in ("train", "validation", "test"):                       # scaled and unscaled rows must align exactly
        a, b = frames[k], frames[k + "_s"]
        if not (a[["Date", "City"]].reset_index(drop=True).equals(b[["Date", "City"]].reset_index(drop=True))):
            raise ValueError(f"{k}: scaled and unscaled files are not row-aligned.")
    if p4:
        rows = p4.get("split", {}).get("rows", {})
        for k in ("train", "validation", "test"):
            if rows.get(k) not in (None, len(frames[k])):
                raise ValueError(f"{k} rows {len(frames[k])} differ from Part 4 manifest {rows.get(k)}.")
    tr, va, te = frames["train"], frames["validation"], frames["test"]
    assert tr["Date"].max() < va["Date"].min() and va["Date"].max() < te["Date"].min(), "Split chronology violated"
    scaler = joblib.load(SCALER_PKL)
    cols = fc["scaled_feature_columns"]
    s = tr.sample(min(2000, len(tr)), random_state=SEED)
    diff = float(np.abs(scaler.transform(s[cols]) - frames["train_s"].loc[s.index, cols].to_numpy()).max())
    print(f"  Scaled files vs scaler.transform(unscaled): max abs difference {diff:.2e} "
          "(CSV rounding only)")
    if diff > 1e-2:
        print("  ! Scaled files do not match the saved scaler closely; check Part 4 outputs.")
    return frames, fc, p4, scaler, diff


def build_design(frames, fc):
    """Shared feature set for ALL models: Part 4 features minus nominal integer codes, plus a one-hot of the
    day-t dominant pollutant (and the K-Means regime, if Part 4 included one)."""
    base = [c for c in fc["feature_columns"] if c not in EXCLUDE_NOMINAL]
    regime_codes = (sorted(frames["train"]["Regime_KMeans"].dropna().unique())
                    if "Regime_KMeans" in frames["train"].columns else [])
    names = list(base) + [f"Dominant_t_{p}" for p in POLLUTANT_ORDER] + [f"Regime_{int(r)}" for r in regime_codes]

    def design(df):
        X = df[base].copy()
        for i, p in enumerate(POLLUTANT_ORDER):
            X[f"Dominant_t_{p}"] = (df["Dominant_Pollutant_t_code"] == i).astype(float)
        for r in regime_codes:
            X[f"Regime_{int(r)}"] = (df["Regime_KMeans"] == r).astype(float)
        return X[names].to_numpy(dtype=np.float32)
    return names, design


def category_code(aqi):
    return np.searchsorted([50, 100, 200, 300, 400], np.asarray(aqi, dtype=float), side="left")


# ----------------------------------------------------------------------------
# 2. Metrics
# ----------------------------------------------------------------------------
def reg_metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    out = {"N": len(y), "MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p))),
           "Bias": float(np.mean(p - y))}
    out["R2"] = r2_score(y, p) if len(y) > 1 and np.var(y) > 0 else np.nan
    return out


def cls_metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return {"N": len(y), "Accuracy": accuracy_score(y, p), "Macro_F1": f1_score(y, p, average="macro", zero_division=0),
            "Balanced_Accuracy": balanced_accuracy_score(y, p)}


def hr_metrics(y, proba, pred):
    y, proba, pred = np.asarray(y).astype(int), np.asarray(proba, float), np.asarray(pred).astype(int)
    out = {"N": len(y), "Positive_Rate": float(y.mean()),
           "Precision": precision_score(y, pred, zero_division=0), "Recall": recall_score(y, pred, zero_division=0),
           "F1": f1_score(y, pred, zero_division=0)}
    if len(np.unique(y)) == 2:
        out["ROC_AUC"] = roc_auc_score(y, proba)
        out["PR_AUC"] = average_precision_score(y, proba)
    else:
        out["ROC_AUC"], out["PR_AUC"] = np.nan, np.nan
    return out


def best_f1_threshold(y, proba):
    grid = np.linspace(0.05, 0.95, 91)
    f1s = [f1_score(y, (proba >= t).astype(int), zero_division=0) for t in grid]
    return float(grid[int(np.argmax(f1s))])


# ----------------------------------------------------------------------------
# 3. Model fitting (train only) and selection (validation only)
# ----------------------------------------------------------------------------
SEL_LOG = []


def log_sel(task, model, params, metric_name, value, secs):
    SEL_LOG.append({"Task": task, "Model": model, "Params": json.dumps(params, default=str),
                    "Validation_Metric": metric_name, "Validation_Value": value, "Fit_Seconds": round(secs, 1)})


def fit_xgb(kind, params, Xtr, ytr, Xva, yva, sample_weight=None):
    est = xgb.XGBRegressor if kind == "reg" else xgb.XGBClassifier
    common = dict(n_estimators=CFG["xgb_rounds"], learning_rate=CFG["xgb_lr"], subsample=0.8, colsample_bytree=0.7,
                  min_child_weight=5, reg_lambda=1.0, tree_method="hist", n_jobs=-1, random_state=SEED, verbosity=0)
    common.update(params)
    fit_kw = dict(eval_set=[(Xva, yva)], verbose=False)
    if sample_weight is not None:
        fit_kw["sample_weight"] = sample_weight
    try:                                                   # xgboost >= 1.6: early stopping set in the constructor
        m = est(early_stopping_rounds=CFG["xgb_early"], **common)
        m.fit(Xtr, ytr, **fit_kw)
    except TypeError:                                      # older versions: pass to fit()
        m = est(**common)
        m.fit(Xtr, ytr, early_stopping_rounds=CFG["xgb_early"], **fit_kw)
    return m


def tune(task, name, candidates, fit_fn, val_fn, higher_better):
    """candidates: list of param dicts. fit_fn(params)->model (train only). val_fn(model)->score (validation only)."""
    best = None
    for params in candidates:
        t0 = time.time()
        m = fit_fn(params)
        score = float(val_fn(m))
        secs = time.time() - t0
        info = dict(params)
        bi = getattr(m, "best_iteration", None)
        if bi is not None:
            info["best_iteration"] = int(bi)
        log_sel(task, name, info, "val_" + ("higher_better" if higher_better else "lower_better"), score, secs)
        print(f"    {task}/{name} {info} -> validation {score:.4f} ({secs:.0f}s)")
        better = best is None or (score > best[0] if higher_better else score < best[0])
        if better:
            best = (score, m, info, secs)
    return best[1], best[2], best[0], best[3]


# ----------------------------------------------------------------------------
# 4. Permutation importance (validation data; joint permutation for groups)
# ----------------------------------------------------------------------------
def permutation_importance_custom(predict_fn, X, y, metric_fn, higher_better, groups, n_repeats, rng):
    base = metric_fn(y, predict_fn(X))
    rows = []
    for gname, idx in groups.items():
        vals = []
        for _ in range(n_repeats):
            Xp = X.copy()
            perm = rng.permutation(len(X))
            Xp[:, idx] = X[perm][:, idx]
            s = metric_fn(y, predict_fn(Xp))
            vals.append((base - s) if higher_better else (s - base))
        rows.append((gname, float(np.mean(vals)), float(np.std(vals))))
    return rows, base


def make_groups(names, fc):
    feat_idx = {n: i for i, n in enumerate(names)}
    singles = {n: [i] for n, i in feat_idx.items()}
    fam = {}
    for v in LAG_VARS:
        cols = [n for n in names if n == v or n.startswith(v + "_lag") or n.startswith(v + "_roll")]
        if cols:
            fam[f"{v} (current+lags+rolling)"] = [feat_idx[c] for c in cols]
    g = fc.get("feature_groups", {})
    extra = {"Calendar & season": g.get("calendar", []) + g.get("calendar_cyclic", []),
             "Interactions": g.get("interaction", []),
             "Location (Lat/Lon)": [c for c in ("Latitude", "Longitude")],
             "Dominant pollutant at t": [n for n in names if n.startswith("Dominant_t_")],
             "Provisional-data flag": ["Provisional_8hr_flag_t"]}
    for k, cols in extra.items():
        idx = [feat_idx[c] for c in cols if c in feat_idx]
        if idx:
            fam[k] = idx
    types = {}
    for k, label in (("current", "Type: current-day values"), ("lag", "Type: lags"),
                     ("rolling_mean", "Type: rolling means"), ("rolling_std", "Type: rolling std")):
        idx = [feat_idx[c] for c in g.get(k, []) if c in feat_idx]
        if idx:
            types[label] = idx
    return singles, fam, types


# ----------------------------------------------------------------------------
# 5. Figures
# ----------------------------------------------------------------------------
FIG_LOG = []


def save_fig(fig, path, title=None, dpi=130):
    if title:
        fig.suptitle(title, fontsize=13, y=0.998)
    fig.text(0.5, 0.002, FOOTER, ha="center", va="bottom", fontsize=7, color="dimgray")
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return path


def run_plot(fn, ctx):
    try:
        paths = fn(ctx)
        paths = [paths] if isinstance(paths, str) else (paths or [])
        FIG_LOG.extend(paths)
        print(f"  [ok] {fn.__name__}")
    except Exception as ex:
        print(f"  [FAILED] {fn.__name__}: {type(ex).__name__}: {ex}")
        FIG_LOG.append(f"FAILED:{fn.__name__}:{type(ex).__name__}")


def _bars(ax, df, metric, models, title):
    x = np.arange(len(models))
    for k, (split, color) in enumerate((("validation", "#9DB4D8"), ("test", "#2F4B7C"))):
        vals = [df[(df["Model"] == m) & (df["Split"] == split)][metric].mean() for m in models]
        ax.bar(x + (k - 0.5) * 0.38, vals, width=0.38, color=color, label=split)
    ax.set_xticks(x, models, rotation=20, ha="right")
    ax.set_title(title)
    ax.legend(fontsize=8)


def plot_model_comparison(ctx):
    m = ctx["metrics"]
    fig, ax = plt.subplots(3, 3, figsize=(17, 13))
    r = m[m["Task"] == "regression"]
    rm = list(dict.fromkeys(r["Model"]))
    for a, met in zip(ax[0], ["MAE", "RMSE", "R2"]):
        _bars(a, r, met, rm, f"Regression: {met}" + (" (lower is better)" if met != "R2" else " (higher is better)"))
    c = m[m["Task"] == "classification"]
    cm = list(dict.fromkeys(c["Model"]))
    for a, met in zip(ax[1], ["Macro_F1", "Balanced_Accuracy", "Accuracy"]):
        _bars(a, c, met, cm, f"Category classification: {met}")
    h = m[(m["Task"] == "highrisk") & (m["Threshold_Rule"].isin(["val_F1_optimal", "binary_persistence"]))]
    hm = list(dict.fromkeys(h["Model"]))
    for a, met in zip(ax[2], ["PR_AUC", "ROC_AUC", "F1"]):
        _bars(a, h, met, hm, f"High-risk (AQI>200 within 3 days): {met}")
    return save_fig(fig, FIG_COMPARISON, "Model comparison (validation vs test)")


def plot_confusions(ctx):
    cms = ctx["confusions"]                                  # (task, model) -> 6x6 count matrix on test
    items = [(k, v) for k, v in cms.items() if k[0] == "classification"]
    n = len(items)
    fig, ax = plt.subplots(2, 2, figsize=(15, 13))
    for a, ((task, model), cm) in zip(ax.ravel(), items):
        rown = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        im = a.imshow(rown, cmap="Blues", vmin=0, vmax=1)
        a.set_xticks(range(6), CATEGORIES, rotation=40, ha="right", fontsize=8)
        a.set_yticks(range(6), CATEGORIES, fontsize=8)
        for i in range(6):
            for j in range(6):
                a.text(j, i, f"{cm[i, j]}\n{100 * rown[i, j]:.0f}%", ha="center", va="center", fontsize=7,
                       color="white" if rown[i, j] > 0.5 else "black")
        a.set_xlabel("Predicted next-day category")
        a.set_ylabel("Actual next-day category")
        a.set_title(f"{model} (test; cells = count and row %)")
    for a in ax.ravel()[n:]:
        a.axis("off")
    p1 = save_fig(fig, FIG_CONFUSION, "Confusion matrices: next-day AQI category (test)")

    hr = [(k, v) for k, v in cms.items() if k[0] == "highrisk"]
    fig, ax = plt.subplots(1, max(len(hr), 1), figsize=(5.2 * max(len(hr), 1), 4.8))
    ax = np.atleast_1d(ax)
    for a, ((task, model), cm) in zip(ax, hr):
        a.imshow(cm / np.maximum(cm.sum(axis=1, keepdims=True), 1), cmap="Oranges", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                a.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=11)
        a.set_xticks([0, 1], ["No", "Yes"])
        a.set_yticks([0, 1], ["No", "Yes"])
        a.set_xlabel("Predicted high risk")
        a.set_ylabel("Actual high risk")
        a.set_title(model)
    p2 = save_fig(fig, os.path.join(P5_FIG_DIR, "confusion_matrices_highrisk.png"),
                  "High-risk confusion matrices (test; validation-optimised threshold)")
    return [p1, p2]


def plot_pred_vs_actual(ctx):
    pva = ctx["pva_test"]
    models = [c[len("Pred_AQI_"):] for c in pva.columns if c.startswith("Pred_AQI_")]
    s = pva.sample(min(8000, len(pva)), random_state=SEED)
    fig, ax = plt.subplots(2, 2, figsize=(14, 13))
    lim = float(max(pva["Actual_AQI_tplus1"].max(), 1))
    for a, m in zip(ax.ravel(), models):
        a.scatter(s["Actual_AQI_tplus1"], s[f"Pred_AQI_{m}"], s=4, alpha=0.25, color=MODEL_COLORS.get(m, "#555"))
        a.plot([0, lim], [0, lim], "k--", lw=0.8)
        mt = reg_metrics(pva["Actual_AQI_tplus1"], pva[f"Pred_AQI_{m}"])
        a.set_title(f"{m}: MAE {mt['MAE']:.1f}, RMSE {mt['RMSE']:.1f}, R2 {mt['R2']:.3f}")
        a.set_xlabel("Actual AQI (t+1)")
        a.set_ylabel("Predicted AQI (t+1)")
    for a in ax.ravel()[len(models):]:
        a.axis("off")
    p1 = save_fig(fig, os.path.join(P5_FIG_DIR, "predicted_vs_actual_scatter.png"),
                  "Predicted vs actual next-day estimated AQI (test, random sample of 8,000 city-days)")

    best = ctx["best_reg"]
    cities = ctx["rep_cities"]
    fig, ax = plt.subplots((len(cities) + 1) // 2, 2, figsize=(16, 3.3 * ((len(cities) + 1) // 2)), sharex=True)
    for a, c in zip(np.atleast_1d(ax).ravel(), cities):
        g = pva[pva["City"] == c].sort_values("Date")
        a.plot(g["Date"], g["Actual_AQI_tplus1"], color="k", lw=1, label="Actual")
        a.plot(g["Date"], g["Pred_AQI_Persistence"], color="#999", lw=0.8, label="Persistence")
        a.plot(g["Date"], g[f"Pred_AQI_{best}"], color=MODEL_COLORS.get(best, "#C44E52"), lw=1, label=best)
        a.set_title(c)
        a.set_ylabel("AQI (t+1)")
    np.atleast_1d(ax).ravel()[0].legend(fontsize=8)
    p2 = save_fig(fig, os.path.join(P5_FIG_DIR, "predicted_vs_actual_timeseries_cities.png"),
                  f"Test-period next-day AQI: actual vs persistence vs {best}")
    return [p1, p2]


def plot_highrisk_curves(ctx):
    hr = ctx["hr_test"]
    y = hr["y"]
    fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
    for m, proba in hr["proba"].items():
        fpr, tpr, _ = roc_curve(y, proba)
        pr, rc, _ = precision_recall_curve(y, proba)
        ax[0].plot(fpr, tpr, color=MODEL_COLORS.get(m), label=f"{m} (AUC {roc_auc_score(y, proba):.3f})")
        ax[1].plot(rc, pr, color=MODEL_COLORS.get(m), label=f"{m} (AP {average_precision_score(y, proba):.3f})")
    ax[0].plot([0, 1], [0, 1], "k--", lw=0.7)
    ax[0].set_xlabel("False-positive rate")
    ax[0].set_ylabel("True-positive rate")
    ax[0].set_title("ROC (test)")
    ax[1].axhline(y.mean(), color="k", ls="--", lw=0.7, label=f"Base rate {y.mean():.2f}")
    ax[1].set_xlabel("Recall")
    ax[1].set_ylabel("Precision")
    ax[1].set_title("Precision-recall (test)")
    for a in ax:
        a.legend()
    return save_fig(fig, os.path.join(P5_FIG_DIR, "highrisk_roc_pr_curves.png"), "High-risk classifier curves (test)")


def plot_group_metrics(ctx):
    out = []
    sw, cw, cat = ctx["season_df"], ctx["city_df"], ctx["category_df"]
    sw = sw[sw["Split"] == "test"]
    cw = cw[cw["Split"] == "test"]
    cat = cat[cat["Split"] == "test"]
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    for a, (task, met) in zip(ax, (("regression", "RMSE"), ("classification", "Macro_F1"), ("highrisk", "PR_AUC"))):
        d = sw[(sw["Task"] == task) & (sw["Model"] != "Persistence-n/a")]
        if task == "highrisk":
            d = d[d["Threshold_Rule"].isin(["val_F1_optimal", "binary_persistence"])]
        models = list(dict.fromkeys(d["Model"]))
        x = np.arange(len(SEASONS))
        for k, m in enumerate(models):
            v = [d[(d["Model"] == m) & (d["Season"] == s)][met].mean() for s in SEASONS]
            a.bar(x + (k - (len(models) - 1) / 2) * 0.8 / len(models), v, width=0.8 / len(models), label=m,
                  color=MODEL_COLORS.get(m))
        a.set_xticks(x, SEASONS)
        a.set_title(f"{task}: {met} by season (test)")
        a.legend(fontsize=7)
    out.append(save_fig(fig, os.path.join(P5_FIG_DIR, "metrics_by_season.png"), "Test performance by season"))

    best = ctx["best_reg"]
    d = cw[(cw["Task"] == "regression") & (cw["Model"] == best)].sort_values("RMSE", ascending=False)
    fig, ax = plt.subplots(1, 2, figsize=(15, 8))
    top = d.head(20).iloc[::-1]
    bot = d.tail(20)
    ax[0].barh(top["City"], top["RMSE"], color="#C44E52")
    ax[0].set_title(f"{best}: 20 cities with highest test RMSE")
    ax[1].barh(bot["City"], bot["RMSE"], color="#4C72B0")
    ax[1].set_title(f"{best}: 20 cities with lowest test RMSE")
    for a in ax:
        a.set_xlabel("RMSE (AQI units)")
    out.append(save_fig(fig, os.path.join(P5_FIG_DIR, "regression_rmse_by_city.png"), "City-wise test error"))

    d = cat[(cat["Task"] == "regression")]
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    models = list(dict.fromkeys(d["Model"]))
    x = np.arange(len(CATEGORIES))
    for k, m in enumerate(models):
        g = d[d["Model"] == m].set_index("AQI_Category").reindex(CATEGORIES)
        ax[0].bar(x + (k - (len(models) - 1) / 2) * 0.8 / len(models), g["MAE"].values, width=0.8 / len(models),
                  label=m, color=MODEL_COLORS.get(m))
    ax[0].set_xticks(x, CATEGORIES, rotation=20)
    ax[0].set_title("Regression MAE by actual next-day category (test)")
    ax[0].legend(fontsize=8)
    d2 = cat[(cat["Task"] == "classification")]
    models = list(dict.fromkeys(d2["Model"]))
    for k, m in enumerate(models):
        g = d2[d2["Model"] == m].set_index("AQI_Category").reindex(CATEGORIES)
        ax[1].bar(x + (k - (len(models) - 1) / 2) * 0.8 / len(models), g["Recall"].values, width=0.8 / len(models),
                  label=m, color=MODEL_COLORS.get(m))
    ax[1].set_xticks(x, CATEGORIES, rotation=20)
    ax[1].set_title("Classification recall by actual next-day category (test)")
    ax[1].legend(fontsize=8)
    out.append(save_fig(fig, os.path.join(P5_FIG_DIR, "metrics_by_aqi_category.png"), "Test performance by AQI category"))
    return out


def plot_importance(ctx):
    imp = ctx["importance"]
    out = []
    pm = imp[(imp["Method"] == "permutation") & (imp["Level"] == "feature")]
    keys = list(dict.fromkeys(zip(pm["Task"], pm["Model"])))
    for task, model in keys:
        d = pm[(pm["Task"] == task) & (pm["Model"] == model)].nlargest(20, "Importance_Mean").iloc[::-1]
        fig, ax = plt.subplots(figsize=(9, 7))
        ax.barh(d["Feature"], d["Importance_Mean"], xerr=d["Importance_Std"], color=MODEL_COLORS.get(model, "#555"))
        ax.set_xlabel(f"Permutation importance (drop in {d['Metric'].iloc[0]} on validation sample)")
        ax.set_title(f"{task}: {model} - top 20 individual features")
        out.append(save_fig(fig, os.path.join(P5_FIG_DIR, f"perm_importance_{task}_{model}.png")))
    gm = imp[(imp["Method"] == "permutation") & (imp["Level"].isin(["variable_family", "feature_type"]))]
    keys = list(dict.fromkeys(zip(gm["Task"], gm["Model"])))
    fig, ax = plt.subplots(len(keys), 1, figsize=(10, 4.3 * len(keys)))
    for a, (task, model) in zip(np.atleast_1d(ax), keys):
        d = gm[(gm["Task"] == task) & (gm["Model"] == model)].nlargest(14, "Importance_Mean").iloc[::-1]
        a.barh(d["Feature"], d["Importance_Mean"], xerr=d["Importance_Std"], color=MODEL_COLORS.get(model, "#555"))
        a.set_title(f"{task}: {model} - grouped permutation importance")
    out.append(save_fig(fig, os.path.join(P5_FIG_DIR, "perm_importance_grouped_all_models.png"),
                        "Grouped permutation importance (validation sample; correlated features permuted together)"))
    return out


# ----------------------------------------------------------------------------
# 6. SHAP (guarded - skipped if the package is missing)
# ----------------------------------------------------------------------------
def _shap_class_view(sv, cls_idx):
    if isinstance(sv, list):
        return np.asarray(sv[cls_idx])
    sv = np.asarray(sv)
    return sv[:, :, cls_idx] if sv.ndim == 3 else sv


def _shap_mean_abs(sv):
    if isinstance(sv, list):
        return np.mean([np.abs(np.asarray(s)).mean(axis=0) for s in sv], axis=0)
    sv = np.asarray(sv)
    return np.abs(sv).mean(axis=(0, 2)) if sv.ndim == 3 else np.abs(sv).mean(axis=0)


def run_shap(models, Xva_tree, names, classes_present):
    """Returns (figure paths, importance rows)."""
    paths, rows = [], []
    if not HAS_SHAP:
        print("  SHAP not installed (pip install shap) - SHAP plots skipped; permutation importance is still reported.")
        return paths, rows
    items = [("regression", "XGBoost", CFG["shap_rows_xgb"]), ("regression", "RandomForest", CFG["shap_rows_rf"]),
             ("classification", "XGBoost", CFG["shap_rows_xgb"]), ("highrisk", "XGBoost", CFG["shap_rows_xgb"]),
             ("highrisk", "RandomForest", CFG["shap_rows_rf"])]
    for task, model, n in items:
        m = models.get((task, model))
        if m is None:
            continue
        try:
            idx = RNG.choice(len(Xva_tree), size=min(n, len(Xva_tree)), replace=False)
            Xs = Xva_tree[idx]
            sv = shap.TreeExplainer(m).shap_values(Xs)
            mean_abs = _shap_mean_abs(sv)
            for f, v in zip(names, mean_abs):
                rows.append({"Task": task, "Model": model, "Level": "feature", "Method": "SHAP_mean_abs", "Feature": f,
                             "Importance_Mean": float(v), "Importance_Std": np.nan, "Metric": "mean |SHAP| (validation sample)"})
            if task == "classification":
                top = np.argsort(mean_abs)[::-1][:20][::-1]
                fig, ax = plt.subplots(figsize=(9, 7))
                ax.barh([names[i] for i in top], mean_abs[top], color=MODEL_COLORS.get(model))
                ax.set_xlabel("Mean |SHAP| averaged over classes")
                ax.set_title(f"SHAP: {model} category classifier - top 20 features")
                paths.append(save_fig(fig, os.path.join(P5_FIG_DIR, f"shap_{task}_{model}_bar.png")))
                cls_pos = list(classes_present).index(3) if 3 in list(classes_present) else len(classes_present) - 1
                plt.figure(figsize=(9, 7))
                shap.summary_plot(_shap_class_view(sv, cls_pos), Xs, feature_names=names, show=False, max_display=20)
                plt.title(f"SHAP: {model}, class '{CATEGORIES[int(classes_present[cls_pos])]}'")
                paths.append(save_fig(plt.gcf(), os.path.join(P5_FIG_DIR, f"shap_{task}_{model}_class_{CATEGORIES[int(classes_present[cls_pos])]}.png")))
            else:
                plt.figure(figsize=(9, 7))
                shap.summary_plot(_shap_class_view(sv, 1) if task == "highrisk" else np.asarray(sv), Xs,
                                  feature_names=names, show=False, max_display=20)
                plt.title(f"SHAP: {task} / {model}")
                paths.append(save_fig(plt.gcf(), os.path.join(P5_FIG_DIR, f"shap_{task}_{model}.png")))
            print(f"  [ok] SHAP {task}/{model}")
        except Exception as ex:
            print(f"  [FAILED] SHAP {task}/{model}: {type(ex).__name__}: {ex}")
            plt.close("all")
    return paths, rows


# ----------------------------------------------------------------------------
# 7. Main pipeline
# ----------------------------------------------------------------------------
def main():
    t_start = time.time()
    banner(f"PART 5 - Persistence baselines and classical ML models (profile: {PROFILE})")
    print(AQI_LABEL)
    print(AQI_CAVEAT)
    print(f"sklearn {sklearn.__version__} | xgboost {xgb.__version__ if HAS_XGB else 'NOT INSTALLED'} | "
          f"shap {shap.__version__ if HAS_SHAP else 'not installed'}")
    if PROFILE == "fast":
        print("FAST profile: smoke test with a train subsample - do NOT report these results.")
    if not HAS_XGB:
        print("  ! xgboost missing: XGBoost models are skipped (pip install xgboost).")

    # ---- load and verify ---------------------------------------------------------------------
    frames, fc, p4, scaler, scale_diff = load_part4()
    hashes_in = {p: sha256_of(p) for p in (TRAIN_CSV, VAL_CSV, TEST_CSV, TRAIN_S_CSV, VAL_S_CSV, TEST_S_CSV, SCALER_PKL)}
    tr, va, te = frames["train"], frames["validation"], frames["test"]
    tr_s, va_s, te_s = frames["train_s"], frames["validation_s"], frames["test_s"]
    if CFG["train_subsample"] and len(tr) > CFG["train_subsample"]:
        keep = np.sort(RNG.choice(len(tr), CFG["train_subsample"], replace=False))
        tr, tr_s = tr.iloc[keep].reset_index(drop=True), tr_s.iloc[keep].reset_index(drop=True)
    print(f"Rows: train {len(tr):,} | validation {len(va):,} | test {len(te):,} | cities {tr['City'].nunique()}")
    print(f"Train {tr['Date'].min().date()}..{tr['Date'].max().date()} | validation {va['Date'].min().date()}.."
          f"{va['Date'].max().date()} | test {te['Date'].min().date()}..{te['Date'].max().date()}")

    names, design = build_design({"train": tr}, fc)
    X = {"tree": {"train": design(tr), "validation": design(va), "test": design(te)},
         "lin": {"train": design(tr_s), "validation": design(va_s), "test": design(te_s)}}
    print(f"Shared feature set: {len(names)} features (nominal integer IDs excluded; dominant-pollutant one-hot added)")

    y_reg = {"train": tr[TARGET_REG].to_numpy(float), "validation": va[TARGET_REG].to_numpy(float), "test": te[TARGET_REG].to_numpy(float)}
    y_cls = {k: d[TARGET_CLS_CODE].to_numpy(int) for k, d in (("train", tr), ("validation", va), ("test", te))}
    hr_mask = {k: d[TARGET_HR].notna().to_numpy() for k, d in (("train", tr), ("validation", va), ("test", te))}
    y_hr = {k: d[TARGET_HR].to_numpy(float) for k, d in (("train", tr), ("validation", va), ("test", te))}
    classes_present = np.array(sorted(np.unique(y_cls["train"])))
    to_idx = lambda y: np.searchsorted(classes_present, y)
    pos_rate = float(np.nanmean(y_hr["train"][hr_mask["train"]]))
    pos_weight = (1 - pos_rate) / max(pos_rate, 1e-9)
    print(f"Train classes present: {[CATEGORIES[i] for i in classes_present]} | high-risk positive rate (train) {pos_rate:.3f}")

    cur_aqi = {"train": tr["Estimated_AQI"].to_numpy(float), "validation": va["Estimated_AQI"].to_numpy(float),
               "test": te["Estimated_AQI"].to_numpy(float)}
    cur_cat = {k: category_code(v) for k, v in cur_aqi.items()}

    # ======================= TUNING (train fit, validation selection; NO test access) =======================
    banner("Tuning on validation (test set is not touched in this stage)")
    models, params, fit_secs = {}, {}, {}
    val_pred = {}

    def register(task, name, m, info, secs):
        models[(task, name)], params[(task, name)], fit_secs[(task, name)] = m, info, secs

    Xtr_l, Xva_l = X["lin"]["train"], X["lin"]["validation"]
    Xtr_t, Xva_t = X["tree"]["train"], X["tree"]["validation"]
    rf_kw = dict(n_estimators=CFG["rf_trees"], max_samples=CFG["rf_max_samples"], n_jobs=-1, random_state=SEED)

    print("  Regression")
    m, info, _, s = tune("regression", "Ridge", [dict(alpha=a) for a in CFG["ridge_alphas"]],
                         lambda p: Ridge(random_state=SEED, **p).fit(Xtr_l, y_reg["train"]),
                         lambda mm: np.sqrt(mean_squared_error(y_reg["validation"], mm.predict(Xva_l))), False)
    register("regression", "Ridge", m, info, s)
    m, info, _, s = tune("regression", "RandomForest", CFG["rf_grid"],
                         lambda p: RandomForestRegressor(**rf_kw, **p).fit(Xtr_t, y_reg["train"]),
                         lambda mm: np.sqrt(mean_squared_error(y_reg["validation"], mm.predict(Xva_t))), False)
    register("regression", "RandomForest", m, info, s)
    if HAS_XGB:
        m, info, _, s = tune("regression", "XGBoost", [dict(max_depth=d, eval_metric="rmse") for d in CFG["xgb_reg_depths"]],
                             lambda p: fit_xgb("reg", p, Xtr_t, y_reg["train"], Xva_t, y_reg["validation"]),
                             lambda mm: np.sqrt(mean_squared_error(y_reg["validation"], mm.predict(Xva_t))), False)
        register("regression", "XGBoost", m, info, s)

    print("  Category classification")
    f1m = lambda y, p: f1_score(y, p, average="macro", zero_division=0)
    lr_cands = [dict(C=C, class_weight=cw) for C, cw in CFG["lr_grid"]]
    m, info, _, s = tune("classification", "LogisticRegression", lr_cands,
                         lambda p: LogisticRegression(max_iter=300, random_state=SEED, **p).fit(Xtr_l, y_cls["train"]),
                         lambda mm: f1m(y_cls["validation"], mm.predict(Xva_l)), True)
    register("classification", "LogisticRegression", m, info, s)
    rf_cls_cands = [dict(class_weight=cw, **g) for g in CFG["rf_grid"] for cw in CFG["rf_class_weights"]]
    m, info, _, s = tune("classification", "RandomForest", rf_cls_cands,
                         lambda p: RandomForestClassifier(**rf_kw, **p).fit(Xtr_t, y_cls["train"]),
                         lambda mm: f1m(y_cls["validation"], mm.predict(Xva_t)), True)
    register("classification", "RandomForest", m, info, s)
    if HAS_XGB:
        known = np.isin(y_cls["validation"], classes_present)
        cands = [dict(max_depth=d, balanced=b) for d, b in CFG["xgb_cls_grid"]]

        def fit_xcls(p):
            sw = compute_sample_weight("balanced", to_idx(y_cls["train"])) if p["balanced"] else None
            return fit_xgb("cls", dict(max_depth=p["max_depth"], objective="multi:softprob", eval_metric="mlogloss"),
                           Xtr_t, to_idx(y_cls["train"]), Xva_t[known], to_idx(y_cls["validation"][known]), sw)
        m, info, _, s = tune("classification", "XGBoost", cands, fit_xcls,
                             lambda mm: f1m(y_cls["validation"], classes_present[np.asarray(mm.predict(Xva_t)).astype(int)]), True)
        register("classification", "XGBoost", m, info, s)

    print("  High-risk (AQI > 200 within next 3 days)")
    mtr, mva = hr_mask["train"], hr_mask["validation"]
    ap = lambda y, p: average_precision_score(y, p)
    hr_rf_cands = [dict(class_weight=cw, **g) for g in CFG["rf_grid"] for cw in (None, "balanced_subsample")] \
        if PROFILE == "standard" else [dict(class_weight="balanced_subsample", **CFG["rf_grid"][0])]
    m, info, _, s = tune("highrisk", "RandomForest", hr_rf_cands,
                         lambda p: RandomForestClassifier(**rf_kw, **p).fit(Xtr_t[mtr], y_hr["train"][mtr].astype(int)),
                         lambda mm: ap(y_hr["validation"][mva], mm.predict_proba(Xva_t[mva])[:, 1]), True)
    register("highrisk", "RandomForest", m, info, s)
    if HAS_XGB:
        cands = [dict(max_depth=d, weighting=w) for d, w in CFG["xgb_hr_grid"]]

        def fit_xhr(p):
            spw = pos_weight if p["weighting"] == "pos_weight" else 1.0
            return fit_xgb("cls", dict(max_depth=p["max_depth"], objective="binary:logistic", eval_metric="aucpr",
                                       scale_pos_weight=spw),
                           Xtr_t[mtr], y_hr["train"][mtr].astype(int), Xva_t[mva], y_hr["validation"][mva].astype(int))
        m, info, _, s = tune("highrisk", "XGBoost", cands, fit_xhr,
                             lambda mm: ap(y_hr["validation"][mva], mm.predict_proba(Xva_t[mva])[:, 1]), True)
        register("highrisk", "XGBoost", m, info, s)

    # ---- prediction helpers ---------------------------------------------------------------------------
    def mat(name, task, split):
        return X["lin" if (task, name) in {("regression", "Ridge"), ("classification", "LogisticRegression")} else "tree"][split]

    def predict(task, name, split, Xmat=None):
        m = models[(task, name)]
        Xm = mat(name, task, split) if Xmat is None else Xmat
        if task == "regression":
            return m.predict(Xm)
        if task == "classification":
            p = np.asarray(m.predict(Xm))
            return classes_present[p.astype(int)] if name == "XGBoost" else p
        return m.predict_proba(Xm)[:, 1]

    # ---- validation predictions and thresholds (selection data) ------------------------------------------
    for (task, name) in list(models):
        val_pred[(task, name)] = predict(task, name, "validation")
    thresholds = {}
    for (task, name) in models:
        if task == "highrisk":
            thresholds[name] = best_f1_threshold(y_hr["validation"][mva], val_pred[(task, name)][mva])
    print("  Validation-optimised F1 thresholds:", {k: round(v, 2) for k, v in thresholds.items()})

    # ======================= SINGLE TEST EVALUATION =======================
    banner("Single test evaluation (each model scored once)")
    test_pred, scored = {}, set()

    def score_test_once(task, name):
        key = (task, name)
        if key in scored:
            raise RuntimeError(f"Test set was already scored for {key}")
        scored.add(key)
        test_pred[key] = predict(task, name, "test")
    for key in models:
        score_test_once(*key)
    print(f"  Test predictions generated once for {len(scored)} models.")

    # persistence baselines (no fitting)
    for split, pred_store in (("validation", val_pred), ("test", test_pred)):
        pred_store[("regression", "Persistence")] = cur_aqi[split]
        pred_store[("classification", "Persistence")] = cur_cat[split]
        pred_store[("highrisk", "Persistence")] = (cur_aqi[split] > 200).astype(float)

    # ======================= METRICS =======================
    banner("Metrics")
    ydict = {"regression": y_reg, "classification": y_cls}
    overall = []
    all_models = {"regression": ["Persistence", "Ridge", "RandomForest", "XGBoost"],
                  "classification": ["Persistence", "LogisticRegression", "RandomForest", "XGBoost"],
                  "highrisk": ["Persistence", "RandomForest", "XGBoost"]}
    for task in all_models:
        all_models[task] = [m for m in all_models[task] if m == "Persistence" or (task, m) in models]

    def meta_row(task, name):
        key = (task, name)
        return {"Task": task, "Model": name, "Best_Params": json.dumps(params.get(key, {}), default=str),
                "Fit_Seconds": round(fit_secs.get(key, 0.0), 1), "Feature_Set": f"{len(names)} shared features",
                "Scaled_Inputs": (name in ("Ridge", "LogisticRegression")) if name != "Persistence" else False}

    for task, mlist in all_models.items():
        for name in mlist:
            for split, store in (("validation", val_pred), ("test", test_pred)):
                p = store[(task, name)]
                if task == "regression":
                    overall.append({**meta_row(task, name), "Split": split, **reg_metrics(y_reg[split], p)})
                elif task == "classification":
                    overall.append({**meta_row(task, name), "Split": split, **cls_metrics(y_cls[split], p)})
                else:
                    mk = hr_mask[split]
                    y = y_hr[split][mk]
                    if name == "Persistence":
                        overall.append({**meta_row(task, name), "Split": split, "Threshold": np.nan,
                                        "Threshold_Rule": "binary_persistence", **hr_metrics(y, p[mk], p[mk])})
                    else:
                        for rule, thr in (("fixed_0.5", 0.5), ("val_F1_optimal", thresholds[name])):
                            overall.append({**meta_row(task, name), "Split": split, "Threshold": thr,
                                            "Threshold_Rule": rule, **hr_metrics(y, p[mk], (p[mk] >= thr).astype(int))})
    metrics_df = pd.DataFrame(overall)
    first = ["Task", "Model", "Split", "N"]
    metrics_df = metrics_df[first + [c for c in metrics_df.columns if c not in first]]

    # ---- grouped metrics -----------------------------------------------------------------------------------
    splits_df = {"validation": va, "test": te}
    city_rows, season_rows, cat_rows, cm_rows = [], [], [], []
    confusions = {}
    for task, mlist in all_models.items():
        for name in mlist:
            for split in ("validation", "test"):
                d = splits_df[split].reset_index(drop=True)
                p = (val_pred if split == "validation" else test_pred)[(task, name)]
                if task == "regression":
                    y = y_reg[split]
                    fn = lambda idx: reg_metrics(y[idx], p[idx])
                    actual_cat = y_cls[split]
                elif task == "classification":
                    y = y_cls[split]
                    fn = lambda idx: cls_metrics(y[idx], p[idx])
                    actual_cat = y_cls[split]
                else:
                    mk = hr_mask[split]
                    y = y_hr[split]
                    thr = thresholds.get(name, 0.5)
                    pred_lab = (p >= thr).astype(int) if name != "Persistence" else p.astype(int)
                    fn = lambda idx: hr_metrics(y[idx][mk[idx]], p[idx][mk[idx]], pred_lab[idx][mk[idx]]) if mk[idx].sum() else {}
                    actual_cat = cur_cat[split]
                base = {"Task": task, "Model": name, "Split": split}
                if task == "highrisk":
                    base["Threshold_Rule"] = "binary_persistence" if name == "Persistence" else "val_F1_optimal"
                for col, store in (("City", city_rows), ("Season", season_rows)):
                    for g, idx in d.groupby(col).indices.items():
                        store.append({**base, col: g, **fn(idx)})
                for ci, cname in enumerate(CATEGORIES):
                    idx = np.where(actual_cat == ci)[0]
                    if len(idx) == 0:
                        continue
                    row = {**base, "AQI_Category": cname,
                           "Category_Basis": "current-day (t) category" if task == "highrisk" else "actual next-day (t+1) category"}
                    if task == "classification":
                        pr, rc, f1c, sup = precision_recall_fscore_support(y, p, labels=[ci], zero_division=0)
                        row.update({"N": int(sup[0]), "Precision": float(pr[0]), "Recall": float(rc[0]), "F1": float(f1c[0])})
                    elif task == "regression":
                        e = reg_metrics(y[idx], p[idx])
                        row.update({k: e[k] for k in ("N", "MAE", "RMSE", "Bias")})
                    else:
                        row.update(fn(idx))
                    cat_rows.append(row)
                if split == "test" and task != "regression":
                    if task == "classification":
                        cm = confusion_matrix(y_cls["test"], p, labels=list(range(6)))
                        labels = CATEGORIES
                    else:
                        mk = hr_mask["test"]
                        lab = (p >= thresholds[name]).astype(int) if name != "Persistence" else p.astype(int)
                        cm = confusion_matrix(y_hr["test"][mk].astype(int), lab[mk], labels=[0, 1])
                        labels = ["No", "Yes"]
                    confusions[(task, name)] = cm
                    for i, a_ in enumerate(labels):
                        for j, b_ in enumerate(labels):
                            cm_rows.append({"Task": task, "Model": name, "Actual": a_, "Predicted": b_, "Count": int(cm[i, j])})
    city_df, season_df, category_df = map(pd.DataFrame, (city_rows, season_rows, cat_rows))

    # ---- model-comparison table -----------------------------------------------------------------------
    keep_hr = metrics_df["Threshold_Rule"].isin(["val_F1_optimal", "binary_persistence"]) | metrics_df["Threshold_Rule"].isna()
    comp = metrics_df[(metrics_df["Split"] == "test") & keep_hr].drop(columns=["Best_Params", "Feature_Set", "Scaled_Inputs"])
    reg_best = (metrics_df[(metrics_df["Task"] == "regression") & (metrics_df["Split"] == "validation") &
                           (metrics_df["Model"] != "Persistence")].sort_values("RMSE").iloc[0]["Model"])
    print(f"Best regression model on validation (by RMSE): {reg_best}")
    for task in all_models:
        t = comp[comp["Task"] == task]
        cols = {"regression": ["Model", "MAE", "RMSE", "R2"], "classification": ["Model", "Accuracy", "Macro_F1", "Balanced_Accuracy"],
                "highrisk": ["Model", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]}[task]
        print(f"\n  {task} - TEST\n" + t[cols].round(4).to_string(index=False))

    # ---- predicted vs actual / high-risk prediction tables ------------------------------------------------
    pva_parts, hr_parts = [], []
    for split, d in splits_df.items():
        d = d.reset_index(drop=True)
        store = val_pred if split == "validation" else test_pred
        t = pd.DataFrame({"Split": split, "Date": d["Date"], "City": d["City"], "Season": d["Season"],
                          "Current_AQI": d["Estimated_AQI"], "Actual_AQI_tplus1": y_reg[split],
                          "Actual_Category_tplus1": d[TARGET_CLS_NAME]})
        for name in all_models["regression"]:
            t[f"Pred_AQI_{name}"] = store[("regression", name)]
        for name in all_models["classification"]:
            t[f"Pred_Category_{name}"] = [CATEGORIES[int(i)] for i in store[("classification", name)]]
        pva_parts.append(t)
        mk = hr_mask[split]
        h = pd.DataFrame({"Split": split, "Date": d["Date"], "City": d["City"], "Season": d["Season"],
                          "Current_AQI": d["Estimated_AQI"], "Actual_HighRisk": y_hr[split]})[mk].reset_index(drop=True)
        for name in all_models["highrisk"]:
            p = store[("highrisk", name)][mk]
            if name == "Persistence":
                h["Pred_Persistence"] = p.astype(int)
            else:
                h[f"Prob_{name}"] = p
                h[f"Pred_{name}_thr0.5"] = (p >= 0.5).astype(int)
                h[f"Pred_{name}_valopt_thr{thresholds[name]:.2f}"] = (p >= thresholds[name]).astype(int)
        hr_parts.append(h)
    pva_all, hr_all = pd.concat(pva_parts, ignore_index=True), pd.concat(hr_parts, ignore_index=True)

    # ======================= IMPORTANCE (validation data only) =======================
    banner("Permutation importance (validation sample) and SHAP")
    singles, fam, types = make_groups(names, fc)
    rows = []
    sub_idx = RNG.choice(len(va), size=min(CFG["perm_rows"], len(va)), replace=False)
    for (task, name), m in models.items():
        t0 = time.time()
        Xm = mat(name, task, "validation")[sub_idx]
        if task == "regression":
            y_sub, pf = y_reg["validation"][sub_idx], (lambda Z, m=m: m.predict(Z))
            metric = lambda y, p: float(np.sqrt(mean_squared_error(y, p)))
            hb, mname = False, "RMSE"
        elif task == "classification":
            y_sub = y_cls["validation"][sub_idx]
            pf = (lambda Z, m=m, n=name: classes_present[np.asarray(m.predict(Z)).astype(int)] if n == "XGBoost" else m.predict(Z))
            metric = lambda y, p: f1_score(y, p, average="macro", zero_division=0)
            hb, mname = True, "Macro_F1"
        else:
            keep = hr_mask["validation"][sub_idx]
            Xm, y_sub = Xm[keep], y_hr["validation"][sub_idx][keep].astype(int)
            pf = (lambda Z, m=m: m.predict_proba(Z)[:, 1])
            metric = lambda y, p: average_precision_score(y, p) if len(np.unique(y)) == 2 else np.nan
            hb, mname = True, "PR_AUC"
        native = None
        if hasattr(m, "feature_importances_"):
            native = np.asarray(m.feature_importances_, float)
        elif hasattr(m, "coef_"):
            native = np.abs(np.asarray(m.coef_, float)).reshape(-1, len(names)).mean(axis=0)
        for level, groups in (("feature", singles), ("variable_family", fam), ("feature_type", types)):
            res, base_score = permutation_importance_custom(pf, Xm, y_sub, metric, hb, groups, CFG["perm_repeats"], RNG)
            for gname, mean_, std_ in res:
                r = {"Task": task, "Model": name, "Level": level, "Method": "permutation", "Feature": gname,
                     "Importance_Mean": mean_, "Importance_Std": std_, "Metric": mname + " (validation sample)"}
                if level == "feature" and native is not None:
                    r["Native_Importance"] = float(native[names.index(gname)])
                rows.append(r)
        print(f"  permutation importance {task}/{name} ({time.time() - t0:.0f}s)")
    shap_paths, shap_rows = run_shap(models, X["tree"]["validation"], names, classes_present)
    importance_df = pd.DataFrame(rows + shap_rows)
    importance_df["Rank"] = importance_df.groupby(["Task", "Model", "Level", "Method"])["Importance_Mean"].rank(ascending=False, method="min")
    importance_df["Computed_On"] = f"validation (permutation sample n={len(sub_idx)}, repeats={CFG['perm_repeats']})"

    # ======================= SAVE =======================
    banner("Saving outputs")
    metrics_df.to_csv(METRICS_CSV, index=False)
    city_df.to_csv(CITY_CSV, index=False)
    season_df.to_csv(SEASON_CSV, index=False)
    category_df.to_csv(CATEGORY_CSV, index=False)
    pva_all.to_csv(PVA_CSV, index=False, date_format="%Y-%m-%d", float_format="%.4f")
    hr_all.to_csv(HR_PRED_CSV, index=False, date_format="%Y-%m-%d", float_format="%.5f")
    importance_df.to_csv(IMPORTANCE_CSV, index=False)
    comp.to_csv(COMPARISON_CSV, index=False)
    pd.DataFrame(SEL_LOG).to_csv(SELECTION_CSV, index=False)
    pd.DataFrame(cm_rows).to_csv(CONFUSION_CSV, index=False)

    model_files = {}
    fname = {("regression", "Ridge"): "ridge_regressor.pkl", ("regression", "RandomForest"): "rf_regressor.pkl",
             ("regression", "XGBoost"): "xgb_regressor.pkl", ("classification", "LogisticRegression"): "logistic_classifier.pkl",
             ("classification", "RandomForest"): "rf_classifier.pkl", ("classification", "XGBoost"): "xgb_classifier.pkl",
             ("highrisk", "RandomForest"): "rf_highrisk_classifier.pkl", ("highrisk", "XGBoost"): "xgb_highrisk_classifier.pkl"}
    for key, m in models.items():
        path = os.path.join(CLASSICAL_DIR, fname[key])
        joblib.dump(m, path, compress=3)
        model_files[f"{key[0]}/{key[1]}"] = path
    index = {"feature_names": names, "uses_scaled_inputs": ["regression/Ridge", "classification/LogisticRegression"],
             "xgb_classifier_class_indices_to_category_codes": [int(c) for c in classes_present],
             "highrisk_validation_optimal_thresholds": thresholds, "categories": CATEGORIES,
             "best_params": {f"{k[0]}/{k[1]}": v for k, v in params.items()}, "model_files": model_files,
             "scaler": SCALER_PKL, "profile": PROFILE, "sklearn_version": sklearn.__version__,
             "xgboost_version": xgb.__version__ if HAS_XGB else None}
    json.dump(index, open(MODEL_INDEX_JSON, "w"), indent=2, default=str)
    for p in (METRICS_CSV, CITY_CSV, SEASON_CSV, CATEGORY_CSV, PVA_CSV, IMPORTANCE_CSV, HR_PRED_CSV, COMPARISON_CSV,
              SELECTION_CSV, CONFUSION_CSV, MODEL_INDEX_JSON):
        print(f"  saved {p}")

    # ======================= FIGURES =======================
    banner("Figures")
    pva_test = pva_all[pva_all["Split"] == "test"]
    rep_pref = ["Delhi", "Mumbai", "Kolkata", "Chennai", "Bengaluru", "Hyderabad", "Lucknow", "Patna"]
    avail = {c.lower().replace("_", " "): c for c in pva_test["City"].unique()}
    rep = [avail[c.lower()] for c in rep_pref if c.lower() in avail][:6]
    for c in pva_test.groupby("City")["Actual_AQI_tplus1"].mean().sort_values().index[::-1]:
        if len(rep) >= 6:
            break
        if c not in rep:
            rep.append(c)
    hr_te = hr_all[hr_all["Split"] == "test"]
    ctx = {"metrics": metrics_df, "confusions": confusions, "pva_test": pva_test, "best_reg": reg_best,
           "rep_cities": rep, "city_df": city_df, "season_df": season_df, "category_df": category_df,
           "importance": importance_df,
           "hr_test": {"y": hr_te["Actual_HighRisk"].to_numpy(int),
                       "proba": {n: hr_te[f"Prob_{n}"].to_numpy() for n in all_models["highrisk"] if n != "Persistence"}}}
    for fn in (plot_model_comparison, plot_confusions, plot_pred_vs_actual, plot_highrisk_curves,
               plot_group_metrics, plot_importance):
        run_plot(fn, ctx)
    FIG_LOG.extend(shap_paths)

    # ======================= INTEGRITY + MANIFEST =======================
    assert all(sha256_of(p) == h for p, h in hashes_in.items()), "A Part 4 split/scaler file was modified!"
    assert len(scored) == len(models), "Test scoring count mismatch"
    data_files = [METRICS_CSV, CITY_CSV, SEASON_CSV, CATEGORY_CSV, PVA_CSV, IMPORTANCE_CSV, HR_PRED_CSV, COMPARISON_CSV,
                  SELECTION_CSV, CONFUSION_CSV, MODEL_INDEX_JSON] + list(model_files.values())
    fig_files = [p for p in FIG_LOG if not str(p).startswith("FAILED:")]
    failed = [p for p in FIG_LOG if str(p).startswith("FAILED:")]
    headline = {}
    for task in all_models:
        t = comp[comp["Task"] == task]
        key = {"regression": "RMSE", "classification": "Macro_F1", "highrisk": "PR_AUC"}[task]
        headline[task] = {r["Model"]: round(float(r[key]), 4) for _, r in t.iterrows()}
    manifest = {
        "part": 5, "total_parts": 7, "run_id": RUN_ID, "seed": SEED, "profile": PROFILE,
        "label": AQI_LABEL, "caveats": AQI_CAVEAT, "base_dir": BASE_DIR,
        "inputs_from_part4": {"sha256": hashes_in, "part4_run_id": p4.get("run_id"),
                              "scaled_vs_scaler_max_abs_diff": scale_diff},
        "protocol": {"fit": "train only", "selection": "validation only",
                     "test": "each model scored once after all tuning; importance/SHAP use validation data",
                     "split_or_scaler_modified": False, "deep_learning_used": False,
                     "high_risk_threshold": "fixed 0.5 and validation-F1-optimal (chosen on validation, applied to test)"},
        "feature_set": {"n_features": len(names), "excluded_nominal": EXCLUDE_NOMINAL,
                        "added": "one-hot of day-t dominant pollutant", "names": names},
        "models_trained": sorted(f"{k[0]}/{k[1]}" for k in models),
        "xgboost_available": HAS_XGB, "shap_available": HAS_SHAP,
        "best_params": {f"{k[0]}/{k[1]}": v for k, v in params.items()},
        "validation_thresholds_highrisk": thresholds,
        "best_regression_model_by_validation_rmse": reg_best,
        "test_headline_metrics": headline,
        "figures_failed": failed,
        "files_created": [{"path": p, "rel_path": os.path.relpath(p, BASE_DIR), "size_bytes": os.path.getsize(p)}
                          for p in data_files + fig_files] +
                         [{"path": MANIFEST_JSON, "rel_path": os.path.relpath(MANIFEST_JSON, BASE_DIR), "size_bytes": None}],
        "next_part_should_load": {"baseline_metrics": METRICS_CSV, "predicted_vs_actual": PVA_CSV,
                                  "high_risk_predictions": HR_PRED_CSV, "scaled_splits": [TRAIN_S_CSV, VAL_S_CSV, TEST_S_CSV],
                                  "feature_columns": FEATURE_COLS_JSON, "classical_models_index": MODEL_INDEX_JSON},
        "runtime_minutes": round((time.time() - t_start) / 60, 1),
    }
    json.dump(manifest, open(MANIFEST_JSON, "w"), indent=2, default=str)

    banner("PART 5 COMPLETE - files created (exact paths for Part 6)")
    for p in data_files + fig_files + [MANIFEST_JSON]:
        print(p)
    if failed:
        print("\nFigures that failed (pipeline continued):", failed)
    print(f"\nBASE_DIR: {BASE_DIR}  | runtime: {manifest['runtime_minutes']} min")
    print("\n" + AQI_LABEL)
    print(AQI_CAVEAT)
    return metrics_df


if __name__ == "__main__" or IN_COLAB:
    metrics_table = main()


PART 5 - Persistence baselines and classical ML models (profile: standard)
Six-pollutant estimated AQI using CPCB sub-index methodology.
NH3 and Pb are unavailable, so this is not the official CPCB AQI. O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct CPCB ground readings.
sklearn 1.6.1 | xgboost 3.4.1 | shap 0.54.1
  Scaled files vs scaler.transform(unscaled): max abs difference 1.30e-04 (CSV rounding only)
Rows: train 99,384 | validation 20,806 | test 21,412 | cities 101
Train 2022-09-03..2025-05-13 | validation 2025-05-21..2025-12-12 | test 2025-12-20..2026-07-19
Shared feature set: 174 features (nominal integer IDs excluded; dominant-pollutant one-hot added)
Train classes present: ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe'] | high-risk positive rate (train) 0.275

Tuning on validation (test set is not touched in this stage)
  Regression
    regression/Ridge {'alpha': 0.1} -> validation 36.4074 (12s)
    regression/Ridge {'alpha': 1.0} -> 

In [8]:
# =============================================================================
# City- and Season-Aware Multi-Pollutant Air Quality Assessment and Forecasting
# Framework for Indian Cities Using Deep Learning
#
# PART 6 of 7: 30-day sequence deep-learning models
#   * Regression      -> Estimated_AQI_tplus1   : LSTM, GRU, 1D-CNN, CNN-LSTM, Bi-LSTM+Attention
#   * Classification  -> AQI category (t+1)     : LSTM, GRU, 1D-CNN, CNN-LSTM, Bi-LSTM+Attention
#   * High-risk       -> P(AQI > 200 within 3 days): LSTM, GRU, Bi-LSTM+Attention
#
# Continues directly from Parts 4-5 (same BASE_DIR). Part 4 files and the saved
# RobustScaler are only read, never modified (verified by hash at the end).
#
# SEQUENCE DESIGN
#   * One sample = the 30 consecutive days ending on the origin date t (day t
#     included), for ONE city. Targets are t+1 (AQI, category) and max(t+1..t+3) > 200.
#   * Per-day inputs: scaled pollutants, scaled 8-hour O3/CO, scaled weather, scaled
#     estimated AQI, provisional-data flag, calendar/season features.
#   * Static inputs: city embedding, region embedding, latitude, longitude, and the
#     K-Means regime (one-hot) if Part 4 included one.
#   * A sample belongs to the split of its ORIGIN date; its 30-day history may reach
#     back into earlier splits (past information only). The 7 purged dates between
#     splits are recovered from features_v1.csv (scaled with the saved scaler) so
#     validation/test sequences are complete and cover the same rows as Part 5.
#
# PROTOCOL: train on train, early stopping / LR schedule / best checkpoint on
# validation loss only; thresholds chosen on validation; each model's TEST
# predictions are generated once, after training has finished.
#
# REAL UNITS: regression models are trained on the AQI target standardised with the
# SAVED RobustScaler's centre/scale for the Estimated_AQI column; predictions are
# converted back to real AQI units with that same scaler before evaluation, category
# derivation and reporting.
#
# LABEL : "Six-pollutant estimated AQI using CPCB sub-index methodology."
# CAVEAT: NH3 and Pb are unavailable, so this is NOT the official CPCB AQI.
#         O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct
#         CPCB ground readings (provisional fallback rows remain flagged).
#
# Colab: use a GPU runtime (Runtime > Change runtime type). TensorFlow/Keras is
# preinstalled. Profiles: AQI_PROFILE=standard (default) | fast (smoke test only).
# =============================================================================

# ----------------------------------------------------------------------------
# 0. Imports, configuration, reproducibility, link to Parts 4-5
# ----------------------------------------------------------------------------
import os
import sys
import json
import time
import random
import hashlib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score,
                             balanced_accuracy_score, confusion_matrix, precision_score, recall_score,
                             roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
                             precision_recall_fscore_support)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.RandomState(SEED)

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
BASE_DIR = "/content/aqi_framework" if IN_COLAB else os.path.abspath("./aqi_framework")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PROFILE = os.environ.get("AQI_PROFILE", "standard")

PROC_DIR = os.path.join(BASE_DIR, "data", "processed")
RES_DIR = os.path.join(BASE_DIR, "results")
FIG_DIR = os.path.join(BASE_DIR, "figures")
DL_FIG_DIR = os.path.join(FIG_DIR, "dl")
MODEL_DIR = os.path.join(BASE_DIR, "models")
DL_MODEL_DIR = os.path.join(MODEL_DIR, "deep_learning")
for d in (RES_DIR, FIG_DIR, DL_FIG_DIR, MODEL_DIR, DL_MODEL_DIR):
    os.makedirs(d, exist_ok=True)

# ---- inputs ----------------------------------------------------------------------------
TRAIN_S_CSV = os.path.join(PROC_DIR, "train_scaled.csv")
VAL_S_CSV = os.path.join(PROC_DIR, "validation_scaled.csv")
TEST_S_CSV = os.path.join(PROC_DIR, "test_scaled.csv")
FEATURES_CSV = os.path.join(PROC_DIR, "features_v1.csv")             # only needed for the 7 purged history days
SCALER_PKL = os.path.join(MODEL_DIR, "standard_scaler.pkl")
FEATURE_COLS_JSON = os.path.join(RES_DIR, "feature_columns.json")
P4_MANIFEST = os.path.join(RES_DIR, "part4_handoff_manifest.json")
P5_METRICS = os.path.join(RES_DIR, "model_metrics.csv")               # optional: for the DL vs classical comparison
P5_PVA = os.path.join(RES_DIR, "predicted_vs_actual.csv")

# ---- outputs ----------------------------------------------------------------------------
METRICS_CSV = os.path.join(RES_DIR, "dl_model_metrics.csv")
CITY_CSV = os.path.join(RES_DIR, "dl_city_wise_metrics.csv")
SEASON_CSV = os.path.join(RES_DIR, "dl_season_wise_metrics.csv")
CATEGORY_CSV = os.path.join(RES_DIR, "dl_aqi_category_wise_metrics.csv")
PVA_CSV = os.path.join(RES_DIR, "dl_predicted_vs_actual.csv")
HR_CSV = os.path.join(RES_DIR, "high_risk_dl_predictions.csv")
HISTORY_CSV = os.path.join(RES_DIR, "dl_training_history.csv")
OCCLUSION_CSV = os.path.join(RES_DIR, "dl_occlusion_importance.csv")
ATTENTION_CSV = os.path.join(RES_DIR, "dl_attention_summary.csv")
COMPARISON_CSV = os.path.join(RES_DIR, "dl_vs_classical_comparison.csv")
CONFUSION_CSV = os.path.join(RES_DIR, "dl_confusion_matrices.csv")
SEQ_SUMMARY_CSV = os.path.join(RES_DIR, "dl_sequence_summary.csv")
MODEL_INDEX_JSON = os.path.join(DL_MODEL_DIR, "dl_models_index.json")
MANIFEST_JSON = os.path.join(RES_DIR, "part6_handoff_manifest.json")
FIG_CURVES = os.path.join(FIG_DIR, "dl_training_curves.png")
FIG_CONFUSION = os.path.join(FIG_DIR, "dl_confusion_matrix_plots.png")
FIG_ATTENTION = os.path.join(FIG_DIR, "dl_attention_heatmaps.png")

# ---- labels -----------------------------------------------------------------------------------
AQI_LABEL = "Six-pollutant estimated AQI using CPCB sub-index methodology."
AQI_CAVEAT = ("NH3 and Pb are unavailable, so this is not the official CPCB AQI. "
              "O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct CPCB ground readings.")
FOOTER = ("Six-pollutant estimated AQI (CPCB sub-index methodology); not official CPCB AQI (NH3, Pb unavailable). "
          "Attention weights and occlusion effects describe what a model relies on; they do not establish causes.")

# ---- schema ------------------------------------------------------------------------------------
CATEGORIES = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
SEASONS = ["Winter", "Summer", "Monsoon", "Post-monsoon"]
WINDOW = 30
BASE_VARS = ["PM2_5", "PM10", "CO", "NO2", "SO2", "O3", "O3_8hr_max", "CO_8hr_max_mg_m3",
             "Temperature_C", "Humidity_pct", "WindSpeed_kmh", "Precipitation_mm", "Estimated_AQI"]
CAL_COLS = ["month_sin", "month_cos", "doy_sin", "doy_cos", "weekday_sin", "weekday_cos", "is_weekend",
            "Season_Winter", "Season_Summer", "Season_Monsoon", "Season_Post_monsoon"]
SEQ_COLS = BASE_VARS + ["Provisional_8hr_flag_t"] + CAL_COLS
STATIC_NUM = ["Latitude", "Longitude"]
TARGET_REG, TARGET_CLS_CODE, TARGET_CLS_NAME, TARGET_HR = (
    "Estimated_AQI_tplus1", "Estimated_AQI_Category_tplus1_code", "Estimated_AQI_Category_tplus1",
    "HighRisk_AQI_gt200_next3d")
ARCHS = {"regression": ["LSTM", "GRU", "CNN1D", "CNN_LSTM", "BiLSTM_Attention"],
         "classification": ["LSTM", "GRU", "CNN1D", "CNN_LSTM", "BiLSTM_Attention"],
         "highrisk": ["LSTM", "GRU", "BiLSTM_Attention"]}
MODEL_COLORS = {"LSTM": "#4C72B0", "GRU": "#55A868", "CNN1D": "#DD8452", "CNN_LSTM": "#8172B2",
                "BiLSTM_Attention": "#C44E52", "Persistence": "#999999", "Ridge": "#8C8C8C", "LogisticRegression": "#8C8C8C",
                "RandomForest": "#B8B8B8", "XGBoost": "#6B6B6B"}

# ---- training configuration -------------------------------------------------------------------------
PROFILES = {
    "standard": dict(units=64, dropout=0.25, batch_size=128, lr=1e-3, epochs=40, patience=8, lr_patience=3,
                     occlusion_rows=600, attention_max_rows=None, train_subsample=None, class_weights=True),
    "fast": dict(units=16, dropout=0.25, batch_size=128, lr=2e-3, epochs=2, patience=2, lr_patience=1,
                 occlusion_rows=120, attention_max_rows=2000, train_subsample=6000, class_weights=True),
}
CFG = PROFILES[PROFILE]
N_OCC_BLOCKS = 6                                                       # 6 blocks of 5 days across the 30-day window
CSV_FLOAT_FORMAT = "%.5g"


def banner(text):
    print("\n" + "=" * 78 + f"\n{text}\n" + "=" * 78)


def sha256_of(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


# ----------------------------------------------------------------------------
# 1. Deep-learning backend (all TensorFlow/Keras code lives in this section)
# ----------------------------------------------------------------------------
def require_tf():
    try:
        import tensorflow as tf
        from tensorflow import keras
        return tf, keras
    except Exception as ex:
        raise ImportError("TensorFlow/Keras is required for Part 6 (preinstalled on Colab; otherwise "
                          "`pip install tensorflow`).") from ex


def build_model(arch, task, n_steps, n_feat, n_static, n_cities, n_regions, cfg):
    """Sequence encoder + city/region embeddings + static numeric inputs -> task head."""
    tf, keras = require_tf()
    L = keras.layers
    u, d = cfg["units"], cfg["dropout"]
    seq = keras.Input((n_steps, n_feat), name="seq")
    city = keras.Input((1,), dtype="int32", name="city")
    region = keras.Input((1,), dtype="int32", name="region")
    static = keras.Input((n_static,), name="static")

    if arch == "LSTM":
        x = L.LSTM(u, return_sequences=True)(seq)
        x = L.Dropout(d)(x)
        x = L.LSTM(max(u // 2, 4))(x)
    elif arch == "GRU":
        x = L.GRU(u, return_sequences=True)(seq)
        x = L.Dropout(d)(x)
        x = L.GRU(max(u // 2, 4))(x)
    elif arch == "CNN1D":
        x = L.Conv1D(u, 3, padding="same", activation="relu")(seq)
        x = L.Conv1D(u, 3, padding="same", activation="relu")(x)
        x = L.Dropout(d)(x)
        x = L.GlobalAveragePooling1D()(x)
    elif arch == "CNN_LSTM":
        x = L.Conv1D(u, 3, padding="same", activation="relu")(seq)
        x = L.MaxPooling1D(2)(x)
        x = L.Dropout(d)(x)
        x = L.LSTM(u)(x)
    elif arch == "BiLSTM_Attention":
        h = L.Bidirectional(L.LSTM(u, return_sequences=True))(seq)
        h = L.Dropout(d)(h)
        s = L.Dense(u, activation="tanh")(h)
        s = L.Dense(1)(s)                                              # one score per time step
        w = L.Softmax(axis=1, name="attn_weights")(s)                  # (batch, 30, 1), sums to 1 over time
        x = L.Flatten()(L.Dot(axes=1)([w, h]))                         # attention-weighted context vector
    else:
        raise ValueError(arch)

    c = L.Flatten()(L.Embedding(n_cities, 8)(city))
    r = L.Flatten()(L.Embedding(n_regions, 4)(region))
    z = L.Concatenate()([x, c, r, static])
    z = L.Dense(64, activation="relu")(z)
    z = L.Dropout(d)(z)
    z = L.Dense(32, activation="relu")(z)
    if task == "regression":
        out, loss, metrics = L.Dense(1, name="out")(z), "mse", ["mae"]
    elif task == "classification":
        out, loss, metrics = L.Dense(len(CATEGORIES), activation="softmax", name="out")(z), \
            "sparse_categorical_crossentropy", ["accuracy"]
    else:
        out, loss, metrics = L.Dense(1, activation="sigmoid", name="out")(z), "binary_crossentropy", ["accuracy"]
    model = keras.Model([seq, city, region, static], out, name=f"{task}_{arch}")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=cfg["lr"]), loss=loss, metrics=metrics)
    return model


def train_model(model, x_tr, y_tr, x_va, y_va, task, cfg, ckpt_path, class_weight=None):
    """Early stopping + ReduceLROnPlateau + best-checkpoint, all on validation loss."""
    tf, keras = require_tf()
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=cfg["patience"], restore_best_weights=True),
           keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=cfg["lr_patience"], min_lr=1e-5),
           keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True)]
    t0 = time.time()
    hist = model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=cfg["epochs"], batch_size=cfg["batch_size"],
                     shuffle=True, class_weight=class_weight, callbacks=cbs, verbose=2)
    secs = time.time() - t0
    best = model
    if os.path.exists(ckpt_path):
        try:
            best = keras.models.load_model(ckpt_path)               # built-in layers only -> no custom_objects needed
        except Exception as ex:
            print(f"    (could not reload checkpoint: {type(ex).__name__}; using in-memory model)")
    h = {k: [float(v) for v in vals] for k, vals in hist.history.items()}
    val_loss = h["val_loss"]
    return {"model": best, "history": h, "best_epoch": int(np.argmin(val_loss)) + 1,
            "best_val_loss": float(np.min(val_loss)), "seconds": secs, "n_params": int(best.count_params())}


def predict_dl(model, inputs, batch_size=512):
    return np.asarray(model.predict(inputs, batch_size=batch_size, verbose=0))


def get_attention(model, inputs, batch_size=512):
    """(N, 30) attention weights from the Bi-LSTM attention layer."""
    tf, keras = require_tf()
    am = keras.Model(model.inputs, model.get_layer("attn_weights").output)
    return np.asarray(am.predict(inputs, batch_size=batch_size, verbose=0))[:, :, 0]


def save_model(model, path):
    model.save(path)


# ----------------------------------------------------------------------------
# 2. Load Part 4 outputs (read-only), scaler handling, sequence building
# ----------------------------------------------------------------------------
def load_part4():
    for p in (TRAIN_S_CSV, VAL_S_CSV, TEST_S_CSV, SCALER_PKL, FEATURE_COLS_JSON):
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Missing Part 4 output: {p}\nRun Parts 1-4 first (same BASE_DIR={BASE_DIR}). "
                                    "In Colab, a restarted runtime loses /content - re-run the earlier parts.")
    fc = json.load(open(FEATURE_COLS_JSON))
    p4 = json.load(open(P4_MANIFEST)) if os.path.isfile(P4_MANIFEST) else {}
    frames = {"train": pd.read_csv(TRAIN_S_CSV, parse_dates=["Date"]),
              "validation": pd.read_csv(VAL_S_CSV, parse_dates=["Date"]),
              "test": pd.read_csv(TEST_S_CSV, parse_dates=["Date"])}
    if p4:
        rows = p4.get("split", {}).get("rows", {})
        for k, df in frames.items():
            if rows.get(k) not in (None, len(df)):
                raise ValueError(f"{k} rows {len(df)} differ from Part 4 manifest {rows.get(k)}.")
    tr, va, te = frames["train"], frames["validation"], frames["test"]
    assert tr["Date"].max() < va["Date"].min() and va["Date"].max() < te["Date"].min(), "Split chronology violated"
    scaler = joblib.load(SCALER_PKL)
    names = list(getattr(scaler, "feature_names_in_", fc["scaled_feature_columns"]))
    if "Estimated_AQI" not in names:
        raise ValueError("Estimated_AQI is not among the scaler's columns.")
    i = names.index("Estimated_AQI")
    aqi_scale = {"center": float(scaler.center_[i]), "scale": float(scaler.scale_[i])}
    for c in SEQ_COLS + ["City_ID", "Region_ID"] + STATIC_NUM:
        if c not in tr.columns:
            raise ValueError(f"Column missing from Part 4 scaled files: {c}")
    return frames, fc, p4, scaler, names, aqi_scale


def load_purged_history(scaler, names, fc):
    """The last 7 dates of train and validation were purged from the split files (to avoid label overlap). They are
    ordinary past observations, so they are used as sequence INPUT history only (never as samples/targets)."""
    if not os.path.isfile(FEATURES_CSV):
        print("  ! features_v1.csv not found: the 7 purged days cannot be recovered, so the first ~37 validation and "
              "test days per city will have incomplete windows and be dropped.")
        return None
    f = pd.read_csv(FEATURES_CSV, parse_dates=["Date"])
    f = f[f["Split"] == "purged"].copy()
    if f.empty:
        return None
    cols = [c for c in names if c in f.columns]
    f[cols] = scaler.transform(f[names])                              # same saved scaler as the split files
    return f


def build_history(frames, purged):
    keep = ["Date", "City"] + SEQ_COLS
    parts = [frames[k][keep] for k in ("train", "validation", "test")]
    if purged is not None:
        parts.append(purged[keep])
    h = pd.concat(parts, ignore_index=True)
    assert not h.duplicated(["City", "Date"]).any(), "Duplicate City-Date rows in the combined history"
    return h


def build_grid(hist):
    cities = sorted(hist["City"].unique())
    dates = pd.date_range(hist["Date"].min(), hist["Date"].max(), freq="D")
    city_ix = {c: i for i, c in enumerate(cities)}
    grid = np.full((len(cities), len(dates), len(SEQ_COLS)), np.nan, dtype=np.float32)
    ci = hist["City"].map(city_ix).to_numpy()
    di = (hist["Date"] - dates[0]).dt.days.to_numpy()
    grid[ci, di, :] = hist[SEQ_COLS].to_numpy(np.float32)
    return grid, city_ix, dates


def extract_sequences(grid, city_ix, dates, rows):
    ci = rows["City"].map(city_ix).to_numpy()
    di = (rows["Date"] - dates[0]).dt.days.to_numpy()
    offs = np.arange(-(WINDOW - 1), 1)
    idx = np.clip(di[:, None] + offs[None, :], 0, grid.shape[1] - 1)
    X = grid[ci[:, None], idx, :]
    valid = (di >= WINDOW - 1) & ~np.isnan(X).any(axis=(1, 2))
    return X, valid


def make_split_data(df, grid, city_ix, dates, aqi_scale, regime_cols):
    X, valid = extract_sequences(grid, city_ix, dates, df)
    n_all = len(df)
    df = df[valid].reset_index(drop=True)
    X = X[valid]
    cur_aqi = X[:, -1, SEQ_COLS.index("Estimated_AQI")] * aqi_scale["scale"] + aqi_scale["center"]   # inverse scaler
    static = df[STATIC_NUM + regime_cols].to_numpy(np.float32)
    y_real = df[TARGET_REG].to_numpy(np.float32)
    data = {"seq": X.astype(np.float32), "city": df["City_ID"].to_numpy(np.int32).reshape(-1, 1),
            "region": df["Region_ID"].to_numpy(np.int32).reshape(-1, 1), "static": static}
    side = {"meta": pd.DataFrame({"Date": df["Date"], "City": df["City"], "Season": df["Season"],
                                  "Current_AQI": cur_aqi, "Actual_AQI_tplus1": y_real,
                                  "Actual_Category_tplus1": df[TARGET_CLS_NAME]}),
            "y_reg": y_real, "y_reg_scaled": ((y_real - aqi_scale["center"]) / aqi_scale["scale"]).astype(np.float32),
            "y_cls": df[TARGET_CLS_CODE].to_numpy(np.int32),
            "y_hr": df[TARGET_HR].to_numpy(np.float32), "n_rows_in_file": n_all}
    side["hr_mask"] = ~np.isnan(side["y_hr"])
    return data, side


def subset(data, mask):
    return {k: v[mask] for k, v in data.items()}


def category_code(aqi):
    return np.searchsorted([50, 100, 200, 300, 400], np.asarray(aqi, dtype=float), side="left")


# ----------------------------------------------------------------------------
# 3. Metrics
# ----------------------------------------------------------------------------
def reg_metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    out = {"N": len(y), "MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p))),
           "Bias": float(np.mean(p - y))}
    out["R2"] = r2_score(y, p) if len(y) > 1 and np.var(y) > 0 else np.nan
    return out


def cls_metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return {"N": len(y), "Accuracy": accuracy_score(y, p), "Macro_F1": f1_score(y, p, average="macro", zero_division=0),
            "Balanced_Accuracy": balanced_accuracy_score(y, p)}


def hr_metrics(y, proba, pred):
    y, proba, pred = np.asarray(y).astype(int), np.asarray(proba, float), np.asarray(pred).astype(int)
    out = {"N": len(y), "Positive_Rate": float(y.mean()), "Precision": precision_score(y, pred, zero_division=0),
           "Recall": recall_score(y, pred, zero_division=0), "F1": f1_score(y, pred, zero_division=0)}
    if len(np.unique(y)) == 2:
        out["ROC_AUC"], out["PR_AUC"] = roc_auc_score(y, proba), average_precision_score(y, proba)
    else:
        out["ROC_AUC"], out["PR_AUC"] = np.nan, np.nan
    return out


def best_f1_threshold(y, proba):
    grid = np.linspace(0.05, 0.95, 91)
    return float(grid[int(np.argmax([f1_score(y, (proba >= t).astype(int), zero_division=0) for t in grid]))])


def class_weights(y, power=0.5):
    """Moderate (square-root) balanced class weights for the training loss only."""
    vals, cnt = np.unique(y, return_counts=True)
    w = (len(y) / (len(vals) * cnt)) ** power
    return {int(v): float(x / w.mean()) for v, x in zip(vals, w)}


# ----------------------------------------------------------------------------
# 4. Occlusion explanations and attention summaries (model-agnostic / descriptive)
# ----------------------------------------------------------------------------
def seq_groups():
    g = {v: [SEQ_COLS.index(v)] for v in BASE_VARS}
    g["Provisional-data flag"] = [SEQ_COLS.index("Provisional_8hr_flag_t")]
    g["Calendar & season"] = [SEQ_COLS.index(c) for c in CAL_COLS]
    return g


def occlusion_importance(score_fn, inputs, baseline, groups, n_blocks=N_OCC_BLOCKS):
    """Replace one feature group within one 5-day block by its training-mean value and record the change in the
    model's score (real-unit AQI for regression, risk probability for classifiers). Model-agnostic."""
    base = score_fn(inputs)
    edges = np.linspace(0, WINDOW, n_blocks + 1).astype(int)
    rows = []
    for gname, idx in groups.items():
        for b in range(n_blocks):
            t0, t1 = edges[b], edges[b + 1]
            X2 = inputs["seq"].copy()
            X2[:, t0:t1, idx] = baseline[idx]
            delta = score_fn({**inputs, "seq": X2}) - base
            rows.append({"Feature_Group": gname, "Block": b, "Days_Before_Origin": f"{WINDOW - t1}-{WINDOW - t0 - 1}",
                         "Mean_Abs_Change": float(np.mean(np.abs(delta))), "Mean_Signed_Change": float(np.mean(delta))})
    return rows


# ----------------------------------------------------------------------------
# 5. Figures
# ----------------------------------------------------------------------------
FIG_LOG = []


def save_fig(fig, path, title=None, dpi=130):
    if title:
        fig.suptitle(title, fontsize=13, y=0.998)
    fig.text(0.5, 0.002, FOOTER, ha="center", va="bottom", fontsize=7, color="dimgray")
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return path


def run_plot(fn, ctx):
    try:
        paths = fn(ctx)
        paths = [paths] if isinstance(paths, str) else (paths or [])
        FIG_LOG.extend(paths)
        print(f"  [ok] {fn.__name__}")
    except Exception as ex:
        print(f"  [FAILED] {fn.__name__}: {type(ex).__name__}: {ex}")
        FIG_LOG.append(f"FAILED:{fn.__name__}:{type(ex).__name__}")


def plot_training_curves(ctx):
    hist = ctx["history_df"]
    tasks = [t for t in ARCHS if t in set(hist["Task"])]
    fig, ax = plt.subplots(len(tasks), 5, figsize=(21, 3.9 * len(tasks)), squeeze=False)
    for r, task in enumerate(tasks):
        for c in range(5):
            a = ax[r, c]
            models = ARCHS[task]
            if c >= len(models):
                a.axis("off")
                continue
            m = models[c]
            h = hist[(hist["Task"] == task) & (hist["Model"] == m)]
            if h.empty:
                a.axis("off")
                continue
            a.plot(h["Epoch"], h["loss"], label="train", color="#4C72B0")
            a.plot(h["Epoch"], h["val_loss"], label="validation", color="#C44E52")
            be = int(h.loc[h["val_loss"].idxmin(), "Epoch"])
            a.axvline(be, color="k", ls=":", lw=0.8, label=f"best epoch {be}")
            a.set_title(f"{task}: {m}", fontsize=10)
            a.set_xlabel("Epoch")
            a.set_ylabel({"regression": "MSE (scaled AQI)", "classification": "Cross-entropy",
                          "highrisk": "Binary cross-entropy"}[task])
            a.legend(fontsize=7)
    return save_fig(fig, FIG_CURVES, "Training and validation loss curves")


def plot_confusions(ctx):
    cms = ctx["confusions"]
    items = [(k, v) for k, v in cms.items() if k[0] == "classification"]
    fig, ax = plt.subplots(2, 3, figsize=(20, 12.5))
    for a, ((task, model), cm) in zip(ax.ravel(), items):
        rown = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        a.imshow(rown, cmap="Blues", vmin=0, vmax=1)
        a.set_xticks(range(6), CATEGORIES, rotation=40, ha="right", fontsize=8)
        a.set_yticks(range(6), CATEGORIES, fontsize=8)
        for i in range(6):
            for j in range(6):
                a.text(j, i, f"{cm[i, j]}\n{100 * rown[i, j]:.0f}%", ha="center", va="center", fontsize=6.5,
                       color="white" if rown[i, j] > 0.5 else "black")
        a.set_title(f"{model} (test)")
        a.set_xlabel("Predicted next-day category")
        a.set_ylabel("Actual next-day category")
    for a in ax.ravel()[len(items):]:
        a.axis("off")
    p1 = save_fig(fig, FIG_CONFUSION, "Deep-learning confusion matrices: next-day AQI category (test; cells = count and row %)")
    hr = [(k, v) for k, v in cms.items() if k[0] == "highrisk"]
    fig, ax = plt.subplots(1, max(len(hr), 1), figsize=(5.2 * max(len(hr), 1), 4.8))
    for a, ((task, model), cm) in zip(np.atleast_1d(ax), hr):
        a.imshow(cm / np.maximum(cm.sum(axis=1, keepdims=True), 1), cmap="Oranges", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                a.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=11)
        a.set_xticks([0, 1], ["No", "Yes"])
        a.set_yticks([0, 1], ["No", "Yes"])
        a.set_title(model)
        a.set_xlabel("Predicted high risk")
        a.set_ylabel("Actual high risk")
    p2 = save_fig(fig, os.path.join(DL_FIG_DIR, "dl_confusion_matrices_highrisk.png"),
                  "Deep-learning high-risk confusion matrices (test; validation-optimised threshold)")
    return [p1, p2]


def plot_pred_vs_actual(ctx):
    pva = ctx["pva_test"]
    models = [c[len("Pred_AQI_"):] for c in pva.columns if c.startswith("Pred_AQI_")]
    s = pva.sample(min(8000, len(pva)), random_state=SEED)
    lim = float(max(pva["Actual_AQI_tplus1"].max(), 1))
    fig, ax = plt.subplots(2, 3, figsize=(19, 12.5))
    for a, m in zip(ax.ravel(), models):
        a.scatter(s["Actual_AQI_tplus1"], s[f"Pred_AQI_{m}"], s=4, alpha=0.25, color=MODEL_COLORS.get(m))
        a.plot([0, lim], [0, lim], "k--", lw=0.8)
        mt = reg_metrics(pva["Actual_AQI_tplus1"], pva[f"Pred_AQI_{m}"])
        a.set_title(f"{m}: MAE {mt['MAE']:.1f}, RMSE {mt['RMSE']:.1f}, R2 {mt['R2']:.3f}")
        a.set_xlabel("Actual AQI (t+1)")
        a.set_ylabel("Predicted AQI (t+1), real units")
    for a in ax.ravel()[len(models):]:
        a.axis("off")
    p1 = save_fig(fig, os.path.join(DL_FIG_DIR, "dl_predicted_vs_actual_scatter.png"),
                  "Deep-learning predicted vs actual next-day AQI (test, random sample of 8,000 city-days)")
    best, cities = ctx["best_reg"], ctx["rep_cities"]
    rows = (len(cities) + 1) // 2
    fig, ax = plt.subplots(rows, 2, figsize=(16, 3.3 * rows), sharex=True)
    for a, c in zip(np.atleast_1d(ax).ravel(), cities):
        g = pva[pva["City"] == c].sort_values("Date")
        a.plot(g["Date"], g["Actual_AQI_tplus1"], color="k", lw=1, label="Actual")
        a.plot(g["Date"], g[f"Pred_AQI_{best}"], color=MODEL_COLORS.get(best), lw=1, label=best)
        a.set_title(c)
        a.set_ylabel("AQI (t+1)")
    np.atleast_1d(ax).ravel()[0].legend(fontsize=8)
    p2 = save_fig(fig, os.path.join(DL_FIG_DIR, "dl_predicted_vs_actual_timeseries_cities.png"),
                  f"Test-period next-day AQI: actual vs {best}")
    return [p1, p2]


def plot_highrisk_curves(ctx):
    hr = ctx["hr_test"]
    fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
    for m, proba in hr["proba"].items():
        fpr, tpr, _ = roc_curve(hr["y"], proba)
        pr, rc, _ = precision_recall_curve(hr["y"], proba)
        ax[0].plot(fpr, tpr, color=MODEL_COLORS.get(m), label=f"{m} (AUC {roc_auc_score(hr['y'], proba):.3f})")
        ax[1].plot(rc, pr, color=MODEL_COLORS.get(m), label=f"{m} (AP {average_precision_score(hr['y'], proba):.3f})")
    ax[0].plot([0, 1], [0, 1], "k--", lw=0.7)
    ax[0].set_xlabel("False-positive rate")
    ax[0].set_ylabel("True-positive rate")
    ax[0].set_title("ROC (test)")
    ax[1].axhline(hr["y"].mean(), color="k", ls="--", lw=0.7, label=f"Base rate {hr['y'].mean():.2f}")
    ax[1].set_xlabel("Recall")
    ax[1].set_ylabel("Precision")
    ax[1].set_title("Precision-recall (test)")
    for a in ax:
        a.legend()
    return save_fig(fig, os.path.join(DL_FIG_DIR, "dl_highrisk_roc_pr_curves.png"), "Deep-learning high-risk curves (test)")


def plot_attention(ctx):
    att = ctx["attention"]                                         # task -> dict(weights (N,30), meta, y_hr)
    tasks = list(att)
    fig, ax = plt.subplots(len(tasks), 2, figsize=(16, 4.3 * len(tasks)), squeeze=False)
    lags = np.arange(WINDOW - 1, -1, -1)
    for r, task in enumerate(tasks):
        w, meta = att[task]["weights"], att[task]["meta"]
        mat = np.vstack([w[(meta["Season"] == s).to_numpy()].mean(axis=0) if (meta["Season"] == s).any()
                         else np.full(WINDOW, np.nan) for s in SEASONS])
        im = ax[r, 0].imshow(mat, aspect="auto", cmap="viridis")
        ax[r, 0].set_yticks(range(4), SEASONS)
        ax[r, 0].set_xticks(range(0, WINDOW, 3), [f"t-{l}" if l else "t" for l in lags[::3]], fontsize=8)
        ax[r, 0].set_title(f"{task}: mean attention by season (test)")
        fig.colorbar(im, ax=ax[r, 0], shrink=0.8, label="attention weight")
        thr = np.nanpercentile(meta["Current_AQI"], 90)
        groups = {"All samples": np.ones(len(w), bool), "Current AQI >= 90th pct": (meta["Current_AQI"] >= thr).to_numpy(),
                  "Current AQI < 90th pct": (meta["Current_AQI"] < thr).to_numpy()}
        for name, mk in groups.items():
            if mk.any():
                ax[r, 1].plot(lags[::-1][::-1], w[mk].mean(axis=0), label=f"{name} (n={int(mk.sum()):,})")
        ax[r, 1].invert_xaxis()
        ax[r, 1].set_xlabel("Days before origin date (0 = origin day t)")
        ax[r, 1].set_ylabel("Mean attention weight")
        ax[r, 1].set_title(f"{task}: attention across the 30-day window")
        ax[r, 1].legend(fontsize=8)
    return save_fig(fig, FIG_ATTENTION, "Bi-LSTM attention heatmaps (descriptive; computed on the test inputs of the final models)")


def plot_occlusion(ctx):
    occ = ctx["occlusion_df"]
    out = []
    for task in [t for t in ARCHS if t in set(occ["Task"])]:
        models = ARCHS[task]
        fig, ax = plt.subplots(1, len(models), figsize=(5.4 * len(models), 6.2), squeeze=False)
        for a, m in zip(ax[0], models):
            d = occ[(occ["Task"] == task) & (occ["Model"] == m)]
            if d.empty:
                a.axis("off")
                continue
            piv = d.pivot(index="Feature_Group", columns="Block", values="Mean_Abs_Change")
            piv = piv.loc[piv.sum(axis=1).sort_values(ascending=False).index]
            im = a.imshow(piv.values, aspect="auto", cmap="magma")
            blocks = d.drop_duplicates("Block").sort_values("Block")["Days_Before_Origin"].tolist()
            a.set_xticks(range(len(blocks)), blocks, rotation=45, ha="right", fontsize=8)
            a.set_yticks(range(len(piv)), piv.index, fontsize=8)
            a.set_xlabel("Occluded block (days before origin)")
            a.set_title(m)
            fig.colorbar(im, ax=a, shrink=0.7)
        unit = {"regression": "mean |change in predicted AQI|", "classification": "mean |change in P(Poor or worse)|",
                "highrisk": "mean |change in high-risk probability|"}[task]
        out.append(save_fig(fig, os.path.join(DL_FIG_DIR, f"dl_occlusion_{task}.png"),
                            f"Occlusion explanations ({task}; {unit}; validation sample)"))
    return out


def plot_group_metrics(ctx):
    sw, cat = ctx["season_df"], ctx["category_df"]
    sw, cat = sw[sw["Split"] == "test"], cat[cat["Split"] == "test"]
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    for a, (task, met) in zip(ax, (("regression", "RMSE"), ("classification", "Macro_F1"), ("highrisk", "PR_AUC"))):
        d = sw[sw["Task"] == task]
        models = list(dict.fromkeys(d["Model"]))
        x = np.arange(len(SEASONS))
        for k, m in enumerate(models):
            v = [d[(d["Model"] == m) & (d["Season"] == s)][met].mean() for s in SEASONS]
            a.bar(x + (k - (len(models) - 1) / 2) * 0.8 / len(models), v, width=0.8 / len(models), label=m,
                  color=MODEL_COLORS.get(m))
        a.set_xticks(x, SEASONS)
        a.set_title(f"{task}: {met} by season (test)")
        a.legend(fontsize=7)
    p1 = save_fig(fig, os.path.join(DL_FIG_DIR, "dl_metrics_by_season.png"), "Deep-learning test performance by season")
    d = cat[cat["Task"] == "regression"]
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    x = np.arange(len(CATEGORIES))
    for task, a, met in (("regression", ax[0], "MAE"), ("classification", ax[1], "Recall")):
        d = cat[cat["Task"] == task]
        models = list(dict.fromkeys(d["Model"]))
        for k, m in enumerate(models):
            g = d[d["Model"] == m].set_index("AQI_Category").reindex(CATEGORIES)
            a.bar(x + (k - (len(models) - 1) / 2) * 0.8 / len(models), g[met].values, width=0.8 / len(models), label=m,
                  color=MODEL_COLORS.get(m))
        a.set_xticks(x, CATEGORIES, rotation=20)
        a.set_title(f"{task}: {met} by actual next-day category (test)")
        a.legend(fontsize=7)
    p2 = save_fig(fig, os.path.join(DL_FIG_DIR, "dl_metrics_by_aqi_category.png"), "Deep-learning test performance by AQI category")
    return [p1, p2]


def plot_comparison(ctx):
    comp = ctx["comparison"]
    if comp is None or comp.empty:
        return []
    fig, ax = plt.subplots(1, 3, figsize=(19, 6))
    for a, (task, met) in zip(ax, (("regression", "RMSE"), ("classification", "Macro_F1"), ("highrisk", "PR_AUC"))):
        d = comp[comp["Task"] == task]
        d = d.sort_values(met, ascending=(task == "regression"))
        colors = {"Persistence": "#BBBBBB", "Classical ML": "#6B8EBF", "Deep learning": "#C44E52"}
        a.barh([f"{r.Model}" for r in d.itertuples()][::-1], d[met].values[::-1], color=[colors[f] for f in d["Family"]][::-1])
        a.set_title(f"{task}: test {met}" + (" (lower is better)" if task == "regression" else " (higher is better)"))
    ax[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in ("#BBBBBB", "#6B8EBF", "#C44E52")],
                 labels=["Persistence", "Classical ML", "Deep learning"], fontsize=8)
    return save_fig(fig, os.path.join(DL_FIG_DIR, "dl_vs_classical_comparison.png"),
                    "Test performance: persistence vs classical ML (Part 5) vs deep learning (Part 6)")


# ----------------------------------------------------------------------------
# 6. Main pipeline
# ----------------------------------------------------------------------------
def main():
    t_start = time.time()
    banner(f"PART 6 - Deep-learning sequence models (profile: {PROFILE})")
    print(AQI_LABEL)
    print(AQI_CAVEAT)
    tf, keras = require_tf()
    if tf is not None:
        keras.utils.set_random_seed(SEED)
        gpus = tf.config.list_physical_devices("GPU")
        print(f"TensorFlow {tf.__version__} | Keras {keras.__version__} | GPUs: {len(gpus)}")
        if not gpus:
            print("  ! No GPU detected. Training 13 models will be slow on CPU - switch to a GPU runtime if possible.")
    if PROFILE == "fast":
        print("FAST profile: smoke test with tiny models and a train subsample - do NOT report these results.")

    # ---- load and verify ---------------------------------------------------------------------
    frames, fc, p4, scaler, scaler_names, aqi_scale = load_part4()
    hash_paths = [TRAIN_S_CSV, VAL_S_CSV, TEST_S_CSV, SCALER_PKL] + ([FEATURES_CSV] if os.path.isfile(FEATURES_CSV) else [])
    hashes_in = {p: sha256_of(p) for p in hash_paths}
    print(f"Saved RobustScaler: Estimated_AQI centre {aqi_scale['center']:.3f}, scale {aqi_scale['scale']:.3f} "
          "(used to standardise the regression target and to convert predictions back to AQI units)")
    purged = load_purged_history(scaler, scaler_names, fc)
    hist = build_history(frames, purged)
    print(f"History table: {len(hist):,} city-days (incl. {0 if purged is None else len(purged):,} purged-day input rows)")
    grid, city_ix, dates = build_grid(hist)

    regime_cols = []
    if "Regime_KMeans" in frames["train"].columns:
        codes = sorted(frames["train"]["Regime_KMeans"].dropna().unique())
        for k in frames.keys():
            for r in codes:
                frames[k][f"Regime_{int(r)}"] = (frames[k]["Regime_KMeans"] == r).astype(float)
        regime_cols = [f"Regime_{int(r)}" for r in codes]
        print(f"K-Means regime labels found: using {len(regime_cols)} one-hot static inputs.")
    else:
        print("K-Means regime labels: not available; static inputs = city/region embeddings + lat/lon.")

    tr_df = frames["train"]
    if CFG["train_subsample"] and len(tr_df) > CFG["train_subsample"]:
        tr_df = tr_df.iloc[np.sort(RNG.choice(len(tr_df), CFG["train_subsample"], replace=False))].reset_index(drop=True)
    D, S = {}, {}
    for name, df in (("train", tr_df), ("validation", frames["validation"]), ("test", frames["test"])):
        D[name], S[name] = make_split_data(df, grid, city_ix, dates, aqi_scale, regime_cols)
    seq_rows = []
    for k in D:
        m = S[k]["meta"]
        seq_rows.append({"split": k, "rows_in_part4_file": S[k]["n_rows_in_file"], "n_sequences": len(m),
                         "date_min": m["Date"].min().date(), "date_max": m["Date"].max().date(),
                         "n_cities": m["City"].nunique(), "shape": str(D[k]["seq"].shape)})
        print(f"{k:>10}: {len(m):,} sequences of {WINDOW} days x {len(SEQ_COLS)} features | "
              f"{m['Date'].min().date()}..{m['Date'].max().date()} | cities {m['City'].nunique()} "
              f"| dropped for incomplete history: {S[k]['n_rows_in_file'] - len(m):,}")
    pd.DataFrame(seq_rows).to_csv(SEQ_SUMMARY_CSV, index=False)
    # each sample sits in the split of its origin date; inputs never extend past the origin date by construction
    assert S["train"]["meta"]["Date"].max() < S["validation"]["meta"]["Date"].min() <= S["validation"]["meta"]["Date"].max() \
        < S["test"]["meta"]["Date"].min(), "Split chronology violated in sequence samples"
    assert set(S["validation"]["meta"]["Date"]).isdisjoint(set(S["test"]["meta"]["Date"]))
    if len(S["test"]["meta"]) != len(frames["test"]):
        print("  ! Some test rows lack a complete 30-day window, so the test set differs from Part 5's.")

    n_cities = int(max(frames[k]["City_ID"].max() for k in frames)) + 1
    n_regions = int(max(frames[k]["Region_ID"].max() for k in frames)) + 1
    n_static = D["train"]["static"].shape[1]
    baseline = np.nanmean(D["train"]["seq"].reshape(-1, len(SEQ_COLS)), axis=0)          # occlusion replacement values
    groups = seq_groups()

    # ======================= TRAINING (validation only for selection; test not touched) =======================
    banner("Training (early stopping / LR schedule / best checkpoint on validation loss)")
    results, trained = {}, {}
    hr_tr, hr_va = S["train"]["hr_mask"], S["validation"]["hr_mask"]
    for task, archs in ARCHS.items():
        for arch in archs:
            t0 = time.time()
            print(f"\n  >> {task} / {arch}")
            model = build_model(arch, task, WINDOW, len(SEQ_COLS), n_static, n_cities, n_regions, CFG)
            if task == "regression":
                x_tr, y_tr, x_va, y_va, cw = D["train"], S["train"]["y_reg_scaled"].reshape(-1, 1), D["validation"], \
                    S["validation"]["y_reg_scaled"].reshape(-1, 1), None
            elif task == "classification":
                x_tr, y_tr, x_va, y_va = D["train"], S["train"]["y_cls"], D["validation"], S["validation"]["y_cls"]
                cw = class_weights(y_tr) if CFG["class_weights"] else None
            else:
                x_tr, y_tr = subset(D["train"], hr_tr), S["train"]["y_hr"][hr_tr].reshape(-1, 1)
                x_va, y_va = subset(D["validation"], hr_va), S["validation"]["y_hr"][hr_va].reshape(-1, 1)
                pos = float(y_tr.mean())
                cw = {0: 1.0, 1: float(np.sqrt((1 - pos) / max(pos, 1e-9)))} if CFG["class_weights"] else None
            ckpt = os.path.join(DL_MODEL_DIR, f"{task}_{arch}.keras")
            res = train_model(model, x_tr, y_tr, x_va, y_va, task, CFG, ckpt, class_weight=cw)
            if not os.path.exists(ckpt):
                save_model(res["model"], ckpt)
            res["path"] = ckpt
            results[(task, arch)] = res
            print(f"     best epoch {res['best_epoch']} | best val loss {res['best_val_loss']:.4f} | "
                  f"{res['n_params']:,} parameters | {time.time() - t0:.0f}s")

    # ======================= PREDICTIONS (test scored once per model) =======================
    banner("Predictions and evaluation (test scored once per model)")
    val_out, test_out, scored = {}, {}, set()
    for key, res in results.items():
        task, arch = key
        m = res["model"]
        pv = predict_dl(m, D["validation"])
        if key in scored:
            raise RuntimeError(f"Test set already scored for {key}")
        scored.add(key)
        pt = predict_dl(m, D["test"])
        if task == "regression":                                  # back to real AQI units with the SAVED scaler
            pv = np.clip(pv.reshape(-1) * aqi_scale["scale"] + aqi_scale["center"], 0, 500)
            pt = np.clip(pt.reshape(-1) * aqi_scale["scale"] + aqi_scale["center"], 0, 500)
        elif task == "classification":
            pv, pt = pv.reshape(len(pv), -1), pt.reshape(len(pt), -1)
        else:
            pv, pt = pv.reshape(-1), pt.reshape(-1)
        val_out[key], test_out[key] = pv, pt
    print(f"  Test predictions generated once for {len(scored)} models.")

    thresholds = {}
    for (task, arch), pv in val_out.items():
        if task == "highrisk":
            thresholds[arch] = best_f1_threshold(S["validation"]["y_hr"][hr_va], pv[hr_va])
    print("  Validation-optimised F1 thresholds:", {k: round(v, 2) for k, v in thresholds.items()})

    def labels(task, arch, split):
        p = (val_out if split == "validation" else test_out)[(task, arch)]
        if task == "classification":
            return p.argmax(axis=1)
        return p

    # ---- overall metrics -------------------------------------------------------------------------
    overall = []
    for (task, arch), res in results.items():
        meta = {"Task": task, "Model": arch, "Epochs_Run": len(res["history"]["loss"]), "Best_Epoch": res["best_epoch"],
                "Best_Val_Loss": res["best_val_loss"], "N_Parameters": res["n_params"],
                "Train_Seconds": round(res["seconds"], 1), "Window_Days": WINDOW, "N_Seq_Features": len(SEQ_COLS)}
        for split in ("validation", "test"):
            s = S[split]
            if task == "regression":
                p = (val_out if split == "validation" else test_out)[(task, arch)]
                overall.append({**meta, "Split": split, **reg_metrics(s["y_reg"], p)})
                pc = category_code(p)                              # category derived from the real-unit AQI prediction
                overall.append({**meta, "Task": "regression_to_category", "Split": split,
                                **cls_metrics(s["y_cls"], pc)})
            elif task == "classification":
                overall.append({**meta, "Split": split, **cls_metrics(s["y_cls"], labels(task, arch, split))})
            else:
                mk = s["hr_mask"]
                p = (val_out if split == "validation" else test_out)[(task, arch)]
                for rule, thr in (("fixed_0.5", 0.5), ("val_F1_optimal", thresholds[arch])):
                    overall.append({**meta, "Split": split, "Threshold": thr, "Threshold_Rule": rule,
                                    **hr_metrics(s["y_hr"][mk], p[mk], (p[mk] >= thr).astype(int))})
    metrics_df = pd.DataFrame(overall)
    first = ["Task", "Model", "Split", "N"]
    metrics_df = metrics_df[first + [c for c in metrics_df.columns if c not in first]]

    # ---- grouped metrics ------------------------------------------------------------------------------
    city_rows, season_rows, cat_rows, cm_rows, confusions = [], [], [], [], {}
    for (task, arch) in results:
        for split in ("validation", "test"):
            s = S[split]
            meta_df = s["meta"]
            p = (val_out if split == "validation" else test_out)[(task, arch)]
            if task == "regression":
                y, fn, actual_cat = s["y_reg"], (lambda idx, p=p, y=s["y_reg"]: reg_metrics(y[idx], p[idx])), s["y_cls"]
            elif task == "classification":
                lab = p.argmax(axis=1)
                y, fn, actual_cat = s["y_cls"], (lambda idx, lab=lab, y=s["y_cls"]: cls_metrics(y[idx], lab[idx])), s["y_cls"]
            else:
                mk, thr = s["hr_mask"], thresholds[arch]
                lab = (p >= thr).astype(int)
                y = s["y_hr"]
                fn = (lambda idx, p=p, lab=lab, y=y, mk=mk: hr_metrics(y[idx][mk[idx]], p[idx][mk[idx]], lab[idx][mk[idx]])
                      if mk[idx].sum() else {})
                actual_cat = category_code(meta_df["Current_AQI"].to_numpy())
            base = {"Task": task, "Model": arch, "Split": split}
            if task == "highrisk":
                base["Threshold_Rule"] = "val_F1_optimal"
            for col, store in (("City", city_rows), ("Season", season_rows)):
                for g, idx in meta_df.groupby(col).indices.items():
                    store.append({**base, col: g, **fn(idx)})
            for ci, cname in enumerate(CATEGORIES):
                idx = np.where(actual_cat == ci)[0]
                if len(idx) == 0:
                    continue
                row = {**base, "AQI_Category": cname,
                       "Category_Basis": "current-day (t) category" if task == "highrisk" else "actual next-day (t+1) category"}
                if task == "classification":
                    pr, rc, f1c, sup = precision_recall_fscore_support(y, lab, labels=[ci], zero_division=0)
                    row.update({"N": int(sup[0]), "Precision": float(pr[0]), "Recall": float(rc[0]), "F1": float(f1c[0])})
                elif task == "regression":
                    e = reg_metrics(y[idx], p[idx])
                    row.update({k: e[k] for k in ("N", "MAE", "RMSE", "Bias")})
                else:
                    row.update(fn(idx))
                cat_rows.append(row)
            if split == "test" and task != "regression":
                if task == "classification":
                    cm, lbls = confusion_matrix(y_true := s["y_cls"], lab, labels=list(range(6))), CATEGORIES
                else:
                    cm, lbls = confusion_matrix(s["y_hr"][s["hr_mask"]].astype(int), lab[s["hr_mask"]], labels=[0, 1]), ["No", "Yes"]
                confusions[(task, arch)] = cm
                for i, a_ in enumerate(lbls):
                    for j, b_ in enumerate(lbls):
                        cm_rows.append({"Task": task, "Model": arch, "Actual": a_, "Predicted": b_, "Count": int(cm[i, j])})
    city_df, season_df, category_df = map(pd.DataFrame, (city_rows, season_rows, cat_rows))

    # ---- console summary ----------------------------------------------------------------------------------
    te = metrics_df[(metrics_df["Split"] == "test")]
    reg_val = metrics_df[(metrics_df["Task"] == "regression") & (metrics_df["Split"] == "validation")]
    best_reg = reg_val.sort_values("RMSE").iloc[0]["Model"]
    print(f"\nBest DL regression model on validation (RMSE): {best_reg}")
    for task, cols in (("regression", ["Model", "MAE", "RMSE", "R2"]),
                       ("regression_to_category", ["Model", "Accuracy", "Macro_F1", "Balanced_Accuracy"]),
                       ("classification", ["Model", "Accuracy", "Macro_F1", "Balanced_Accuracy"]),
                       ("highrisk", ["Model", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"])):
        t = te[te["Task"] == task]
        if task == "highrisk":
            t = t[t["Threshold_Rule"] == "val_F1_optimal"]
        print(f"\n  {task} - TEST\n" + t[cols].round(4).to_string(index=False))

    # ---- predicted-vs-actual tables -----------------------------------------------------------------------------
    pva_parts, hr_parts = [], []
    for split in ("validation", "test"):
        s = S[split]
        t = s["meta"].copy()
        t.insert(0, "Split", split)
        t["Actual_Category_code"] = s["y_cls"]
        store = val_out if split == "validation" else test_out
        for arch in ARCHS["regression"]:
            p = store[("regression", arch)]
            t[f"Pred_AQI_{arch}"] = p
            t[f"Pred_CategoryFromAQI_{arch}"] = [CATEGORIES[i] for i in category_code(p)]
        for arch in ARCHS["classification"]:
            pr = store[("classification", arch)]
            t[f"Pred_Category_{arch}"] = [CATEGORIES[i] for i in pr.argmax(axis=1)]
            t[f"Prob_PoorOrWorse_{arch}"] = pr[:, 3:].sum(axis=1)
        pva_parts.append(t)
        mk = s["hr_mask"]
        h = s["meta"][["Date", "City", "Season", "Current_AQI"]].copy()
        h.insert(0, "Split", split)
        h["Actual_HighRisk"] = s["y_hr"]
        for arch in ARCHS["highrisk"]:
            p = store[("highrisk", arch)]
            h[f"Prob_{arch}"] = p
            h[f"Pred_{arch}_thr0.5"] = (p >= 0.5).astype(int)
            h[f"Pred_{arch}_valopt_thr{thresholds[arch]:.2f}"] = (p >= thresholds[arch]).astype(int)
        hr_parts.append(h[mk].reset_index(drop=True))
    pva_all, hr_all = pd.concat(pva_parts, ignore_index=True), pd.concat(hr_parts, ignore_index=True)

    # ---- explanations: occlusion (validation sample) and attention (descriptive) --------------------------------------
    banner("Explanations: occlusion (validation sample) and attention")
    occ_rows = []
    n_occ = min(CFG["occlusion_rows"], len(S["validation"]["meta"]))
    occ_idx = RNG.choice(len(S["validation"]["meta"]), size=n_occ, replace=False)
    for (task, arch), res in results.items():
        m = res["model"]
        inp = subset(D["validation"], occ_idx) if task != "highrisk" else subset(D["validation"], occ_idx)
        if task == "regression":
            score = lambda x, m=m: np.clip(predict_dl(m, x).reshape(-1) * aqi_scale["scale"] + aqi_scale["center"], 0, 500)
        elif task == "classification":
            score = lambda x, m=m: predict_dl(m, x).reshape(len(x["seq"]), -1)[:, 3:].sum(axis=1)
        else:
            score = lambda x, m=m: predict_dl(m, x).reshape(-1)
        t0 = time.time()
        for r in occlusion_importance(score, inp, baseline, groups):
            occ_rows.append({"Task": task, "Model": arch, "N_Samples": n_occ, **r})
        print(f"  occlusion {task}/{arch} ({time.time() - t0:.0f}s)")
    occlusion_df = pd.DataFrame(occ_rows)

    attention, att_rows = {}, []
    amax = CFG["attention_max_rows"]
    for task in ARCHS:
        key = (task, "BiLSTM_Attention")
        if key not in results:
            continue
        x_t = D["test"] if amax is None else subset(D["test"], np.arange(min(amax, len(S["test"]["meta"]))))
        w = get_attention(results[key]["model"], x_t)
        meta_t = S["test"]["meta"].iloc[:len(w)].reset_index(drop=True)
        attention[task] = {"weights": w, "meta": meta_t}
        lags = np.arange(WINDOW - 1, -1, -1)
        for gname, mk in [("All", np.ones(len(w), bool))] + [(s_, (meta_t["Season"] == s_).to_numpy()) for s_ in SEASONS]:
            if mk.any():
                for j, lag in enumerate(lags):
                    att_rows.append({"Task": task, "Group": gname, "Days_Before_Origin": int(lag),
                                     "Mean_Attention": float(w[mk][:, j].mean()), "N": int(mk.sum())})
    attention_df = pd.DataFrame(att_rows)

    # ---- comparison with Part 5 -----------------------------------------------------------------------------------------
    comparison, same_rows = None, None
    if os.path.isfile(P5_METRICS):
        p5 = pd.read_csv(P5_METRICS)
        p5 = p5[p5["Split"] == "test"]
        p5 = p5[p5["Task"].isin(["regression", "classification", "highrisk"])]
        p5 = p5[(p5["Task"] != "highrisk") | p5["Threshold_Rule"].isin(["val_F1_optimal", "binary_persistence"])]
        p5 = p5.assign(Family=np.where(p5["Model"] == "Persistence", "Persistence", "Classical ML"))
        dl = metrics_df[(metrics_df["Split"] == "test") & metrics_df["Task"].isin(["regression", "classification", "highrisk"])]
        dl = dl[(dl["Task"] != "highrisk") | (dl["Threshold_Rule"] == "val_F1_optimal")].assign(Family="Deep learning")
        keep = ["Task", "Family", "Model", "N", "MAE", "RMSE", "R2", "Accuracy", "Macro_F1", "Balanced_Accuracy",
                "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]
        comparison = pd.concat([p5.reindex(columns=keep), dl.reindex(columns=keep)], ignore_index=True)
        if os.path.isfile(P5_PVA):
            p5p = pd.read_csv(P5_PVA, usecols=["Split", "Date", "City"], parse_dates=["Date"])
            p5p = p5p[p5p["Split"] == "test"]
            a = set(zip(p5p["City"], p5p["Date"]))
            b = set(zip(S["test"]["meta"]["City"], S["test"]["meta"]["Date"]))
            same_rows = bool(a == b)
            print(f"\nTest rows identical to Part 5's test rows: {same_rows} "
                  f"(Part 5 {len(a):,} vs DL {len(b):,})")
            if not same_rows:
                print("  ! Compare models on the common rows only; the Part 5 and Part 6 test sets differ.")
        print("\nTest comparison (persistence vs classical vs deep learning):")
        for task, met in (("regression", "RMSE"), ("classification", "Macro_F1"), ("highrisk", "PR_AUC")):
            t = comparison[comparison["Task"] == task].sort_values(met, ascending=(task == "regression"))
            print(f"  {task} by {met}: " + " | ".join(f"{r.Model}[{r.Family[:2]}] {getattr(r, met):.3f}" for r in t.itertuples()))
    else:
        print("\nPart 5 results not found; skipping the DL vs classical comparison.")

    # ======================= SAVE =======================
    banner("Saving outputs")
    metrics_df.to_csv(METRICS_CSV, index=False)
    city_df.to_csv(CITY_CSV, index=False)
    season_df.to_csv(SEASON_CSV, index=False)
    category_df.to_csv(CATEGORY_CSV, index=False)
    pva_all.to_csv(PVA_CSV, index=False, date_format="%Y-%m-%d", float_format=CSV_FLOAT_FORMAT)
    hr_all.to_csv(HR_CSV, index=False, date_format="%Y-%m-%d", float_format=CSV_FLOAT_FORMAT)
    pd.DataFrame(cm_rows).to_csv(CONFUSION_CSV, index=False)
    hist_rows = []
    for (task, arch), res in results.items():
        for ep in range(len(res["history"]["loss"])):
            hist_rows.append({"Task": task, "Model": arch, "Epoch": ep + 1,
                              **{k: v[ep] for k, v in res["history"].items() if len(v) > ep}})
    history_df = pd.DataFrame(hist_rows)
    history_df.to_csv(HISTORY_CSV, index=False)
    occlusion_df.to_csv(OCCLUSION_CSV, index=False)
    attention_df.to_csv(ATTENTION_CSV, index=False)
    if comparison is not None:
        comparison.to_csv(COMPARISON_CSV, index=False)
    index = {"window_days": WINDOW, "sequence_features": SEQ_COLS, "static_numeric": STATIC_NUM + regime_cols,
             "embeddings": {"City_ID": n_cities, "Region_ID": n_regions},
             "aqi_target_scaling": {**aqi_scale, "source": "saved RobustScaler column 'Estimated_AQI'",
                                    "inverse": "AQI = prediction * scale + center, clipped to [0, 500]"},
             "highrisk_validation_optimal_thresholds": thresholds, "categories": CATEGORIES,
             "models": {f"{t}/{a}": r["path"] for (t, a), r in results.items()},
             "training": {k: CFG[k] for k in ("units", "dropout", "batch_size", "lr", "epochs", "patience", "lr_patience")},
             "profile": PROFILE, "scaler": SCALER_PKL,
             "input_dict_keys": ["seq", "city", "region", "static"]}
    json.dump(index, open(MODEL_INDEX_JSON, "w"), indent=2, default=str)
    data_files = [METRICS_CSV, CITY_CSV, SEASON_CSV, CATEGORY_CSV, PVA_CSV, HR_CSV, CONFUSION_CSV, HISTORY_CSV,
                  OCCLUSION_CSV, ATTENTION_CSV, SEQ_SUMMARY_CSV] + ([COMPARISON_CSV] if comparison is not None else []) \
        + [MODEL_INDEX_JSON]
    for p in data_files:
        print(f"  saved {p}")

    # ======================= FIGURES =======================
    banner("Figures")
    pva_test = pva_all[pva_all["Split"] == "test"]
    pref = ["Delhi", "Mumbai", "Kolkata", "Chennai", "Bengaluru", "Hyderabad", "Lucknow", "Patna"]
    avail = {c.lower().replace("_", " "): c for c in pva_test["City"].unique()}
    rep = [avail[c.lower()] for c in pref if c.lower() in avail][:6]
    for c in pva_test.groupby("City")["Actual_AQI_tplus1"].mean().sort_values().index[::-1]:
        if len(rep) >= 6:
            break
        if c not in rep:
            rep.append(c)
    hr_te = hr_all[hr_all["Split"] == "test"]
    ctx = {"history_df": history_df, "confusions": confusions, "pva_test": pva_test, "best_reg": best_reg, "rep_cities": rep,
           "attention": attention, "occlusion_df": occlusion_df, "season_df": season_df, "category_df": category_df,
           "comparison": comparison,
           "hr_test": {"y": hr_te["Actual_HighRisk"].to_numpy(int),
                       "proba": {a: hr_te[f"Prob_{a}"].to_numpy() for a in ARCHS["highrisk"]}}}
    for fn in (plot_training_curves, plot_confusions, plot_pred_vs_actual, plot_highrisk_curves, plot_attention,
               plot_occlusion, plot_group_metrics, plot_comparison):
        run_plot(fn, ctx)

    # ======================= INTEGRITY + MANIFEST =======================
    assert all(sha256_of(p) == h for p, h in hashes_in.items()), "A Part 4 file or the scaler was modified!"
    assert len(scored) == len(results)
    model_files = [r["path"] for r in results.values()]
    fig_files = [p for p in FIG_LOG if not str(p).startswith("FAILED:")]
    failed = [p for p in FIG_LOG if str(p).startswith("FAILED:")]
    headline = {}
    for task, met in (("regression", "RMSE"), ("classification", "Macro_F1"), ("highrisk", "PR_AUC")):
        t = te[te["Task"] == task]
        if task == "highrisk":
            t = t[t["Threshold_Rule"] == "val_F1_optimal"]
        headline[task] = {r.Model: round(float(getattr(r, met)), 4) for r in t.itertuples()}
    manifest = {
        "part": 6, "total_parts": 7, "run_id": RUN_ID, "seed": SEED, "profile": PROFILE,
        "label": AQI_LABEL, "caveats": AQI_CAVEAT, "base_dir": BASE_DIR,
        "framework": {"tensorflow": getattr(tf, "__version__", None), "keras": getattr(keras, "__version__", None)},
        "inputs_from_part4": {"sha256": hashes_in, "part4_run_id": p4.get("run_id"),
                              "purged_history_rows_used_as_input_only": 0 if purged is None else int(len(purged))},
        "sequence_design": {"window_days": WINDOW, "sequence_features": SEQ_COLS, "static_numeric": STATIC_NUM + regime_cols,
                            "embeddings": ["City_ID", "Region_ID"], "samples": seq_rows,
                            "history_rule": "sample belongs to the split of its origin date; its history is past data only",
                            "same_test_rows_as_part5": same_rows},
        "protocol": {"train": "train sequences only", "selection": "validation loss (early stopping, LR schedule, checkpoint)",
                     "thresholds": "validation-F1-optimal and fixed 0.5", "test": "each model scored once after training",
                     "class_weights_in_training_loss": CFG["class_weights"],
                     "split_or_scaler_modified": False, "predictions_converted_with_saved_scaler": True},
        "training_config": {k: CFG[k] for k in ("units", "dropout", "batch_size", "lr", "epochs", "patience", "lr_patience")},
        "best_dl_regression_model_by_validation_rmse": best_reg,
        "test_headline_metrics": headline,
        "figures_failed": failed,
        "files_created": [{"path": p, "rel_path": os.path.relpath(p, BASE_DIR), "size_bytes": os.path.getsize(p)}
                          for p in data_files + model_files + fig_files] +
                         [{"path": MANIFEST_JSON, "rel_path": os.path.relpath(MANIFEST_JSON, BASE_DIR), "size_bytes": None}],
        "next_part_should_load": {"dl_predicted_vs_actual": PVA_CSV, "high_risk_dl_predictions": HR_CSV,
                                  "dl_metrics": METRICS_CSV, "classical_predicted_vs_actual": P5_PVA,
                                  "classical_metrics": P5_METRICS, "dl_models_index": MODEL_INDEX_JSON,
                                  "part3_daily_with_causes": os.path.join(RES_DIR, "daily_aqi_dominant_pollutant_with_causes.csv")},
        "runtime_minutes": round((time.time() - t_start) / 60, 1),
    }
    json.dump(manifest, open(MANIFEST_JSON, "w"), indent=2, default=str)

    banner("PART 6 COMPLETE - files created (exact paths for Part 7)")
    for p in data_files + model_files + fig_files + [MANIFEST_JSON]:
        print(p)
    if failed:
        print("\nFigures that failed (pipeline continued):", failed)
    print(f"\nBASE_DIR: {BASE_DIR} | runtime: {manifest['runtime_minutes']} min")
    print("\n" + AQI_LABEL)
    print(AQI_CAVEAT)
    return metrics_df


if __name__ == "__main__" or IN_COLAB:
    dl_metrics_table = main()


PART 6 - Deep-learning sequence models (profile: standard)
Six-pollutant estimated AQI using CPCB sub-index methodology.
NH3 and Pb are unavailable, so this is not the official CPCB AQI. O3/CO 8-hour values are Open-Meteo/CAMS-based estimates, not direct CPCB ground readings.
TensorFlow 2.21.0 | Keras 3.13.2 | GPUs: 0
  ! No GPU detected. Training 13 models will be slow on CPU - switch to a GPU runtime if possible.
Saved RobustScaler: Estimated_AQI centre 147.000, scale 77.000 (used to standardise the regression target and to convert predictions back to AQI units)
History table: 143,016 city-days (incl. 1,414 purged-day input rows)
K-Means regime labels: not available; static inputs = city/region embeddings + lat/lon.
     train: 96,455 sequences of 30 days x 25 features | 2022-10-02..2025-05-13 | cities 101 | dropped for incomplete history: 2,929
validation: 20,806 sequences of 30 days x 25 features | 2025-05-21..2025-12-12 | cities 101 | dropped for incomplete history: 0
      test:

In [2]:
# =============================================================================
# City- and Season-Aware Multi-Pollutant Air Quality Assessment and Forecasting
# Framework for Indian Cities Using Deep Learning
#
# PART 7 of 7: final synthesis
#   1. one harmonised comparison of persistence, classical ML and deep learning
#   2. best model per task (chosen on VALIDATION), with block-bootstrap uncertainty
#   3. observed and predicted City x Season tables
#   4. feature-group ablation (base -> +weather -> +season/calendar -> +city -> +regime)
#   5. final explainability, high-risk alerts, pollution-episode case studies
#   6. final figures, CSVs, Markdown report and handoff manifest
#
# Continues directly from Parts 1-6 (same BASE_DIR). All earlier outputs are only
# read, never modified (verified by hash). Existing models are NOT retrained; the
# only new fitting is (a) a small, fast gradient-boosting model per ablation step and
# (b) a K-Means regime model fitted on the TRAIN split only - both needed to answer
# the ablation / regime-awareness questions. Seed 42 throughout.
#
# LABEL : "Six-pollutant estimated AQI using CPCB sub-index methodology."
# CAVEAT: NH3 and Pb are unavailable, so this is NOT the official CPCB AQI.
#         O3/CO values are Open-Meteo/CAMS-based estimates, not direct CPCB
#         ground-station observations.
# =============================================================================

# ----------------------------------------------------------------------------
# 0. Imports, configuration, reproducibility, link to Parts 1-6
# ----------------------------------------------------------------------------
import os
import re
import sys
import json
import time
import shutil
import random
import hashlib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score,
                             balanced_accuracy_score, confusion_matrix, precision_score, recall_score,
                             roc_auc_score, average_precision_score, silhouette_score)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.RandomState(SEED)

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
BASE_DIR = "/content/aqi_framework" if IN_COLAB else os.path.abspath("./aqi_framework")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PROFILE = os.environ.get("AQI_PROFILE", "standard")

PROC_DIR = os.path.join(BASE_DIR, "data", "processed")
RES_DIR = os.path.join(BASE_DIR, "results")
FIG_DIR = os.path.join(BASE_DIR, "figures", "final")
VERSION_DIR = os.path.join(RES_DIR, "final_versions", f"part7_{RUN_ID}")
for d in (RES_DIR, FIG_DIR, VERSION_DIR):
    os.makedirs(d, exist_ok=True)

# ---- required inputs (results from Parts 3, 5, 6) -------------------------------------------
REQUIRED = {
    "model_metrics": "model_metrics.csv", "dl_model_metrics": "dl_model_metrics.csv",
    "city_wise": "city_wise_metrics.csv", "season_wise": "season_wise_metrics.csv",
    "dl_city_wise": "dl_city_wise_metrics.csv", "dl_season_wise": "dl_season_wise_metrics.csv",
    "feature_importance": "feature_importance.csv", "pva": "predicted_vs_actual.csv",
    "dl_pva": "dl_predicted_vs_actual.csv", "hr": "high_risk_predictions.csv",
    "dl_hr": "high_risk_dl_predictions.csv", "city_season": "city_season_dominant_pollutant_summary.csv",
    "daily": "daily_aqi_dominant_pollutant_with_causes.csv"}
REQ_PATHS = {k: os.path.join(RES_DIR, v) for k, v in REQUIRED.items()}
# ---- optional inputs --------------------------------------------------------------------------
OPT = {"dl_attention": os.path.join(RES_DIR, "dl_attention_summary.csv"),
       "dl_occlusion": os.path.join(RES_DIR, "dl_occlusion_importance.csv"),
       "quality": os.path.join(PROC_DIR, "data_quality_report.csv"),
       "rule_freq": os.path.join(RES_DIR, "cause_policy_rule_frequencies.csv"),
       "feature_cols": os.path.join(RES_DIR, "feature_columns.json")}
SPLIT_FILES = {k: os.path.join(PROC_DIR, f"{k}.csv") for k in ("train", "validation", "test")}
SPLIT_S_FILES = {k: os.path.join(PROC_DIR, f"{k}_scaled.csv") for k in ("train", "validation", "test")}
MANIFESTS = {i: os.path.join(RES_DIR if i > 1 else PROC_DIR, f"part{i}_handoff_manifest.json") for i in range(1, 7)}

# ---- required final outputs ----------------------------------------------------------------------
OBSERVED_CSV = os.path.join(RES_DIR, "observed_city_season_table.csv")
PREDICTED_CSV = os.path.join(RES_DIR, "predicted_city_season_table.csv")
FINAL_COMPARISON_CSV = os.path.join(RES_DIR, "final_model_comparison.csv")
FINAL_ABLATION_CSV = os.path.join(RES_DIR, "final_ablation_results.csv")
FINAL_IMPORTANCE_CSV = os.path.join(RES_DIR, "final_feature_importance.csv")
FINAL_ALERTS_CSV = os.path.join(RES_DIR, "final_high_risk_alerts.csv")
FINAL_SUMMARY_CSV = os.path.join(RES_DIR, "final_city_season_dominant_pollutant_summary.csv")
FINAL_REPORT_MD = os.path.join(RES_DIR, "final_report.md")
# ---- supporting outputs ----------------------------------------------------------------------------
BOOTSTRAP_CSV = os.path.join(RES_DIR, "final_bootstrap_comparisons.csv")
GROUP_METRICS_CSV = os.path.join(RES_DIR, "final_group_metrics.csv")
ALERT_SUMMARY_CSV = os.path.join(RES_DIR, "final_high_risk_alert_summary_by_city.csv")
CASES_CSV = os.path.join(RES_DIR, "final_case_studies.csv")
CASE_SERIES_CSV = os.path.join(RES_DIR, "final_case_study_series.csv")
REGIME_CSV = os.path.join(RES_DIR, "final_regime_labels.csv")
REGIME_PROFILE_CSV = os.path.join(RES_DIR, "final_regime_profiles.csv")
CONSISTENCY_CSV = os.path.join(RES_DIR, "final_consistency_checks.csv")
MANIFEST_JSON = os.path.join(RES_DIR, "part7_handoff_manifest.json")

# ---- labels ---------------------------------------------------------------------------------------------
AQI_LABEL = "Six-pollutant estimated AQI using CPCB sub-index methodology."
AQI_CAVEAT = ("NH3 and Pb are unavailable, so this is not the official CPCB AQI. "
              "O3/CO values are Open-Meteo/CAMS-based estimates, not direct CPCB ground-station observations.")
FOOTER = ("Six-pollutant estimated AQI (CPCB sub-index methodology); not official CPCB AQI (NH3, Pb unavailable). "
          "Decision support only; no definitive source attribution.")

# ---- schema ---------------------------------------------------------------------------------------------------
CATEGORIES = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
CAT_CODE = {c: i for i, c in enumerate(CATEGORIES)}
SEASONS = ["Winter", "Summer", "Monsoon", "Post-monsoon"]
POLLUTANT_ORDER = ["PM2_5", "PM10", "NO2", "SO2", "O3", "CO"]
POLL_COLORS = {"PM2_5": "#C44E52", "PM10": "#DD8452", "NO2": "#4C72B0", "SO2": "#937860", "O3": "#55A868", "CO": "#8172B2"}
FAMILY_COLORS = {"Persistence": "#BBBBBB", "Classical ML": "#6B8EBF", "Deep learning": "#C44E52"}
P_VARS = ["PM2_5", "PM10", "CO", "NO2", "SO2", "O3", "O3_8hr_max", "CO_8hr_max_mg_m3", "Estimated_AQI"]
W_VARS = ["Temperature_C", "Humidity_pct", "WindSpeed_kmh", "Precipitation_mm"]
POLL_INTERACTIONS = ["PM2_5_to_PM10_ratio", "NO2_plus_CO", "SO2_to_PM2_5_ratio"]
WEATHER_INTERACTIONS = ["Temperature_x_O3", "WindSpeed_x_PM2_5", "Rainfall_x_PM10", "Humidity_x_PM2_5"]
REGIME_BASE = ["PM2_5", "PM10", "NO2", "SO2", "O3", "CO", "Temperature_C", "Humidity_pct", "WindSpeed_kmh", "Precipitation_mm"]
TASK_PRIMARY = {"regression": ("RMSE", False), "classification": ("Macro_F1", True), "highrisk": ("PR_AUC", True)}

PROFILES = {
    "standard": dict(hgb_iter=250, hgb_lr=0.07, hgb_depth=6, ablation_train_rows=None, n_boot=500,
                     perm_rows=4000, perm_repeats=3, ablation_tasks=["regression", "classification", "highrisk"]),
    "fast": dict(hgb_iter=40, hgb_lr=0.15, hgb_depth=4, ablation_train_rows=12000, n_boot=100,
                 perm_rows=800, perm_repeats=2, ablation_tasks=["regression", "highrisk"]),
}
CFG = PROFILES[PROFILE]
BLOCK_LEN = 7                                                           # moving-block bootstrap, 7 days
ALERT_HIGH_CONF = 0.80


def banner(text):
    print("\n" + "=" * 78 + f"\n{text}\n" + "=" * 78)


def sha256_of(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


def fmt(x, nd=3):
    try:
        return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{float(x):.{nd}f}"
    except Exception:
        return str(x)


def md_table(df, nd=3):
    if df is None or len(df) == 0:
        return "_(no data)_"
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    for _, r in df.iterrows():
        cells = []
        for c in cols:
            v = r[c]
            cells.append(fmt(v, nd) if isinstance(v, (float, np.floating)) else str(v))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


def category_code(aqi):
    return np.searchsorted([50, 100, 200, 300, 400], np.asarray(aqi, dtype=float), side="left")


# ----------------------------------------------------------------------------
# 1. Metrics and per-date sufficient statistics (for the block bootstrap)
# ----------------------------------------------------------------------------
def reg_metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"N": len(y), "MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "R2": r2_score(y, p) if len(y) > 1 and np.var(y) > 0 else np.nan, "Bias": float(np.mean(p - y))}


def cls_metrics(y, p):
    return {"N": len(y), "Accuracy": accuracy_score(y, p), "Macro_F1": f1_score(y, p, average="macro", zero_division=0),
            "Balanced_Accuracy": balanced_accuracy_score(y, p)}


def hr_metrics(y, proba, pred):
    y, proba, pred = np.asarray(y).astype(int), np.asarray(proba, float), np.asarray(pred).astype(int)
    out = {"N": len(y), "Positive_Rate": float(y.mean()), "Precision": precision_score(y, pred, zero_division=0),
           "Recall": recall_score(y, pred, zero_division=0), "F1": f1_score(y, pred, zero_division=0)}
    if len(np.unique(y)) == 2:
        out["ROC_AUC"], out["PR_AUC"] = roc_auc_score(y, proba), average_precision_score(y, proba)
    else:
        out["ROC_AUC"], out["PR_AUC"] = np.nan, np.nan
    return out


def best_f1_threshold(y, proba):
    grid = np.linspace(0.05, 0.95, 91)
    return float(grid[int(np.argmax([f1_score(y, (proba >= t).astype(int), zero_division=0) for t in grid]))])


def per_date_stats(task, dates, y, pred):
    """Additive per-date statistics so any metric can be recomputed on resampled date blocks."""
    ud, inv = np.unique(np.asarray(dates), return_inverse=True)
    D = len(ud)
    y, pred = np.asarray(y), np.asarray(pred)
    if task == "regression":
        err2 = (y.astype(float) - pred.astype(float)) ** 2
        return np.column_stack([np.bincount(inv, weights=err2, minlength=D), np.bincount(inv, minlength=D)])
    if task == "classification":
        idx = inv * 36 + y.astype(int) * 6 + pred.astype(int)
        return np.bincount(idx, minlength=D * 36).reshape(D, 36).astype(float)
    y, pred = y.astype(int), pred.astype(int)
    tp = np.bincount(inv, weights=((y == 1) & (pred == 1)).astype(float), minlength=D)
    fp = np.bincount(inv, weights=((y == 0) & (pred == 1)).astype(float), minlength=D)
    fn = np.bincount(inv, weights=((y == 1) & (pred == 0)).astype(float), minlength=D)
    return np.column_stack([tp, fp, fn])


def metric_from_stats(task, S):
    """S: (..., D, k) -> metric over the last-but-one axis (summed)."""
    tot = S.sum(axis=-2)
    if task == "regression":
        return np.sqrt(tot[..., 0] / np.maximum(tot[..., 1], 1))
    if task == "classification":
        cm = tot.reshape(tot.shape[:-1] + (6, 6))
        tp = np.einsum("...ii->...i", cm)
        sup, pr = cm.sum(axis=-1), cm.sum(axis=-2)
        f1 = np.where(sup + pr > 0, 2 * tp / np.maximum(sup + pr, 1), np.nan)
        return np.nanmean(f1, axis=-1)
    tp, fp, fn = tot[..., 0], tot[..., 1], tot[..., 2]
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)


def block_bootstrap_delta(task, stats_a, stats_b, n_boot=500, seed=SEED):
    """Moving-block (circular) bootstrap over dates of metric(A) - metric(B) on the SAME resampled dates."""
    D = stats_a.shape[0]
    rng = np.random.RandomState(seed)
    n_blocks = int(np.ceil(D / BLOCK_LEN))
    starts = rng.randint(0, D, size=(n_boot, n_blocks))
    idx = ((starts[:, :, None] + np.arange(BLOCK_LEN)[None, None, :]) % D).reshape(n_boot, -1)[:, :D]
    da = metric_from_stats(task, stats_a[idx]) - metric_from_stats(task, stats_b[idx])
    point = float(metric_from_stats(task, stats_a) - metric_from_stats(task, stats_b))
    return point, float(np.nanpercentile(da, 2.5)), float(np.nanpercentile(da, 97.5))


# ----------------------------------------------------------------------------
# 2. Load and harmonise Parts 3, 5 and 6 results
# ----------------------------------------------------------------------------
def load_all():
    missing = [p for p in REQ_PATHS.values() if not os.path.isfile(p)]
    if missing:
        raise FileNotFoundError("Missing required result files (run the earlier parts in the same BASE_DIR="
                                f"{BASE_DIR}):\n  " + "\n  ".join(missing))
    R = {}
    for k, p in REQ_PATHS.items():
        if k == "daily":
            use = ["Date", "City", "Season", "PM2_5", "PM10", "CO", "NO2", "SO2", "O3", "Temperature_C", "Humidity_pct",
                   "WindSpeed_kmh", "Precipitation_mm", "Estimated_AQI", "AQI_Category", "Dominant_Pollutant",
                   "Severe_Capped_Flag", "AQI_Uses_Provisional_8hr", "Likely_Cause", "Recommended_Actions"]
            R[k] = pd.read_csv(p, parse_dates=["Date"], usecols=lambda c: c in use)
        elif k in ("pva", "dl_pva", "hr", "dl_hr"):
            R[k] = pd.read_csv(p, parse_dates=["Date"])
        else:
            R[k] = pd.read_csv(p)
    for k, p in OPT.items():
        if os.path.isfile(p):
            R[k] = json.load(open(p)) if p.endswith(".json") else pd.read_csv(p)
    R["manifests"] = {i: json.load(open(p)) for i, p in MANIFESTS.items() if os.path.isfile(p)}
    return R


def harmonise(R):
    """Inner-join Part 5 and Part 6 predictions on (Split, Date, City) so every model is scored on identical rows."""
    keys = ["Split", "Date", "City"]
    a, b = R["pva"], R["dl_pva"]
    ml_models = [c[len("Pred_AQI_"):] for c in a.columns if c.startswith("Pred_AQI_") and not c.endswith("Persistence")]
    dl_models = [c[len("Pred_AQI_"):] for c in b.columns if c.startswith("Pred_AQI_")]
    fam = {"Persistence": "Persistence", **{m: "Classical ML" for m in ml_models}, **{m: "Deep learning" for m in dl_models}}
    ca = keys + ["Season", "Current_AQI", "Actual_AQI_tplus1", "Actual_Category_tplus1"] + \
        [c for c in a.columns if c.startswith(("Pred_AQI_", "Pred_Category_"))]
    cb = keys + ["Actual_AQI_tplus1"] + [c for c in b.columns if c.startswith(("Pred_AQI_", "Pred_Category_"))]
    H = a[ca].merge(b[cb].rename(columns={"Actual_AQI_tplus1": "Actual_dl"}), on=keys, how="inner")
    diff = float(np.abs(H["Actual_AQI_tplus1"] - H["Actual_dl"]).max()) if len(H) else np.nan
    info = {"rows_part5": len(a), "rows_part6": len(b), "rows_common": len(H), "max_abs_actual_diff": diff}
    if diff > 0.5:
        print(f"  ! Actual AQI values differ between Part 5 and Part 6 files (max {diff:.3f}).")
    H["Actual_Cat_code"] = H["Actual_Category_tplus1"].map(CAT_CODE)
    H["Current_Cat_code"] = category_code(H["Current_AQI"])
    reg_models = [m for m in fam if f"Pred_AQI_{m}" in H.columns]
    cls_models = [m for m in fam if f"Pred_Category_{m}" in H.columns]
    for m in cls_models:
        H[f"CatCode_{m}"] = H[f"Pred_Category_{m}"].map(CAT_CODE)

    ha, hb = R["hr"], R["dl_hr"]
    hk = keys
    prob_a = [c for c in ha.columns if c.startswith("Prob_")]
    prob_b = [c for c in hb.columns if c.startswith("Prob_")]
    thr_cols = {}
    for c in list(ha.columns) + list(hb.columns):
        mm = re.match(r"^Pred_(.+)_valopt_thr([0-9.]+)$", c)
        if mm:
            thr_cols[mm.group(1)] = (c, float(mm.group(2)))
    Hh = ha[hk + ["Season", "Current_AQI", "Actual_HighRisk", "Pred_Persistence"] + prob_a +
            [c for c, _ in thr_cols.values() if c in ha.columns]].merge(
        hb[hk + prob_b + [c for c, _ in thr_cols.values() if c in hb.columns]], on=hk, how="inner")
    hr_models = ["Persistence"] + [c[len("Prob_"):] for c in prob_a + prob_b]
    hr_fam = {"Persistence": "Persistence", **{c[len("Prob_"):]: "Classical ML" for c in prob_a},
              **{c[len("Prob_"):]: "Deep learning" for c in prob_b}}
    info.update(hr_rows_common=len(Hh))
    return H, Hh, fam, hr_fam, reg_models, cls_models, hr_models, thr_cols, info


# ----------------------------------------------------------------------------
# 3. Comparison table, best models, bootstrap
# ----------------------------------------------------------------------------
def build_comparison(H, Hh, fam, hr_fam, reg_models, cls_models, hr_models, thr_cols):
    rows = []
    for m in reg_models:
        for split in ("validation", "test"):
            d = H[H["Split"] == split]
            rows.append({"Task": "regression", "Model": m, "Family": fam[m], "Split": split,
                         **reg_metrics(d["Actual_AQI_tplus1"], d[f"Pred_AQI_{m}"])})
    for m in cls_models:
        for split in ("validation", "test"):
            d = H[H["Split"] == split]
            rows.append({"Task": "classification", "Model": m, "Family": fam[m], "Split": split,
                         **cls_metrics(d["Actual_Cat_code"], d[f"CatCode_{m}"])})
    thresholds = {}
    for m in hr_models:
        for split in ("validation", "test"):
            d = Hh[(Hh["Split"] == split) & Hh["Actual_HighRisk"].notna()]
            if m == "Persistence":
                p = d["Pred_Persistence"].to_numpy(float)
                rows.append({"Task": "highrisk", "Model": m, "Family": "Persistence", "Split": split, "Threshold": np.nan,
                             **hr_metrics(d["Actual_HighRisk"], p, p)})
            else:
                pr = d[f"Prob_{m}"].to_numpy(float)
                if m in thr_cols and thr_cols[m][0] in d.columns:
                    thr = thr_cols[m][1]
                else:
                    thr = 0.5
                thresholds[m] = thr
                rows.append({"Task": "highrisk", "Model": m, "Family": hr_fam[m], "Split": split, "Threshold": thr,
                             **hr_metrics(d["Actual_HighRisk"], pr, (pr >= thr).astype(int))})
    comp = pd.DataFrame(rows)
    # best by VALIDATION (persistence excluded from the candidates)
    comp["Rank_by_Validation"] = np.nan
    comp["Best_By_Validation"] = False
    for task, (met, hb) in TASK_PRIMARY.items():
        v = comp[(comp["Task"] == task) & (comp["Split"] == "validation") & (comp["Model"] != "Persistence")]
        order = v.sort_values(met, ascending=not hb)["Model"].tolist()
        for i, m in enumerate(order, 1):
            comp.loc[(comp["Task"] == task) & (comp["Model"] == m), "Rank_by_Validation"] = i
        if order:
            comp.loc[(comp["Task"] == task) & (comp["Model"] == order[0]), "Best_By_Validation"] = True
    first = ["Task", "Family", "Model", "Split", "N"]
    comp = comp[first + [c for c in comp.columns if c not in first]]
    best = {t: comp[(comp["Task"] == t) & comp["Best_By_Validation"]]["Model"].iloc[0] for t in TASK_PRIMARY
            if comp[(comp["Task"] == t) & comp["Best_By_Validation"]].shape[0]}
    return comp, best, thresholds


def best_in_family(comp, task, family):
    met, hb = TASK_PRIMARY[task]
    v = comp[(comp["Task"] == task) & (comp["Split"] == "validation") & (comp["Family"] == family)]
    if v.empty:
        return None
    return v.sort_values(met, ascending=not hb).iloc[0]["Model"]


def model_stats(task, model, H, Hh, thresholds, split="test"):
    if task == "regression":
        d = H[H["Split"] == split]
        return per_date_stats(task, d["Date"], d["Actual_AQI_tplus1"], d[f"Pred_AQI_{model}"])
    if task == "classification":
        d = H[H["Split"] == split]
        return per_date_stats(task, d["Date"], d["Actual_Cat_code"], d[f"CatCode_{model}"])
    d = Hh[(Hh["Split"] == split) & Hh["Actual_HighRisk"].notna()]
    pred = d["Pred_Persistence"] if model == "Persistence" else (d[f"Prob_{model}"] >= thresholds.get(model, 0.5))
    return per_date_stats(task, d["Date"], d["Actual_HighRisk"], pred.astype(int))


def build_bootstrap(comp, best, H, Hh, thresholds):
    rows = []
    for task, (met, hb) in TASK_PRIMARY.items():
        if task not in best:
            continue
        pairs = [(best[task], "Persistence", "best vs persistence")]
        b_ml, b_dl = best_in_family(comp, task, "Classical ML"), best_in_family(comp, task, "Deep learning")
        if b_ml and b_dl:
            pairs.append((b_dl, b_ml, "best deep learning vs best classical ML"))
        for a, b, label in pairs:
            sa, sb = model_stats(task, a, H, Hh, thresholds), model_stats(task, b, H, Hh, thresholds)
            if sa.shape != sb.shape:
                continue
            tmet = {"regression": "RMSE", "classification": "Macro_F1", "highrisk": "F1 (val-opt threshold)"}[task]
            point, lo, hi = block_bootstrap_delta(task, sa, sb, CFG["n_boot"])
            better = (hi < 0) if task == "regression" else (lo > 0)
            worse = (lo > 0) if task == "regression" else (hi < 0)
            rows.append({"Task": task, "Comparison": label, "Model_A": a, "Model_B": b, "Metric": tmet,
                         "Delta_A_minus_B": point, "CI95_low": lo, "CI95_high": hi,
                         "Reading": "A clearly better" if better else ("A clearly worse" if worse else "no clear difference"),
                         "Bootstrap": f"{CFG['n_boot']} draws, {BLOCK_LEN}-day moving blocks over test dates"})
    return pd.DataFrame(rows)


def group_metrics(H, Hh, best, fam, thresholds, region_map):
    """Test-period metrics by season / city / region for persistence, best ML, best DL and the overall best model."""
    rows = []
    sel = {"regression": ["Persistence"], "classification": ["Persistence"], "highrisk": ["Persistence"]}
    for task in sel:
        for m in {best.get(task)} | {best_in_family(COMP_GLOBAL, task, "Classical ML"), best_in_family(COMP_GLOBAL, task, "Deep learning")}:
            if m and m not in sel[task]:
                sel[task].append(m)
    for task, models in sel.items():
        d0 = (H if task != "highrisk" else Hh[Hh["Actual_HighRisk"].notna()])
        d0 = d0[d0["Split"] == "test"].copy()
        d0["Region"] = d0["City"].map(region_map)
        for m in models:
            for gcol in ("Season", "City", "Region"):
                for g, d in d0.groupby(gcol):
                    if task == "regression":
                        r = reg_metrics(d["Actual_AQI_tplus1"], d[f"Pred_AQI_{m}"])
                    elif task == "classification":
                        r = cls_metrics(d["Actual_Cat_code"], d[f"CatCode_{m}"])
                    else:
                        if m == "Persistence":
                            p = d["Pred_Persistence"].to_numpy(float)
                            r = hr_metrics(d["Actual_HighRisk"], p, p)
                        else:
                            pr = d[f"Prob_{m}"].to_numpy(float)
                            r = hr_metrics(d["Actual_HighRisk"], pr, (pr >= thresholds.get(m, 0.5)).astype(int))
                    rows.append({"Task": task, "Model": m, "Family": (fam.get(m) or HR_FAM_GLOBAL.get(m, "")),
                                 "Group_Type": gcol, "Group": g, **r})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# 4. Regime awareness (K-Means fitted on TRAIN only) and ablation
# ----------------------------------------------------------------------------
def load_splits(fcols):
    meta_cols = fcols["meta_columns"]
    feats = fcols["feature_columns"]
    targets = fcols["target_regression"] + fcols["target_classification"]
    use = list(dict.fromkeys(meta_cols + feats + targets))
    S = {k: pd.read_csv(p, parse_dates=["Date"], usecols=lambda c: c in use) for k, p in SPLIT_FILES.items()}
    sc = {k: pd.read_csv(p, parse_dates=["Date"], usecols=lambda c: c in (["Date", "City"] + REGIME_BASE)) for k, p in SPLIT_S_FILES.items()}
    for k in S:
        if not S[k][["Date", "City"]].reset_index(drop=True).equals(sc[k][["Date", "City"]].reset_index(drop=True)):
            raise ValueError(f"{k}: scaled and unscaled split files are not row-aligned.")
    return S, sc


def fit_regimes(S, sc):
    """Daily pollution-weather regimes: K-Means on TRAIN-split scaled day-t profiles; applied to all splits."""
    if "Regime_KMeans" in S["train"].columns:
        print("  Using the K-Means regime labels already present in the Part 4 files.")
        return S, None, None
    Xtr = np.clip(sc["train"][REGIME_BASE].to_numpy(float), -5, 10)
    sub = RNG.choice(len(Xtr), size=min(30000, len(Xtr)), replace=False)
    sil_idx = RNG.choice(len(sub), size=min(5000, len(sub)), replace=False)
    scores = {}
    for k in range(3, 8):
        km = KMeans(n_clusters=k, n_init=5, random_state=SEED).fit(Xtr[sub])
        scores[k] = float(silhouette_score(Xtr[sub][sil_idx], km.labels_[sil_idx]))
    k_best = max(scores, key=scores.get)
    km = KMeans(n_clusters=k_best, n_init=10, random_state=SEED).fit(Xtr)
    order = np.argsort(-pd.Series(Xtr[:, 0]).groupby(km.labels_).mean().reindex(range(k_best)).to_numpy())  # high PM2.5 first
    rank = np.empty(k_best, int)
    rank[order[::-1]] = np.arange(k_best)                                  # 0 = cleanest PM2.5 regime
    s2 = float(np.median(np.min(((Xtr[:2000, None, :] - km.cluster_centers_[None]) ** 2).sum(-1), axis=1)))
    prof_rows, lab_frames = [], []
    for k, df in S.items():
        X = np.clip(sc[k][REGIME_BASE].to_numpy(float), -5, 10)
        lab = rank[km.predict(X)]
        d2 = ((X[:, None, :] - km.cluster_centers_[None]) ** 2).sum(-1)
        pr = np.exp(-(d2 - d2.min(axis=1, keepdims=True)) / (2 * max(s2, 1e-6)))
        pr = pr / pr.sum(axis=1, keepdims=True)
        pr = pr[:, np.argsort(rank)]                                       # columns ordered by new label
        df["Regime_KMeans"] = lab
        for j in range(k_best):
            df[f"Regime_prob_{j}"] = pr[:, j]
        lab_frames.append(df[["Date", "City", "Regime_KMeans"] + [f"Regime_prob_{j}" for j in range(k_best)]].assign(Split=k))
    labels = pd.concat(lab_frames, ignore_index=True)
    tr = S["train"]
    prof = tr.groupby("Regime_KMeans")[["PM2_5", "PM10", "NO2", "SO2", "O3", "CO", "Temperature_C", "Humidity_pct",
                                        "WindSpeed_kmh", "Precipitation_mm", "Estimated_AQI"]].mean()
    prof["Share_of_train_days_pct"] = 100 * tr["Regime_KMeans"].value_counts(normalize=True).sort_index()
    seas = pd.crosstab(tr["Regime_KMeans"], tr["Season"], normalize="index").reindex(columns=SEASONS).fillna(0) * 100
    prof = prof.join(seas.add_prefix("Pct_in_"))
    prof.insert(0, "n_regimes", k_best)
    prof["Silhouette_for_chosen_k"] = scores[k_best]
    prof = prof.reset_index()
    print(f"  K-Means regimes fitted on TRAIN only: k = {k_best} (silhouette by k: "
          + ", ".join(f"{k}:{v:.3f}" for k, v in scores.items()) + ")")
    return S, labels, prof


def design_frame(df, regime_cols):
    X = df.copy()
    for i, p in enumerate(POLLUTANT_ORDER):
        X[f"Dominant_t_{p}"] = (X["Dominant_Pollutant_t_code"] == i).astype(float) if "Dominant_Pollutant_t_code" in X else 0.0
    return X


def feature_sets(fcols, regime_cols):
    feats = fcols["feature_columns"]
    groups = fcols.get("feature_groups", {})

    def fam_cols(vs):
        return [c for c in feats if any(c == v or c.startswith(v + "_lag") or c.startswith(v + "_roll") for v in vs)]
    dom = [f"Dominant_t_{p}" for p in POLLUTANT_ORDER]
    base = fam_cols(P_VARS) + [c for c in POLL_INTERACTIONS if c in feats] + \
        [c for c in ["Provisional_8hr_flag_t"] if c in feats] + dom
    weather = fam_cols(W_VARS) + [c for c in WEATHER_INTERACTIONS if c in feats]
    cal = [c for c in groups.get("calendar", []) + groups.get("calendar_cyclic", []) if c in feats]
    city = [c for c in ["City_ID", "Region_ID", "Latitude", "Longitude"] if c in feats]
    steps = {"1_base_pollutant_lags_rolling": list(dict.fromkeys(base))}
    steps["2_plus_weather"] = list(dict.fromkeys(steps["1_base_pollutant_lags_rolling"] + weather))
    steps["3_plus_season_calendar"] = list(dict.fromkeys(steps["2_plus_weather"] + cal))
    steps["4_plus_city"] = list(dict.fromkeys(steps["3_plus_season_calendar"] + city))
    steps["5_full_plus_regime"] = list(dict.fromkeys(steps["4_plus_city"] + regime_cols))
    used = set(steps["5_full_plus_regime"])
    leftover = [c for c in feats if c not in used and c != "Dominant_Pollutant_t_code"]
    return steps, leftover, {"base": base, "weather": weather, "calendar": cal, "city": city, "regime": regime_cols}


def make_hgb(task, cat_features):
    kw = dict(max_iter=CFG["hgb_iter"], learning_rate=CFG["hgb_lr"], max_depth=CFG["hgb_depth"], min_samples_leaf=40,
              l2_regularization=1.0, early_stopping=False, random_state=SEED, categorical_features=cat_features or None)
    return HistGradientBoostingRegressor(**kw) if task == "regression" else HistGradientBoostingClassifier(**kw)


def prep(df, cols, cat_cols):
    X = df[cols].copy()
    for c in cols:
        X[c] = X[c].astype(int) if c in cat_cols else X[c].astype(np.float32)
    return X


def run_ablation(S, fcols, steps, regime_present):
    cats_all = {"City_ID", "Region_ID", "Season_code", "Regime_KMeans"}
    D = {k: design_frame(S[k], None) for k in S}
    tr = D["train"]
    if CFG["ablation_train_rows"] and len(tr) > CFG["ablation_train_rows"]:
        tr = tr.iloc[np.sort(RNG.choice(len(tr), CFG["ablation_train_rows"], replace=False))]
    ycol = {"regression": "Estimated_AQI_tplus1", "classification": "Estimated_AQI_Category_tplus1_code",
            "highrisk": "HighRisk_AQI_gt200_next3d"}
    rows, preds, models, stats_store = [], {}, {}, {}
    group_added = {"1_base_pollutant_lags_rolling": "pollutant lags/rolling statistics (incl. AQI history)",
                   "2_plus_weather": "weather lags/rolling + weather x pollutant interactions",
                   "3_plus_season_calendar": "calendar, season, cyclic terms",
                   "4_plus_city": "City_ID, Region_ID, latitude, longitude",
                   "5_full_plus_regime": "K-Means regime label and probabilities"}
    for task in CFG["ablation_tasks"]:
        y_tr_all = tr[ycol[task]]
        mtr = y_tr_all.notna().to_numpy()
        prev_stats, prev_step = None, None
        for step, cols in steps.items():
            if step.startswith("5_") and not regime_present:
                continue
            cat_cols = [c for c in cols if c in cats_all]
            Xtr = prep(tr[mtr], cols, set(cat_cols))
            t0 = time.time()
            mdl = make_hgb(task, cat_cols).fit(Xtr, tr.loc[mtr, ycol[task]].astype(int if task != "regression" else float))
            models[(task, step)] = (mdl, cols, set(cat_cols))
            out = {"Task": task, "Step": step, "Added_Feature_Group": group_added[step], "N_Features": len(cols),
                   "Fit_Seconds": round(time.time() - t0, 1)}
            for split in ("validation", "test"):
                df = D[split]
                mk = df[ycol[task]].notna().to_numpy()
                X = prep(df[mk], cols, set(cat_cols))
                y = df.loc[mk, ycol[task]].to_numpy()
                if task == "regression":
                    p = np.clip(mdl.predict(X), 0, 500)
                    met = reg_metrics(y, p)
                    preds[(task, step, split)] = (df.loc[mk, "Date"].to_numpy(), y, p)
                elif task == "classification":
                    P = mdl.predict_proba(X)
                    full = np.zeros((len(X), 6))
                    full[:, mdl.classes_] = P
                    p = full.argmax(axis=1)
                    met = cls_metrics(y.astype(int), p)
                    preds[(task, step, split)] = (df.loc[mk, "Date"].to_numpy(), y.astype(int), p)
                else:
                    pr = mdl.predict_proba(X)[:, 1]
                    if split == "validation":
                        thr = best_f1_threshold(y.astype(int), pr)
                        out["Threshold_val_F1_optimal"] = thr
                    met = hr_metrics(y.astype(int), pr, (pr >= out["Threshold_val_F1_optimal"]).astype(int))
                    preds[(task, step, split)] = (df.loc[mk, "Date"].to_numpy(), y.astype(int),
                                                  (pr >= out["Threshold_val_F1_optimal"]).astype(int))
                for k, v in met.items():
                    out[f"{split}_{k}"] = v
            st = per_date_stats(task, *[preds[(task, step, "test")][i] for i in (0, 1, 2)])
            if prev_stats is not None and prev_stats.shape == st.shape:
                point, lo, hi = block_bootstrap_delta(task, st, prev_stats, CFG["n_boot"])
                prim = {"regression": "RMSE", "classification": "Macro_F1", "highrisk": "F1"}[task]
                out.update({"Delta_vs_previous_step_metric": prim, "Delta_vs_previous_step": point,
                            "Delta_CI95_low": lo, "Delta_CI95_high": hi})
                better = (hi < 0) if task == "regression" else (lo > 0)
                worse = (lo > 0) if task == "regression" else (hi < 0)
                out["Verdict"] = ("improves (CI excludes 0)" if better else
                                  "worsens (CI excludes 0)" if worse else "no clear change (CI includes 0)")
            else:
                out["Verdict"] = "reference step"
            out["Compared_with"] = prev_step or ""
            rows.append(out)
            prev_stats, prev_step = st, step
            print(f"    ablation {task:>14} / {step:<32} {len(cols):>4} features  ({out['Fit_Seconds']:.0f}s)")
    return pd.DataFrame(rows), models, D


# ----------------------------------------------------------------------------
# 5. Explainability (grouped permutation, held-out validation + test rows)
# ----------------------------------------------------------------------------
def importance_groups(cols, parts):
    g = {}
    for v in P_VARS + W_VARS:
        sel = [c for c in cols if c == v or c.startswith(v + "_lag") or c.startswith(v + "_roll")]
        if sel:
            g[f"{v} (current+lags+rolling)"] = sel
    for name, lst in (("Calendar & season", parts["calendar"]), ("Weather x pollutant interactions", WEATHER_INTERACTIONS),
                      ("Pollutant interactions", POLL_INTERACTIONS), ("City identity (ID/region/lat/lon)", parts["city"]),
                      ("Regime (K-Means)", parts["regime"]),
                      ("Dominant pollutant at t", [c for c in cols if c.startswith("Dominant_t_")]),
                      ("Provisional-data flag", ["Provisional_8hr_flag_t"])):
        sel = [c for c in lst if c in cols]
        if sel:
            g[name] = sel
    return g


def grouped_permutation(model, cols, cat_cols, df, ycol, groups, n_rows, repeats, rng):
    mk = df[ycol].notna().to_numpy()
    d = df[mk]
    if len(d) > n_rows:
        d = d.iloc[np.sort(rng.choice(len(d), n_rows, replace=False))]
    X = prep(d, cols, cat_cols)
    y = d[ycol].to_numpy(float)
    dtypes = X.dtypes
    A = X.to_numpy()
    colpos = {c: i for i, c in enumerate(cols)}

    def pred(arr):
        return np.clip(model.predict(pd.DataFrame(arr, columns=cols).astype(dtypes)), 0, 500)
    base = float(np.sqrt(mean_squared_error(y, pred(A))))
    out = []
    for gname, gcols in groups.items():
        idx = [colpos[c] for c in gcols if c in colpos]
        vals = []
        for _ in range(repeats):
            Ap = A.copy()
            Ap[:, idx] = A[rng.permutation(len(A))][:, idx]
            vals.append(float(np.sqrt(mean_squared_error(y, pred(Ap)))) - base)
        out.append((gname, float(np.mean(vals)), float(np.std(vals))))
    return out, base, len(d)


def local_occlusion(model, cols, cat_cols, x_row, train_means, groups):
    A = prep(x_row, cols, cat_cols)
    base = float(model.predict(A)[0])
    res = []
    for gname, gcols in groups.items():
        B = A.copy()
        for c in gcols:
            if c in B.columns and c not in cat_cols:
                B[c] = np.float32(train_means.get(c, 0.0))
        res.append((gname, float(model.predict(B)[0]) - base))
    return base, sorted(res, key=lambda t: -abs(t[1]))


# ----------------------------------------------------------------------------
# 6. Figures
# ----------------------------------------------------------------------------
FIG_LOG = []


def save_fig(fig, name, title=None, dpi=130):
    if title:
        fig.suptitle(title, fontsize=13, y=0.998)
    fig.text(0.5, 0.002, FOOTER, ha="center", va="bottom", fontsize=7, color="dimgray")
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return path


def run_plot(fn, ctx):
    try:
        paths = fn(ctx)
        paths = [paths] if isinstance(paths, str) else (paths or [])
        FIG_LOG.extend(paths)
        print(f"  [ok] {fn.__name__}")
    except Exception as ex:
        print(f"  [FAILED] {fn.__name__}: {type(ex).__name__}: {ex}")
        FIG_LOG.append(f"FAILED:{fn.__name__}:{type(ex).__name__}")


def fig_model_comparison(ctx):
    comp = ctx["comp"]
    fig, ax = plt.subplots(3, 3, figsize=(19, 14))
    spec = [("regression", ["RMSE", "MAE", "R2"]), ("classification", ["Macro_F1", "Balanced_Accuracy", "Accuracy"]),
            ("highrisk", ["PR_AUC", "ROC_AUC", "F1"])]
    for r, (task, mets) in enumerate(spec):
        for c, met in enumerate(mets):
            a = ax[r, c]
            d = comp[(comp["Task"] == task) & (comp["Split"] == "test")].copy()
            asc = task == "regression" and met != "R2"
            d = d.sort_values(met, ascending=not asc if met in ("R2",) else asc)
            colors = [FAMILY_COLORS[f] for f in d["Family"]]
            bars = a.barh(d["Model"], d[met], color=colors, edgecolor=["k" if b else "none" for b in d["Best_By_Validation"]],
                          linewidth=1.8)
            a.set_title(f"{task}: test {met}" + (" (lower is better)" if met in ("RMSE", "MAE") else " (higher is better)"))
            a.invert_yaxis()
    ax[0, 0].legend(handles=[Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()] +
                    [Patch(facecolor="white", edgecolor="k", linewidth=1.8, label="best on validation")], fontsize=8)
    return save_fig(fig, "f01_model_comparison.png", "Final model comparison on identical test rows (persistence vs classical ML vs deep learning)")


def fig_region_city(ctx):
    g = ctx["group_metrics"]
    d = g[(g["Task"] == "regression")]
    fig, ax = plt.subplots(2, 2, figsize=(18, 12.5))
    reg = d[d["Group_Type"] == "Region"]
    models = list(dict.fromkeys(reg["Model"]))
    regions = sorted(reg["Group"].dropna().unique())
    x = np.arange(len(regions))
    for k, m in enumerate(models):
        v = [reg[(reg["Model"] == m) & (reg["Group"] == r)]["RMSE"].mean() for r in regions]
        ax[0, 0].bar(x + (k - (len(models) - 1) / 2) * 0.8 / len(models), v, width=0.8 / len(models), label=m,
                     color=FAMILY_COLORS.get(ctx["fam"].get(m, "Persistence")), alpha=0.5 + 0.5 * (k / max(len(models) - 1, 1)))
    ax[0, 0].set_xticks(x, regions, rotation=30, ha="right")
    ax[0, 0].set_title("Regression RMSE by region (test)")
    ax[0, 0].legend(fontsize=8)
    c = g[(g["Task"] == "classification") & (g["Group_Type"] == "Region")]
    for k, m in enumerate(list(dict.fromkeys(c["Model"]))):
        v = [c[(c["Model"] == m) & (c["Group"] == r)]["Macro_F1"].mean() for r in regions]
        ax[0, 1].bar(x + (k - 1) * 0.27, v, width=0.27, label=m, color=FAMILY_COLORS.get(ctx["fam"].get(m, "Persistence")),
                     alpha=0.5 + 0.25 * k)
    ax[0, 1].set_xticks(x, regions, rotation=30, ha="right")
    ax[0, 1].set_title("Category macro-F1 by region (test)")
    ax[0, 1].legend(fontsize=8)
    best = ctx["best"]["regression"]
    cty = d[(d["Group_Type"] == "City") & (d["Model"] == best)].sort_values("RMSE", ascending=False)
    ax[1, 0].barh(cty.head(15)["Group"][::-1], cty.head(15)["RMSE"][::-1], color="#C44E52")
    ax[1, 0].set_title(f"{best}: 15 cities with highest test RMSE")
    ax[1, 1].barh(cty.tail(15)["Group"][::-1], cty.tail(15)["RMSE"][::-1], color="#4C72B0")
    ax[1, 1].set_title(f"{best}: 15 cities with lowest test RMSE")
    return save_fig(fig, "f02_region_city_performance.png", "Region- and city-wise test performance")


def fig_season(ctx):
    g = ctx["group_metrics"]
    g = g[g["Group_Type"] == "Season"]
    fig, ax = plt.subplots(1, 3, figsize=(19, 5.5))
    for a, (task, met) in zip(ax, (("regression", "RMSE"), ("classification", "Macro_F1"), ("highrisk", "PR_AUC"))):
        d = g[g["Task"] == task]
        seasons = [s for s in SEASONS if s in set(d["Group"])]
        models = list(dict.fromkeys(d["Model"]))
        x = np.arange(len(seasons))
        for k, m in enumerate(models):
            v = [d[(d["Model"] == m) & (d["Group"] == s)][met].mean() for s in seasons]
            a.bar(x + (k - (len(models) - 1) / 2) * 0.8 / len(models), v, width=0.8 / len(models), label=m,
                  color=FAMILY_COLORS.get(ctx["fam"].get(m, "Persistence")), alpha=0.45 + 0.55 * (k / max(len(models) - 1, 1)))
        a.set_xticks(x, seasons)
        a.set_title(f"{task}: test {met} by season")
        a.legend(fontsize=7)
    return save_fig(fig, "f03_season_performance.png", "Season-wise test performance (the test period contains no post-monsoon days if absent here)")


def fig_pred_actual(ctx):
    H = ctx["H"][ctx["H"]["Split"] == "test"]
    models = ["Persistence", ctx["best_ml"], ctx["best_dl"], ctx["best"]["regression"]]
    models = list(dict.fromkeys([m for m in models if m]))
    s = H.sample(min(8000, len(H)), random_state=SEED)
    lim = float(max(H["Actual_AQI_tplus1"].max(), 1))
    fig, ax = plt.subplots(2, 2, figsize=(14, 13))
    for a, m in zip(ax.ravel(), models):
        a.scatter(s["Actual_AQI_tplus1"], s[f"Pred_AQI_{m}"], s=4, alpha=0.25, color=FAMILY_COLORS.get(ctx["fam"][m]))
        a.plot([0, lim], [0, lim], "k--", lw=0.8)
        mt = reg_metrics(H["Actual_AQI_tplus1"], H[f"Pred_AQI_{m}"])
        a.set_title(f"{m} ({ctx['fam'][m]}): MAE {mt['MAE']:.1f}, RMSE {mt['RMSE']:.1f}, R2 {mt['R2']:.3f}")
        a.set_xlabel("Actual AQI (t+1)")
        a.set_ylabel("Predicted AQI (t+1)")
    for a in ax.ravel()[len(models):]:
        a.axis("off")
    return save_fig(fig, "f04_predicted_vs_actual.png", "Predicted vs actual next-day estimated AQI (test, random sample of 8,000)")


def fig_confusions(ctx):
    H = ctx["H"][ctx["H"]["Split"] == "test"]
    models = list(dict.fromkeys(["Persistence", best_in_family(ctx["comp"], "classification", "Classical ML"),
                                 best_in_family(ctx["comp"], "classification", "Deep learning"), ctx["best"]["classification"]]))
    models = [m for m in models if m]
    fig, ax = plt.subplots(2, 2, figsize=(15, 13))
    for a, m in zip(ax.ravel(), models):
        cm = confusion_matrix(H["Actual_Cat_code"], H[f"CatCode_{m}"], labels=list(range(6)))
        rown = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        a.imshow(rown, cmap="Blues", vmin=0, vmax=1)
        a.set_xticks(range(6), CATEGORIES, rotation=40, ha="right", fontsize=8)
        a.set_yticks(range(6), CATEGORIES, fontsize=8)
        for i in range(6):
            for j in range(6):
                a.text(j, i, f"{cm[i, j]}\n{100 * rown[i, j]:.0f}%", ha="center", va="center", fontsize=6.5,
                       color="white" if rown[i, j] > 0.5 else "black")
        a.set_title(f"{m} ({ctx['fam'][m]})")
        a.set_xlabel("Predicted next-day category")
        a.set_ylabel("Actual next-day category")
    for a in ax.ravel()[len(models):]:
        a.axis("off")
    return save_fig(fig, "f05_category_confusion_matrices.png", "AQI-category confusion matrices (test; count and row %)")


def fig_importance(ctx):
    out = []
    imp = ctx["final_importance"]
    g = imp[(imp["Scope"] == "Global") & (imp["Source"] == "Part7_explainer")].sort_values("Importance", ascending=False).head(15)
    fig, ax = plt.subplots(1, 2, figsize=(18, 7))
    ax[0].barh(g["Feature"][::-1], g["Importance"][::-1], xerr=g["Importance_Std"][::-1], color="#4C72B0")
    ax[0].set_xlabel("RMSE increase when the group is permuted (held-out rows)")
    ax[0].set_title("Global grouped importance (Part 7 explainer model)")
    p5 = imp[(imp["Source"] == "Part5") & (imp["Method"] == "permutation") & (imp["Level"] == "feature") & (imp["Task"] == "regression")]
    if not p5.empty:
        m = p5["Model"].iloc[0]
        for cand in ("XGBoost", "RandomForest", "Ridge"):
            if cand in set(p5["Model"]):
                m = cand
                break
        q = p5[p5["Model"] == m].sort_values("Importance", ascending=False).head(15)
        ax[1].barh(q["Feature"][::-1], q["Importance"][::-1], color="#55A868")
        ax[1].set_title(f"Part 5 permutation importance: {m} (individual features)")
    return save_fig(fig, "f06_global_feature_importance.png", "Global feature importance (associations learned by the models)")


def fig_city_season_importance(ctx):
    imp = ctx["final_importance"]
    out = []
    cs = imp[(imp["Scope"] == "City") & (imp["Source"] == "Part7_explainer")]
    if not cs.empty:
        order = ctx["city_order"][:25]
        piv = cs.pivot_table(index="Group", columns="Feature", values="Importance_Share", aggfunc="mean").reindex(order)
        piv = piv[piv.mean().sort_values(ascending=False).index[:12]]
        fig, ax = plt.subplots(figsize=(14, 10))
        im = ax.imshow(piv.values * 100, aspect="auto", cmap="YlOrRd")
        ax.set_xticks(range(piv.shape[1]), piv.columns, rotation=60, ha="right", fontsize=8)
        ax.set_yticks(range(piv.shape[0]), piv.index, fontsize=8)
        fig.colorbar(im, ax=ax, label="share of positive importance (%)")
        out.append(save_fig(fig, "f07_city_feature_importance.png", "City-wise feature-group importance (25 highest-AQI cities)"))
    ss = imp[(imp["Scope"] == "Season") & (imp["Source"] == "Part7_explainer")]
    if not ss.empty:
        piv = ss.pivot_table(index="Group", columns="Feature", values="Importance_Share", aggfunc="mean").reindex(SEASONS).dropna(how="all")
        piv = piv[piv.mean().sort_values(ascending=False).index[:14]]
        fig, ax = plt.subplots(figsize=(14, 4.5))
        im = ax.imshow(piv.values * 100, aspect="auto", cmap="YlOrRd")
        ax.set_xticks(range(piv.shape[1]), piv.columns, rotation=60, ha="right", fontsize=8)
        ax.set_yticks(range(piv.shape[0]), piv.index)
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                ax.text(j, i, f"{100 * piv.values[i, j]:.0f}", ha="center", va="center", fontsize=7)
        fig.colorbar(im, ax=ax, label="share of positive importance (%)")
        out.append(save_fig(fig, "f08_season_feature_importance.png", "Season-wise feature-group importance"))
    return out


def fig_attention(ctx):
    att = ctx.get("attention")
    if att is None or att.empty:
        return []
    tasks = list(dict.fromkeys(att["Task"]))
    fig, ax = plt.subplots(1, len(tasks), figsize=(7 * len(tasks), 4.6), squeeze=False)
    for a, t in zip(ax[0], tasks):
        d = att[att["Task"] == t]
        groups = [g for g in ["All"] + SEASONS if g in set(d["Group"])]
        mat = np.vstack([d[d["Group"] == g].sort_values("Days_Before_Origin", ascending=False)["Mean_Attention"].to_numpy() for g in groups])
        im = a.imshow(mat, aspect="auto", cmap="viridis")
        a.set_yticks(range(len(groups)), groups)
        lags = sorted(d["Days_Before_Origin"].unique(), reverse=True)
        a.set_xticks(range(0, len(lags), 3), [f"t-{l}" if l else "t" for l in lags[::3]], fontsize=8)
        a.set_title(f"{t}: Bi-LSTM attention")
        fig.colorbar(im, ax=a, shrink=0.8)
    return save_fig(fig, "f09_attention_heatmaps.png", "Attention weights across the 30-day window (descriptive; Part 6)")


def fig_dominant_heatmap(ctx):
    s = ctx["observed"]
    reg = s.drop_duplicates("City").set_index("City")["Region"]
    piv = s.pivot(index="City", columns="Season", values="Observed_Dominant_Pollutant")[SEASONS]
    order = sorted(piv.index, key=lambda c: (str(reg[c]), c))
    code = piv.loc[order].apply(lambda col: col.map({p: i for i, p in enumerate(POLLUTANT_ORDER)})).astype(float)
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap([POLL_COLORS[p] for p in POLLUTANT_ORDER])
    cmap.set_bad("#DDDDDD")
    fig, ax = plt.subplots(figsize=(7.5, 0.22 * len(order) + 3))
    ax.imshow(np.ma.masked_invalid(code.values), cmap=cmap, vmin=-0.5, vmax=5.5, aspect="auto")
    ax.set_xticks(range(4), SEASONS)
    ax.set_yticks(range(len(order)), [f"{c} ({reg[c]})" for c in order], fontsize=6.5)
    ax.xaxis.tick_top()
    ax.legend(handles=[Patch(color=POLL_COLORS[p], label=p) for p in POLLUTANT_ORDER], loc="upper center",
              bbox_to_anchor=(0.5, -0.01), ncol=6, fontsize=8)
    return save_fig(fig, "f10_city_season_dominant_pollutant_heatmap.png",
                    "Observed most frequent dominant pollutant by city and season (suggestive, rule-based context)")


def fig_ablation(ctx):
    ab = ctx["ablation"]
    if ab is None or ab.empty:
        return []
    tasks = list(dict.fromkeys(ab["Task"]))
    fig, ax = plt.subplots(1, len(tasks), figsize=(6.4 * len(tasks), 5.2), squeeze=False)
    for a, t in zip(ax[0], tasks):
        d = ab[ab["Task"] == t]
        met = {"regression": "RMSE", "classification": "Macro_F1", "highrisk": "PR_AUC"}[t]
        x = np.arange(len(d))
        a.bar(x - 0.2, d[f"validation_{met}"], width=0.4, color="#9DB4D8", label="validation")
        a.bar(x + 0.2, d[f"test_{met}"], width=0.4, color="#2F4B7C", label="test")
        a.set_xticks(x, [s.split("_", 1)[0] + ":" + s.split("_", 1)[1][:14] for s in d["Step"]], rotation=35, ha="right", fontsize=8)
        a.set_title(f"{t}: {met}" + (" (lower is better)" if t == "regression" else " (higher is better)"))
        lo, hi = float(d[[f"validation_{met}", f"test_{met}"]].min().min()), float(d[[f"validation_{met}", f"test_{met}"]].max().max())
        a.set_ylim(lo - 0.15 * (hi - lo + 1e-9), hi + 0.1 * (hi - lo + 1e-9))
        a.legend(fontsize=8)
    return save_fig(fig, "f11_ablation_results.png", "Ablation: base -> +weather -> +season/calendar -> +city -> +regime")


def fig_cases(ctx):
    cases, series = ctx["cases"], ctx["case_series"]
    if cases is None or cases.empty:
        return []
    n = len(cases)
    fig, ax = plt.subplots(n, 2, figsize=(17, 3.9 * n), squeeze=False, gridspec_kw={"width_ratios": [2.2, 1]})
    for i, c in cases.reset_index(drop=True).iterrows():
        s = series[series["Case_ID"] == c["Case_ID"]].sort_values("Target_Date")
        a = ax[i, 0]
        a.plot(s["Target_Date"], s["Observed_AQI"], "k-", lw=1.6, label="Observed AQI")
        a.plot(s["Target_Date"], s["Predicted_AQI_best_model"], color="#C44E52", lw=1.2, label=f"{ctx['best']['regression']} (1-day ahead)")
        a.plot(s["Target_Date"], s["Persistence_AQI"], color="#999", lw=1, ls="--", label="Persistence")
        a.axhline(200, color="#F58220", ls=":", lw=1)
        a.axvline(pd.Timestamp(c["Peak_Target_Date"]), color="#7E0023", ls=":", lw=1)
        a.set_title(f"Case {i + 1}: {c['City']} ({c['Season']}) - peak {pd.Timestamp(c['Peak_Target_Date']).date()}, "
                    f"observed AQI {c['Observed_Peak_AQI']:.0f} ({c['Observed_Peak_Category']})", fontsize=10)
        a.set_ylabel("AQI")
        if i == 0:
            a.legend(fontsize=8)
        b = ax[i, 1]
        b.axis("off")
        txt = (f"Dominant pollutant on origin day: {c['Dominant_Pollutant_Origin_Day']}\n"
               f"Weather (origin day): {c['Temperature_C']:.1f} C, {c['Humidity_pct']:.0f}% RH,\n"
               f"  wind {c['WindSpeed_kmh']:.1f} km/h, rain {c['Precipitation_mm']:.1f} mm\n"
               f"High-risk probability: {c['HighRisk_Probability_best_model']:.2f}\n"
               f"Explainer-model drivers (occlusion):\n  {c['Top_Local_Drivers']}\n"
               f"Rule-based cause (suggestive):\n  " + "\n  ".join(_wrap(c["Likely_Cause"], 52)))
        b.text(0, 1, txt, va="top", ha="left", fontsize=8)
    return save_fig(fig, "f12_pollution_episode_case_studies.png", "Pollution-episode case studies (held-out period)")


def _wrap(text, width):
    import textwrap
    return textwrap.wrap(str(text), width=width)[:5]


def fig_alerts(ctx):
    al, summ = ctx["alerts_all_test"], ctx["alert_summary"]
    fig, ax = plt.subplots(1, 3, figsize=(19, 5.5))
    by_s = al.groupby("Season").agg(alerts=("Alert", "sum"), n=("Alert", "size"), pos=("Actual_HighRisk", "sum"))
    by_s = by_s.reindex([s for s in SEASONS if s in by_s.index])
    x = np.arange(len(by_s))
    ax[0].bar(x - 0.2, 100 * by_s["alerts"] / by_s["n"], width=0.4, label="% days with alert", color="#C44E52")
    ax[0].bar(x + 0.2, 100 * by_s["pos"] / by_s["n"], width=0.4, label="% days actually high-risk", color="#4C72B0")
    ax[0].set_xticks(x, by_s.index)
    ax[0].legend()
    ax[0].set_title("Alert rate vs observed high-risk rate (test)")
    top = summ.sort_values("Alerts", ascending=False).head(20)[::-1]
    ax[1].barh(top["City"], top["Alerts"], color="#F58220")
    ax[1].set_title("Cities with most alert days (test)")
    ax[2].scatter(summ["Precision"], summ["Recall"], s=18, alpha=0.7, color="#4C72B0")
    ax[2].set_xlabel("Precision")
    ax[2].set_ylabel("Recall")
    ax[2].set_title("City-wise alert precision vs recall (cities with >=1 high-risk day)")
    return save_fig(fig, "f13_high_risk_alerts.png", f"High-risk warning ({ctx['best']['highrisk']}; validation-optimised threshold)")


def fig_regimes(ctx):
    prof = ctx.get("regime_profile")
    if prof is None or prof.empty:
        return []
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    ax[0].bar(prof["Regime_KMeans"].astype(str), prof["Share_of_train_days_pct"], color="#8172B2")
    ax[0].set_title("Share of training city-days per regime (%)")
    ax[0].set_xlabel("Regime (0 = lowest mean PM2.5)")
    cols = ["PM2_5", "PM10", "NO2", "SO2", "O3", "CO"]
    x = np.arange(len(prof))
    for k, c in enumerate(["PM2_5", "PM10", "NO2", "O3"]):
        ax[1].bar(x + (k - 1.5) * 0.2, prof[c], width=0.2, label=c, color=POLL_COLORS[c])
    ax[1].set_xticks(x, prof["Regime_KMeans"].astype(str))
    ax[1].set_title("Mean concentrations per regime (train, ug/m3)")
    ax[1].legend()
    return save_fig(fig, "f14_regime_profiles.png", "K-Means pollution-weather regimes (fitted on the training split only)")


# ----------------------------------------------------------------------------
# 7. Report
# ----------------------------------------------------------------------------
def pick(df, **kw):
    d = df
    for k, v in kw.items():
        d = d[d[k] == v]
    return d


def write_report(ctx):
    R, M = ctx["R"], ctx["R"]["manifests"]
    comp, best, boot, ab = ctx["comp"], ctx["best"], ctx["bootstrap"], ctx["ablation"]
    daily, obs = R["daily"], ctx["observed"]
    n_cities, n_rows = daily["City"].nunique(), len(daily)
    d0, d1 = daily["Date"].min().date(), daily["Date"].max().date()
    prov = int(daily["AQI_Uses_Provisional_8hr"].astype(str).str.lower().isin(["true", "1"]).sum()) if "AQI_Uses_Provisional_8hr" in daily else None
    prov_pct = 100 * prov / n_rows if prov is not None else None
    p1, p2, p4 = M.get(1, {}), M.get(2, {}), M.get(4, {})
    q = R.get("quality")

    def qv(check, item=None):
        if q is None:
            return None
        s = q[q["check"] == check]
        if item is not None:
            s = s[s["item"] == item]
        return None if s.empty else s.iloc[0]["value"]
    ident_all = qv("identical_pairs_all_pollutants")
    ident_pm = qv("identical_pairs_single_pollutant", "PM2_5")
    pm_gt = qv("PM2_5 > PM10 rows")
    ext_pm10 = qv("n_flagged (per-city Q3+3*IQR)", "PM10")
    capped = p2.get("results", {}).get("severe_capped_rows")
    sev_top = p2.get("aqi_method", {}).get("severe_band_top_assumed")
    test_c = comp[comp["Split"] == "test"]

    def row(task, model):
        d = test_c[(test_c["Task"] == task) & (test_c["Model"] == model)]
        return None if d.empty else d.iloc[0]
    rb, cb, hb = (row("regression", best.get("regression")), row("classification", best.get("classification")),
                  row("highrisk", best.get("highrisk")))
    rp, cp, hp = row("regression", "Persistence"), row("classification", "Persistence"), row("highrisk", "Persistence")
    split = p4.get("split", {})
    seasons_test = ctx["test_seasons"]

    def boot_line(task):
        b = boot[(boot["Task"] == task) & (boot["Comparison"] == "best vs persistence")] if boot is not None and len(boot) else pd.DataFrame()
        if b.empty:
            return "No bootstrap comparison with persistence was available."
        r = b.iloc[0]
        return (f"On the test dates, the {r['Metric']} difference (best model minus persistence) is {fmt(r['Delta_A_minus_B'], 3)} "
                f"(95% block-bootstrap interval {fmt(r['CI95_low'], 3)} to {fmt(r['CI95_high'], 3)}), which indicates: {r['Reading']}.")

    def top_global(n=6):
        g = ctx["final_importance"]
        g = g[(g["Scope"] == "Global") & (g["Source"] == "Part7_explainer")].sort_values("Importance", ascending=False).head(n)
        return ", ".join(f"{r.Feature} ({r.Importance:.2f})" for r in g.itertuples()) if len(g) else "n/a"

    ab_tab = pd.DataFrame()
    if ab is not None and len(ab):
        met = {"regression": "RMSE", "classification": "Macro_F1", "highrisk": "PR_AUC"}
        ab_tab = pd.concat([ab[ab["Task"] == t][["Task", "Step", "N_Features", f"validation_{met[t]}", f"test_{met[t]}", "Verdict"]]
                            .rename(columns={f"validation_{met[t]}": "Validation_metric", f"test_{met[t]}": "Test_metric"})
                            for t in ab["Task"].unique()])
        ab_tab["Metric"] = ab_tab["Task"].map(met)
    lines = []
    A = lines.append
    A(f"# City- and Season-Aware Multi-Pollutant Air Quality Assessment and Forecasting Framework for Indian Cities Using Deep Learning\n")
    A(f"*Final report (auto-generated by Part 7, run {RUN_ID}). {AQI_LABEL}*\n")
    A(f"> **Scope note.** {AQI_CAVEAT} The framework provides decision support, not definitive source attribution or regulatory enforcement.\n")

    A("## 1. Abstract\n")
    abs_txt = (f"This study builds a city- and season-aware framework for estimating and forecasting air quality across {n_cities} Indian cities "
               f"({d0} to {d1}; {n_rows:,} city-days) using Open-Meteo/CAMS-based pollutant and weather estimates. A six-pollutant estimated AQI "
               f"(CPCB sub-index methodology; NH3 and Pb unavailable) is computed from daily means and 8-hour maxima, and the dominant pollutant is "
               f"summarised for every city and season (404 city-season cells) with rule-based, suggestive weather explanations. Persistence, classical "
               f"machine-learning and deep-learning models (LSTM, GRU, 1D-CNN, CNN-LSTM, Bi-LSTM with attention) were compared on one chronological "
               f"train/validation/test split. ")
    if rb is not None and rp is not None:
        abs_txt += (f"For next-day AQI regression the model selected on validation was {best['regression']} "
                    f"({ctx['fam'][best['regression']]}; test RMSE {fmt(rb['RMSE'], 2)} vs {fmt(rp['RMSE'], 2)} for persistence). ")
    if cb is not None and cp is not None:
        abs_txt += f"For category classification the selected model was {best['classification']} (test macro-F1 {fmt(cb['Macro_F1'])} vs {fmt(cp['Macro_F1'])} for persistence). "
    if hb is not None and hp is not None:
        abs_txt += f"For the 3-day high-risk warning (AQI > 200) the selected model was {best['highrisk']} (test PR-AUC {fmt(hb['PR_AUC'])}). "
    abs_txt += ("Results are reported with uncertainty intervals and should be read as predictive performance on this dataset, not as causal evidence.")
    A(abs_txt + "\n")

    A("## 2. Introduction\n")
    A("Air-quality management benefits from short-term forecasts that also say *which pollutant* is likely to drive a poor-air day and *why that "
      "may be so*. This project combines (i) an estimated multi-pollutant AQI, (ii) a city x season description of dominant pollutants with "
      "weather-based, suggestive explanations, and (iii) forecasting models for next-day AQI, next-day category and a 3-day high-risk warning. "
      "All statements about causes are interpretations of associations and are phrased accordingly.\n")

    A("## 3. Dataset and source limitations\n")
    A(f"The daily dataset covers {n_cities} cities from {d0} to {d1} ({n_rows:,} city-days) with PM2.5, PM10, CO, NO2, SO2, O3 (daily means, ug/m3), "
      "temperature, humidity, wind speed and rainfall. **The values are Open-Meteo/CAMS-based estimates, not direct CPCB ground-station observations.** "
      "Original files were never modified; every processed output is versioned.\n")
    if pm_gt is not None:
        A(f"- Part 1 validation found {pm_gt} rows with PM2.5 > PM10, which indicates estimation inconsistency; they were flagged, not deleted.")
    if ident_all is not None:
        A(f"- {ident_all} city pairs have identical series across all pollutants and {ident_pm} pairs for PM2.5 alone; this is consistent with a coarse spatial grid "
          "assigning the same cell to nearby cities (not a definitive explanation).")
    if ext_pm10 is not None:
        A(f"- Extreme observations were flagged and retained; {ext_pm10} PM10 city-days exceed the per-city Q3 + 3 x IQR fence.")
    A("")

    A("## 4. Methodology\n")
    A("The framework is a seven-part pipeline: (1) loading, validation and hourly O3/CO retrieval; (2) estimated AQI and EDA; (3) weather-based explanations "
      "and policy rules; (4) leakage-safe features, chronological split and train-only scaling; (5) classical ML; (6) 30-day sequence deep learning; "
      "(7) synthesis, ablation, explainability and reporting.\n")

    A("## 5. Hourly O3/CO retrieval and 8-hour maximum calculation\n")
    A("Hourly ozone and carbon monoxide were requested from the Open-Meteo Air Quality API for each unique city coordinate (local time, Asia/Kolkata). "
      "A trailing 8-hour mean (at least 6 valid hours) is computed per city; the daily maximum of these means (at least 18 valid windows per day) gives "
      "O3_8hr_max (ug/m3) and CO_8hr_max, converted to mg/m3 for the CO sub-index. Original daily means are left unchanged.")
    if prov is not None:
        if prov == 0:
            A("All city-days used hourly-derived 8-hour values.")
        else:
            A(f"**Coverage note:** {prov:,} of {n_rows:,} city-days ({prov_pct:.1f}%) still use the provisional daily-mean fallback for O3 and/or CO "
              "(hourly data could not be retrieved for those days). Their O3/CO sub-indices are therefore provisional and likely understate 8-hour peaks.")
    A("")

    A("## 6. AQI estimation\n")
    A(f"**{AQI_LABEL}** Sub-indices for PM2.5, PM10, NO2 and SO2 use 24-hour (daily) means; O3 uses the 8-hour maximum (ug/m3) and CO the 8-hour maximum "
      "(mg/m3). Estimated AQI is the maximum sub-index and the dominant pollutant is the pollutant attaining it (ties go to PM2.5, PM10, NO2, SO2, O3, CO "
      "in that order). Categories: Good 0-50, Satisfactory 51-100, Moderate 101-200, Poor 201-300, Very Poor 301-400, Severe 401-500. "
      "NH3 and Pb are unavailable, so this is not the official CPCB AQI.")
    if sev_top:
        A(f"The CPCB tables are open-ended in the Severe band; the upper ends used for interpolation and the 500 cap are an assumption of this framework: {sev_top}"
          + (f" ({capped:,} city-days reached the cap)." if capped is not None else "."))
    A("")

    A("## 7. Feature engineering\n")
    nf = p4.get("feature_design", {}).get("n_features")
    A(f"Past-only features per city{f' ({nf} features)' if nf else ''}: current-day values, lags (1, 2, 3, 7 days), rolling means (3, 7, 14, 30 days) and standard deviations "
      "(7, 30 days) computed after a one-day shift, calendar and cyclic terms, season indicators, interaction terms (PM2.5/PM10, NO2+CO, SO2/PM2.5, "
      "temperature x O3, wind x PM2.5, rainfall x PM10, humidity x PM2.5), and city/region context. Targets are the AQI and category at t+1 and a binary "
      "indicator of AQI > 200 within the next three days. Perturbation tests confirmed that scrambling future observations leaves earlier features unchanged.\n")

    A("## 8. City and season awareness\n")
    A("City awareness enters through a city identifier (embedding for deep models), region, latitude and longitude; season awareness through season "
      "indicators and cyclic calendar terms. Regime awareness uses K-Means pollution-weather regimes fitted on the training split only"
      + (f" (k = {int(ctx['regime_profile']['n_regimes'].iloc[0])})" if ctx.get("regime_profile") is not None and len(ctx["regime_profile"]) else "")
      + ". The primary descriptive product is the 101 x 4 city-season table (observed dominant pollutant, mean and median concentrations, mean AQI, share of "
        "days with AQI > 200, likely cause).\n")

    A("## 9. Weather-based dominant-pollutant explanation\n")
    dom = obs.groupby(["Season", "Observed_Dominant_Pollutant"]).size().unstack(fill_value=0).reindex(SEASONS).fillna(0).astype(int)
    A("Rule-based statements link simultaneous weather and concentration patterns (e.g., low wind and little rain with elevated particulates) to possible "
      "contributors. They **suggest** plausible influences and are **consistent with** known meteorology; they do not attribute emissions to sources. "
      "Counts of cities by observed most-frequent dominant pollutant:\n")
    A(md_table(dom.reset_index(), 0) + "\n")

    A("## 10. Models\n")
    A("Baselines: persistence (tomorrow equals today). Classical ML: Ridge / Logistic regression, Random Forest, XGBoost. Deep learning on 30-day sequences "
      "with city and region embeddings: LSTM, GRU, 1D-CNN, CNN-LSTM and Bi-LSTM with attention (regression, category classification; LSTM, GRU and "
      "Bi-LSTM-attention for the high-risk task). Deep-learning regression outputs are converted to AQI units with the saved RobustScaler.\n")

    A("## 11. Experimental setup\n")
    if split:
        A(f"One global chronological split (all cities together): train {split.get('train_dates', ['?', '?'])[0]} to {split.get('train_dates', ['?', '?'])[1]}, "
          f"validation {split.get('validation_dates', ['?', '?'])[0]} to {split.get('validation_dates', ['?', '?'])[1]}, "
          f"test {split.get('test_dates', ['?', '?'])[0]} to {split.get('test_dates', ['?', '?'])[1]}; the last {split.get('purge_days', 7)} dates of train and "
          "validation were purged so that multi-day labels cannot overlap the next split. ")
    A("Scaling used a RobustScaler fitted on training data only. Models were fitted on train, tuned or early-stopped on validation, and scored once on test. "
      "All models were compared on identical test rows. Uncertainty is shown with a moving-block bootstrap over test dates "
      f"({CFG['n_boot']} draws, {BLOCK_LEN}-day blocks). Seed 42.")
    A(f"The test period contains the seasons: {', '.join(seasons_test)}; seasonal conclusions are limited to these.\n")

    A("## 12. Results\n")
    show = test_c[test_c["Task"] == "regression"][["Family", "Model", "MAE", "RMSE", "R2"]]
    A("**Next-day AQI regression (test)**\n")
    A(md_table(show) + "\n")
    A("**Next-day category classification (test)**\n")
    A(md_table(test_c[test_c["Task"] == "classification"][["Family", "Model", "Accuracy", "Macro_F1", "Balanced_Accuracy"]]) + "\n")
    A("**High-risk warning, AQI > 200 within 3 days (test; validation-optimised threshold)**\n")
    A(md_table(test_c[test_c["Task"] == "highrisk"][["Family", "Model", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]]) + "\n")
    A(f"Best models on validation: regression **{best.get('regression')}**, classification **{best.get('classification')}**, "
      f"high-risk **{best.get('highrisk')}**. Regression: {boot_line('regression')} Classification: {boot_line('classification')} High-risk: {boot_line('highrisk')}\n")
    if boot is not None and len(boot):
        A(md_table(boot[["Task", "Comparison", "Model_A", "Model_B", "Metric", "Delta_A_minus_B", "CI95_low", "CI95_high", "Reading"]]) + "\n")
    A(f"A one-day-ahead AQI series is strongly autocorrelated, so persistence is a demanding baseline; gains over it should be judged against the intervals above. "
      f"{ctx['alert_text']}\n")

    A("## 13. Ablation study\n")
    if len(ab_tab):
        A("A fixed gradient-boosting learner was refitted on nested feature sets (base = pollutant lags/rolling incl. AQI history; + weather; + season/calendar; "
          "+ city; + regime). Verdicts use the block-bootstrap interval of the change versus the previous step on test dates.\n")
        A(md_table(ab_tab[["Metric", "Step", "N_Features", "Validation_metric", "Test_metric", "Verdict"]]) + "\n")
        reg_ab = ab[ab["Task"] == "regression"]
        if len(reg_ab):
            imp_steps = reg_ab[reg_ab["Verdict"].str.startswith("improves")]["Step"].tolist()
            A("Reading (regression): " + ("steps with a clear improvement in RMSE: " + ", ".join(imp_steps) + ". " if imp_steps else
              "no added feature group produced a clear RMSE improvement beyond the previous step. ") +
              "Because AQI history already carries most of the signal, small or insignificant gains are plausible; the table should be read as indicating, "
              "not proving, the contribution of each awareness component.\n")
        if not ctx["regime_in_ablation"]:
            A("_Regime labels were not available, so step 5 was skipped._\n")
    else:
        A("_Ablation was not run._\n")

    A("## 14. Explainability\n")
    A(f"Grouped permutation importance on held-out (validation and test) rows for the full-feature explainer model ranks: {top_global()}. "
      "City-wise and season-wise importance tables (final_feature_importance.csv) show how the leading groups shift across cities and seasons; shifts "
      "are consistent with different drivers, but they remain model associations.")
    p5s = ctx["final_importance"]
    shap_rows = p5s[(p5s["Source"] == "Part5") & (p5s["Method"] == "SHAP_mean_abs")]
    if len(shap_rows):
        t = shap_rows[shap_rows["Task"] == "regression"].sort_values("Importance", ascending=False).head(5)
        A("Part 5 SHAP (validation sample) highlights: " + ", ".join(f"{r.Feature}" for r in t.itertuples()) + ".")
    att = ctx.get("attention")
    if att is not None and len(att):
        a_all = att[(att["Group"] == "All") & (att["Task"] == att["Task"].iloc[0])].sort_values("Mean_Attention", ascending=False).head(3)
        A("Bi-LSTM attention is concentrated on the most recent days (highest weights at lags "
          + ", ".join(str(int(x)) for x in a_all["Days_Before_Origin"]) + " days before the origin date), indicating the model relies mostly on recent history.")
    A("\nCase studies of major episodes (held-out period):\n")
    for _, c in ctx["cases"].iterrows():
        A(f"- **{c['City']} ({c['Season']}, peak {pd.Timestamp(c['Peak_Target_Date']).date()})**: observed AQI {c['Observed_Peak_AQI']:.0f} "
          f"({c['Observed_Peak_Category']}); one-day-ahead prediction {c['Predicted_AQI_best_model']:.0f}; dominant pollutant on the origin day "
          f"{c['Dominant_Pollutant_Origin_Day']}. Rule-based note (suggestive): {c['Likely_Cause']}")
    A("")

    A("## 15. Limitations\n")
    for s in ["Data are Open-Meteo/CAMS-based estimates, not direct CPCB ground-station observations.",
              "AQI is a six-pollutant estimated AQI, not the official CPCB AQI.",
              "NH3 and Pb are unavailable.",
              "O3 and CO AQI sub-indices use 8-hour rolling maxima from re-fetched hourly Open-Meteo data" +
              (f" (with {prov:,} city-days, {prov_pct:.1f}%, on a provisional daily-mean fallback)." if prov else "."),
              "Identical city-series pairs may occur due to coarse spatial grid resolution.",
              "Extreme PM10 values were retained (flagged, not removed), so error metrics are influenced by them.",
              "The framework provides decision support, not definitive source attribution or regulatory enforcement.",
              f"The test period spans {len(seasons_test)} season(s) ({', '.join(seasons_test)}) of a roughly four-year record; seasonal generalisation is limited.",
              "Cause and policy statements are rule-based and threshold-dependent (thresholds are dataset-relative); they were not validated against emission inventories.",
              "Deep-learning results come from a single seed; GPU non-determinism and tuning limited to early stopping mean small differences between models should not be over-interpreted."]:
        A(f"- {s}")
    A("")
    A("## 16. Future work\n")
    A("Use verified CPCB station data for calibration and validation; add NH3 and Pb when available; use finer spatial reanalysis to avoid duplicated city series; "
      "extend the record so every season appears in each split (or use rolling-origin evaluation); test multi-seed and probabilistic forecasts; add "
      "uncertainty-aware alerts and evaluate policy rules with local authorities; include emission-inventory and fire-count data for better-grounded attribution studies.\n")
    A("## 17. Conclusion\n")
    A(f"Within the limits of Open-Meteo/CAMS-based estimates and a six-pollutant estimated AQI, the framework describes dominant pollutants by city and season, "
      f"forecasts next-day AQI and category, and issues a 3-day high-risk warning. {ctx['conclusion_sentence']} The weather- and city-aware components are reported with "
      "ablation evidence and uncertainty intervals so that readers can judge how much each contributes. The outputs support decision-making and do not replace "
      "regulatory monitoring.\n")
    A("---\n*Reproducibility:* seed 42; all file paths and hashes are listed in part7_handoff_manifest.json.")
    with open(FINAL_REPORT_MD, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return FINAL_REPORT_MD


# ----------------------------------------------------------------------------
# 8. Main pipeline
# ----------------------------------------------------------------------------
COMP_GLOBAL, HR_FAM_GLOBAL = None, {}


def main():
    global COMP_GLOBAL, HR_FAM_GLOBAL
    t_start = time.time()
    banner(f"PART 7 - Final synthesis (profile: {PROFILE})")
    print(AQI_LABEL)
    print(AQI_CAVEAT)
    if PROFILE == "fast":
        print("FAST profile: smoke test only - do NOT report these results.")

    # ---- load -----------------------------------------------------------------------------------
    R = load_all()
    in_paths = list(REQ_PATHS.values()) + [p for k, p in OPT.items() if os.path.isfile(p)]
    hashes_in = {p: sha256_of(p) for p in in_paths}
    fcols = R.get("feature_cols")
    print(f"Loaded {len(REQ_PATHS)} required result files and {sum(k in R for k in OPT)} optional ones.")
    H, Hh, fam, hr_fam, reg_models, cls_models, hr_models, thr_cols, hinfo = harmonise(R)
    HR_FAM_GLOBAL = hr_fam
    print(f"Harmonised rows: Part 5 {hinfo['rows_part5']:,} | Part 6 {hinfo['rows_part6']:,} | common {hinfo['rows_common']:,} "
          f"(high-risk common {hinfo['hr_rows_common']:,}) | max actual-AQI difference {hinfo['max_abs_actual_diff']:.4f}")
    if hinfo["rows_common"] < min(hinfo["rows_part5"], hinfo["rows_part6"]):
        print("  ! Part 5 and Part 6 do not cover identical rows; all comparisons use the common rows only.")

    # ---- 1-2. comparison, best models, bootstrap ----------------------------------------------------
    banner("Final model comparison (identical rows) and best models (chosen on validation)")
    comp, best, thresholds = build_comparison(H, Hh, fam, hr_fam, reg_models, cls_models, hr_models, thr_cols)
    COMP_GLOBAL = comp
    best_ml, best_dl = best_in_family(comp, "regression", "Classical ML"), best_in_family(comp, "regression", "Deep learning")
    print("Best on validation:", best)
    for task, (met, hb) in TASK_PRIMARY.items():
        t = comp[(comp["Task"] == task) & (comp["Split"] == "test")].sort_values(met, ascending=not hb)
        extra = {"regression": ["MAE", "R2"], "classification": ["Accuracy", "Balanced_Accuracy"], "highrisk": ["F1", "ROC_AUC", "Precision", "Recall"]}[task]
        cols = ["Family", "Model", "N", met] + [c for c in extra if c != met]
        print(f"\n  {task} - TEST (sorted by {met})\n" + t[cols].round(4).to_string(index=False))
    boot = build_bootstrap(comp, best, H, Hh, thresholds)
    print("\nBlock-bootstrap comparisons:\n" + boot[["Task", "Comparison", "Model_A", "Model_B", "Delta_A_minus_B", "CI95_low", "CI95_high", "Reading"]].round(3).to_string(index=False))

    # consistency with the files written by Parts 5-6
    chk = []
    for src, key in (("model_metrics", "Persistence"), ("dl_model_metrics", None)):
        m = R[src]
        m = m[(m["Split"] == "test") & (m["Task"] == "regression")]
        for _, r in m.iterrows():
            mine = comp[(comp["Task"] == "regression") & (comp["Model"] == r["Model"]) & (comp["Split"] == "test")]
            if len(mine):
                chk.append({"Source_File": REQUIRED[src], "Model": r["Model"], "Stored_Test_RMSE": r["RMSE"],
                            "Recomputed_Test_RMSE": mine.iloc[0]["RMSE"], "Abs_Diff": abs(r["RMSE"] - mine.iloc[0]["RMSE"])})
    chk = pd.DataFrame(chk)
    if len(chk):
        print(f"\nConsistency with Parts 5-6 stored metrics: max |RMSE difference| = {chk['Abs_Diff'].max():.4f} "
              "(non-zero only if rows were dropped by the common-row join)")

    # ---- 3. observed / predicted city x season tables ------------------------------------------------------
    banner("City x season tables")
    cs = R["city_season"].copy()
    rename = {"Dominant_Pollutant": "Observed_Dominant_Pollutant", "Percent_Days_Dominant": "Observed_Percent_Days_Dominant",
              "Mean_AQI": "Observed_Mean_AQI", "Percent_Days_AQI_Above200": "Observed_Percent_Days_AQI_Above200",
              "Mean_AQI_Category": "Observed_Mean_AQI_Category"}
    observed = cs.rename(columns={k: v for k, v in rename.items() if k in cs.columns})
    lead = ["City", "Season", "Region", "Observed_Dominant_Pollutant", "Observed_Percent_Days_Dominant", "Dominant_Pollutant_Mean_Concentration",
            "Mean_PM2_5", "Mean_PM10", "Mean_CO", "Mean_NO2", "Mean_SO2", "Mean_O3", "Observed_Mean_AQI", "Observed_Mean_AQI_Category",
            "Observed_Percent_Days_AQI_Above200", "Mean_Temperature", "Mean_Humidity", "Mean_WindSpeed", "Mean_Rainfall", "Likely_Cause"]
    lead = [c for c in lead if c in observed.columns]
    observed = observed[lead + [c for c in observed.columns if c not in lead]]
    assert len(observed) == observed["City"].nunique() * 4, "Observed table is not 101 x 4"
    region_map = observed.drop_duplicates("City").set_index("City")["Region"].to_dict() if "Region" in observed else {}
    print(f"Observed table: {len(observed)} rows ({observed['City'].nunique()} cities x 4 seasons)")

    bm = best["regression"]
    bc = best["classification"]
    Ht = H[H["Split"] == "test"].copy()
    daily = R["daily"]
    nxt = daily[["City", "Date", "Dominant_Pollutant", "Estimated_AQI"]].rename(
        columns={"Date": "Target_Date", "Dominant_Pollutant": "Target_Day_Dominant_Pollutant", "Estimated_AQI": "Target_AQI_check"})
    Ht["Target_Date"] = Ht["Date"] + pd.Timedelta(days=1)
    Ht = Ht.merge(nxt, on=["City", "Target_Date"], how="left")
    Ht["Pred_Best_AQI"] = Ht[f"Pred_AQI_{bm}"]
    Ht["Pred_Best_Cat_FromAQI"] = category_code(Ht["Pred_Best_AQI"])
    pred_rows = []
    for (city, season), d in Ht.groupby(["City", "Season"]):
        mode_obs = d["Target_Day_Dominant_Pollutant"].value_counts().reindex(POLLUTANT_ORDER).fillna(0)
        pred_rows.append({
            "City": city, "Season": season, "Region": region_map.get(city), "N_Test_Days": len(d),
            "Best_Model": bm, "Observed_Mean_AQI_Test": d["Actual_AQI_tplus1"].mean(), "Predicted_Mean_AQI_Test": d["Pred_Best_AQI"].mean(),
            "Mean_Error_Pred_minus_Obs": (d["Pred_Best_AQI"] - d["Actual_AQI_tplus1"]).mean(),
            "MAE_Test": mean_absolute_error(d["Actual_AQI_tplus1"], d["Pred_Best_AQI"]),
            "RMSE_Test": float(np.sqrt(mean_squared_error(d["Actual_AQI_tplus1"], d["Pred_Best_AQI"]))),
            "Persistence_MAE_Test": mean_absolute_error(d["Actual_AQI_tplus1"], d["Pred_AQI_Persistence"]),
            "Observed_Mean_AQI_Category_Test": CATEGORIES[int(category_code(d["Actual_AQI_tplus1"].mean()))],
            "Predicted_Mean_AQI_Category_Test": CATEGORIES[int(category_code(d["Pred_Best_AQI"].mean()))],
            "Observed_Most_Frequent_Category_Test": d["Actual_Category_tplus1"].mode().iloc[0],
            "Predicted_Most_Frequent_Category_Test": CATEGORIES[int(pd.Series(d["Pred_Best_Cat_FromAQI"]).mode().iloc[0])],
            "Category_Agreement_Pct_Daily": 100 * float((d["Pred_Best_Cat_FromAQI"] == d["Actual_Cat_code"]).mean()),
            "Observed_Pct_Days_AQI_Above200_Test": 100 * float((d["Actual_AQI_tplus1"] > 200).mean()),
            "Predicted_Pct_Days_AQI_Above200_Test": 100 * float((d["Pred_Best_AQI"] > 200).mean()),
            "Observed_Dominant_Pollutant_Test_Period": mode_obs.idxmax() if mode_obs.sum() > 0 else np.nan,
            "Observed_Dominant_Pollutant_FullPeriod": observed[(observed["City"] == city) & (observed["Season"] == season)]["Observed_Dominant_Pollutant"].iloc[0]})
    predicted = pd.DataFrame(pred_rows)
    test_seasons = [s for s in SEASONS if s in set(predicted["Season"])]
    print(f"Predicted table ({bm}, test period): {len(predicted)} City x Season cells; seasons present in test: {test_seasons}")
    final_summary = observed.merge(predicted.drop(columns=["Region"]).add_prefix("Test_").rename(
        columns={"Test_City": "City", "Test_Season": "Season"}), on=["City", "Season"], how="left")

    # ---- group metrics -------------------------------------------------------------------------------------
    gm = group_metrics(H, Hh, best, fam, thresholds, region_map)
    gm.to_csv(GROUP_METRICS_CSV, index=False)

    # ---- 7. high-risk alerts ---------------------------------------------------------------------------------
    banner("High-risk alerts")
    bh = best["highrisk"]
    Hht = Hh[(Hh["Split"] == "test")].copy()
    thr_h = thresholds.get(bh, 0.5)
    Hht["Probability"] = Hht[f"Prob_{bh}"]
    Hht["Alert"] = (Hht["Probability"] >= thr_h).astype(int)
    Hht["Alert_Level"] = np.where(Hht["Probability"] >= max(ALERT_HIGH_CONF, thr_h), "High-confidence alert",
                                  np.where(Hht["Alert"] == 1, "Alert", "No alert"))
    ctx_cols = daily[["City", "Date", "Dominant_Pollutant", "AQI_Category", "Likely_Cause", "Recommended_Actions"]].rename(
        columns={"Dominant_Pollutant": "Dominant_Pollutant_Origin_Day", "AQI_Category": "Current_AQI_Category",
                 "Likely_Cause": "Likely_Cause_Origin_Day", "Recommended_Actions": "Recommended_Actions_Origin_Day"})
    Hht = Hht.merge(ctx_cols, on=["City", "Date"], how="left")
    Hht["Outcome"] = np.select([(Hht["Alert"] == 1) & (Hht["Actual_HighRisk"] == 1), (Hht["Alert"] == 1) & (Hht["Actual_HighRisk"] == 0),
                                (Hht["Alert"] == 0) & (Hht["Actual_HighRisk"] == 1)], ["True alert", "False alert", "Missed event"], "No alert, no event")
    Hht["Region"] = Hht["City"].map(region_map)
    alerts = Hht[Hht["Alert"] == 1][["Date", "City", "Region", "Season", "Current_AQI", "Current_AQI_Category", "Probability", "Alert_Level",
                                     "Actual_HighRisk", "Outcome", "Dominant_Pollutant_Origin_Day", "Likely_Cause_Origin_Day",
                                     "Recommended_Actions_Origin_Day"]].copy()
    alerts.insert(2, "Model", bh)
    alerts.insert(3, "Threshold", thr_h)
    alerts = alerts.sort_values(["Date", "Probability"], ascending=[True, False])
    asum = Hht.groupby("City").agg(Alerts=("Alert", "sum"), High_Risk_Days=("Actual_HighRisk", "sum"), Days=("Alert", "size"),
                                   True_Alerts=("Outcome", lambda s: int((s == "True alert").sum())),
                                   Missed=("Outcome", lambda s: int((s == "Missed event").sum()))).reset_index()
    asum["Precision"] = np.where(asum["Alerts"] > 0, asum["True_Alerts"] / asum["Alerts"].replace(0, np.nan), np.nan)
    asum["Recall"] = np.where(asum["High_Risk_Days"] > 0, asum["True_Alerts"] / asum["High_Risk_Days"].replace(0, np.nan), np.nan)
    tot = Hht["Outcome"].value_counts()
    alert_text = (f"For the high-risk warning ({bh}, threshold {thr_h:.2f} chosen on validation), the test period contains {int(Hht['Actual_HighRisk'].sum()):,} "
                  f"high-risk city-days out of {len(Hht):,}; the model issued {int(Hht['Alert'].sum()):,} alerts ({int(tot.get('True alert', 0)):,} true, "
                  f"{int(tot.get('False alert', 0)):,} false) and missed {int(tot.get('Missed event', 0)):,} events.")
    print(alert_text)

    # ---- 4. regime + ablation ------------------------------------------------------------------------------------
    banner("Regime awareness and ablation (new fitting limited to these steps)")
    ablation, models, D, regime_profile, labels, steps, leftover, parts = None, {}, None, None, None, {}, [], {}
    regime_in_ablation = False
    if fcols is None or not all(os.path.isfile(p) for p in list(SPLIT_FILES.values()) + list(SPLIT_S_FILES.values())):
        print("  ! Part 4 split files or feature_columns.json missing: ablation and explainer model skipped.")
    else:
        S, sc = load_splits(fcols)
        S, labels, regime_profile = fit_regimes(S, sc)
        regime_cols = [c for c in S["train"].columns if c == "Regime_KMeans" or c.startswith("Regime_prob_")]
        regime_in_ablation = len(regime_cols) > 0
        fcols_aug = dict(fcols)
        fcols_aug["feature_columns"] = list(fcols["feature_columns"]) + [c for c in regime_cols if c not in fcols["feature_columns"]]
        steps, leftover, parts = feature_sets(fcols_aug, regime_cols)
        if leftover:
            print(f"  Note: {len(leftover)} Part 4 columns are not used by any step: {leftover[:6]}")
        ablation, models, D = run_ablation(S, fcols_aug, steps, regime_in_ablation)
        print("\n" + ablation[["Task", "Step", "N_Features", "validation_" + "RMSE" if "validation_RMSE" in ablation else "Task",
                               "test_RMSE" if "test_RMSE" in ablation else "Task", "Verdict"]].to_string(index=False))
        if labels is not None:
            labels.to_csv(REGIME_CSV, index=False, date_format="%Y-%m-%d", float_format="%.5g")
            regime_profile.to_csv(REGIME_PROFILE_CSV, index=False)
    if ablation is not None:
        ablation.to_csv(FINAL_ABLATION_CSV, index=False)

    # ---- 5. explainability -------------------------------------------------------------------------------------------
    banner("Explainability")
    imp_rows = []

    def add(scope, group, source, task, model, method, level, feature, value, std=np.nan):
        imp_rows.append({"Scope": scope, "Group": group, "Source": source, "Task": task, "Model": model, "Method": method,
                         "Level": level, "Feature": feature, "Importance": value, "Importance_Std": std})
    fi = R["feature_importance"]
    for _, r in fi.iterrows():
        add("Global", "All", "Part5", r["Task"], r["Model"], r["Method"], r["Level"], r["Feature"], r["Importance_Mean"], r.get("Importance_Std", np.nan))
    if "dl_occlusion" in R:
        oc = R["dl_occlusion"].groupby(["Task", "Model", "Feature_Group"])["Mean_Abs_Change"].sum().reset_index()
        for _, r in oc.iterrows():
            add("Global", "All", "Part6", r["Task"], r["Model"], "occlusion_sum_over_blocks", "feature_group", r["Feature_Group"], r["Mean_Abs_Change"])
    attention = R.get("dl_attention")
    if attention is not None:
        for _, r in attention.iterrows():
            add("Global" if r["Group"] == "All" else "Season", r["Group"], "Part6", r["Task"], "BiLSTM_Attention", "attention", "lag_day",
                f"t-{int(r['Days_Before_Origin'])}", r["Mean_Attention"])
    city_order = list(observed.groupby("City")["Observed_Mean_AQI"].mean().sort_values(ascending=False).index)
    cases_df, case_series = pd.DataFrame(), pd.DataFrame()
    if models:
        mdl, cols, cats = models[("regression", "5_full_plus_regime" if regime_in_ablation else "4_plus_city")] \
            if ("regression", "5_full_plus_regime" if regime_in_ablation else "4_plus_city") in models else list(models.values())[0]
        groups = importance_groups(cols, parts)
        held = pd.concat([D["validation"], D["test"]], ignore_index=True)
        held = held[held["Estimated_AQI_tplus1"].notna()].reset_index(drop=True)
        res, base_rmse, n_used = grouped_permutation(mdl, cols, cats, held, "Estimated_AQI_tplus1", groups, CFG["perm_rows"], CFG["perm_repeats"], RNG)
        print(f"  Global grouped permutation (explainer model, {n_used} held-out rows, base RMSE {base_rmse:.2f})")
        pos = sum(max(v, 0) for _, v, _ in res) or 1.0
        glob_rows = []
        for g, v, s in res:
            add("Global", "All", "Part7_explainer", "regression", "HGB_full_features", "permutation_grouped", "variable_family", g, v, s)
        t0 = time.time()
        for city in city_order:
            d = held[held["City"] == city]
            if len(d) < 30:
                continue
            r_, _, _ = grouped_permutation(mdl, cols, cats, d, "Estimated_AQI_tplus1", groups, 400, 2, RNG)
            for g, v, s in r_:
                add("City", city, "Part7_explainer", "regression", "HGB_full_features", "permutation_grouped", "variable_family", g, v, s)
        print(f"  City-wise grouped permutation done ({time.time() - t0:.0f}s)")
        for season in SEASONS:
            d = held[held["Season"] == season]
            if len(d) < 200:
                continue
            r_, _, _ = grouped_permutation(mdl, cols, cats, d, "Estimated_AQI_tplus1", groups, CFG["perm_rows"], CFG["perm_repeats"], RNG)
            for g, v, s in r_:
                add("Season", season, "Part7_explainer", "regression", "HGB_full_features", "permutation_grouped", "variable_family", g, v, s)

        # ---- case studies: peak episodes in held-out data, different seasons/cities ----------------------------------------
        Hh_all = H[H["Split"].isin(["validation", "test"])].copy()
        Hh_all["Target_Date"] = Hh_all["Date"] + pd.Timedelta(days=1)
        used_cities, used_regions, chosen = set(), set(), []
        order_seasons = SEASONS + ["Winter"]
        for season in order_seasons:
            d = Hh_all[Hh_all["Season"] == season].sort_values("Actual_AQI_tplus1", ascending=False)
            for pref_new_region in (True, False):
                hit = None
                for _, r in d.iterrows():
                    if r["City"] in used_cities:
                        continue
                    if pref_new_region and region_map.get(r["City"]) in used_regions:
                        continue
                    hit = r
                    break
                if hit is not None:
                    chosen.append(hit)
                    used_cities.add(hit["City"])
                    used_regions.add(region_map.get(hit["City"]))
                    break
            if len(chosen) >= 5:
                break
        train_means = D["train"][cols].mean(numeric_only=True).to_dict()
        hr_prob = Hh.set_index(["City", "Date"]) if f"Prob_{bh}" in Hh.columns else None
        case_rows, series_rows = [], []
        for i, r in enumerate(chosen, 1):
            cid = f"case{i}"
            origin = r["Date"]
            win = Hh_all[(Hh_all["City"] == r["City"]) & (Hh_all["Date"] >= origin - pd.Timedelta(days=11)) &
                         (Hh_all["Date"] <= origin + pd.Timedelta(days=9))]
            for _, w in win.iterrows():
                series_rows.append({"Case_ID": cid, "City": r["City"], "Origin_Date": w["Date"], "Target_Date": w["Target_Date"],
                                    "Observed_AQI": w["Actual_AQI_tplus1"], "Predicted_AQI_best_model": w[f"Pred_AQI_{bm}"],
                                    "Persistence_AQI": w["Pred_AQI_Persistence"]})
            dctx = daily[(daily["City"] == r["City"]) & (daily["Date"] == origin)]
            dctx = dctx.iloc[0] if len(dctx) else None
            xrow = held[(held["City"] == r["City"]) & (held["Date"] == origin)]
            drivers, own_pred = "n/a", np.nan
            if len(xrow):
                own_pred, ranked = local_occlusion(mdl, cols, cats, xrow.head(1), train_means, groups)
                drivers = "; ".join(f"{g} ({v:+.1f})" for g, v in ranked[:3])
                for g, v in ranked[:8]:
                    add("Episode", cid, "Part7_explainer", "regression", "HGB_full_features", "occlusion_local", "variable_family", g, v)
            prob = np.nan
            if hr_prob is not None and (r["City"], origin) in hr_prob.index:
                prob = float(hr_prob.loc[(r["City"], origin), f"Prob_{bh}"])
            case_rows.append({
                "Case_ID": cid, "City": r["City"], "Region": region_map.get(r["City"]), "Season": r["Season"], "Split": r["Split"],
                "Origin_Date": origin, "Peak_Target_Date": r["Target_Date"], "Observed_Peak_AQI": r["Actual_AQI_tplus1"],
                "Observed_Peak_Category": r["Actual_Category_tplus1"], "Predicted_AQI_best_model": r[f"Pred_AQI_{bm}"],
                "Persistence_AQI": r["Pred_AQI_Persistence"], "Explainer_Model_Prediction": own_pred,
                "Dominant_Pollutant_Origin_Day": dctx["Dominant_Pollutant"] if dctx is not None else np.nan,
                "Temperature_C": dctx["Temperature_C"] if dctx is not None else np.nan,
                "Humidity_pct": dctx["Humidity_pct"] if dctx is not None else np.nan,
                "WindSpeed_kmh": dctx["WindSpeed_kmh"] if dctx is not None else np.nan,
                "Precipitation_mm": dctx["Precipitation_mm"] if dctx is not None else np.nan,
                "HighRisk_Probability_best_model": prob, "Top_Local_Drivers": drivers,
                "Likely_Cause": dctx["Likely_Cause"] if dctx is not None else "n/a",
                "Recommended_Actions": dctx["Recommended_Actions"] if dctx is not None else "n/a",
                "Interpretation": "Rule-based and occlusion-based statements suggest, but do not establish, the contributors to this episode."})
        cases_df, case_series = pd.DataFrame(case_rows), pd.DataFrame(series_rows)
        cases_df.to_csv(CASES_CSV, index=False, date_format="%Y-%m-%d")
        case_series.to_csv(CASE_SERIES_CSV, index=False, date_format="%Y-%m-%d")
        print(f"  {len(cases_df)} case studies: " + "; ".join(f"{c.City} ({c.Season})" for c in cases_df.itertuples()))
    fimp = pd.DataFrame(imp_rows)
    pos_part = fimp.assign(_pos=fimp["Importance"].clip(lower=0))
    fimp["Importance_Share"] = pos_part["_pos"] / pos_part.groupby(["Scope", "Group", "Source", "Task", "Model", "Method", "Level"])["_pos"].transform("sum").replace(0, np.nan)
    fimp["Rank"] = fimp.groupby(["Scope", "Group", "Source", "Task", "Model", "Method", "Level"])["Importance"].rank(ascending=False, method="min")
    fimp["Basis"] = np.where(fimp["Source"] == "Part7_explainer", "held-out validation+test rows (descriptive; no selection based on it)",
                             np.where(fimp["Source"] == "Part5", "validation sample", "Part 6 outputs"))

    # ---- save final CSVs -------------------------------------------------------------------------------------------------------
    banner("Saving final outputs")
    comp.to_csv(FINAL_COMPARISON_CSV, index=False)
    boot.to_csv(BOOTSTRAP_CSV, index=False)
    chk.to_csv(CONSISTENCY_CSV, index=False)
    observed.to_csv(OBSERVED_CSV, index=False)
    predicted.to_csv(PREDICTED_CSV, index=False)
    final_summary.to_csv(FINAL_SUMMARY_CSV, index=False)
    fimp.to_csv(FINAL_IMPORTANCE_CSV, index=False)
    alerts.to_csv(FINAL_ALERTS_CSV, index=False, date_format="%Y-%m-%d", float_format="%.4f")
    asum.to_csv(ALERT_SUMMARY_CSV, index=False)

    # ---- figures -----------------------------------------------------------------------------------------------------------------
    banner("Figures")
    ctx = {"R": R, "comp": comp, "best": best, "fam": fam, "H": H, "best_ml": best_ml, "best_dl": best_dl, "boot": boot,
           "group_metrics": gm, "final_importance": fimp, "city_order": city_order, "attention": attention,
           "observed": observed, "ablation": ablation, "cases": cases_df, "case_series": case_series,
           "alerts_all_test": Hht, "alert_summary": asum, "regime_profile": regime_profile}
    for fn in (fig_model_comparison, fig_region_city, fig_season, fig_pred_actual, fig_confusions, fig_importance,
               fig_city_season_importance, fig_attention, fig_dominant_heatmap, fig_ablation, fig_cases, fig_alerts, fig_regimes):
        run_plot(fn, ctx)

    # ---- report ------------------------------------------------------------------------------------------------------------------------
    banner("Final report")
    concl = ""
    if "regression" in best and "regression" in TASK_PRIMARY:
        b_row = boot[(boot["Task"] == "regression") & (boot["Comparison"] == "best vs persistence")]
        if len(b_row):
            concl = ("For next-day AQI the selected model is "
                     + ("clearly better than persistence on the test dates. " if b_row.iloc[0]["Reading"] == "A clearly better"
                        else "not clearly distinguishable from persistence on the test dates, so persistence remains a competitive baseline. "))
    ctx.update({"bootstrap": boot, "test_seasons": test_seasons, "alert_text": alert_text, "regime_in_ablation": regime_in_ablation,
                "conclusion_sentence": concl})
    write_report(ctx)
    print(f"  report written: {FINAL_REPORT_MD} ({len(open(FINAL_REPORT_MD, encoding='utf-8').read().split()):,} words)")

    # ---- integrity, versioned copies, manifest -------------------------------------------------------------------------------------------
    assert all(sha256_of(p) == h for p, h in hashes_in.items()), "An earlier output file was modified!"
    final_data = [FINAL_COMPARISON_CSV, FINAL_ABLATION_CSV if ablation is not None else None, FINAL_IMPORTANCE_CSV, FINAL_ALERTS_CSV,
                  FINAL_SUMMARY_CSV, OBSERVED_CSV, PREDICTED_CSV, BOOTSTRAP_CSV, GROUP_METRICS_CSV, ALERT_SUMMARY_CSV, CONSISTENCY_CSV,
                  CASES_CSV if len(cases_df) else None, CASE_SERIES_CSV if len(case_series) else None,
                  REGIME_CSV if labels is not None else None, REGIME_PROFILE_CSV if regime_profile is not None else None, FINAL_REPORT_MD]
    final_data = [p for p in final_data if p and os.path.isfile(p)]
    version_files = []
    for p in final_data:
        dst = os.path.join(VERSION_DIR, os.path.basename(p))
        shutil.copy2(p, dst)
        version_files.append(dst)
    fig_files = [p for p in FIG_LOG if not str(p).startswith("FAILED:")]
    failed = [p for p in FIG_LOG if str(p).startswith("FAILED:")]
    manifest = {
        "part": 7, "total_parts": 7, "run_id": RUN_ID, "seed": SEED, "profile": PROFILE,
        "label": AQI_LABEL, "caveats": AQI_CAVEAT, "base_dir": BASE_DIR,
        "inputs_sha256": hashes_in, "earlier_parts_modified": False,
        "harmonisation": hinfo, "best_models_by_validation": best,
        "best_regression_by_family": {"Classical ML": best_ml, "Deep learning": best_dl},
        "highrisk_thresholds": thresholds,
        "bootstrap": boot.to_dict(orient="records"),
        "ablation": {"run": ablation is not None, "tasks": CFG["ablation_tasks"], "learner": "sklearn HistGradientBoosting (fixed settings)",
                     "regime_in_ablation": regime_in_ablation,
                     "regime_note": "K-Means fitted on the TRAIN split only (day-t scaled pollutant and weather profile)" if labels is not None else "regime labels taken from Part 4 files or unavailable"},
        "retraining": "No Part 5/6 model was retrained; new fitting limited to ablation learners and the K-Means regime model.",
        "test_seasons_present": test_seasons,
        "figures_failed": failed,
        "files_created": [{"path": p, "rel_path": os.path.relpath(p, BASE_DIR), "size_bytes": os.path.getsize(p)}
                          for p in final_data + fig_files + version_files] +
                         [{"path": MANIFEST_JSON, "rel_path": os.path.relpath(MANIFEST_JSON, BASE_DIR), "size_bytes": None}],
        "versioned_copy_dir": VERSION_DIR,
        "project_manifests_found": sorted(R["manifests"].keys()),
        "runtime_minutes": round((time.time() - t_start) / 60, 1)}
    json.dump(manifest, open(MANIFEST_JSON, "w"), indent=2, default=str)
    shutil.copy2(MANIFEST_JSON, os.path.join(VERSION_DIR, os.path.basename(MANIFEST_JSON)))

    banner("PART 7 COMPLETE - final output files")
    for p in final_data + fig_files + [MANIFEST_JSON]:
        print(p)
    print(f"\nVersioned copies: {VERSION_DIR}")
    if failed:
        print("Figures that failed (pipeline continued):", failed)
    print(f"BASE_DIR: {BASE_DIR} | runtime: {manifest['runtime_minutes']} min")
    print("\n" + AQI_LABEL)
    print(AQI_CAVEAT)
    return comp


if __name__ == "__main__" or IN_COLAB:
    final_comparison = main()



PART 7 - Final synthesis (profile: standard)
Six-pollutant estimated AQI using CPCB sub-index methodology.
NH3 and Pb are unavailable, so this is not the official CPCB AQI. O3/CO values are Open-Meteo/CAMS-based estimates, not direct CPCB ground-station observations.


FileNotFoundError: Missing required result files (run the earlier parts in the same BASE_DIR=/content/aqi_framework):
  /content/aqi_framework/results/model_metrics.csv
  /content/aqi_framework/results/dl_model_metrics.csv
  /content/aqi_framework/results/city_wise_metrics.csv
  /content/aqi_framework/results/season_wise_metrics.csv
  /content/aqi_framework/results/dl_city_wise_metrics.csv
  /content/aqi_framework/results/dl_season_wise_metrics.csv
  /content/aqi_framework/results/feature_importance.csv
  /content/aqi_framework/results/predicted_vs_actual.csv
  /content/aqi_framework/results/dl_predicted_vs_actual.csv
  /content/aqi_framework/results/high_risk_predictions.csv
  /content/aqi_framework/results/high_risk_dl_predictions.csv
  /content/aqi_framework/results/city_season_dominant_pollutant_summary.csv
  /content/aqi_framework/results/daily_aqi_dominant_pollutant_with_causes.csv